# Setup

Jina-reranker-v3 require vllm >=0.20.0

Caution: be aware with error libcudart.so.13

## vLLM

In [1]:
# GPU Setup
!uv venv --python 3.12 --seed --managed-python --clear -q
!source .venv/bin/activate -q
!uv pip uninstall torch torchvision torchaudio -q
# --torch-backend=cu129 only installs the pytorch+cu129 versions
# it seems that vllm 0.20.0 is more stable with cu129 than higher version
# the following will install vllm+cu129 (fix vllm install bug)
!uv pip install vllm --extra-index-url https://wheels.vllm.ai/0.20.0/cu129 --torch-backend=cu129 -q

In [2]:
import torch
import vllm
print(f"vLLM Version: {vllm.__version__}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Active CUDA Version: {torch.version.cuda}")
print(f"GPU Detected: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

from vllm import LLM

vLLM Version: 0.20.0
PyTorch Version: 2.11.0+cu129
Active CUDA Version: 12.9
GPU Detected: True
GPU Name: Tesla T4


Run below helper functions

In [3]:
import time
import torch
import gc

def clear_gpu_memory():
    # 1. Xóa các biến chứa mô hình khỏi bộ nhớ toàn cục (Globals)
    global llm, llm06, llm4, llmbge, llmjina
    if 'llm' in globals():
        del llm
    if 'llm06' in globals():
        del llm06
    if 'llm4' in globals():
        del llm4
    if 'llmbge' in globals():
        del llmbge
    if 'llmjina' in globals():
        del llmjina

    # 2. Ép Python thu gom rác (Garbage Collection)
    gc.collect()

    # 3. Xóa cache PyTorch trên GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    # 4. [QUAN TRỌNG NHẤT] Phá hủy môi trường phân tán của vLLM
    # Bước này giúp tránh lỗi "CUDA is initialized" khi gọi LLM() lần tiếp theo
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment
        destroy_model_parallel()
        destroy_distributed_environment()
    except Exception as e:
        pass # Bỏ qua nếu chưa import vLLM

def evaluate_model(model_name, llm, test_cases, eval, top_k=3):
    total_mrr = 0
    total_hit = 0
    total_recall = 0
    total_time = 0
    total_docs = 0  # Dùng để đo Throughput (Số tài liệu / giây)
    num_queries = len(test_cases)
    qwen_models = ["Qwen/Qwen3-Reranker-0.6B", "Qwen/Qwen3-Reranker-4B"]

    print(f"\nĐánh giá mô hình: {model_name}")
    print("="*60)

    # 1. Reset thống kê VRAM trước khi đo để lấy số liệu chính xác cho phiên chạy này
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    for query, docs in test_cases.items():
        ground_truth_indices = [0, 1, 2]
        total_docs += len(docs)

        # Đo thời gian bắt đầu
        start_time = time.time()

        # Make sure Qwen models use chat_template and the others dont
        if model_name in qwen_models:
          outputs = llm.score(query, docs, chat_template=chat_template)
        else: outputs = llm.score(query, docs)
        scores = [out.outputs.score for out in outputs]

        # Thời gian kết thúc
        total_time += (time.time() - start_time)

        # Xếp hạng và tính chỉ số độ chính xác
        combined = [(i, score) for i, score in enumerate(scores)]
        ranked_results = sorted(combined, key=lambda x: x[1], reverse=True)
        ranked_indices = [x[0] for x in ranked_results]

        # Hit Rate
        top_k_indices = ranked_indices[:top_k]
        # if any(idx in ground_truth_indices for idx in top_k_indices):
        #     total_hit += 1

        # Đếm xem có BAO NHIÊU tài liệu HIGH lọt vào Top K
        hits_in_top = sum(1 for idx in top_k_indices if idx in ground_truth_indices)
        query_recall = hits_in_top / len(ground_truth_indices)
        total_recall += query_recall

        # MRR
        for rank, idx in enumerate(ranked_indices, start=1):
            if idx in ground_truth_indices:
                total_mrr += (1.0 / rank)
                break

    avg_recall = (total_recall / num_queries) * 100
    # 2. Tính toán System Metrics (Thông số hệ thống)
    avg_mrr = total_mrr / num_queries
    hit_rate = (total_hit / num_queries) * 100
    avg_latency = (total_time / num_queries) * 1000 # ms/query

    # Throughput: Số Query xử lý được trong 1 giây (QPS) và Số Document xử lý trong 1 giây
    qps = num_queries / total_time
    docs_per_sec = total_docs / total_time

    # VRAM Usage
    used_vram_gb = 0
    if torch.cuda.is_available():
      free_mem, total_mem = torch.cuda.mem_get_info()
      used_vram_gb = (total_mem - free_mem) / (1024 ** 3)

    # 3. In Báo cáo
    # print(f"Kết quả Ranking Quality:")
    # print(f"  - Hit Rate @{top_k}:         {hit_rate:.1f}%")
    # print(f"  - Recall @{top_k}:           {avg_recall:.1f}%")
    # print(f"  - MRR:                 {avg_mrr:.4f}")

    # print(f"\nKết quả System Performance:")
    # print(f"  - Độ trễ (Latency):    {avg_latency:.1f} ms/query")
    # print(f"  - Thông lượng (QPS):   {qps:.2f} queries/second")
    # print(f"  - Tốc độ xử lý Docs:   {docs_per_sec:.2f} docs/second")
    # print(f"  - VRAM tiêu thụ (Peak): {used_vram_gb:.2f} GB")
    # print("="*60)

    # Store eval
    eval.append({
        "model": model_name,
        "recall": avg_recall,
        "mrr": avg_mrr,
        "latency": avg_latency,
        "qps": qps,
        "docs_per_sec": docs_per_sec,
        "used_vram_gb": used_vram_gb
    })
    # Gọi hàm để dọn dẹp
    clear_gpu_memory()

def test_model(model_name, llm, test_cases):
  qwen_models = ["Qwen/Qwen3-Reranker-0.6B", "Qwen/Qwen3-Reranker-4B"]
  print(f"\n*** Testing {model_name} ***")
  print('')
  for ids, (query, docs) in enumerate(test_cases.items(), start=1):
    # Make sure Qwen models use chat_template and the others dont
    if model_name in qwen_models:
      outputs = llm.score(query, docs, chat_template=chat_template)
    else: outputs = llm.score(query, docs)
    scores = [out.outputs.score for out in outputs]

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)
    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)

chat_template = """<|im_start|>system
You are an expert banking and financial evaluator. Judge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>
<|im_start|>user
<Instruct>: {{ instruction | default(instruct | default(messages | selectattr("role", "eq", "system") | map(attribute="content") | first | default("Given a banking or financial query, retrieve highly relevant documents that provide direct answers, operational procedures, or legal compliance guidelines.", true), true), true) }}
<Query>: {{ messages | selectattr("role", "eq", "query") | map(attribute="content") | first }}
<Document>: {{ messages | selectattr("role", "eq", "document") | map(attribute="content") | first }}<|im_end|>
<|im_start|>assistant
<think>

</think>"""

## CrossEncoder

In [ ]:
!pip install sentence_transformers -q

# Simple test

In [ ]:
# ==========================================
# TEST CASE 1: Mortgage / Home Loan
# User style: Vietnamese, casual, missing punctuation
# ==========================================
query_1 = "hồ sơ vay mua nhà cần giấy tờ gì"

documents_1 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "To apply for a home loan, you will need to provide your last two years of tax returns, recent pay stubs, and bank statements.",
    "Lenders typically require proof of income, a government-issued ID, W-2 forms, and details of your current debts and assets.",
    "Hồ sơ vay vốn mua nhà bao gồm giấy tờ tùy thân, sao kê lương 3 tháng gần nhất, hợp đồng lao động và giấy tờ chứng minh tài sản đảm bảo.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Getting pre-approved for a mortgage gives you a clear idea of how much house you can afford before you start looking.",
    "A fixed-rate mortgage keeps the same interest rate for the life of the loan, whereas an adjustable-rate mortgage (ARM) can fluctuate.",
    "Phí thẩm định giá tài sản và phí công chứng hợp đồng thế chấp là những khoản khách hàng phải chịu khi làm hồ sơ vay.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "To avoid overdraft fees, you can link your primary checking account to a savings account for automatic transfers.",
    "Cashback credit cards offer a percentage of your daily purchases returned to you as statement credit.",
    "Khách hàng có thể mở tài khoản chứng khoán trực tuyến bằng phương thức định danh eKYC chỉ trong vài phút.",
]


# ==========================================
# TEST CASE 2: International Wire Transfers
# User style: English, keyword-only, missing "how long"
# ==========================================
query_2 = "time international wire transfer swift"

documents_2 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "International wire transfers sent via the SWIFT network generally take between 1 to 5 business days to arrive in the recipient's account.",
    "Depending on the destination country and intermediary banks, overseas transfers are typically processed within 48 to 72 hours.",
    "Thời gian chuyển tiền quốc tế qua hệ thống SWIFT thường mất từ 2 đến 5 ngày làm việc, tùy thuộc vào múi giờ và ngân hàng trung gian.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "To send money abroad, you will need the recipient's full name, account number, bank address, and the bank's SWIFT/BIC code.",
    "Exchange rate markups and outbound wire fees can significantly increase the cost of sending money overseas.",
    "Mã SWIFT là mã định danh duy nhất của mỗi ngân hàng, được yêu cầu bắt buộc khi thực hiện giao dịch chuyển tiền xuyên biên giới.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "A Certificate of Deposit (CD) offers a fixed interest rate in exchange for locking your money away for a specific term length.",
    "If you forget your online banking password, click the 'Forgot Password' link and follow the email verification steps.",
    "Bảo hiểm nhân thọ liên kết đầu tư cho phép khách hàng vừa bảo vệ rủi ro sức khỏe vừa tham gia vào các quỹ mở.",
]


# ==========================================
# TEST CASE 3: Joint Bank Accounts
# User style: English, lazy syntax, highly specific but poorly formatted
# ==========================================
query_3 = "remove person joint account no permission"

documents_3 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "No, you generally cannot remove a co-owner from a joint checking account without their written consent and signature.",
    "If you want to separate finances from a joint account holder who won't agree to be removed, your only option is usually to close the account entirely.",
    "Bạn không thể tự ý xóa tên người đồng sở hữu khỏi tài khoản chung mà không có sự đồng ý và chữ ký xác nhận của họ tại quầy giao dịch.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Both account holders have equal access to the funds in a joint account, regardless of who actually deposited the money.",
    "In a joint account with rights of survivorship, if one owner passes away, the remaining funds automatically go to the surviving owner.",
    "Để mở tài khoản ngân hàng chung, cả hai cá nhân đều phải cung cấp giấy tờ tùy thân và cùng ký vào thỏa thuận mở tài khoản.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "Early withdrawal from your 401(k) retirement plan before age 59½ may result in a 10% penalty tax.",
    "An auto loan term usually ranges from 36 to 72 months, with longer terms offering lower monthly payments but higher total interest.",
    "Khách hàng doanh nghiệp cần cung cấp giấy phép đăng ký kinh doanh và mẫu dấu khi mở tài khoản thanh toán tại ngân hàng.",
]


# ==========================================
# TEST CASE 4: ATM Operations / Physical Cards
# User style: Vietnamese, frantic, short phrase
# ==========================================
query_4 = "atm nuốt thẻ phải làm sao"

documents_4 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "If your card is captured by the machine, go inside the branch immediately if it is open, or call the bank's customer service number on the ATM screen.",
    "Banks usually destroy swallowed cards for security reasons, so you will likely need to request a replacement debit card via your banking app.",
    "Nếu máy ATM nuốt thẻ, bạn cần mang CMND/CCCD vào chi nhánh ngân hàng quản lý máy đó để làm thủ tục nhận lại hoặc phát hành thẻ mới.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Entering the wrong PIN three consecutive times will lock your debit card to protect you from unauthorized access.",
    "Many modern ATMs allow cardless withdrawals using Apple Pay, Google Wallet, or a QR code generated by your banking app.",
    "Hạn mức rút tiền mặt tối đa tại máy ATM thường là 10 triệu đồng mỗi lần và 50 triệu đồng mỗi ngày đối với thẻ ghi nợ tiêu chuẩn.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "A cashier's check is guaranteed by the bank's own funds rather than yours, making it a secure payment method for large purchases like real estate.",
    "Wealth management services provide personalized investment strategies, tax planning, and estate management for high-net-worth individuals.",
    "Phí thường niên của thẻ tín dụng hạng Platinum thường cao hơn thẻ chuẩn, nhưng đi kèm với nhiều đặc quyền như phòng chờ sân bay.",
]

# ==========================================
# TEST CASE 5: Credit Card Fraud Dispute
# User style: Bilingual mix (Vietlish), combining tech terms with local language
# ==========================================
query_5 = "dispute giao dịch giả mạo thẻ tín dụng"

documents_5 = [
    # --- HIGH SIMILARITY (Direct answers to the query) ---
    "To dispute a fraudulent charge, you must contact your card issuer immediately by calling the number on the back of your card and filing a formal fraud report.",
    "If you notice an unauthorized transaction, log into your mobile banking app, select the specific transaction, and click 'Dispute' within 60 days.",
    "Quy trình khiếu nại giao dịch thẻ tín dụng giả mạo yêu cầu khách hàng điền mẫu đơn tra soát và gửi đến ngân hàng trong vòng 45 ngày.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap, but different intent) ---
    "Visa Zero Liability policy ensures that you won't be held responsible for unauthorized charges made with your account or account information.",
    "If your credit card is lost or stolen, you should temporarily lock the card via the app to prevent any unauthorized transactions from occurring.",
    "Purchase protection is a credit card benefit that covers eligible items against theft or accidental damage for up to 90 days after purchase.",
    "Bảo hiểm gian lận thẻ tín dụng sẽ bồi thường cho bạn những tổn thất tài chính nếu thông tin thẻ của bạn bị đánh cắp trên mạng.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "A credit score is calculated based on your payment history, credit utilization, length of credit history, and new credit inquiries.",
    "Lãi suất tiền gửi tiết kiệm có kỳ hạn 12 tháng tại các ngân hàng thương mại hiện đang dao động từ 5% đến 6% một năm.",
    "Khi thanh lý hợp đồng vay thế chấp, khách hàng phải thanh toán toàn bộ dư nợ gốc, lãi và phí phạt trả nợ trước hạn nếu có.",
]

dic = {
    query_1: documents_1,
    query_2: documents_2,
    query_3: documents_3,
    query_4: documents_4,
    query_5: documents_5
}

## Qwen 3 Ranker

#### 0.6B

Converted model

In [ ]:
#llm = LLM(model="tomaarsen/Qwen3-Reranker-0.6B-seq-cls", runner="pooling")

Original pipeline

In [ ]:
model_name = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=model_name,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=1024,
        gpu_memory_utilization=0.4,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
for ids, (query, docs) in enumerate(dic.items(), start=1):
    outputs = llm.score(query, docs, chat_template=chat_template)
    scores = [out.outputs.score for out in outputs]

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)
    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)

INFO 06-19 03:11:22 [utils.py:238] non-default args: {'runner': 'pooling', 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'model': 'Qwen/Qwen3-Reranker-0.6B'}
INFO 06-19 03:11:24 [config.py:882] Found sentence-transformers tokenize configuration.
INFO 06-19 03:11:41 [model.py:856] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-19 03:11:41 [model.py:531] Resolved architecture: Qwen3ForSequenceClassification
INFO 06-19 03:11:41 [config.py:770] Found sentence-transformers modules configuration.
WARNING 06-19 03:11:41 [model.py:1839] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-19 03:11:41 [model.py:1892] Casting torch.bfloat16 to torch.float16.
INFO 06-19 03:11:41 [model.py:1554] Using max model len 4096

Rendering prompts:   0%|          | 0/9 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: hồ sơ vay mua nhà cần giấy tờ gì
[3] Score: 0.9991
Hồ sơ vay vốn mua nhà bao gồm giấy tờ tùy thân, sao kê lương 3 tháng gần nhất, hợp đồng lao động và giấy tờ chứng minh tài sản đảm bảo.
----------------------------------------------------------------------
[1] Score: 0.9973
To apply for a home loan, you will need to provide your last two years of tax returns, recent pay stubs, and bank statements.
----------------------------------------------------------------------
[2] Score: 0.9489
Lenders typically require proof of income, a government-issued ID, W-2 forms, and details of your current debts and assets.
----------------------------------------------------------------------
[6] Score: 0.5804
Phí thẩm định giá tài sản và phí công chứng hợp đồng thế chấp là những khoản khách hàng phải chịu khi làm hồ sơ vay.
----------------------------------------------------------------------
[4] Score: 0.1319
Getting pre-approved for a mortgage gives you a clear idea of how much house you

Rendering prompts:   0%|          | 0/9 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (6 outputs left out)


Query 4: atm nuốt thẻ phải làm sao
[3] Score: 0.9996
Nếu máy ATM nuốt thẻ, bạn cần mang CMND/CCCD vào chi nhánh ngân hàng quản lý máy đó để làm thủ tục nhận lại hoặc phát hành thẻ mới.
----------------------------------------------------------------------
[5] Score: 0.9967
Many modern ATMs allow cardless withdrawals using Apple Pay, Google Wallet, or a QR code generated by your banking app.
----------------------------------------------------------------------
[1] Score: 0.9921
If your card is captured by the machine, go inside the branch immediately if it is open, or call the bank's customer service number on the ATM screen.
----------------------------------------------------------------------
[2] Score: 0.9782
Banks usually destroy swallowed cards for security reasons, so you will likely need to request a replacement debit card via your banking app.
----------------------------------------------------------------------
[6] Score: 0.3666
Hạn mức rút tiền mặt tối đa tại máy ATM thườn

Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 5: dispute giao dịch giả mạo thẻ tín dụng
[1] Score: 0.9993
To dispute a fraudulent charge, you must contact your card issuer immediately by calling the number on the back of your card and filing a formal fraud report.
----------------------------------------------------------------------
[3] Score: 0.9992
Quy trình khiếu nại giao dịch thẻ tín dụng giả mạo yêu cầu khách hàng điền mẫu đơn tra soát và gửi đến ngân hàng trong vòng 45 ngày.
----------------------------------------------------------------------
[2] Score: 0.9851
If you notice an unauthorized transaction, log into your mobile banking app, select the specific transaction, and click 'Dispute' within 60 days.
----------------------------------------------------------------------
[7] Score: 0.9507
Bảo hiểm gian lận thẻ tín dụng sẽ bồi thường cho bạn những tổn thất tài chính nếu thông tin thẻ của bạn bị đánh cắp trên mạng.
----------------------------------------------------------------------
[5] Score: 0.1535
If your cred

####4B

vLLM

In [ ]:
model_name = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
    model=model_name,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=1024,
    gpu_memory_utilization=0.4,
    hf_overrides={
        "architectures": ["Qwen3ForSequenceClassification"],
        "classifier_from_token": ["no", "yes"],
        "is_original_qwen3_reranker": True,
    },
)
for ids, (query, docs) in enumerate(dic.items(), start=1):
    outputs = llm.score(query, docs, chat_template=chat_template)
    scores = [out.outputs.score for out in outputs]

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)
    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)

INFO 06-19 09:57:24 [utils.py:238] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 2048, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Reranker-4B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


sentence_bert_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

INFO 06-19 09:57:25 [config.py:882] Found sentence-transformers tokenize configuration.
INFO 06-19 09:57:49 [model.py:856] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-19 09:58:07 [model.py:531] Resolved architecture: Qwen3ForSequenceClassification


... (21 outputs left out)


Query 4: atm nuốt thẻ phải làm sao
[3] Score: 0.5136
Nếu máy ATM nuốt thẻ, bạn cần mang CMND/CCCD vào chi nhánh ngân hàng quản lý máy đó để làm thủ tục nhận lại hoặc phát hành thẻ mới.
----------------------------------------------------------------------
[1] Score: 0.4763
If your card is captured by the machine, go inside the branch immediately if it is open, or call the bank's customer service number on the ATM screen.
----------------------------------------------------------------------
[2] Score: 0.4471
Banks usually destroy swallowed cards for security reasons, so you will likely need to request a replacement debit card via your banking app.
----------------------------------------------------------------------
[5] Score: 0.1672
Many modern ATMs allow cardless withdrawals using Apple Pay, Google Wallet, or a QR code generated by your banking app.
----------------------------------------------------------------------
[4] Score: 0.1265
Entering the wrong PIN three consecutive time

Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 5: dispute giao dịch giả mạo thẻ tín dụng
[3] Score: 0.5069
Quy trình khiếu nại giao dịch thẻ tín dụng giả mạo yêu cầu khách hàng điền mẫu đơn tra soát và gửi đến ngân hàng trong vòng 45 ngày.
----------------------------------------------------------------------
[7] Score: 0.4904
Bảo hiểm gian lận thẻ tín dụng sẽ bồi thường cho bạn những tổn thất tài chính nếu thông tin thẻ của bạn bị đánh cắp trên mạng.
----------------------------------------------------------------------
[1] Score: 0.4853
To dispute a fraudulent charge, you must contact your card issuer immediately by calling the number on the back of your card and filing a formal fraud report.
----------------------------------------------------------------------
[2] Score: 0.3664
If you notice an unauthorized transaction, log into your mobile banking app, select the specific transaction, and click 'Dispute' within 60 days.
----------------------------------------------------------------------
[4] Score: 0.2686
Visa Zero Li

CrossEncoder

In [ ]:
from sentence_transformers import CrossEncoder
import torch

model = CrossEncoder("Qwen/Qwen3-Reranker-4B")


for ids, (query, docs) in enumerate(dic.items(), start=1):
    scores = model.predict([(query, doc) for doc in docs], activation_fn=torch.nn.Sigmoid())

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)
    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Query 1: hồ sơ vay mua nhà cần giấy tờ gì
[3] Score: 1.0000
Hồ sơ vay vốn mua nhà bao gồm giấy tờ tùy thân, sao kê lương 3 tháng gần nhất, hợp đồng lao động và giấy tờ chứng minh tài sản đảm bảo.
----------------------------------------------------------------------
[1] Score: 0.9961
To apply for a home loan, you will need to provide your last two years of tax returns, recent pay stubs, and bank statements.
----------------------------------------------------------------------
[6] Score: 0.4688
Phí thẩm định giá tài sản và phí công chứng hợp đồng thế chấp là những khoản khách hàng phải chịu khi làm hồ sơ vay.
----------------------------------------------------------------------
[2] Score: 0.2334
Lenders typically require proof of income, a government-issued ID, W-2 forms, and details of your current debts and assets.
----------------------------------------------------------------------
[4] Score: 0.0260
Getting pre-approved for a mortgage gives you a clear idea of how much house you

## BAAI/bge-reranker-v2-m3

In [ ]:
model_name = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=model_name,
          runner="score",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          gpu_memory_utilization=0.4,)

for ids, (query, docs) in enumerate(dic.items(), start=1):
    outputs = llm.score(query, docs)
    scores = [out.outputs.score for out in outputs]

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)

    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)

INFO 06-19 03:42:08 [utils.py:238] non-default args: {'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

INFO 06-19 03:42:35 [model.py:804] Resolved `--runner auto` to `--runner pooling`. Pass the value explicitly to silence this message.
INFO 06-19 03:42:35 [model.py:531] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-19 03:42:35 [model.py:1889] Downcasting torch.float32 to torch.float16.


tokenizer_config.json: 0.00B [00:00, ?B/s]

INFO 06-19 03:42:35 [model.py:1554] Using max model len 8192
INFO 06-19 03:42:36 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 06-19 03:42:36 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-19 03:42:36 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-19 03:42:36 [vllm.py:957] Cudagraph is disabled under eager mode


... (15 outputs left out)


Query 4: atm nuốt thẻ phải làm sao
[3] Score: 0.9861
Nếu máy ATM nuốt thẻ, bạn cần mang CMND/CCCD vào chi nhánh ngân hàng quản lý máy đó để làm thủ tục nhận lại hoặc phát hành thẻ mới.
----------------------------------------------------------------------
[2] Score: 0.2387
Banks usually destroy swallowed cards for security reasons, so you will likely need to request a replacement debit card via your banking app.
----------------------------------------------------------------------
[1] Score: 0.0386
If your card is captured by the machine, go inside the branch immediately if it is open, or call the bank's customer service number on the ATM screen.
----------------------------------------------------------------------
[5] Score: 0.0032
Many modern ATMs allow cardless withdrawals using Apple Pay, Google Wallet, or a QR code generated by your banking app.
----------------------------------------------------------------------
[6] Score: 0.0014
Hạn mức rút tiền mặt tối đa tại máy ATM thườn

Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 5: dispute giao dịch giả mạo thẻ tín dụng
[3] Score: 0.8903
Quy trình khiếu nại giao dịch thẻ tín dụng giả mạo yêu cầu khách hàng điền mẫu đơn tra soát và gửi đến ngân hàng trong vòng 45 ngày.
----------------------------------------------------------------------
[1] Score: 0.8700
To dispute a fraudulent charge, you must contact your card issuer immediately by calling the number on the back of your card and filing a formal fraud report.
----------------------------------------------------------------------
[2] Score: 0.0704
If you notice an unauthorized transaction, log into your mobile banking app, select the specific transaction, and click 'Dispute' within 60 days.
----------------------------------------------------------------------
[7] Score: 0.0499
Bảo hiểm gian lận thẻ tín dụng sẽ bồi thường cho bạn những tổn thất tài chính nếu thông tin thẻ của bạn bị đánh cắp trên mạng.
----------------------------------------------------------------------
[5] Score: 0.0011
If your cred

## Jina Reranker v3

vLLM

In [ ]:
model_name = "jinaai/jina-reranker-v3"
llm = LLM(
    model=model_name,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=1024,
    #gpu_memory_utilization=0.8,
)
for ids, (query, docs) in enumerate(dic.items(), start=1):
    outputs = llm.score(query, docs)
    scores = [out.outputs.score for out in outputs]

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)
    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)

INFO 06-22 08:11:10 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'jinaai/jina-reranker-v3'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/828 [00:00<?, ?B/s]

INFO 06-22 08:11:51 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 06-22 08:11:51 [nixl_utils.py:34] NIXL is not available
WARNING 06-22 08:11:51 [nixl_utils.py:44] NIXL agent config is not available
INFO 06-22 08:11:51 [model.py:555] Resolved architecture: JinaForRanking
WARNING 06-22 08:11:51 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-22 08:11:51 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-22 08:11:51 [model.py:1680] Using max model len 1024
INFO 06-22 08:11:51 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-22 08:11:51 [vllm.py:840] Asynchronous scheduling is disabled.
WARNING 06-22 08:11:51 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WA

tokenizer_config.json:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (12 outputs left out)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 4: atm nuốt thẻ phải làm sao
[3] Score: 0.4117
Nếu máy ATM nuốt thẻ, bạn cần mang CMND/CCCD vào chi nhánh ngân hàng quản lý máy đó để làm thủ tục nhận lại hoặc phát hành thẻ mới.
----------------------------------------------------------------------
[1] Score: 0.1113
If your card is captured by the machine, go inside the branch immediately if it is open, or call the bank's customer service number on the ATM screen.
----------------------------------------------------------------------
[2] Score: 0.0161
Banks usually destroy swallowed cards for security reasons, so you will likely need to request a replacement debit card via your banking app.
----------------------------------------------------------------------
[4] Score: -0.0717
Entering the wrong PIN three consecutive times will lock your debit card to protect you from unauthorized access.
----------------------------------------------------------------------
[5] Score: -0.0749
Many modern ATMs allow cardless withdrawals using

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 5: dispute giao dịch giả mạo thẻ tín dụng
[3] Score: 0.3760
Quy trình khiếu nại giao dịch thẻ tín dụng giả mạo yêu cầu khách hàng điền mẫu đơn tra soát và gửi đến ngân hàng trong vòng 45 ngày.
----------------------------------------------------------------------
[1] Score: 0.2528
To dispute a fraudulent charge, you must contact your card issuer immediately by calling the number on the back of your card and filing a formal fraud report.
----------------------------------------------------------------------
[2] Score: 0.0374
If you notice an unauthorized transaction, log into your mobile banking app, select the specific transaction, and click 'Dispute' within 60 days.
----------------------------------------------------------------------
[7] Score: -0.0259
Bảo hiểm gian lận thẻ tín dụng sẽ bồi thường cho bạn những tổn thất tài chính nếu thông tin thẻ của bạn bị đánh cắp trên mạng.
----------------------------------------------------------------------
[4] Score: -0.0791
Visa Zero 

# Advanced test

In [ ]:
# ==========================================
# TEST CASE 6: Retail Banking - Credit Appraisal (Behavioral vs Traditional)
# User style: Vietlish, keyword heavy, comparing two specific banking terms
# ==========================================
query_6 = "thẩm định tín dụng b-score khác gì cic normal"

documents_6 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "B-Score (Behavioral Score) là điểm tín dụng hành vi được ngân hàng tự xây dựng dựa trên lịch sử giao dịch nội bộ (số dư, dòng tiền, thanh toán), trong khi báo cáo CIC truyền thống chỉ phản ánh lịch sử nợ và dư nợ tín dụng tại hệ thống ngân hàng trung ương.",
    "Unlike the standard CIC report which tracks external borrowing history across all national institutions, a Behavioral Scorecard (B-Score) analyzes a customer's internal transactional habits, deposit consistency, and app usage to predict default risk during the appraisal process.",
    "Trong quy trình cấp tín dụng, CIC cho biết khách hàng có đang nợ xấu ở tổ chức khác hay không, còn B-Score giúp ngân hàng đánh giá độ rủi ro dựa trên thói quen tài chính hàng ngày của khách hàng ngay cả khi họ chưa từng vay vốn (thin-file customers).",

    # --- NORMAL SIMILARITY (Related domain, partial overlap - Tricky!) ---
    "Hệ thống thông tin tín dụng quốc gia (CIC) lưu trữ dữ liệu của khách hàng từ 3 đến 5 năm, bao gồm các nhóm nợ từ Nhóm 1 (nợ đủ tiêu chuẩn) đến Nhóm 5 (nợ có khả năng mất vốn).",
    "Credit scoring models in modern banking utilize machine learning algorithms, such as XGBoost and Random Forest, to analyze hundreds of variables to approve or reject uncollateralized personal loan applications.",
    "Khách hàng có lịch sử nợ xấu trên CIC sẽ bị các ngân hàng từ chối hồ sơ vay thế chấp bất động sản, bất kể thu nhập hiện tại của họ có cao đến mức nào.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "Hệ thống an ninh giám sát tại chi nhánh sử dụng camera nhận diện khuôn mặt có thể phát hiện các cá nhân nằm trong danh sách đen ngay khi họ bước qua cửa tự động.",
    "To open a corporate checking account, the company director must provide the certificate of business registration, the company charter, and the appointment letter of the chief accountant.",
    "Mutual funds allow retail investors to pool their money together to purchase a diversified portfolio of stocks and bonds managed by professional portfolio managers."
]

# ==========================================
# TEST CASE 7: Tech Operations - eKYC Issues
# User style: Vietnamese, short phrase reporting a bug/issue
# ==========================================
query_7 = "lỗi nhận diện khuôn mặt ekyc ánh sáng yếu"

documents_7 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "Khi thực hiện eKYC trong điều kiện thiếu sáng, thuật toán Liveness Detection có thể không ghi nhận được độ sâu của khuôn mặt, dẫn đến lỗi từ chối sinh trắc học. Vui lòng di chuyển đến khu vực có ánh sáng tự nhiên và thử lại.",
    "Facial recognition failures during the digital onboarding (eKYC) process are frequently caused by poor illumination, extreme backlighting, or glare on eyeglasses, which prevents the OCR model from extracting key facial landmarks.",
    "Để khắc phục lỗi không khớp khuôn mặt khi mở tài khoản trực tuyến, người dùng cần đảm bảo không đeo khẩu trang, tháo kính râm, và giữ thiết bị ngang tầm mắt ở nơi có cường độ ánh sáng đạt chuẩn trên 300 lux.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Quy trình eKYC toàn diện bao gồm bước chụp ảnh mặt trước, mặt sau của thẻ Căn cước công dân gắn chip, sau đó hệ thống OCR sẽ tự động bóc tách các trường thông tin như họ tên, ngày sinh, và quê quán.",
    "NFC technology allows the banking app to read the encrypted demographic data directly from the chip embedded in the national ID card, bypassing the need for optical character recognition.",
    "Theo quy định của Ngân hàng Nhà nước, các giao dịch chuyển tiền trên 10 triệu đồng hiện nay đều bắt buộc phải xác thực khuôn mặt để phòng chống gian lận tài khoản ảo.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "Mật khẩu ngân hàng điện tử (Internet Banking) của bạn sẽ bị khóa tự động nếu bạn nhập sai quá 5 lần liên tiếp để bảo vệ tài khoản khỏi các cuộc tấn công brute-force.",
    "Interest on standard savings accounts is calculated based on the actual number of days the funds are held, divided by 365 days, multiplied by the annual percentage rate.",
    "Chủ thẻ tín dụng có thể tận hưởng đặc quyền miễn phí thường niên năm đầu tiên và hoàn tiền 5% cho mọi chi tiêu tại siêu thị vào dịp cuối tuần."
]

# ==========================================
# TEST CASE 8: Retail Banking - Mortgage Penalties
# User style: Keywords only, specific intent
# ==========================================
query_8 = "phí phạt trả nợ trước hạn vay mua nhà"

documents_8 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "If you pay off your mortgage early, the bank may charge a prepayment penalty, typically calculated as 1% to 3% of the remaining principal balance, depending on how many years are left on the contract.",
    "Hầu hết các hợp đồng vay thế chấp bất động sản đều quy định mức phí phạt trả nợ trước hạn từ 1.5% đến 2% trên số tiền gốc trả trước trong 3 năm đầu tiên, và sẽ được miễn phí hoàn toàn từ năm thứ 5 trở đi.",
    "When a borrower liquidates their home loan prior to the maturity date, the financial institution incurs a loss of expected interest revenue, which is offset by applying an early settlement fee based on the outstanding loan amount.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Lãi suất vay mua nhà thường được cố định ở mức ưu đãi (ví dụ 6.9%/năm) trong 12 tháng đầu, sau đó sẽ thả nổi dựa trên lãi suất cơ sở cộng với biên độ dao động từ 3% đến 3.5%.",
    "Late payment fees are applied when a borrower misses their monthly EMI (Equated Monthly Installment) deadline, which also negatively impacts their credit score.",
    "Khi tất toán hồ sơ vay, khách hàng cần nhận lại Giấy chứng nhận quyền sử dụng đất bản chính và mang đến phòng công chứng để thực hiện thủ tục xóa đăng ký giao dịch bảo đảm.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "Cash advance fees for credit cards are generally very expensive, often charging an immediate 4% of the withdrawn amount plus a high daily interest rate.",
    "Overdraft protection is a financial service that automatically transfers funds from a linked savings account to prevent a checking account transaction from being declined.",
    "Người lao động nước ngoài muốn chuyển tiền lương về quốc gia gốc cần cung cấp hợp đồng lao động, sao kê đóng thuế thu nhập cá nhân và hộ chiếu hợp lệ."
]

# ==========================================
# TEST CASE 9: AI/Data Engineering - Information Extraction
# User style: English tech keywords, no grammar
# ==========================================
query_9 = "extract table data bank statement ocr pdf"

documents_9 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "To extract structured tabular data such as transaction dates, descriptions, and amounts from an unstructured PDF bank statement, developers typically employ layout parsing models combined with advanced OCR engines like Tesseract or AWS Textract.",
    "Hệ thống bóc tách dữ liệu thông minh sử dụng các mô hình học sâu để nhận diện các đường viền bảng (bounding boxes) trong tệp PDF scan, từ đó chuyển đổi thông tin sao kê ngân hàng thành định dạng JSON hoặc CSV có cấu trúc.",
    "Table extraction from financial documents is challenging because bank statements lack standardized formats; solutions often require passing the OCR output through a Large Language Model to semantically map the columns correctly.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Core banking systems automatically generate monthly electronic bank statements in secure PDF format, which are then emailed directly to the account holders with a password protection layer.",
    "Optical Character Recognition (OCR) is widely used in mobile banking applications to allow customers to auto-fill payment details simply by taking a photograph of a physical utility bill.",
    "Data engineers build ETL pipelines to migrate massive volumes of transaction records from legacy SQL databases into a centralized cloud data warehouse for business intelligence analysis.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "A cover letter for an AI Engineer position should highlight specific projects, such as deploying computer vision models or fine-tuning language models, to demonstrate practical technical competency.",
    "Quantum search algorithms, like Grover's algorithm, provide a quadratic speedup for unstructured database searches compared to classical linear search techniques.",
    "Phí duy trì tài khoản thanh toán hàng tháng sẽ được ngân hàng tự động miễn phí nếu số dư bình quân trong tháng của khách hàng luôn duy trì ở mức trên 2 triệu đồng."
]

# ==========================================
# TEST CASE 10: Retail Banking - Corporate Accounts
# User style: Vietnamese, very lazy syntax, abbreviation ("cty tnhh", "ko")
# ==========================================
query_10 = "mở tài khoản cty tnhh cần uỷ quyền ko"

documents_10 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "Khi mở tài khoản cho công ty TNHH, Giám đốc (Người đại diện theo pháp luật) bắt buộc phải ký trực tiếp. Nếu Giám đốc không thể đến ngân hàng, họ có thể lập Giấy ủy quyền hợp lệ cho một cá nhân khác để thực hiện thủ tục mở tài khoản.",
    "For a Limited Liability Company (LLC) account opening, the legal representative must sign the mandate. However, they can delegate this authority to a deputy director or chief accountant via a notarized Power of Attorney.",
    "Việc đăng ký chữ ký mẫu và tài khoản thanh toán doanh nghiệp đòi hỏi quyết định bổ nhiệm Kế toán trưởng. Kế toán trưởng có thể không cần ủy quyền nếu điều lệ công ty đã quy định rõ quyền hạn của họ trong giao dịch ngân hàng.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Hồ sơ mở tài khoản tổ chức yêu cầu Bản sao chứng thực Giấy chứng nhận đăng ký doanh nghiệp, Thông báo mẫu dấu, và Căn cước công dân của tất cả những người có liên quan đến tài khoản.",
    "Khách hàng cá nhân không thể ủy quyền cho người khác mở tài khoản ngân hàng thay mình, vì quy định của Ngân hàng Nhà nước bắt buộc chủ tài khoản phải hiện diện trực tiếp để định danh eKYC.",
    "A corporate credit line requires a comprehensive review of the company's audited financial statements for the past two fiscal years, cash flow projections, and business plans.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "Quá trình giải ngân khoản vay mua ô tô chỉ được thực hiện sau khi khách hàng đã hoàn tất việc đăng ký xe, lấy biển số và ký hợp đồng thế chấp cavet xe tại ngân hàng.",
    "Phishing emails often impersonate bank officials, asking you to click on a link to verify your account details in an attempt to steal your login credentials.",
    "Giao dịch hợp đồng tương lai và quyền chọn trên thị trường chứng khoán phái sinh mang lại cơ hội phòng ngừa rủi ro (hedging) cho danh mục đầu tư cơ sở."
]

# ==========================================
# TEST CASE 11: Trade Finance - L/C (Letter of Credit)
# User style: Jargon heavy, typo ("upance" instead of usance)
# ==========================================
query_11 = "lc at sight khác gì upance lc letter of credit"

documents_11 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "The primary difference is the timing of payment: an 'At Sight' Letter of Credit requires the issuing bank to release funds immediately upon presentation of compliant shipping documents, whereas a 'Usance' (deferred) L/C allows the buyer a grace period, such as 90 days, to pay.",
    "Trong tài trợ thương mại, L/C trả ngay (At Sight) ép buộc nhà nhập khẩu thanh toán lập tức khi chứng từ hợp lệ được xuất trình. Ngược lại, L/C trả chậm (Usance L/C) cho phép nhà nhập khẩu nhận hàng trước và thanh toán sau một khoảng thời gian nhất định.",
    "A Usance Letter of Credit is essentially a financing tool that grants the importer credit terms, while an At Sight L/C is purely a payment guarantee mechanism heavily favoring the exporter's cash flow.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "A Bill of Lading (B/L) is a crucial transport document issued by a carrier to acknowledge receipt of cargo, and it is almost always required as part of the document presentation under a Letter of Credit.",
    "Bank guarantees and standby letters of credit (SBLC) serve as a safety net, paying out to the beneficiary only if the applicant fails to fulfill their contractual obligations.",
    "Tỷ giá mua bán ngoại tệ giao ngay (Spot Rate) được áp dụng cho các giao dịch thanh toán quốc tế có ngày giá trị trong vòng 2 ngày làm việc kể từ ngày giao kết hợp đồng.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "Bảo hiểm liên kết đơn vị (Unit-linked insurance) không cam kết mức lãi suất tối thiểu, toàn bộ rủi ro đầu tư sẽ do bên mua bảo hiểm tự chịu dựa trên kết quả của quỹ.",
    "In object-oriented programming, inheritance allows a new class to inherit properties and methods from an existing class, promoting code reusability.",
    "Phí chuyển tiền nhanh 24/7 qua hệ thống Napas giữa các ngân hàng trong nước hiện nay thường được miễn phí hoàn toàn trên các ứng dụng ngân hàng số."
]

# ==========================================
# TEST CASE 12: Retail Banking - Credit Card Foreign Exchange
# User style: Bilingual mix
# ==========================================
query_12 = "thẻ tín dụng charge phí chuyển đổi ngoại tệ foreign transaction fee"

documents_12 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "When you use your credit card to make a purchase in a currency other than your home currency, banks typically charge a foreign transaction fee (FX markup) ranging from 2.5% to 4% of the total transaction amount.",
    "Phí chuyển đổi ngoại tệ trên thẻ tín dụng được tính khi khách hàng quẹt thẻ ở nước ngoài hoặc thanh toán trực tuyến trên các trang web quốc tế (như Agoda, Amazon). Mức phí này bao gồm phí xử lý của tổ chức thẻ (Visa/Mastercard) và biên độ rủi ro tỷ giá của ngân hàng phát hành.",
    "To calculate the total cost of an overseas purchase, the bank first converts the foreign amount to your local currency using the daily Visa/Mastercard exchange rate, then adds the foreign transaction fee percentage on top of that converted amount.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Dynamic Currency Conversion (DCC) is a service offered by overseas merchants that allows you to pay in your home currency, but it almost always features a significantly worse exchange rate than letting your bank handle the conversion.",
    "Phí rút tiền mặt tại ATM bằng thẻ tín dụng thường rất cao, lên tới 4% số tiền rút, và bị tính lãi suất ngay lập tức kể từ thời điểm giao dịch thành công.",
    "When receiving an inbound international wire transfer, the beneficiary bank may deduct a receiving fee from the total principal amount before crediting the funds to your account.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "A balloon payment is an unusually large payment due at the end of a loan term, common in certain types of commercial real estate financing or auto leasing.",
    "Khách hàng có thể đăng ký phát hành thẻ ghi nợ phi vật lý (virtual card) ngay trên ứng dụng mobile banking và sử dụng ngay lập tức cho các giao dịch thương mại điện tử.",
    "Mã xác thực OTP (One Time Password) được gửi qua SMS hoặc ứng dụng Smart OTP để đảm bảo người thực hiện giao dịch chính là chủ sở hữu tài khoản hợp pháp."
]

# ==========================================
# TEST CASE 13: Technical/AI - System Architecture
# User style: Tech jargon, troubleshooting memory limits
# ==========================================
query_13 = "qwen 4b max_model_len context window OOM fix"

documents_13 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "When experiencing Out-Of-Memory (OOM) crashes loading models like Qwen 4B on a 16GB GPU, you must manually reduce the `max_model_len` parameter (e.g., from 40k to 1024) to prevent the vLLM engine from pre-allocating an excessively large KV cache for the context window.",
    "Lỗi CUDA OOM khi chạy LLM thường xảy ra do mô hình cố gắng dành trước bộ nhớ cho toàn bộ context window. Để khắc phục, hãy hạ `max_model_len` xuống giới hạn văn bản thực tế của bạn, và kích hoạt `enforce_eager=True` để vô hiệu hóa CUDA Graphs ngốn RAM.",
    "Adjusting the `gpu_memory_utilization` flag to a higher strict limit (like 0.95) combined with drastically cutting the `max_model_len` is the most effective fix to fit 4-billion parameter language models into edge consumer hardware without triggering allocation failures.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Large Language Models (LLMs) utilize RoPE (Rotary Positional Embeddings) to handle dynamic context lengths, allowing them to extrapolate text relationships far beyond the sequence length they were originally trained on.",
    "To fine-tune a massive neural network on consumer GPUs, engineers deploy LoRA (Low-Rank Adaptation) and 4-bit Quantization (QLoRA) techniques to drastically reduce the VRAM footprint required for backpropagation.",
    "Hệ thống RAG (Retrieval-Augmented Generation) hoạt động bằng cách chia nhỏ các tài liệu dài thành các chunk (khoảng 500 tokens) để đưa vào context window của LLM, giúp mô hình trả lời chính xác mà không bị quá tải bộ nhớ.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "Thẻ tín dụng nội địa NAPAS chỉ có thể sử dụng để thanh toán tại các máy POS và trang web có cổng thanh toán trong phạm vi lãnh thổ Việt Nam.",
    "To close a joint bank account, both account holders typically need to present themselves at a physical branch location and sign a mutual account closure form.",
    "Thu nhập từ lãi tiền gửi tiết kiệm của cá nhân tại các tổ chức tín dụng hiện nay đang được nhà nước miễn thuế thu nhập cá nhân."
]

# ==========================================
# TEST CASE 14: Computer Vision - AI Operations
# User style: English/Vietnamese mix, specific to industrial camera logic
# ==========================================
query_14 = "yolov8 detect intrusion zone camera an ninh"

documents_14 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "To build a safety monitoring system, you can deploy a YOLOv8 object detection model on camera feeds, explicitly defining Polygon Region of Interests (ROIs). When the bounding box of a detected 'person' intersects with the predefined intrusion zone coordinates, the system triggers an alert.",
    "Ứng dụng AI vào camera an ninh ngân hàng hoặc nhà máy yêu cầu thiết lập các vùng cấm (restricted zones). Khi mô hình YOLOv8 phát hiện sự xâm nhập, logic lập trình sẽ sử dụng hàm tính toán đa giác (point-in-polygon) để xác định xem tọa độ tâm của đối tượng có nằm trong khu vực vi phạm hay không.",
    "Real-time intrusion detection relies on feeding RTSP streams from CCTV cameras into a modular computer vision pipeline, utilizing YOLOv8 for rapid frame-by-frame inference and OpenCV for drawing the visual warning boundaries over the restricted zones.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Convolutional Neural Networks (CNNs) are the backbone of facial recognition systems used at ATM terminals to verify if the individual withdrawing cash matches the account holder's registered identity.",
    "Hệ thống phát hiện gian lận thẻ tín dụng sử dụng các mô hình học máy để phân tích hành vi bất thường, ví dụ như phát hiện một giao dịch tại Việt Nam và một giao dịch khác tại Mỹ chỉ cách nhau vỏn vẹn 30 phút.",
    "To process video streams efficiently on edge devices, AI engineers often convert standard PyTorch models into TensorRT or ONNX formats to maximize frame-rate throughput while minimizing latency.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "Mở sổ tiết kiệm đứng tên hai người (đồng sở hữu) yêu cầu cả hai bên phải cùng ký vào sổ, và khi rút tiền cũng cần sự hiện diện của cả hai người trừ khi có giấy ủy quyền đặc biệt.",
    "For freshers interviewing for a Data Engineer position, expect technical questions focused on writing complex SQL window functions and designing robust ETL architectures.",
    "The compound interest formula calculates accumulated wealth by applying the interest rate not only to the initial principal but also to the accumulated interest of previous periods."
]

# ==========================================
# TEST CASE 15: Retail Banking - Disputed Charges
# User style: Bilingual mix, describing a common merchant glitch
# ==========================================
query_15 = "tiền bị trừ 2 lần double charge shopee bank"

documents_15 = [
    # --- HIGH SIMILARITY (Direct answers) ---
    "A 'double charge' on an e-commerce platform like Shopee usually occurs due to a network timeout during the payment gateway process. One charge is the actual settled transaction, while the other is merely a temporary authorization hold (phong tỏa tài khoản) that the bank will automatically drop within 7 to 14 business days.",
    "Khi thanh toán trực tuyến bị lỗi trừ tiền 2 lần, ngân hàng thực chất chỉ ghi nhận 1 giao dịch thành công. Khoản tiền thứ hai đang bị hệ thống thanh toán quốc tế treo dưới dạng 'Pending'. Nếu Shopee không gửi lệnh quyết toán (settlement), số dư của bạn sẽ tự động hoàn lại mà không cần khiếu nại.",
    "If your credit or debit card statement shows duplicate transactions for a single purchase, do not panic immediately. Contact the merchant first to confirm they only received one order, and the duplicate bank hold will expire and fall off your account naturally.",

    # --- NORMAL SIMILARITY (Related domain, partial overlap) ---
    "Để thực hiện khiếu nại (dispute) một giao dịch thẻ tín dụng giả mạo do bị lộ thông tin, khách hàng phải liên hệ tổng đài khóa thẻ khẩn cấp và nộp đơn tra soát trong vòng 45 ngày kể từ ngày phát sinh giao dịch.",
    "Refunds originating from canceled orders on e-commerce platforms can take up to 45 days to reflect on your credit card statement, depending entirely on the merchant's payment processor and the issuing bank's policies.",
    "Bảo mật thanh toán 3D-Secure yêu cầu khách hàng nhập mã OTP gửi về điện thoại trước khi giao dịch trực tuyến được phê duyệt, giúp hạn chế rủi ro bị ăn cắp thông tin thẻ.",

    # --- LOW SIMILARITY (Different topic entirely) ---
    "In Transformer architectures, the Self-Attention mechanism computes a weighted average of all words in a sentence, determining how much focus to place on surrounding context words to understand the meaning.",
    "Tài sản bảo đảm cho khoản vay tiêu dùng có thể là sổ tiết kiệm, giấy tờ có giá, hoặc số dư trên tài khoản thanh toán được phong tỏa theo thỏa thuận giữa ngân hàng và khách hàng.",
    "Vesting schedules for startup employee stock options (ESOP) typically involve a one-year cliff, meaning the employee must stay for at least 12 months before earning the right to exercise any shares."
]

dic = {
    query_6: documents_6,
    query_7: documents_7,
    query_8: documents_8,
    query_9: documents_9,
    query_10: documents_10,
    query_11: documents_11,
    query_12: documents_12,
    query_13: documents_13,
    query_14: documents_14,
    query_15: documents_15,
}

## Qwen 3 Reranker

### 0.6B

In [ ]:
model_name = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=model_name,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=1024,
        gpu_memory_utilization=0.8,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
for ids, (query, docs) in enumerate(dic.items(), start=1):
    outputs = llm.score(query, docs, chat_template=chat_template)
    scores = [out.outputs.score for out in outputs]

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)
    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)
clear_gpu_memory()

INFO 06-23 02:29:46 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Reranker-0.6B'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

INFO 06-23 02:29:49 [config.py:960] Found sentence-transformers tokenize configuration.
INFO 06-23 02:30:09 [model.py:924] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-23 02:30:42 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 06-23 02:30:42 [nixl_utils.py:34] NIXL is not available
WARNING 06-23 02:30:42 [nixl_utils.py:44] NIXL agent config is not available
INFO 06-23 02:30:42 [model.py:555] Resolved architecture: Qwen3ForSequenceClassification


modules.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

... (24 outputs left out)

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 9: yolov8 detect intrusion zone camera an ninh
[3] Score: 0.9996
Real-time intrusion detection relies on feeding RTSP streams from CCTV cameras into a modular computer vision pipeline, utilizing YOLOv8 for rapid frame-by-frame inference and OpenCV for drawing the visual warning boundaries over the restricted zones.
----------------------------------------------------------------------
[1] Score: 0.9994
To build a safety monitoring system, you can deploy a YOLOv8 object detection model on camera feeds, explicitly defining Polygon Region of Interests (ROIs). When the bounding box of a detected 'person' intersects with the predefined intrusion zone coordinates, the system triggers an alert.
----------------------------------------------------------------------
[2] Score: 0.9993
Ứng dụng AI vào camera an ninh ngân hàng hoặc nhà máy yêu cầu thiết lập các vùng cấm (restricted zones). Khi mô hình YOLOv8 phát hiện sự xâm nhập, logic lập trình sẽ sử dụng hàm tính toán đa giác (point-in-p

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 10: tiền bị trừ 2 lần double charge shopee bank
[1] Score: 0.9999
A 'double charge' on an e-commerce platform like Shopee usually occurs due to a network timeout during the payment gateway process. One charge is the actual settled transaction, while the other is merely a temporary authorization hold (phong tỏa tài khoản) that the bank will automatically drop within 7 to 14 business days.
----------------------------------------------------------------------
[2] Score: 0.9997
Khi thanh toán trực tuyến bị lỗi trừ tiền 2 lần, ngân hàng thực chất chỉ ghi nhận 1 giao dịch thành công. Khoản tiền thứ hai đang bị hệ thống thanh toán quốc tế treo dưới dạng 'Pending'. Nếu Shopee không gửi lệnh quyết toán (settlement), số dư của bạn sẽ tự động hoàn lại mà không cần khiếu nại.
----------------------------------------------------------------------
[4] Score: 0.0812
Để thực hiện khiếu nại (dispute) một giao dịch thẻ tín dụng giả mạo do bị lộ thông tin, khách hàng phải liên hệ tổng đài khóa th

### 4B

In [ ]:
model_name = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
    model=model_name,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=1024,
    gpu_memory_utilization=0.8,
    hf_overrides={
        "architectures": ["Qwen3ForSequenceClassification"],
        "classifier_from_token": ["no", "yes"],
        "is_original_qwen3_reranker": True,
    },
)
for ids, (query, docs) in enumerate(dic.items(), start=1):
    outputs = llm.score(query, docs, chat_template=chat_template)
    scores = [out.outputs.score for out in outputs]

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)
    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)

clear_gpu_memory()

INFO 06-23 02:39:40 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Reranker-4B'}


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

INFO 06-23 02:39:41 [config.py:960] Found sentence-transformers tokenize configuration.
INFO 06-23 02:39:41 [model.py:924] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-23 02:39:41 [model.py:555] Resolved architecture: Qwen3ForSequenceClassification


modules.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

INFO 06-23 02:39:41 [config.py:848] Found sentence-transformers modules configuration.
WARNING 06-23 02:39:41 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-23 02:39:41 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-23 02:39:41 [model.py:1680] Using max model len 1024
WARNING 06-23 02:39:41 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-23 02:39:41 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-23 02:39:41 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-23 02:39:41 [vllm.py:1089] Cudagraph is disabled under eager mode


... (24 outputs left out)

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 9: yolov8 detect intrusion zone camera an ninh
[2] Score: 0.4783
Ứng dụng AI vào camera an ninh ngân hàng hoặc nhà máy yêu cầu thiết lập các vùng cấm (restricted zones). Khi mô hình YOLOv8 phát hiện sự xâm nhập, logic lập trình sẽ sử dụng hàm tính toán đa giác (point-in-polygon) để xác định xem tọa độ tâm của đối tượng có nằm trong khu vực vi phạm hay không.
----------------------------------------------------------------------
[3] Score: 0.4549
Real-time intrusion detection relies on feeding RTSP streams from CCTV cameras into a modular computer vision pipeline, utilizing YOLOv8 for rapid frame-by-frame inference and OpenCV for drawing the visual warning boundaries over the restricted zones.
----------------------------------------------------------------------
[1] Score: 0.4380
To build a safety monitoring system, you can deploy a YOLOv8 object detection model on camera feeds, explicitly defining Polygon Region of Interests (ROIs). When the bounding box of a detected 'person' 

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 10: tiền bị trừ 2 lần double charge shopee bank
[2] Score: 0.5336
Khi thanh toán trực tuyến bị lỗi trừ tiền 2 lần, ngân hàng thực chất chỉ ghi nhận 1 giao dịch thành công. Khoản tiền thứ hai đang bị hệ thống thanh toán quốc tế treo dưới dạng 'Pending'. Nếu Shopee không gửi lệnh quyết toán (settlement), số dư của bạn sẽ tự động hoàn lại mà không cần khiếu nại.
----------------------------------------------------------------------
[1] Score: 0.4548
A 'double charge' on an e-commerce platform like Shopee usually occurs due to a network timeout during the payment gateway process. One charge is the actual settled transaction, while the other is merely a temporary authorization hold (phong tỏa tài khoản) that the bank will automatically drop within 7 to 14 business days.
----------------------------------------------------------------------
[3] Score: 0.3263
If your credit or debit card statement shows duplicate transactions for a single purchase, do not panic immediately. Contact the

## bge-reranker-v3-m3

In [ ]:
model_name = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=model_name,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          gpu_memory_utilization=0.4,)

for ids, (query, docs) in enumerate(dic.items(), start=1):
    outputs = llm.score(query, docs)
    scores = [out.outputs.score for out in outputs]

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)

    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)

clear_gpu_memory()

INFO 06-23 02:50:00 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

INFO 06-23 02:50:22 [model.py:555] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-23 02:50:22 [model.py:2015] Downcasting torch.float32 to torch.float16.


tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

INFO 06-23 02:50:22 [model.py:1680] Using max model len 1024
WARNING 06-23 02:50:22 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-23 02:50:22 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-23 02:50:22 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-23 02:50:22 [vllm.py:1089] Cudagraph is disabled under eager mode


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

... (19 outputs left out)

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 9: yolov8 detect intrusion zone camera an ninh
[2] Score: 0.9853
Ứng dụng AI vào camera an ninh ngân hàng hoặc nhà máy yêu cầu thiết lập các vùng cấm (restricted zones). Khi mô hình YOLOv8 phát hiện sự xâm nhập, logic lập trình sẽ sử dụng hàm tính toán đa giác (point-in-polygon) để xác định xem tọa độ tâm của đối tượng có nằm trong khu vực vi phạm hay không.
----------------------------------------------------------------------
[1] Score: 0.9813
To build a safety monitoring system, you can deploy a YOLOv8 object detection model on camera feeds, explicitly defining Polygon Region of Interests (ROIs). When the bounding box of a detected 'person' intersects with the predefined intrusion zone coordinates, the system triggers an alert.
----------------------------------------------------------------------
[3] Score: 0.9498
Real-time intrusion detection relies on feeding RTSP streams from CCTV cameras into a modular computer vision pipeline, utilizing YOLOv8 for rapid frame-by-frame i

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 10: tiền bị trừ 2 lần double charge shopee bank
[2] Score: 0.9389
Khi thanh toán trực tuyến bị lỗi trừ tiền 2 lần, ngân hàng thực chất chỉ ghi nhận 1 giao dịch thành công. Khoản tiền thứ hai đang bị hệ thống thanh toán quốc tế treo dưới dạng 'Pending'. Nếu Shopee không gửi lệnh quyết toán (settlement), số dư của bạn sẽ tự động hoàn lại mà không cần khiếu nại.
----------------------------------------------------------------------
[1] Score: 0.7635
A 'double charge' on an e-commerce platform like Shopee usually occurs due to a network timeout during the payment gateway process. One charge is the actual settled transaction, while the other is merely a temporary authorization hold (phong tỏa tài khoản) that the bank will automatically drop within 7 to 14 business days.
----------------------------------------------------------------------
[3] Score: 0.0056
If your credit or debit card statement shows duplicate transactions for a single purchase, do not panic immediately. Contact the

##Jina Reranker v3

In [ ]:
model_name = "jinaai/jina-reranker-v3"
llm = LLM(
    model=model_name,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=1024,
    gpu_memory_utilization=0.4,
)
for ids, (query, docs) in enumerate(dic.items(), start=1):
    outputs = llm.score(query, docs)
    scores = [out.outputs.score for out in outputs]

    combined_data = [(i, doc, score) for i, (doc, score) in enumerate(zip(docs, scores))]
    ranked_results = sorted(combined_data, key=lambda x: x[2], reverse=True)
    print(f"\nQuery {ids}: {query}")
    print("="*70)
    for original_idx, doc, score in ranked_results:
        print(f"[{original_idx + 1}] Score: {score:.4f}")
        print(f"{doc}")
        print("-" * 70)

clear_gpu_memory()

INFO 06-23 02:53:10 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'jinaai/jina-reranker-v3'}


config.json:   0%|          | 0.00/828 [00:00<?, ?B/s]

INFO 06-23 02:53:27 [model.py:555] Resolved architecture: JinaForRanking
WARNING 06-23 02:53:27 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-23 02:53:27 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-23 02:53:27 [model.py:1680] Using max model len 1024
WARNING 06-23 02:53:27 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-23 02:53:27 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-23 02:53:27 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-23 02:53:27 [vllm.py:1089] Cudagraph is disabled under eager mode


tokenizer_config.json:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (21 outputs left out)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 9: yolov8 detect intrusion zone camera an ninh
[2] Score: 0.2840
Ứng dụng AI vào camera an ninh ngân hàng hoặc nhà máy yêu cầu thiết lập các vùng cấm (restricted zones). Khi mô hình YOLOv8 phát hiện sự xâm nhập, logic lập trình sẽ sử dụng hàm tính toán đa giác (point-in-polygon) để xác định xem tọa độ tâm của đối tượng có nằm trong khu vực vi phạm hay không.
----------------------------------------------------------------------
[1] Score: 0.2306
To build a safety monitoring system, you can deploy a YOLOv8 object detection model on camera feeds, explicitly defining Polygon Region of Interests (ROIs). When the bounding box of a detected 'person' intersects with the predefined intrusion zone coordinates, the system triggers an alert.
----------------------------------------------------------------------
[3] Score: 0.2178
Real-time intrusion detection relies on feeding RTSP streams from CCTV cameras into a modular computer vision pipeline, utilizing YOLOv8 for rapid frame-by-frame i

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 10: tiền bị trừ 2 lần double charge shopee bank
[2] Score: 0.4860
Khi thanh toán trực tuyến bị lỗi trừ tiền 2 lần, ngân hàng thực chất chỉ ghi nhận 1 giao dịch thành công. Khoản tiền thứ hai đang bị hệ thống thanh toán quốc tế treo dưới dạng 'Pending'. Nếu Shopee không gửi lệnh quyết toán (settlement), số dư của bạn sẽ tự động hoàn lại mà không cần khiếu nại.
----------------------------------------------------------------------
[1] Score: 0.2364
A 'double charge' on an e-commerce platform like Shopee usually occurs due to a network timeout during the payment gateway process. One charge is the actual settled transaction, while the other is merely a temporary authorization hold (phong tỏa tài khoản) that the bank will automatically drop within 7 to 14 business days.
----------------------------------------------------------------------
[3] Score: 0.0066
If your credit or debit card statement shows duplicate transactions for a single purchase, do not panic immediately. Contact the

# Comparison

## Simple

Please run test cases cell in the **"Simple test"** section in advance

In [ ]:
bge_m3 = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=bge_m3,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          gpu_memory_utilization=0.4,
          disable_log_stats=True,
          )
evaluate_model(bge_m3, llm, dic, simple_eval)

INFO 06-23 07:12:46 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}
INFO 06-23 07:12:46 [model.py:555] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-23 07:12:46 [model.py:2015] Downcasting torch.float32 to torch.float16.
INFO 06-23 07:12:46 [model.py:1680] Using max model len 1024
WARNING 06-23 07:12:46 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-23 07:12:46 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-23 07:12:46 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-23 07:12:46 [vllm.py:1089] Cudagr

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Kết quả Ranking Quality:
  - Recall @3:           100.0%
  - MRR:                 1.0000

Kết quả System Performance:
  - Độ trễ (Latency):    347.6 ms/query
  - Thông lượng (QPS):   2.88 queries/second
  - Tốc độ xử lý Docs:   26.47 docs/second
  - VRAM tiêu thụ (Peak): 1.47 GB


In [ ]:
qwen3_06 = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_06,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=1024,
        gpu_memory_utilization=0.4,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_06, llm, dic, simple_eval)

INFO 06-23 07:13:25 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Reranker-0.6B'}
INFO 06-23 07:13:26 [model.py:924] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-23 07:13:26 [model.py:555] Resolved architecture: Qwen3ForSequenceClassification
WARNING 06-23 07:13:26 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-23 07:13:26 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-23 07:13:26 [model.py:1680] Using max model len 1024
WARNING 06-23 07:13:26 [vllm.py:896] Enforce eager set, disabling torc

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Kết quả Ranking Quality:
  - Recall @3:           66.7%
  - MRR:                 0.9000

Kết quả System Performance:
  - Độ trễ (Latency):    456.0 ms/query
  - Thông lượng (QPS):   2.19 queries/second
  - Tốc độ xử lý Docs:   20.17 docs/second
  - VRAM tiêu thụ (Peak): 6.35 GB


In [ ]:
qwen3_4 = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_4,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=1024,
        gpu_memory_utilization=0.8,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_4, llm, dic, simple_eval)

INFO 06-23 07:14:08 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Reranker-4B'}
INFO 06-23 07:14:09 [model.py:924] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-23 07:14:09 [model.py:555] Resolved architecture: Qwen3ForSequenceClassification
WARNING 06-23 07:14:09 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-23 07:14:09 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-23 07:14:09 [model.py:1680] Using max model len 1024
WARNING 06-23 07:14:09 [vllm.py:896] Enforce eager set, disabling torch.

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Kết quả Ranking Quality:
  - Recall @3:           53.3%
  - MRR:                 0.9000

Kết quả System Performance:
  - Độ trễ (Latency):    568.2 ms/query
  - Thông lượng (QPS):   1.76 queries/second
  - Tốc độ xử lý Docs:   16.19 docs/second
  - VRAM tiêu thụ (Peak): 11.81 GB


In [ ]:
jina_v3 = "jinaai/jina-reranker-v3"
llm = LLM(
    model=jina_v3,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=1024,
    gpu_memory_utilization=0.4,
    disable_log_stats=True,
)
evaluate_model(jina_v3, llm, dic, simple_eval)

INFO 06-23 07:15:15 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'jinaai/jina-reranker-v3'}
INFO 06-23 07:15:16 [model.py:555] Resolved architecture: JinaForRanking
WARNING 06-23 07:15:16 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-23 07:15:16 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-23 07:15:16 [model.py:1680] Using max model len 1024
WARNING 06-23 07:15:16 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-23 07:15:16 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-23 07:15:16 [kernel.p

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Kết quả Ranking Quality:
  - Recall @3:           100.0%
  - MRR:                 1.0000

Kết quả System Performance:
  - Độ trễ (Latency):    332.3 ms/query
  - Thông lượng (QPS):   3.01 queries/second
  - Tốc độ xử lý Docs:   27.68 docs/second
  - VRAM tiêu thụ (Peak): 6.47 GB


## Advance

Please run test cases cell in the **"Advanced test"** section in advance

In [ ]:
bge_m3 = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=bge_m3,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          gpu_memory_utilization=0.4,
          disable_log_stats=True,
          )
evaluate_model(bge_m3, llm, dic, advance_eval)

INFO 06-23 07:18:02 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}
INFO 06-23 07:18:03 [model.py:555] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-23 07:18:03 [model.py:2015] Downcasting torch.float32 to torch.float16.
INFO 06-23 07:18:03 [model.py:1680] Using max model len 1024
WARNING 06-23 07:18:03 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-23 07:18:03 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-23 07:18:03 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-23 07:18:03 [vllm.py:1089] Cudagr

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (2 outputs left out)

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Kết quả Ranking Quality:
  - Recall @3:           96.7%
  - MRR:                 1.0000

Kết quả System Performance:
  - Độ trễ (Latency):    303.0 ms/query
  - Thông lượng (QPS):   3.30 queries/second
  - Tốc độ xử lý Docs:   29.70 docs/second
  - VRAM tiêu thụ (Peak): 1.49 GB


In [ ]:
jina_v3 = "jinaai/jina-reranker-v3"
llm = LLM(
    model=jina_v3,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=1024,
    gpu_memory_utilization=0.4,
    disable_log_stats=True,
)
evaluate_model(jina_v3, llm, dic, advance_eval)

INFO 06-23 07:18:41 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'jinaai/jina-reranker-v3'}
INFO 06-23 07:18:42 [model.py:555] Resolved architecture: JinaForRanking
WARNING 06-23 07:18:42 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-23 07:18:42 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-23 07:18:42 [model.py:1680] Using max model len 1024
WARNING 06-23 07:18:42 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-23 07:18:42 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-23 07:18:42 [kernel.p

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (2 outputs left out)

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Kết quả Ranking Quality:
  - Recall @3:           100.0%
  - MRR:                 1.0000

Kết quả System Performance:
  - Độ trễ (Latency):    212.9 ms/query
  - Thông lượng (QPS):   4.70 queries/second
  - Tốc độ xử lý Docs:   42.27 docs/second
  - VRAM tiêu thụ (Peak): 6.47 GB


In [ ]:
qwen3_06 = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_06,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=1024,
        gpu_memory_utilization=0.4,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_06, llm, dic, advance_eval)

INFO 06-23 07:19:16 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Reranker-0.6B'}
INFO 06-23 07:19:16 [model.py:924] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-23 07:19:16 [model.py:555] Resolved architecture: Qwen3ForSequenceClassification
WARNING 06-23 07:19:16 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-23 07:19:16 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-23 07:19:16 [model.py:1680] Using max model len 1024
WARNING 06-23 07:19:16 [vllm.py:896] Enforce eager set, disabling torc

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (2 outputs left out)

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Kết quả Ranking Quality:
  - Recall @3:           70.0%
  - MRR:                 1.0000

Kết quả System Performance:
  - Độ trễ (Latency):    213.9 ms/query
  - Thông lượng (QPS):   4.68 queries/second
  - Tốc độ xử lý Docs:   42.08 docs/second
  - VRAM tiêu thụ (Peak): 6.37 GB


In [ ]:
qwen3_4 = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_4,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=1024,
        gpu_memory_utilization=0.8,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_4, llm, dic, advance_eval)

INFO 06-23 07:19:51 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Reranker-4B'}
INFO 06-23 07:19:51 [model.py:924] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-23 07:19:51 [model.py:555] Resolved architecture: Qwen3ForSequenceClassification
WARNING 06-23 07:19:51 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-23 07:19:51 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-23 07:19:51 [model.py:1680] Using max model len 1024
WARNING 06-23 07:19:51 [vllm.py:896] Enforce eager set, disabling torch.

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (2 outputs left out)

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Kết quả Ranking Quality:
  - Recall @3:           56.7%
  - MRR:                 0.6976

Kết quả System Performance:
  - Độ trễ (Latency):    483.1 ms/query
  - Thông lượng (QPS):   2.07 queries/second
  - Tốc độ xử lý Docs:   18.63 docs/second
  - VRAM tiêu thụ (Peak): 11.85 GB


## Extreme

These test cases base completely on internal documents used in Trainee Program of Shinhan Bank VN.

###1. Retail Deposit

In [ ]:
clear_gpu_memory()

# ==============================================================================
# TEST CASE 1: Mở tài khoản và đăng ký CIF cho người nước ngoài
# ==============================================================================
query_1 = "Hồ sơ và quy định đăng ký CIF để mở tài khoản cho khách nước ngoài tại Việt Nam, cần gì để chứng minh cư trú"

documents_1 = [
    # Chunk 1 (Rank 1 - Rất cao): Trực tiếp trả lời loại CIF và hồ sơ giấy tờ tùy thân, thị thực.
    """Khách hàng cá nhân người nước ngoài được phân loại vào CIF Type 2-Person. Trong đó, CIF Class 110 dành cho khách hàng người Hàn Quốc và CIF Class 120 dành cho khách hàng người nước ngoài không phải người Hàn Quốc. Hồ sơ yêu cầu đối với khách hàng người nước ngoài bao gồm Giấy tờ tùy thân (Hộ chiếu) và Giấy tờ có giá trị thay thế thị thực. Cụ thể, khách hàng cần cung cấp Thị thực, Giấy chứng nhận tạm trú (Thẻ tạm trú), Thẻ thường trú, Giấy miễn thị thực, hoặc Thẻ đi lại doanh nhân APEC & Con dấu nhập cảnh. Các giấy tờ chứng minh địa chỉ nơi ở hiện tại ở Việt Nam cũng bắt buộc đối với người cư trú.""",

    # Chunk 2 (Rank 2 - Cao): Giải thích chi tiết về thời hạn cư trú và con dấu nhập cảnh.
    """Người nước ngoài được phép cư trú tại Việt Nam với thời hạn từ 12 tháng (360 ngày) trở lên được xem là người cư trú. Tình trạng cư trú được xác định dựa trên thời hạn cư trú phép tại Việt Nam thể hiện trên giấy tờ tùy thân. Đối với Thẻ tạm trú, tình trạng cư trú có giá trị trong thời hạn của thẻ. Đối với khách hàng người nước ngoài (bao gồm người Hàn Quốc) sử dụng Thị thực điện tử (E-visa), thị thực này được phê duyệt trước khi nhập cảnh với thời hạn tối đa 90 ngày. Con dấu nhập cảnh bao gồm thời hạn cư trú được phép tại Việt Nam. Thời hạn cư trú sẽ là ngày nhập cảnh và ngày cuối cùng được phép cư trú tại Việt Nam trên Con dấu nhập cảnh.""",

    # Chunk 3 (Rank 3 - Khá): Đề cập đến biểu mẫu điền mở tài khoản chung cho cá nhân.
    """Sơ lược về mẫu biểu liên quan đến hoạt động mở tài khoản và tiền gửi bao gồm: Deposit 01a (Đơn đăng ký thông tin, mở tài khoản và dịch vụ dành cho khách hàng cá nhân). Biểu mẫu này dùng chung cho khách hàng Việt Nam và người nước ngoài. Khách hàng phải điền đầy đủ Thông tin Chủ tài khoản, trong đó Phần 3 dành riêng cho Khách hàng cá nhân đa quốc tịch, bao gồm việc xác nhận có mang quốc tịch thứ ba, thông tin mã số thuế nước ngoài hoặc có đang cư trú ở quốc gia mạng quốc tịch thứ ba hay không. Ngoài ra, biểu mẫu yêu cầu chữ ký mẫu và chữ ký phải nhất quán với hồ sơ.""",

    # Chunk 4 (Rank 4 - Trung bình): Đề cập đến một loại tài khoản đặc thù liên quan đến nước ngoài (K-Study)
    """Tài khoản K-Study là tài khoản Thanh toán dành cho khách hàng cá nhân Việt Nam có nhu cầu nhận Giấy xác nhận Tiền Gửi Đảm Bảo phí du học Hàn Quốc. Đối tượng là khách hàng cá nhân người Việt Nam, loại tiền tệ áp dụng là VND. Số tiền bảo đảm tương đương 8,000,000 KRW hoặc 10,000,000 KRW trở lên. Đối với tài khoản này, không áp dụng ủy quyền. Hồ sơ yêu cầu bao gồm Thẻ Căn cước công dân gắn chip, Hộ chiếu có số định danh cá nhân, và bản scan Thẻ đăng ký người nước ngoài tại Hàn Quốc được gia hạn (khi cần giải tỏa tiền ký quỹ sau 06 tháng đầu tiên).""",

    # Chunk 5 (Rank 5 - Thấp): Quy định về nhận diện khách hàng Việt Nam (chỉ để đối chiếu với người nước ngoài).
    """Đối với khách hàng người Việt Nam, giấy tờ tùy thân bao gồm Thẻ Căn cước công dân gắn chip hoặc Thẻ Căn cước. Thông tin lưu trữ trên chip đã bao gồm số CMND 09 số, nhân viên ngân hàng kiểm tra bằng máy đọc chip tại quầy giao dịch. Trong trường hợp khách hàng sử dụng Chứng minh nhân dân 9 hoặc 12 số, các loại giấy tờ này chỉ có giá trị sử dụng đến hết ngày 31/12/2024. Từ 01/07/2024, Thẻ Căn cước mới bắt đầu được ban hành.""",

    # Chunk 6 (Rank 6 - Thấp): Thông tin về đại diện hợp pháp cho khách hàng vị thành niên.
    """Khách hàng vị thành niên là người dưới 18 tuổi. Khách hàng từ đủ 15 tuổi đến dưới 18 tuổi có năng lực hành vi dân sự đầy đủ theo quy định của pháp luật có thể tự mở tài khoản thanh toán mà không cần người đại diện theo pháp luật. Khách hàng dưới 15 tuổi phải thông qua người đại diện theo pháp luật (bố mẹ, người giám hộ). Hồ sơ mở tài khoản cho người vị thành niên cần giấy tờ pháp lý chứng minh mối quan hệ giữa trẻ vị thành niên và người đại diện (giấy khai sinh, giấy tờ hợp pháp hóa lãnh sự đối với người nước ngoài).""",

    # Chunk 7 (Rank 7 - Rất thấp): Tài khoản S-Payroll.
    """Nhóm tài khoản không kỳ hạn bao gồm tài khoản S-Payroll. Đây là tài khoản thanh toán với những quyền lợi tối ưu dành cho khách hàng cá nhân có nhu cầu nhận lương. S-Payroll miễn phí hoàn toàn các dịch vụ như: Rút tiền tại ATM Shinhan và các ngân hàng khác, Chuyển tiền trong nước, chuyển tiền 24/7 qua IB/MB/ATM, thanh toán hóa đơn (điện, nước, internet), dịch vụ thông báo qua Zalo và dịch vụ ngân hàng điện tử.""",

    # Chunk 8 (Rank 8 - Rất thấp): Thẻ ATM và mã PIN.
    """Mã số xác định chủ thẻ PIN (6 số) có thể đổi qua các kênh như Chi nhánh, ATM, hệ thống trả lời tự động, ứng dụng SOL. Thẻ sẽ tạm khóa nếu nhập sai PIN quá 5 lần. Thời hạn sử dụng thẻ không vượt quá thời hạn cư trú tại Việt Nam đối với Chủ thẻ là người nước ngoài. Cần cập nhật thông tin để tiếp tục sử dụng dịch vụ hoặc phát hành lại thẻ mới để tiếp tục sử dụng dịch vụ.""",

    # Chunk 9 (Rank 9 - Không liên quan): Tài khoản đầu tư gián tiếp (IIA).
    """Tài khoản đầu tư gián tiếp (IIA) là tài khoản thanh toán bằng đồng Việt Nam được sử dụng để thực hiện các giao dịch chuyển vốn hợp pháp khác liên quan đến hoạt động đầu tư gián tiếp vào đối tượng khách hàng cá nhân nước ngoài là người không cư trú. Nộp tiền mặt vào tài khoản IIA không được phép. Tài khoản đầu tư gián tiếp chung không áp dụng. Số dư trên IIA không được dùng để mở tiền gửi có kỳ hạn hoặc tiền gửi tích lũy.""",

    # Chunk 10 (Rank 10 - Không liên quan): Xếp hạng Tops Club.
    """Chương trình Tops Club là chương trình khách hàng thân thiết dành cho những khách hàng gắn bó lâu năm và sử dụng nhiều sản phẩm/dịch vụ của Ngân hàng Shinhan. Cơ chế tính điểm và xếp hạng dựa trên tổng số dư trung bình 3 tháng tiền gửi có kỳ hạn, tiền gửi tích lũy, tiền gửi thanh toán, các khoản vay cá nhân, giao dịch chuyển tiền ngoại tệ, và thẻ tín dụng. Các hạng bao gồm Premier, ACE, BEST, và CLASSIC với chu kỳ đánh giá xếp hạng diễn ra vào ngày 15 các tháng 1, 4, 7, 10."""
]

# ==============================================================================
# TEST CASE 2: Tài khoản S-Payroll
# ==============================================================================
query_2 = "S-Payroll là gì, khách cần đáp ứng điều kiện nào để được hưởng ưu đãi và các ưu đãi đó là gì?"

documents_2 = [
    # Chunk 1 (Rank 1 - Rất cao): Định nghĩa, điều kiện S-Payroll.
    """Nhóm tài khoản không kỳ hạn S-Payroll là tài khoản thanh toán với những quyền lợi tối ưu dành cho cá nhân có nhu cầu nhận lương hoặc có thu nhập khác hàng tháng. Đối tượng là khách hàng cá nhân từ đủ 15 tuổi trở lên, loại tiền tệ VND hoặc USD. Điều kiện hưởng ưu đãi: Có ít nhất 01 giao dịch nhận lương hoặc thu nhập khác từ 5 triệu đồng trở lên mỗi tháng. Đối với thu nhập khác, khách hàng phải chọn 1 ngày để nhận thu nhập hàng tháng. Giao dịch hợp lệ được tính từ ngày đăng ký S-Payroll cộng/trừ 5 ngày làm việc.""",

    # Chunk 2 (Rank 2 - Cao): Các ưu đãi miễn phí của S-Payroll.
    """Tài khoản S-Payroll cung cấp các ưu đãi miễn phí hoàn toàn cho khách hàng, bao gồm: Miễn phí phát hành thẻ ATM, thẻ tín dụng, thẻ phát hành rút tiền tại ATM Shinhan và các ngân hàng khác; Miễn phí chuyển tiền (chuyển tiền trong nước, chuyển khoản 24/7 qua Internet Banking/Mobile Banking/ATM); Miễn phí thanh toán hóa đơn điện, nước, internet, truyền hình cáp; Miễn phí dịch vụ tin nhắn thông báo qua Zalo; Miễn phí dịch vụ ngân hàng điện tử (Internet Banking, Mobile Banking).""",

    # Chunk 3 (Rank 3 - Khá): Giai đoạn trải nghiệm ưu đãi S-Payroll.
    """Ngân hàng áp dụng giai đoạn trải nghiệm ưu đãi cho tài khoản S-Payroll. Giai đoạn trải nghiệm tính từ ngày mở tài khoản cho đến hết ngày 15 dương lịch của tháng thứ 4 kể từ tháng mở tài khoản. Trong giai đoạn này, khách hàng mặc định được hưởng tất cả các ưu đãi của tài khoản S-Payroll mà không cần xét điều kiện thu nhập. Nguyên tắc kiểm tra điều kiện để áp dụng ưu đãi cho các chu kỳ tiếp theo: Từ ngày 16 của tháng hiện tại đến ngày 15 của tháng tiếp theo dựa trên việc kiểm tra giao dịch của chu kỳ trước đó.""",

    # Chunk 4 (Rank 4 - Trung bình): So sánh với My SOL Package (cũng liên quan ưu đãi tài khoản thanh toán).
    """My SOL Package là gói sản phẩm ưu đãi đặc biệt dành cho khách hàng cá nhân sử dụng tài khoản thanh toán, dịch vụ Ngân hàng trực tuyến và Thẻ ghi nợ quốc tế Mastercard. Khách hàng cá nhân hiện hữu hoặc mới đang sử dụng đồng thời 3 sản phẩm trên (trong đó tài khoản thanh toán có thể là tài khoản thường, Payroll, S-Payroll, 4E, PWM...) ở trạng thái đang hoạt động sẽ được hưởng các ưu đãi miễn phí như phí quản lý tài khoản, phí chuyển tiền nhanh, SMS Zalo. Lưu ý, chương trình không áp dụng đồng thời với các gói sản phẩm Tops Club.""",

    # Chunk 5 (Rank 5 - Thấp): Khái niệm tài khoản thanh toán thông thường và phí quản lý.
    """Đối với tài khoản thanh toán thông thường, phí quản lý tài khoản hàng tháng được thu khi khách hàng không duy trì số dư trung bình tháng tối thiểu. Số dư trung bình tháng tối thiểu là 1 triệu VND hoặc 50 USD. Mức phí quản lý tài khoản là 11,000 VND/tháng/tài khoản và được thu vào mỗi chủ nhật hàng tuần. Tuy nhiên, khách hàng sử dụng Shinhan Package (bao gồm tài khoản không kỳ hạn, thẻ tín dụng và dịch vụ ngân hàng điện tử) sẽ được miễn phí quản lý này.""",

    # Chunk 6 (Rank 6 - Thấp): Hạn mức giao dịch chuyển tiền trên ATM.
    """Đối với tài khoản thanh toán, hạn mức giao dịch trên ATM được quy định như sau: Rút tiền tối đa 50,000,000 VND trên 1 giao dịch và 100,000,000 VND trên 1 ngày. Chuyển tiền nội bộ Shinhan tối đa 100,000,000 VND/ngày. Chuyển khoản liên ngân hàng tối đa 50,000,000 VND/giao dịch và 100,000,000 VND/ngày. Thanh toán hàng hóa dịch vụ tại POS tối đa 50,000,000 VND/giao dịch. Có thể yêu cầu tạm ngừng cung cấp dịch vụ tại máy ATM nếu cần.""",

    # Chunk 7 (Rank 7 - Rất thấp): Quản lý tài khoản và tài khoản không hoạt động.
    """Sau khi được phân loại thành Tài khoản không hoạt động (không phát sinh bất kỳ giao dịch nào trong suốt 12 tháng và số dư dưới 100,000 VND), tài khoản sẽ không được chi trả lãi suất kể từ ngày bị chuyển trạng thái. Ngân hàng sẽ thu phí duy trì tài khoản bằng cách trích tự động từ tài khoản này cho đến khi số dư bằng 0. Tài khoản không hoạt động sẽ tự động đóng tài khoản trong trường hợp từ 18 tháng trở lên và số dư tài khoản bằng 0.""",

    # Chunk 8 (Rank 8 - Rất thấp): K-Study.
    """Tài khoản K-Study là Tài khoản Thanh toán dành cho khách hàng cá nhân Việt Nam có nhu cầu xin Visa du học Hàn Quốc. Tính năng nổi bật là khách hàng sẽ được phong tỏa một khoản tiền ký quỹ (ví dụ 10 triệu KRW) để chứng minh tài chính với Đại sứ quán Hàn Quốc. Không áp dụng ủy quyền cho loại tài khoản này và bắt buộc cung cấp Căn cước công dân gắn chip hoặc Hộ chiếu định danh cá nhân.""",

    # Chunk 9 (Rank 9 - Không liên quan): Tops Club.
    """Chương trình Tops Club phân loại khách hàng thành 4 hạng: Premier, ACE, BEST, CLASSIC. Khách hàng thuộc hạng Premier sẽ được miễn phí dịch vụ rút tiền mặt tại các ATM, miễn phí chuyển tiền trong nước, miễn phí tin nhắn SMS và Zalo, cũng như giảm 50% phí phát hành chứng nhận tài khoản. Tiêu chí để đạt Tops Club được tính điểm theo quy mô giao dịch tiền gửi, vay, và thẻ tín dụng.""",

    # Chunk 10 (Rank 10 - Không liên quan): Quy trình kiểm tra sinh trắc học khi mở tài khoản.
    """Tại Bước 6 của quy trình Đăng ký CIF và Mở tài khoản: Đăng ký CIF, đối chiếu thông tin sinh trắc học và mở tài khoản. Nhân viên cần đăng ký chấp thuận xử lý dữ liệu cá nhân trên màn hình 0044, đăng ký thông tin CIF mới, kiểm tra đối chiếu thông tin sinh trắc học của Chủ tài khoản ở màn hình 6316. Đối với khách hàng người Việt Nam, luồng đối chiếu sinh trắc học thực hiện qua màn hình 0042 hoặc qua ứng dụng Shinhan SOL Mobile Banking."""
]

# ==============================================================================
# TEST CASE 3: Rút trước hạn tiền gửi có kỳ hạn
# ==============================================================================
query_3 = "rút tiền trước hạn đối với tài khoản tiền gửi có kỳ hạn, khách có được rút một phần tiền không và lãi suất áp dụng ra sao"

documents_3 = [
    # Chunk 1 (Rank 1 - Rất cao): Chi tiết rút từng phần và rút trước hạn toàn phần, lãi suất áp dụng.
    """Khách hàng có thể rút tiền gửi có kỳ hạn trước hạn toàn bộ hoặc rút từng phần. Rút từng phần cho phép khách hàng rút một phần nhiều lần vào bất kỳ thời điểm nào trước ngày đến hạn, miễn là tổng số tiền được rút không vượt quá số dư còn lại trên tài khoản tiền gửi đó. Khi được rút từng phần, số tiền rút trước hạn sẽ được áp dụng mức lãi suất tối đa bằng mức lãi suất tiền gửi không kỳ hạn thấp nhất (0.1%), số tiền còn lại tiếp tục duy trì cho tới ngày đáo hạn với lãi suất đang áp dụng cho khoản tiền gửi đó. Đối với rút trước hạn toàn bộ số dư, khách hàng cũng chỉ được hưởng mức lãi suất tiền gửi không kỳ hạn thấp nhất (0.1%).""",

    # Chunk 2 (Rank 2 - Cao): Chỉ định/lựa chọn khi đáo hạn của tiền gửi có kỳ hạn.
    """Khi mở tài khoản tiền gửi có kỳ hạn, khách hàng được quyền lựa chọn phương thức xử lý tài khoản khi đáo hạn. Các lựa chọn Tái tục (Chỉ thị đáo hạn) áp dụng bao gồm: Tự động tái tục gốc và lãi (tất cả gốc và lãi được chuyển sang kỳ hạn mới tương đương); Tự động tái tục gốc (chỉ số tiền gốc được gia hạn, tiền lãi chuyển vào tài khoản thanh toán liên kết); hoặc Không tái tục, ghi có Gốc & Lãi vào Tài khoản liên kết (đóng tài khoản tiền gửi có kỳ hạn khi đến hạn).""",

    # Chunk 3 (Rank 3 - Khá): Thông tin chung về Tiền gửi có kỳ hạn (kỳ hạn, tiền tệ).
    """Tiền gửi có kỳ hạn là khoản tiền của khách hàng gửi tại Ngân hàng trong một thời hạn nhất định theo thỏa thuận giữa Khách hàng và Ngân hàng. Đối tượng mở là cá nhân Việt Nam và cá nhân nước ngoài cư trú hoặc không cư trú. Loại ngoại tệ hỗ trợ gồm: VND, USD, EUR, JPY, AUD, GBP, CAD, SGD, NZD. Kỳ hạn linh hoạt từ 1-3W, 1-5M, 6M, 9M, 12M, 18M, 24M, 36M, 48M, đến 60M. Số dư tối thiểu để mở tài khoản là 2,000,000 VND hoặc 50,000 JPY, 100 USD/EUR... Không được phát hành thẻ ATM hay Séc trên tài khoản này.""",

    # Chunk 4 (Rank 4 - Trung bình): Tài khoản tiền gửi Tích lũy (Installment Deposit) - phân biệt với có kỳ hạn.
    """Nhóm Tài khoản Tích lũy là tài khoản hướng đến các khách hàng muốn có một khoản tiền tích lũy nhất định trong tương lai bằng việc gửi góp định kỳ. Có thể trích tiền tự động theo chu kỳ hàng tháng. Hệ thống tự động hạch toán vào ngày trích tiền cố định hàng tháng. Nếu số dư tài khoản liên kết không đủ, hệ thống sẽ tiếp tục kiểm tra hàng ngày cho đến khi trích thành công. Khách hàng cũng có thể chọn nộp tiền tích lũy thủ công từng lần. Rút từng phần không được phép áp dụng đối với Tiền gửi Tích lũy thông thường.""",

    # Chunk 5 (Rank 5 - Thấp): Đặc điểm tài khoản thanh toán không kỳ hạn.
    """Tài khoản Tiền gửi Thanh toán là một loại tài khoản tiền gửi Không kỳ hạn được sử dụng để giữ ngoại tệ, tiền mặt và thực hiện các giao dịch không dùng tiền mặt (chuyển khoản, thanh toán thẻ, nộp rút tiền mặt). Lãi suất áp dụng là lãi suất không kỳ hạn, được tính hàng ngày trên cơ sở 1 năm 365 ngày và trả vào ngày Thứ bảy của tuần thứ ba hàng tháng. Trái ngược với tiền gửi có kỳ hạn, khách hàng có thể rút tiền mặt hoặc chuyển khoản bất kỳ lúc nào mà không bị phạt giảm lãi suất.""",

    # Chunk 6 (Rank 6 - Thấp): Quản lý tài khoản chung.
    """Tài khoản đồng sở hữu là tài khoản có ít nhất hai chủ thể trở lên cùng đứng tên mở. Các chủ tài khoản đồng sở hữu có quyền và nghĩa vụ ngang nhau và không được phép thay đổi (bổ sung thêm hoặc bớt đi) bất kỳ Chủ tài khoản đồng sở hữu nào sau khi mở. Chi áp dụng cho các thành viên trong gia đình. Việc đóng/rút tiền khoản tiền gửi có kỳ hạn đồng sở hữu yêu cầu sự hiện diện hoặc xác nhận của tất cả các đồng sở hữu chủ.""",

    # Chunk 7 (Rank 7 - Rất thấp): Hạn mức giao dịch thẻ ATM.
    """Hạn mức giao dịch đối với thẻ ghi nợ (ATM) Shinhan: Hạn mức rút tiền mặt tối đa tại ATM Shinhan là 50 triệu VND/giao dịch và 100 triệu VND/ngày. Hạn mức thanh toán tại POS/trực tuyến tối đa là 50 triệu VND/giao dịch. Các giới hạn này được thiết lập nhằm bảo vệ số dư tài khoản thanh toán của khách hàng khỏi rủi ro gian lận. Thẻ ATM không thể liên kết trực tiếp với tài khoản tiền gửi có kỳ hạn để thực hiện rút tiền mặt tại máy ATM.""",

    # Chunk 8 (Rank 8 - Rất thấp): Hồ sơ Mở CIF khách hàng.
    """Hồ sơ yêu cầu mở CIF và tài khoản: Khách hàng cần điền mẫu Deposit 01a đối với khách hàng cá nhân mở tài khoản thanh toán, Deposit 04a đối với yêu cầu mở tài khoản tiền gửi thanh toán (form Tiếng Việt/Anh/Hàn). Đối với yêu cầu mở tài khoản Tiền gửi có kỳ hạn hoặc Tiền gửi tiết kiệm tích lũy, khách hàng sử dụng biểu mẫu Deposit 04b. Mẫu Deposit 50 áp dụng cho yêu cầu mở tài khoản tiền gửi tích lũy linh động bậc thang.""",

    # Chunk 9 (Rank 9 - Không liên quan): Ưu đãi Tops Club.
    """Chương trình Tops Club chia khách hàng thành 4 hạng. Khách hàng hạng Premier (cao nhất) cần đạt 7,000 điểm trở lên trong kỳ xét duyệt. Điểm tích lũy dựa trên số dư bình quân 3 tháng của tiền gửi không kỳ hạn (500,000 VND = 5 điểm), tiền gửi có kỳ hạn (5,000,000 VND = 3 điểm), dư nợ vay bình quân, doanh số sử dụng thẻ tín dụng... Khách hàng hạng Premier được miễn phí duy trì tài khoản, SMS Zalo, chuyển tiền quốc tế, và giảm 50% phí thường niên thẻ tín dụng.""",

    # Chunk 10 (Rank 10 - Không liên quan): Quy trình Mở tài khoản tại quầy.
    """Chi tiết quy trình đăng ký CIF và mở tài khoản tiền gửi tại quầy: Bước 1: Nhận hồ sơ, kiểm tra tính hợp pháp, đối chiếu khuôn mặt khách hàng thực tế với giấy tờ. Bước 2: Tra cứu CIF trên màn hình 0030. Bước 3: Kiểm tra thông tin FATCA/AML. Bước 4: Tư vấn điền mẫu biểu. Bước 5: Đối chiếu thông tin trên hệ thống. Nếu có sai lệch thông tin (như sai số thẻ căn cước, sai ngày sinh), nhân viên phải yêu cầu khách hàng làm thủ tục cập nhật thông tin trước khi thực hiện giao dịch mở tài khoản mới."""
]

# ==============================================================================
# TEST CASE 4: Quy trình mở CIF và Tài khoản
# ==============================================================================
query_4 = "quy trình đăng ký CIF và mở tài khoản tiền gửi cho khách hàng tại quầy giao dịch"

documents_4 = [
    # Chunk 1 (Rank 1 - Rất cao): Tổng quan quy trình 9 bước mở CIF/Tài khoản.
    """Quy trình đăng ký CIF và mở tài khoản/tiền gửi bao gồm 9 bước chính: (1) Nhận hồ sơ từ Khách hàng; (2) Tra cứu số CIF bằng màn hình 0030; (3) Kiểm tra hồ sơ Khách hàng (nhận dạng, KYC, FATCA, AML); (4) Tư vấn & cung cấp mẫu biểu cho khách hàng; (5) Kiểm tra & đối chiếu thông tin điền trên form; (6) Đăng ký CIF, đối chiếu thông tin sinh trắc học và mở tài khoản; (7) Cung cấp số tài khoản cho khách; (8) Phê duyệt hồ sơ trên hệ thống; (9) Đăng tải và lưu trữ hồ sơ bằng màn hình 0950 & thư chấp thuận.""",

    # Chunk 2 (Rank 2 - Cao): Chi tiết các bước đầu (1-5) kiểm tra tính hợp lệ và FATCA.
    """Tại bước 1, nhân viên cần nhận diện khách hàng, xác định mục đích mở tài khoản, xác định tình trạng FATCA của Khách hàng và tra cứu hướng dẫn liên quan đến FATCA, AML của phòng Phòng, chống rửa tiền. Bước 2, tiến hành tra cứu số CIF trên thẻ Căn cước hoặc Hộ chiếu xem khách hàng đã tồn tại CIF chưa để tránh trùng lặp. Ở bước 5, kiểm tra tính đầy đủ, chính xác, khớp đúng giữa các thông tin khách hàng điền trên biểu mẫu và hồ sơ, đảm bảo khách hàng cung cấp chữ ký khớp đúng với chữ ký mẫu đã đăng ký.""",

    # Chunk 3 (Rank 3 - Khá): Chi tiết bước 6-9 (Đăng ký sinh trắc học, mở tài khoản, phê duyệt).
    """Ở Bước 6, nhân viên thực hiện Đăng ký "Chấp thuận về xử lý dữ liệu cá nhân" trên màn hình 0044. Đăng ký thông tin CDD ở màn hình 6803 và FATCA ở màn hình 6316. Kiểm tra đối chiếu khớp đúng thông tin sinh trắc học (màn hình 0042 đối với CMND/CCCD hoặc qua ứng dụng SOL đối với Căn cước công dân gắn chip). Mở tài khoản bằng màn hình 0110. Sau đó, thu thập chữ ký và dấu vân tay sinh trắc học. Bước 8, Giao dịch viên đưa hồ sơ cho Kiểm soát viên phê duyệt trên màn hình và phê duyệt tay trên hồ sơ giấy.""",

    # Chunk 4 (Rank 4 - Trung bình): Hồ sơ nhận diện đối với người Việt Nam và khách hàng dưới 14 tuổi.
    """Khi thực hiện quy trình mở CIF, yêu cầu về hồ sơ khách hàng là người Việt Nam phải có Giấy tờ tùy thân hợp lệ (Thẻ căn cước, Thẻ CCCD gắn chip). Hệ thống sẽ yêu cầu đọc thông tin lưu trữ trên chip. Đối với Khách hàng vị thành niên dưới 14 tuổi chưa có thẻ căn cước, có thể sử dụng Giấy khai sinh và số định danh cá nhân để thay thế, kèm theo hồ sơ của người đại diện hợp pháp (bố mẹ/người giám hộ). Người đại diện hợp pháp phải đăng ký CIF và mở tài khoản trước khi thực hiện quy trình cho khách hàng vị thành niên.""",

    # Chunk 5 (Rank 5 - Thấp): Điều kiện đăng ký biểu mẫu theo uỷ quyền.
    """Trong trường hợp đăng ký CIF qua người ủy quyền, người được ủy quyền phải được cấp giấy CIF tại Ngân hàng và chỉ được thực hiện đăng ký đối với các trường hợp được pháp luật cho phép (như đối với tài khoản không liên quan đến yêu cầu nhận diện bắt buộc trực tiếp). Tuy nhiên, ngoài việc thực hiện đăng ký CIF cho người ủy quyền, nhân viên bắt buộc phải kiểm tra thông tin người ủy quyền và tiến hành khóa chức năng thẻ ghi nợ (nếu có). Người đại diện ký thay vào các mẫu Deposit 01b.""",

    # Chunk 6 (Rank 6 - Thấp): Quy định quản lý tài khoản hoạt động/đóng tài khoản.
    """Khi mở tài khoản thanh toán, khách hàng cần nắm quy định về đóng tài khoản. Một tài khoản có thể bị đóng theo yêu cầu tự nguyện của Chủ tài khoản, hoặc Ngân hàng tự động đóng khi tài khoản rơi vào trạng thái "Tài khoản không hoạt động" trong thời gian dài (từ 18 tháng trở lên) và có số dư bằng 0. Khách hàng cũng được phân loại tình trạng tài khoản từ hoạt động bình thường sang tài khoản tạm dừng giao dịch nếu giấy tờ tùy thân hết hiệu lực pháp luật theo thông tin lưu trữ.""",

    # Chunk 7 (Rank 7 - Rất thấp): Tài khoản S-Payroll.
    """Trong quá trình tư vấn mở tài khoản ở Bước 4, nhân viên có thể giới thiệu sản phẩm S-Payroll nếu khách hàng là người đi làm có thu nhập. S-Payroll là tài khoản tối ưu lợi ích nhận lương, miễn phí hoàn toàn phí chuyển khoản ngoại mạng, phí rút tiền ATM, thanh toán hóa đơn. Khách hàng chỉ cần có ít nhất 01 giao dịch nhận lương/thu nhập khác >= 5 triệu VND/tháng vào tài khoản S-Payroll để duy trì các ưu đãi này.""",

    # Chunk 8 (Rank 8 - Rất thấp): Rút trước hạn tiền gửi.
    """Khách hàng mở CIF cũng có thể mở tiền gửi có kỳ hạn. Trường hợp Khách hàng cần tiền gấp, Ngân hàng Shinhan cho phép rút trước hạn toàn bộ hoặc rút từng phần đối với tiền gửi có kỳ hạn. Tuy nhiên, khi rút trước hạn, số tiền rút ra sẽ chỉ được áp dụng mức lãi suất không kỳ hạn thấp nhất (0.1%/năm) thay vì lãi suất có kỳ hạn như cam kết ban đầu. Việc rút này có thể thực hiện tại quầy sau khi tra cứu hồ sơ CIF.""",

    # Chunk 9 (Rank 9 - Không liên quan): Tài khoản IIA.
    """Tài khoản vốn đầu tư gián tiếp (IIA) phục vụ khách hàng nước ngoài tham gia thị trường chứng khoán Việt Nam. Việc nộp tiền mặt vào tài khoản IIA là không được phép. Khách hàng phải sử dụng nguồn ngoại tệ chuyển khoản từ nước ngoài vào để đổi sang VND hoặc chuyển VND từ tài khoản thanh toán chuyên dùng khác. Tất cả các giao dịch mua, bán chứng khoán, góp vốn, nhận cổ tức đều phải thực hiện thông qua tài khoản này.""",

    # Chunk 10 (Rank 10 - Không liên quan): Phân hạng khách hàng Tops Club.
    """Tops Club của Shinhan Bank đánh giá khách hàng dựa trên tổng giao dịch hàng quý. Khách hàng duy trì số dư bình quân 3 tháng lớn sẽ nhận được các điểm quy đổi: Tiền gửi không kỳ hạn 500.000 VND = 5 điểm, tiền gửi có kỳ hạn 5.000.000 VND = 3 điểm. Khách hàng cá nhân hạng Premier, ACE được giảm 50% hoặc miễn phí chuyển tiền ra nước ngoài, ưu đãi tỷ giá khi mua bán ngoại tệ, và được tặng quà tri ân nhân dịp sinh nhật."""
]

# ==============================================================================
# TEST CASE 5: Chương trình Tops Club
# ==============================================================================
query_5 = "Tops Club có những hạng thành viên nào, cơ chế tính điểm và xét duyệt hạng được tính dựa trên các tiêu chí nào"

documents_5 = [
    # Chunk 1 (Rank 1 - Rất cao): Hạng thành viên và thời gian xét duyệt.
    """Chương trình Tops Club là chương trình khách hàng thân thiết của Ngân hàng Shinhan. Chương trình phân loại khách hàng thành 4 hạng từ cao xuống thấp: PREMIER, ACE, BEST, và CLASSIC. Việc đánh giá và xếp hạng được thực hiện định kỳ theo quý. Cụ thể, kỳ xét duyệt diễn ra vào ngày 15 của các tháng 1, 4, 7 và 10 hàng năm. Thời gian hiệu lực của mỗi hạng Tops Club kéo dài 3 tháng kể từ ngày đánh giá. Thời gian đóng góp để đánh giá là chu kỳ 3 tháng trước tháng xét duyệt.""",

    # Chunk 2 (Rank 2 - Cao): Cơ chế tính điểm dư nợ tiền gửi và cho vay.
    """Cơ chế tính điểm của Tops Club dựa trên nhiều loại hình giao dịch. Đối với tiền gửi Không kỳ hạn, mỗi 500,000 VND số dư trung bình 3 tháng được tính 5 điểm. Đối với tiền gửi Có kỳ hạn/Tiền gửi tích lũy, mỗi 5,000,000 VND số dư trung bình 3 tháng tính 3 điểm. Khách hàng có tài khoản tiền gửi đầu tư hưởng 10 điểm cho mỗi 5,000,000 VND số dư trung bình. Đối với các khoản vay cá nhân, mỗi 500,000 VND dư nợ trung bình 3 tháng quy đổi thành 10 điểm.""",

    # Chunk 3 (Rank 3 - Khá): Cơ chế tính điểm Thẻ tín dụng và Chuyển tiền ngoại tệ.
    """Bên cạnh tiền gửi và khoản vay, điểm Tops Club còn được tính từ giao dịch thẻ tín dụng và chuyển tiền/mua bán ngoại tệ. Đối với Thẻ tín dụng, tổng số tiền thanh toán thẻ tín dụng trong 3 tháng (chỉ tính giao dịch POS) được cộng 1 điểm cho mỗi 500,000 VND thanh toán. Đối với Giao dịch Chuyển khoản, tổng số giao dịch chuyển tiền ngoại tệ trong 3 tháng sẽ mang lại 10 điểm cho mỗi 1 giao dịch. Khách hàng thực hiện mua/bán ngoại tệ mang lại 2 điểm cho mỗi 500 USD giao dịch thành công.""",

    # Chunk 4 (Rank 4 - Trung bình): Quyền lợi dịch vụ tài khoản theo hạng Tops Club.
    """Khách hàng thân thiết thuộc Tops Club được hưởng nhiều quyền lợi ưu đãi phí dịch vụ tùy theo hạng. Hạng Premier và ACE được miễn phí toàn bộ phí rút tiền mặt tại ATM, phí chuyển tiền trong nước, phí phát hành chứng nhận tài khoản, và phí nhận/chuyển tiền từ nước ngoài. Hạng BEST và CLASSIC được giảm 30% đến miễn phí đối với một số dịch vụ như phí tin nhắn SMS/Zalo, phí phát hành thẻ ghi nợ hoặc thường niên thẻ. Ngoài ra, hạng cao cấp còn được ưu tiên phục vụ tại quầy giao dịch.""",

    # Chunk 5 (Rank 5 - Thấp): My SOL Package (một chương trình ưu đãi khác).
    """Tương tự như khái niệm ưu đãi, My SOL Package là một gói đặc quyền cho khách hàng dùng tài khoản Thanh toán, dịch vụ Ngân hàng trực tuyến, thẻ ghi nợ quốc tế Shinhan. My SOL Package miễn phí quản lý tài khoản hàng tháng, chuyển tiền trực tuyến, gửi thông báo qua Zalo. Lưu ý quan trọng: Các tiện ích của My SOL Package không áp dụng đồng thời với ưu đãi của khách hàng có hạng Tops Club (Premier/ACE/BEST/CLASSIC). Nếu khách hàng có hạng Tops Club, hệ thống ưu tiên áp dụng ưu đãi của Tops Club thay vì gói My SOL.""",

    # Chunk 6 (Rank 6 - Thấp): Khái niệm số dư trung bình tháng (phục vụ thu phí tài khoản thường).
    """Số dư trung bình tháng của tài khoản thanh toán là chỉ tiêu quan trọng để Ngân hàng thu phí quản lý tài khoản. Khách hàng duy trì số dư trung bình tháng dưới 1,000,000 VND hoặc 50 USD sẽ bị tính phí quản lý 11,000 VND/tháng. Mức thu nhập này được trích tự động hàng tuần. Dù số dư trung bình cũng được dùng để xét hạng Tops Club, nhưng ở quy mô tài khoản thông thường, nó chủ yếu dùng để quyết định việc thu hồi phí dịch vụ cơ bản của ngân hàng.""",

    # Chunk 7 (Rank 7 - Rất thấp): Tài khoản Tích lũy linh động.
    """Sản phẩm Tiền gửi Tích lũy Linh động Bậc thang hỗ trợ khách hàng tích tiểu thành đại. Kỳ hạn gửi đa dạng từ 6 tháng đến 60 tháng. Số tiền tối thiểu yêu cầu cho mỗi lần nộp là 100,000 VND hoặc 10 USD. Khách hàng có thể nộp bất cứ lúc nào trong kỳ hạn gửi (trừ 1/4 thời gian cuối kỳ hạn, số tiền nộp không được vượt quá 1/2 tổng số tiền đã nộp). Tài khoản này không phát hành thẻ Séc hay thẻ ATM.""",

    # Chunk 8 (Rank 8 - Rất thấp): Rút tiền trước hạn tiền gửi.
    """Đối với khoản Tiền gửi có kỳ hạn, nếu khách hàng thực hiện rút tiền toàn bộ hoặc từng phần trước ngày đáo hạn (được thỏa thuận trước), lãi suất phạt rút trước hạn sẽ được áp dụng. Mức lãi suất quy đổi là lãi suất không kỳ hạn do ngân hàng công bố tại thời điểm rút (thường ở mức 0.1%/năm). Lãi suất được tính theo số ngày thực gửi trên cơ sở năm 365 ngày. Do vậy, việc tất toán trước hạn khoản tiền gửi có thể ảnh hưởng lớn đến lợi nhuận của khách hàng.""",

    # Chunk 9 (Rank 9 - Không liên quan): Hồ sơ mở tài khoản Khách hàng Việt Nam.
    """Khách hàng người Việt Nam từ 15 tuổi trở lên có đầy đủ năng lực hành vi dân sự được mở tài khoản mà không cần người giám hộ. Hồ sơ bắt buộc gồm Căn cước công dân gắn chip hoặc Thẻ Căn cước mới. Thông tin chip sẽ được giao dịch viên xác thực bằng thiết bị đọc tại quầy để trích xuất dữ liệu sinh trắc học và kiểm tra độ chính xác, ngăn ngừa rủi ro sử dụng giấy tờ giả mạo khi mở CIF.""",

    # Chunk 10 (Rank 10 - Không liên quan): Tài khoản S-Payroll.
    """S-Payroll là tài khoản nhận lương chuyên biệt của Shinhan. Khách hàng mở tài khoản S-Payroll sẽ được hưởng quyền lợi hoàn toàn miễn phí khi chuyển tiền ngoài hệ thống, phí duy trì thẻ ATM và Zalo banking. Điều kiện duy trì vô cùng đơn giản: chỉ cần có 1 giao dịch chuyển tiền với nội dung trả lương hoặc chuyển khoản nội bộ/khác ngân hàng trị giá từ 5.000.000 VND trở lên trong vòng 1 tháng tính từ chu kỳ sao kê."""
]

dic = {
    query_1: documents_1,
    query_2: documents_2,
    query_3: documents_3,
    query_4: documents_4,
    query_5: documents_5,
}

# extreme evaluation store
retail_deposit_eval = []

# Test models
bge_m3 = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=bge_m3,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          #gpu_memory_utilization=0.4,
          disable_log_stats=True,
          )
evaluate_model(bge_m3, llm, dic, retail_deposit_eval)

jina_v3 = "jinaai/jina-reranker-v3"
llm = LLM(
    model=jina_v3,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=1024,
    #gpu_memory_utilization=0.6,
    disable_log_stats=True,
)
evaluate_model(jina_v3, llm, dic, retail_deposit_eval)

qwen3_06 = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_06,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=1024,
        #gpu_memory_utilization=0.4,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_06, llm, dic, retail_deposit_eval)

qwen3_4 = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_4,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=1024,
        #gpu_memory_utilization=0.8,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_4, llm, dic, retail_deposit_eval)

INFO 06-24 04:06:18 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}
INFO 06-24 04:06:19 [model.py:555] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-24 04:06:19 [model.py:2015] Downcasting torch.float32 to torch.float16.
INFO 06-24 04:06:19 [model.py:1680] Using max model len 1024
WARNING 06-24 04:06:19 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-24 04:06:19 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-24 04:06:19 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-24 04:06:19 [vllm.py:1089] Cudagraph is disabled under eager mod

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (14 outputs left out)

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

###2. Consumer cards

In [ ]:
clear_gpu_memory()

# TEST CASE 1: NGHIỆP VỤ PHÁT HÀNH THẺ VÀ ĐIỀU KIỆN THU NHẬP
query_1 = "Điều kiện thu nhập tối thiểu và hồ sơ yêu cầu để mở các dòng thẻ tín dụng cá nhân và doanh nghiệp"

documents_1 = [
    # High similarity (Direct answer - Income)
    "Để đủ điều kiện mở thẻ tín dụng cá nhân tại Shinhan Bank, khách hàng cần đáp ứng yêu cầu về thu nhập hàng tháng. Đối với khách hàng người Việt Nam, thu nhập yêu cầu là 8.000.000 VNĐ cho thẻ Classic/Lotte, 12.000.000 VNĐ cho thẻ Gold, 20.000.000 VNĐ cho thẻ Platinum và 30.000.000 VNĐ đối với thẻ hạng Signature. Đối với người Hàn Quốc, mức thu nhập tối thiểu từ 20.000.000 VNĐ đến 30.000.000 VNĐ tùy hạng thẻ. Người nước ngoài khác cần thu nhập tối thiểu 33.000.000 VNĐ cho mọi hạng thẻ.",

    # High similarity (Direct answer - Business Owner)
    "Đối với thẻ tín dụng Shinhan Business Owner (khách hàng doanh nghiệp/hộ kinh doanh), điều kiện phát hành thẻ tín chấp yêu cầu độ tuổi từ 20-65, là chủ doanh nghiệp tư nhân hoặc hộ kinh doanh hoạt động ít nhất 1 năm trong ngành. Thu nhập tối thiểu yêu cầu là từ 20 triệu VNĐ. Đối với hình thức thẻ có tài sản đảm bảo, khách hàng từ 18 tuổi cần có sổ tiết kiệm hoặc tài khoản tiền gửi có kỳ hạn từ 3 tháng với số dư từ 200.000.000 VNĐ tại Shinhan Bank.",

    # Medium-High similarity (Direct answer - Documents)
    "Hồ sơ yêu cầu để mở thẻ tín dụng bao gồm: Giấy tờ chứng minh nhân thân và xác nhận nơi cư trú của người nộp đơn, đơn đăng ký phát hành thẻ tín dụng của Shinhan Bank. Tùy thuộc vào đối tượng khách hàng mục tiêu, ngân hàng sẽ yêu cầu bổ sung các giấy tờ chứng minh công việc và chứng minh thu nhập hợp lệ. Đối với khách hàng doanh nghiệp, hồ sơ yêu cầu bổ sung thêm Giấy phép đăng ký kinh doanh.",

    # Medium similarity (Specific card conditions)
    "Ngoài điều kiện thu nhập, một số dòng thẻ tín dụng liên kết yêu cầu điều kiện thành viên cụ thể. Ví dụ, để mở thẻ tín dụng Shinhan Lotte Mart, khách hàng bắt buộc phải có thẻ Thành viên Lotte Mart và nộp cho ngân hàng cùng lúc với đơn đăng ký thẻ. Tương tự, đối với thẻ Korean Air - Shinhan Consumer, khách hàng phải có mã thành viên SKYPASS của Korean Air, với điều kiện tên đăng ký phải trùng khớp với tên trên hệ thống Shinhan.",

    # Medium-Low similarity (Card Activation)
    "Sau khi hồ sơ được duyệt và nhận thẻ, khách hàng có thể kích hoạt thẻ tín dụng Shinhan qua nhiều kênh khác nhau. Khách hàng có thể đăng nhập vào dịch vụ Ngân hàng trực tuyến Internet Banking hoặc ứng dụng Mobile Banking SOL để kích hoạt. Ngoài ra, khách hàng cũng có thể gửi tin nhắn SMS theo cú pháp quy định đến tổng đài 6089, hoặc liên hệ trung tâm dịch vụ khách hàng 1900 1577 để được hỗ trợ kích hoạt và đổi mã PIN.",

    # Low-Medium similarity (General account opening)
    "Ngân hàng Shinhan cung cấp đa dạng các dịch vụ tài khoản cá nhân như Tài khoản tiền gửi không kỳ hạn, tài khoản S-Payroll dành cho khách hàng nhận lương, tài khoản K-Study dành cho du học sinh. Khách hàng có thể mở Tài khoản tiền gửi có kỳ hạn và Tài khoản tiền gửi tích lũy để nhận mức lãi suất hấp dẫn. Các dịch vụ này được tích hợp với hệ thống ngân hàng điện tử để dễ dàng quản lý.",

    # Low similarity (Loan products)
    "Bên cạnh thẻ tín dụng, Shinhan Bank cung cấp các sản phẩm vay cá nhân trực tuyến đa dạng nhằm đáp ứng nhu cầu tài chính của khách hàng. Các sản phẩm bao gồm Vay mua nhà, vay mua xe mới và xe đã qua sử dụng, vay tiêu dùng cá nhân. Khách hàng có thể tham gia chương trình trả nợ khoản vay thế chấp bất động sản tại ngân hàng khác để hưởng lãi suất ưu đãi từ Shinhan.",

    # Low similarity (Corporate accounts)
    "Đối với mảng doanh nghiệp, Shinhan Bank hỗ trợ mở tài khoản thanh toán nội địa và ngoại tệ, tiền gửi có kỳ hạn, số tài khoản may mắn. Doanh nghiệp có thể sử dụng các dịch vụ tín dụng doanh nghiệp như Hỗ trợ vay ngắn hạn, trung và dài hạn, dịch vụ bảo lãnh, bao thanh toán và cho vay thế chấp khoản phải thu. Các giao dịch có thể thực hiện thông qua hệ thống CMS dành riêng cho doanh nghiệp.",

    # Very Low similarity (Cashback features)
    "Thẻ tín dụng Shinhan Visa Cash-Back mang đến ưu đãi hoàn tiền không giới hạn cho mọi chi tiêu. Hạng thẻ Platinum có mức hoàn tiền cơ bản là 0.4%, trong khi hạng Gold và Classic là 0.3%. Khách hàng còn nhận được quà tặng hoàn tiền chào mừng khi kích hoạt và sử dụng thẻ lần lượt là 1.000.000 VNĐ, 550.000 VNĐ và 350.000 VNĐ tương ứng với các hạng thẻ Platinum, Gold và Classic.",

    # Very Low similarity (Airport lounge)
    "Đặc quyền phòng chờ sân bay là một tiện ích cao cấp dành cho chủ thẻ chính của các dòng thẻ hạng Platinum trở lên. Khách hàng có thể sử dụng ứng dụng DragonPass, kết hợp với Hộ chiếu, Thẻ lên máy bay và Thẻ tín dụng hợp lệ (như Shinhan PWM Platinum, Travel Platinum hoặc Korean Air Shinhan) để được miễn phí sử dụng phòng chờ VIP tại nhiều sân bay quốc tế."
]


# TEST CASE 2: NGHIỆP VỤ TÍNH PHÍ VÀ LÃI SUẤT
query_2 = "biểu phí thường niên, phí giao dịch ngoại tệ và lãi suất phạt chậm thanh toán của các dòng thẻ tín dụng"

documents_2 = [
    # High similarity (Annual Fees)
    "Biểu phí thường niên đối với thẻ tín dụng cá nhân của Shinhan Bank được quy định khác nhau tùy theo hạng thẻ. Thẻ PWM Platinum được miễn phí thường niên. Thẻ Signature có phí 2.500.000 VNĐ/năm, thẻ Travel Platinum là 1.500.000 VNĐ, thẻ Platinum (Cashback, Hi-Point) là 1.100.000 VNĐ, thẻ Gold là 550.000 VNĐ và thẻ Classic là 350.000 VNĐ. Khách hàng có thể được miễn phí thường niên từ năm thứ 2 trở đi nếu đạt tổng doanh số chi tiêu theo quy định (ví dụ 110 triệu VNĐ đối với thẻ Platinum).",

    # High similarity (FX & Interest Rates)
    "Lãi suất thẻ tín dụng thông thường dao động từ 24.99% đến 31.8% một năm tùy thuộc vào loại thẻ. Phí giao dịch ngoại tệ áp dụng cho các giao dịch quốc tế là 0.99% đối với thẻ PWM và Travel Platinum, 1.99% cho thẻ Signature, và 2.9% đối với các dòng thẻ Platinum thông thường, Gold, Classic và thẻ doanh nghiệp. Phí chậm thanh toán được tính bằng 4% trên số tiền tối thiểu chưa thanh toán, với mức phạt tối thiểu là 250.000 VNĐ và tối đa là 999.000 VNĐ.",

    # Medium-High similarity (Advance Cash fees)
    "Khi sử dụng thẻ tín dụng để rút tiền mặt, khách hàng sẽ phải chịu phí ứng trước tiền mặt là 4% trên số tiền giao dịch (tối thiểu 50.000 VNĐ, tối đa 999.000 VNĐ). Hạn mức ứng tiền mặt tối đa là 25% hạn mức tín dụng được cấp. Ngoài ra, nếu rút tiền tại ATM của ngân hàng khác, khách hàng chịu thêm phí 2.2% (tối thiểu 22.000 VNĐ), trong khi giao dịch tại ATM của Shinhan Bank hoàn toàn miễn phí.",

    # Medium similarity (Other service fees)
    "Ngân hàng áp dụng các khoản phí dịch vụ khác nhằm quản lý tài khoản thẻ tín dụng. Phí cung cấp lại Bảng sao kê là 110.000 VNĐ, phí cấp lại thẻ vật lý là 220.000 VNĐ, và phí yêu cầu tăng hạn mức tín dụng là 100.000 VNĐ. Nếu khách hàng khiếu nại và yêu cầu tra soát sự cố mà lỗi thuộc về chủ thẻ, phí là 110.000 VNĐ/giao dịch. Dịch vụ nhận thông báo qua SMS có phí 22.000 VNĐ/kỳ sao kê (nếu dưới 30 tin nhắn).",

    # Medium-Low similarity (Installment & Flexi-cash fees)
    "Đối với dịch vụ Tiền Mặt Linh Hoạt (Flexi-Cash), khách hàng phải trả phí đăng ký ban đầu là 400.000 VNĐ. Lãi suất trả góp Tiền Mặt Linh Hoạt là 23%/năm (kỳ hạn 12 tháng), 21% (24 tháng) và 18% (36 tháng). Nếu khách hàng muốn tất toán sớm trước hạn, phí trả nợ trước hạn là 2% trên dư nợ gốc còn lại (tối thiểu 200.000 VNĐ). Lãi suất trả góp ưu đãi khi mua sắm dao động từ 20.5% đến 21.5% tùy kỳ hạn.",

    # Low-Medium similarity (Corporate Card Fees)
    "Thẻ tín dụng doanh nghiệp Shinhan (Shinhan Business Owner, SME, Corporate) có phí thường niên dao động từ 220.000 VNĐ đến 550.000 VNĐ. Phí giao dịch ngoại tệ cho thẻ tín dụng doanh nghiệp là 2.6% đến 2.9%. Mức phí chậm thanh toán đối với khách hàng doanh nghiệp là 4% với mức tối thiểu 50.000 VNĐ và tối đa 1.000.000 VNĐ. Lãi suất trễ hạn có thể lên tới 150% lãi suất thông thường từ tháng thứ 3 trở đi nếu không thanh toán.",

    # Low similarity (Debit Card Fees)
    "Biểu phí đối với thẻ ghi nợ (Debit card) nhẹ nhàng hơn nhiều so với thẻ tín dụng. Hầu hết các dòng thẻ ghi nợ của Shinhan đều miễn phí thường niên và phí cấp lại mã PIN. Phí rút tiền tại ATM ngân hàng khác là 2.200 VNĐ/giao dịch, tuy nhiên sẽ được miễn phí nếu khách hàng thuộc nhóm PWM, Tops Club hoặc sử dụng gói tài khoản S-payroll. Phí giao dịch ngoại tệ của thẻ ghi nợ dao động từ 2.86% đến 3.85%.",

    # Low similarity (Points mechanism)
    "Thẻ Hi-Point tập trung vào việc tích lũy điểm thưởng. Điểm cơ bản được tích lũy không giới hạn với tỷ lệ 0.3% - 0.4% trên mọi chi tiêu không yêu cầu mức chi tối thiểu. Điểm thưởng đặc biệt I lên tới 5% tại các đối tác liên kết như Samsung, Grab, Agoda. Điểm thưởng đặc biệt II từ 1-2% cho chi tiêu siêu thị và ẩm thực. 1 điểm Shinhan tương đương 1 VNĐ và có thể quy đổi thành tiền mặt.",

    # Very Low similarity (Digital App utilities)
    "Ngân hàng điện tử SOL App cung cấp tính năng Quản lý cửa hàng dành cho khách hàng Hộ kinh doanh/SME. Tính năng này giúp minh bạch nguồn thu, quản lý dòng tiền và thao tác tiện lợi, hỗ trợ đắc lực trong kinh doanh thời đại số. Ngoài ra, khách hàng có thể trải nghiệm các dịch vụ thanh toán không chạm như Google Pay, Samsung Pay, Garmin Pay liên kết trực tiếp với thẻ tín dụng Shinhan.",

    # Very Low similarity (Korean Visa Privilege)
    "Đặc quyền xin thị thực Hàn Quốc dành cho chủ thẻ Platinum, Signature bao gồm việc đơn giản hóa các thủ tục khi nộp hồ sơ xin Visa du lịch Hàn Quốc, miễn chứng minh tài chính. Đặc quyền này được áp dụng cho chủ thẻ chính, giúp khách hàng tiết kiệm thời gian chuẩn bị hồ sơ và tăng tỷ lệ đậu Visa, tạo điều kiện thuận lợi cho các chuyến công tác và du lịch quốc tế."
]


# TEST CASE 3: NGHIỆP VỤ TÍCH LŨY PHẦN THƯỞNG (REWARDS, CASHBACK, MILES)
query_3 = "cơ chế hoàn tiền, tích lũy Shinhan Points và quy đổi dặm bay của các dòng thẻ tín dụng"

documents_3 = [
    # High similarity (Point accumulation mechanism)
    "Chương trình tích điểm Shinhan Point áp dụng cho thẻ Hi-Point với cơ chế tích lũy đa tầng. Thẻ Platinum tích lũy điểm cơ bản 0.4%, thẻ Gold và Classic là 0.3% cho mọi chi tiêu không giới hạn. Khi chi tiêu tối thiểu 3.000.000 VNĐ/kỳ sao kê, khách hàng nhận Điểm đặc biệt I với tỷ lệ 5% (tối đa 400.000 điểm) tại các đối tác Shopee, Grab, Agoda, và Điểm đặc biệt II từ 1%-2% (tối đa 200.000 điểm) tại Siêu thị, Cửa hàng tiện lợi.",

    # High similarity (Miles accumulation)
    "Đối với dòng thẻ Travel Platinum và Korean Air, khách hàng được tích lũy dặm bay cho mọi chi tiêu hợp lệ. Thẻ Travel Platinum tích lũy 1 dặm Shinhan cho mỗi 25.000 VNĐ chi tiêu, và nhân 3 lần dặm (tối đa 2000 dặm) khi chi tiêu quốc tế vào cuối tuần (Thứ 7 & Chủ Nhật) với điều kiện tổng chi tiêu đạt 30.000.000 VNĐ. Dặm Shinhan có thể quy đổi sang dặm thưởng của Vietnam Airlines (Lotusmiles), Singapore Airlines hoặc Cathay Pacific.",

    # Medium-High similarity (Cashback mechanism)
    "Cơ chế hoàn tiền của thẻ Shinhan Visa Cash-Back rất linh hoạt. Hoàn tiền cơ bản không giới hạn ở mức 0.4% cho hạng Platinum và 0.3% cho hạng Gold/Classic trên mọi chi tiêu. Đặc biệt, khách hàng được hoàn tiền 5% cho chi tiêu Ẩm thực (F&B) vào cuối tuần. Để nhận hoàn tiền 5% (tối đa 300.000 VNĐ cho Platinum), tổng chi tiêu trong kỳ sao kê phải đạt từ 10.000.000 VNĐ trở lên. Nếu không đạt, chi tiêu cuối tuần sẽ áp dụng mức hoàn cơ bản.",

    # Medium similarity (Partner specific points - Lotte/Tiki)
    "Thẻ liên kết Lotte Mart và Tiki mang lại điểm thưởng hoặc chiết khấu chuyên biệt. Thẻ Shinhan Lotte Mart tích lũy 2.5% điểm Lotte Mart khi chi tiêu tại hệ thống siêu thị này, và 0.2% tại các đơn vị chấp nhận thẻ khác. Điểm sẽ tự động chuyển đổi sang tài khoản Lotte Mart trong 15 ngày. Thẻ Tiki tích lũy 3% Shinhan point (tối đa 100.000 điểm/kỳ) cho các chi tiêu ngoài Tiki, và tặng các mã giảm giá, miễn phí vận chuyển trực tiếp trên sàn Tiki.",

    # Medium-Low similarity (Redemption rules)
    "Điểm Shinhan Point có thể được quy đổi một cách dễ dàng với tỷ lệ 1 điểm = 1 VNĐ. Khách hàng có thể yêu cầu quy đổi điểm thành tiền mặt nhận tại quầy giao dịch, hoặc hoàn tiền trực tiếp vào tài khoản thanh toán Shinhan thông qua Internet Banking, Mobile Banking hoặc gọi Tổng đài 1900 1577. Mức quy đổi tối thiểu là 15.000 điểm. Điểm cũng có thể quy đổi sang dặm bay Lotusmiles với tỷ lệ 280 điểm = 1 dặm.",

    # Low-Medium similarity (Excluded transactions)
    "Cần lưu ý rằng không phải mọi giao dịch đều được tính để tích lũy điểm thưởng, dặm bay hay hoàn tiền. Các giao dịch bị loại trừ khỏi chương trình phần thưởng bao gồm: giao dịch rút tiền mặt, thanh toán hóa đơn điện nước qua ngân hàng, giao dịch trả góp với lãi suất 0%, ứng tiền mặt linh hoạt (Flexi-cash), và các loại phí phát sinh như phí thường niên, phí chậm thanh toán hay lãi suất thẻ tín dụng.",

    # Low similarity (PWM special cashback)
    "Khách hàng cao cấp sở hữu thẻ Shinhan Visa PWM Platinum cũng được hưởng đặc quyền hoàn tiền ưu việt. Thẻ cung cấp mức hoàn tiền cơ bản 0.5% không giới hạn cho mọi giao dịch hợp lệ. Tương tự thẻ Cashback, thẻ PWM cũng hoàn tiền 5% cho chi tiêu ẩm thực cuối tuần với mức hoàn tối đa 300.000 VNĐ khi tổng chi tiêu trong tháng đạt từ 10.000.000 VNĐ trở lên. Thẻ này được phát hành miễn phí cho khách hàng định danh PWM.",

    # Low similarity (Business Owner Points)
    "Đối với thẻ tín dụng Shinhan Business Owner, khách hàng doanh nghiệp tích lũy 0.1% điểm cơ bản cho mọi giao dịch. Điểm thưởng đặc biệt lên tới 5% cho chi tiêu Ẩm thực và 2% cho chi tiêu Quảng cáo & Trực tuyến (tối đa 500.000 điểm/kỳ sao kê, yêu cầu chi tiêu tối thiểu 3 triệu). Ngoài ra, khách hàng còn nhận điểm ưu đãi 0.5% khi giao dịch thẻ qua máy POS của Shinhan Bank Việt Nam.",

    # Very Low similarity (Card Eligibility)
    "Để mở thẻ tín dụng, cá nhân từ 20 đến 65 tuổi cần nộp hồ sơ chứng minh thu nhập và cư trú. Thu nhập yêu cầu tối thiểu là 8 triệu đồng/tháng đối với thẻ hạng chuẩn dành cho người Việt Nam. Việc phê duyệt hạn mức thẻ tín dụng sẽ được bộ phận thẩm định của ngân hàng đánh giá dựa trên lịch sử tín dụng CIC, mức lương chuyển khoản và tính ổn định của công việc hiện tại của khách hàng.",

    # Very Low similarity (Payment due date)
    "Ngày đến hạn thanh toán thẻ tín dụng Shinhan được xác định dựa trên chu kỳ sao kê mà khách hàng đã chọn. Có các ngày đến hạn thanh toán vào ngày 5, 10, 11, 15, 20, 21, 25 hàng tháng. Chu kỳ sao kê thường kéo dài khoảng 30 ngày và thời gian ân hạn (grace period) là từ 14 đến 15 ngày kể từ ngày chốt sao kê. Khách hàng cần thanh toán Số tiền thanh toán tối thiểu trước 16h30 ngày đến hạn để tránh bị phạt."
]


# TEST CASE 4: NGHIỆP VỤ THANH TOÁN VÀ TRẢ GÓP THẺ TÍN DỤNG
query_4 = "phương thức thanh toán dư nợ thẻ tín dụng là gì và quy định về việc trích nợ tự động, thời gian cập nhật hạn mức thẻ"

documents_4 = [
    # High similarity (Auto-debit payment)
    "Phương thức thanh toán tiện lợi nhất là Thanh toán tự động (Auto-debit). Dịch vụ này áp dụng cho khách hàng có tài khoản tiền gửi thanh toán tại Shinhan Bank. Ngân hàng sẽ tự động trích nợ số tiền tương ứng với tỷ lệ thanh toán (tối thiểu hoặc toàn bộ) mà khách hàng đã đăng ký từ tài khoản vào Ngày đến hạn. Khách hàng cần duy trì số dư đủ để thanh toán trước 16h30 ngày đến hạn để tránh ghi nhận thanh toán trễ.",

    # High similarity (Payoo & Over counter)
    "Khách hàng có thể thanh toán dư nợ thẻ tín dụng miễn phí thông qua hệ thống Payoo bằng 3 cách: Nộp tiền mặt trực tiếp tại các cửa hàng liên kết như Circle K, FamilyMart, GS25; Thanh toán qua ứng dụng Payoo trên điện thoại di động; hoặc thanh toán trên website Payoo. Khách hàng chỉ cần nhập mã Khách hàng (Member No.). Ngoài ra, khách hàng có thể nộp tiền mặt thanh toán tại quầy giao dịch chi nhánh Shinhan trước 16h30 ngày đến hạn.",

    # Medium-High similarity (Bank transfer payment)
    "Thanh toán qua hình thức chuyển khoản từ ngân hàng khác cũng được hỗ trợ. Để đảm bảo việc thanh toán đúng hạn, khách hàng nên thực hiện chuyển khoản trước Ngày đến hạn ít nhất 02 ngày làm việc. Khi sử dụng dịch vụ Chuyển tiền nhanh liên ngân hàng qua số thẻ (Express domestic transfer service), số tiền chuyển sẽ được ghi có trực tiếp vào Thẻ tín dụng của khách hàng mà không cần liên kết với tài khoản tiền gửi Shinhan (DDA).",

    # Medium similarity (Limit update timing)
    "Thời gian cập nhật lại hạn mức tín dụng phụ thuộc vào kênh thanh toán. Nếu thanh toán qua dịch vụ Ngân hàng trực tuyến/SOL, hạn mức sẽ được cập nhật ngay lập tức sau 3 phút. Nếu nộp tiền mặt tại quầy hoặc qua Payoo, hạn mức cũng được cập nhật trong khoảng thời gian tương tự. Tuy nhiên, nếu thanh toán vào cuối tuần, ngày lễ hoặc sau 18h00, giao dịch có thể sẽ được hạch toán và cập nhật vào ngày làm việc tiếp theo.",

    # Medium-Low similarity (Drop box payment)
    "Ngân hàng Shinhan cung cấp Dịch vụ Hộp thanh toán (Drop box) tại các chi nhánh. Khách hàng điền thông tin vào phong bì gửi tiền, bỏ tiền mặt vào (tối đa 30.000.000 VNĐ/phong bì và 100.000.000 VNĐ/ngày), niêm phong và bỏ vào hộp Drop box tại chi nhánh trước 16h30 ngày đến hạn thanh toán. Đây là giải pháp thanh toán dư nợ thẻ tín dụng nhanh chóng cho khách hàng không muốn chờ đợi tại quầy giao dịch.",

    # Low-Medium similarity (Statement logic calculation)
    "Khách hàng cần theo dõi Bảng sao kê thẻ tín dụng để thanh toán đúng hạn. Bảng sao kê thể hiện Dư nợ kỳ trước, số tiền đã thanh toán, các giao dịch phát sinh trong kỳ, và các loại phí/lãi. Dư nợ cuối kỳ là tổng số tiền cần thanh toán. Số tiền thanh toán tối thiểu (Minimum Payment Amount) thường là 5% của dư nợ cuối kỳ. Nếu khách hàng không thanh toán toàn bộ dư nợ, lãi suất thẻ tín dụng sẽ được áp dụng.",

    # Low similarity (Installment conversion)
    "Shinhan cung cấp dịch vụ trả góp linh hoạt cho các giao dịch thẻ tín dụng. Khách hàng có thể chuyển đổi giao dịch mua sắm thành trả góp với lãi suất ưu đãi 20.5% - 21.5%/năm cho các kỳ hạn từ 3 đến 36 tháng. Hoặc khách hàng có thể chọn trả góp 0% lãi suất với một khoản phí chuyển đổi thu 1 lần (2.5% cho 3 tháng, 4.5% cho 6 tháng, 6.5% cho 9 tháng). Mọi thẻ tín dụng cá nhân đều được áp dụng.",

    # Low similarity (Flexi Cash overview)
    "Dịch vụ Ứng tiền mặt linh hoạt (Flexi-Cash) cho phép khách hàng chuyển đổi hạn mức thẻ tín dụng thành tiền mặt giải ngân vào tài khoản thanh toán, lên đến 80% hạn mức thẻ. Khoản tiền ứng sẽ được trả góp hàng tháng với mức lãi suất từ 18% đến 23%/năm tùy theo kỳ hạn đăng ký từ 12 đến 36 tháng. Phí đăng ký dịch vụ một lần là 400.000 VNĐ. Đây là giải pháp hỗ trợ tài chính khẩn cấp cho chủ thẻ.",

    # Very Low similarity (Annual Fees)
    "Phí thường niên thẻ tín dụng được thu hàng năm sau khi thẻ được kích hoạt. Thẻ Bạch kim có phí 1.100.000 VNĐ cho thẻ chính và 700.000 VNĐ cho thẻ phụ. Phí thường niên sẽ được tính vào kỳ sao kê đầu tiên của khách hàng. Từ năm thứ 2, ngân hàng có chính sách miễn phí thường niên nếu tổng chi tiêu năm trước đạt mức yêu cầu, ví dụ như 110 triệu VNĐ đối với thẻ Bạch Kim.",

    # Very Low similarity (Travel privileges)
    "Bảo hiểm du lịch toàn cầu là đặc quyền đi kèm miễn phí cho các chủ thẻ tín dụng Shinhan hạng Platinum, Signature và Travel. Khách hàng và người thân đi cùng sẽ được bảo vệ với mức bồi thường tối đa lên đến 12 tỷ VNĐ cho các tai nạn trong chuyến đi, bồi thường chậm chuyến bay, mất hành lý. Để kích hoạt bảo hiểm, khách hàng phải sử dụng thẻ Shinhan để thanh toán ít nhất 80% chi phí chuyến đi (vé máy bay, tour)."
]


# TEST CASE 5: NGHIỆP VỤ KHÁCH HÀNG DOANH NGHIỆP VÀ HỘ KINH DOANH (SME)
query_5 = "Shinhan cung cấp những sản phẩm thẻ tín dụng, gói vay và dịch vụ thanh toán nào cho KHDN và HKD SME"

documents_5 = [
    # High similarity (SME Credit Cards)
    "Dành cho khách hàng doanh nghiệp và hộ kinh doanh, Shinhan cung cấp các dòng thẻ tín dụng chuyên biệt như Thẻ tín dụng Shinhan Business Owner và thẻ Shinhan SME. Thẻ tín dụng doanh nghiệp giúp tách bạch chi tiêu cá nhân và doanh nghiệp, quản lý dòng tiền hiệu quả. Thẻ Business Owner (hạng Bạch Kim) cung cấp ưu đãi hoàn tiền 5% cho ẩm thực và 2% cho quảng cáo trực tuyến, đặc biệt phù hợp cho các chủ cơ sở kinh doanh vừa và nhỏ.",

    # High similarity (Corporate Loans)
    "Về giải pháp tài chính, Shinhan Bank cung cấp đa dạng các gói Vay tài trợ kinh doanh cho SME và Hộ kinh doanh. Các sản phẩm tín dụng nổi bật bao gồm: Hỗ trợ vay bổ sung vốn lưu động ngắn hạn, Vay vốn đầu tư kinh doanh trung và dài hạn. Ngân hàng cũng cung cấp dịch vụ bảo lãnh, dịch vụ bao thanh toán, cho vay thế chấp khoản phải thu, và dịch vụ giải ngân trực tuyến vô cùng thuận tiện trên hệ thống điện tử.",

    # Medium-High similarity (POS and Packages)
    "Để hỗ trợ việc kinh doanh và thu hộ, Shinhan cung cấp Dịch vụ chấp nhận thanh toán thẻ qua máy POS với tỷ lệ chiết khấu cạnh tranh. Khi đăng ký lắp đặt POS, doanh nghiệp có thể tham gia gói SOHO Biz Package hoặc gói ưu đãi Shinhan SOHO Merchant để nhận nhiều quyền lợi miễn phí giao dịch và ưu đãi lãi suất tiền gửi, giúp tiết kiệm chi phí vận hành và quản lý bán hàng dễ dàng hơn.",

    # Medium similarity (Import/Export and FX)
    "Đối với doanh nghiệp hoạt động thương mại quốc tế, Shinhan cung cấp các dịch vụ Xuất Nhập Khẩu toàn diện. Doanh nghiệp có thể thực hiện phát hành và điều chỉnh thư Tín dụng (L/C) trực tuyến thông qua kênh Ngân hàng trực tuyến một cách dễ dàng. Ngân hàng cũng cung cấp dịch vụ giao dịch ngoại hối (mua bán ngoại tệ kỳ hạn, hoán đổi lãi suất, dịch vụ ngoại tệ trực tuyến) để giúp doanh nghiệp quản trị rủi ro tỷ giá.",

    # Medium-Low similarity (Global Trade Platform GTPP)
    "Shinhan Bank ra mắt chương trình ưu đãi dành cho doanh nghiệp giao dịch trên Nền tảng thanh toán thương mại toàn cầu GTPP. Khách hàng doanh nghiệp có thể nhận đến 5 triệu điểm thưởng Shinhan khi thực hiện thanh toán bằng thẻ tín dụng hoặc thẻ ghi nợ doanh nghiệp Shinhan Visa trên nền tảng GTPP. Đây là giải pháp thanh toán B2B quốc tế an toàn, giúp tối ưu hóa luồng tiền và nhận lại giá trị phần thưởng lớn.",

    # Low-Medium similarity (Corporate Accounts & CMS)
    "Khách hàng doanh nghiệp có thể mở các loại tài khoản tiền gửi không kỳ hạn và có kỳ hạn với lãi suất cạnh tranh. Ngân hàng cung cấp dịch vụ 'Số tài khoản may mắn doanh nghiệp' giúp tăng nhận diện thương hiệu. Để quản lý dòng tiền quy mô lớn, dịch vụ Shinhan CMS (Hệ thống quản lý dòng tiền) và Ngân hàng điện tử doanh nghiệp sẽ hỗ trợ việc truy vấn, chuyển tiền lô lớn và trả lương nhân viên hiệu quả.",

    # Low similarity (SOL Shop Management)
    "Ứng dụng Mobile Banking SOL của Shinhan không chỉ dành cho cá nhân mà còn tích hợp tính năng 'Quản lý cửa hàng' (Store Management) là giải pháp vàng cho kinh doanh thời đại số. Tính năng này cho phép các hộ kinh doanh cá thể quản lý doanh thu minh bạch, theo dõi các giao dịch thanh toán QR và POS dễ dàng ngay trên điện thoại di động, từ đó đưa ra các quyết định kinh doanh kịp thời.",

    # Low similarity (Corporate card fee)
    "Biểu phí thẻ tín dụng doanh nghiệp Shinhan được thiết kế hợp lý. Phí thường niên của thẻ Shinhan hạng World, hạng Vàng, và thẻ SME là 220.000 VNĐ/năm, và được miễn phí từ năm thứ 2 nếu chi tiêu năm trước đạt tối thiểu 105 triệu VNĐ. Phí giao dịch ngoại tệ của dòng thẻ này là 2.6%, rẻ hơn so với các dòng thẻ cá nhân (2.9%), giúp doanh nghiệp tiết kiệm chi phí khi đi công tác nước ngoài hoặc thanh toán quảng cáo quốc tế.",

    # Very Low similarity (Consumer personal loans)
    "Đối với khách hàng cá nhân (không phải doanh nghiệp), Shinhan Bank nổi bật với các sản phẩm Vay mua nhà (lãi suất ưu đãi cố định lên đến 5 năm), Vay mua xe ô tô mới, và Vay tiêu dùng trực tuyến thông qua ứng dụng SOL không cần thế chấp tài sản. Hạn mức vay tiêu dùng cá nhân có thể lên đến 900 triệu đồng với thời hạn vay linh hoạt đến 60 tháng, duyệt hồ sơ nhanh chóng dựa trên sao kê lương.",

    # Very Low similarity (Tiki and Lotte Consumer Cards)
    "Thẻ tín dụng tiêu dùng liên kết như Tiki-Shinhan và Shinhan-Lotte Mart hướng tới đối tượng cá nhân thích mua sắm. Thẻ Tiki miễn phí thường niên năm đầu và tặng các gói quyền lợi ưu đãi mua sắm trị giá hàng trăm nghìn đồng mỗi tháng trên ứng dụng Tiki. Trong khi đó thẻ Lotte Mart cho phép tích lũy điểm thưởng trực tiếp vào thẻ thành viên Lotte để trừ tiền khi mua sắm tại siêu thị, không áp dụng cho mục đích chi tiêu doanh nghiệp."
]

dic = {
    query_1: documents_1,
    query_2: documents_2,
    query_3: documents_3,
    query_4: documents_4,
    query_5: documents_5,
}

# extreme evaluation store
consumer_card_eval = []

# Test models
bge_m3 = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=bge_m3,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          #gpu_memory_utilization=0.4,
          disable_log_stats=True,
          )
evaluate_model(bge_m3, llm, dic, consumer_card_eval)

jina_v3 = "jinaai/jina-reranker-v3"
llm = LLM(
    model=jina_v3,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=2024,
    #gpu_memory_utilization=0.6,
    disable_log_stats=True,
)
evaluate_model(jina_v3, llm, dic, consumer_card_eval)

qwen3_06 = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_06,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.4,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_06, llm, dic, consumer_card_eval)

qwen3_4 = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_4,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.8,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_4, llm, dic, consumer_card_eval)

INFO 06-24 04:09:20 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}
INFO 06-24 04:09:21 [model.py:555] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-24 04:09:21 [model.py:2015] Downcasting torch.float32 to torch.float16.
INFO 06-24 04:09:21 [model.py:1680] Using max model len 1024
WARNING 06-24 04:09:21 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-24 04:09:21 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-24 04:09:21 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-24 04:09:21 [vllm.py:1089] Cudagraph is disabled under eager mod

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (14 outputs left out)

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

###3. Corporate Deposit

In [ ]:
clear_gpu_memory()

# TEST CASE 1: Đăng ký chữ ký và vai trò của Kế toán trưởng/Người đại diện hợp pháp
query_1 = "đăng ký chữ ký của KTT và người đại diện hợp pháp trên chứng từ giao dịch thanh toán của doanh nghiệp được thực hiệc thế nào"

documents_1 = [
    # Chunk 1: Độ tương đồng cao nhất - Trả lời trực tiếp quy định ký chứng từ và ngoại lệ
    "Theo quy định tại Nghị định 174/2016/NĐ-CP và hướng dẫn nội bộ, chứng từ giao dịch của khách hàng tổ chức phải có đủ chữ ký của Người đại diện theo pháp luật và Kế toán trưởng (hoặc người phụ trách kế toán) cùng dấu của đơn vị. Tại ô dành cho Kế toán trưởng sẽ do người được bổ nhiệm ký. Tuy nhiên, doanh nghiệp siêu nhỏ theo quy định pháp luật về hỗ trợ doanh nghiệp nhỏ và vừa không bắt buộc phải bố trí Kế toán trưởng. Đối với chữ ký, Kế toán trưởng không được ủy quyền lại cho người khác.",

    # Chunk 2: Độ tương đồng rất cao - Quy định về các đối tượng miễn trừ Kế toán trưởng
    "Trong quá trình mở tài khoản và đăng ký mẫu chữ ký, một số tổ chức không bắt buộc phải đăng ký chữ ký Kế toán trưởng hoặc Người phụ trách kế toán. Các đối tượng này bao gồm: Văn phòng đại diện của Nhà đầu tư nước ngoài trong hợp đồng BCC, Văn phòng điều hành của Nhà thầu nước ngoài, Văn phòng đại diện của các công ty nước ngoài và trong nước, Tổ chức phi lợi nhuận, và Ban quản trị nhà chung cư. Các tổ chức này có thể đánh dấu vào tùy chọn không bắt buộc bố trí Kế toán trưởng trên biểu mẫu.",

    # Chunk 3: Độ tương đồng cao - Xử lý khi hết hạn giấy tờ ủy quyền hoặc giấy tờ tùy thân
    "Căn cứ Thông tư 17/2024/TT-NHNN, việc tạm dừng giao dịch thanh toán, rút tiền trên tài khoản tổ chức sẽ được áp dụng khi giấy tờ tùy thân của người có liên quan hết hiệu lực. Cụ thể, nếu giấy tờ tùy thân của Người đại diện hợp pháp hoặc Kế toán trưởng hết hạn, ngân hàng sẽ tạm dừng trên toàn bộ giao dịch thanh toán của tài khoản. Nếu giấy tờ của Người được ủy quyền hết hạn, ngân hàng chỉ tạm dừng các giao dịch do chính người được ủy quyền đó yêu cầu, các người có thẩm quyền khác vẫn giao dịch bình thường.",

    # Chunk 4: Tương đồng khá - Tình huống thay đổi Kế toán trưởng/Chủ tịch HĐTV
    "Trong các trường hợp doanh nghiệp thay đổi nhân sự quản lý như Kế toán trưởng hoặc Chủ tịch Hội đồng thành viên, các quyết định bổ nhiệm cũ có thể vẫn còn hiệu lực cho đến khi có văn bản bãi nhiệm thay thế. Ví dụ, việc công ty thay đổi Chủ tịch HĐTV không tự động làm chấm dứt hiệu lực của Quyết định bổ nhiệm kế toán trưởng do Chủ tịch cũ ký. Tuy nhiên, khách hàng có trách nhiệm đăng ký thay đổi thông tin kế toán trưởng và cung cấp hồ sơ cập nhật cho chi nhánh ngân hàng (SHBVN) theo đúng quy định pháp luật.",

    # Chunk 5: Tương đồng trung bình - Quy định về xác minh hồ sơ và tài liệu mở tài khoản
    "Khi thiết lập mối quan hệ với khách hàng, chứng từ và tài liệu nộp cho ngân hàng phải đảm bảo tính pháp lý. Các tài liệu bằng giấy phải là bản chính hoặc bản sao được chứng thực, bản sao cấp từ sổ gốc, hoặc bản sao kèm bản chính để đối chiếu. Tài liệu do cơ quan nước ngoài cấp phải được hợp pháp hóa lãnh sự và dịch sang tiếng Việt. Đối với chứng từ điện tử, phải có chữ ký số của cơ quan thẩm quyền, đảm bảo tính vẹn toàn, tính xác thực và có thể truy cập được khi cần thiết.",

    # Chunk 6: Tương đồng trung bình thấp - Trách nhiệm của nhân viên Teller/Giao dịch viên
    "Nhân viên ngân hàng có trách nhiệm kiểm tra cẩn thận các lệnh, yêu cầu của khách hàng và chứng từ kèm theo phù hợp với quy định hiện hành và luật áp dụng. Nhân viên phải xác nhận chữ ký và mẫu dấu (nếu có) của tài khoản so với chữ ký, mẫu dấu đã đăng ký với Ngân hàng. Một khi nhận lệnh hợp lệ, nhân viên có trách nhiệm thực hiện hạch toán các giao dịch vào hệ thống. Việc kiểm tra này thiết lập nhằm mục đích bảo vệ an toàn tiền gửi của khách hàng tại Ngân hàng.",

    # Chunk 7: Tương đồng thấp - Quy trình thực hiện KYC và kiểm tra FATCA
    "Quy trình định danh khách hàng (KYC) yêu cầu nhân viên phải tìm hiểu mục đích mở tài khoản, kiểm tra thông tin khách hàng trên các nguồn thông tin chính phủ như website đăng ký kinh doanh và hệ thống mã số thuế. Khách hàng cũng phải khai báo thông tin FATCA và chi nhánh thực hiện tra soát thông tin theo Luật Phòng, chống rửa tiền. Sau khi thu thập đủ giấy tờ thành lập, giấy tờ tùy thân, chi nhánh sẽ tiến hành tạo mã khách hàng (CIF) trên hệ thống ngân hàng lõi.",

    # Chunk 8: Tương đồng rất thấp - Phân loại CIF cho các tổ chức
    "Trên hệ thống quản lý, mã khách hàng (CIF) được phân loại thành nhiều nhóm (CIF Type và CIF Class) để quản lý. Nhóm 2 dành cho Cá nhân (Person) bao gồm Cá nhân người Việt Nam, Cá nhân người nước ngoài, Hộ gia đình, Doanh nghiệp tư nhân. Nhóm 3 dành cho Tổ chức (Corporation) bao gồm Công ty TNHH, Công ty Cổ phần, Doanh nghiệp có vốn đầu tư nước ngoài. Việc chọn sai CIF Type của khách hàng sẽ bị báo cáo lỗi về ngân hàng mẹ (Mother Bank) và gây sai lệch hệ thống dữ liệu báo cáo.",

    # Chunk 9: Tương đồng cực thấp - Định nghĩa tổ chức cư trú
    "Để mở tài khoản đúng mục đích, cần phân biệt rõ trình trạng cư trú. Tổ chức cư trú (Resident) bao gồm các tổ chức tín dụng, chi nhánh ngân hàng nước ngoài được thành lập tại Việt Nam, các tổ chức kinh tế, cơ quan nhà nước, đơn vị lực lượng vũ trang, các tổ chức chính trị - xã hội, quỹ xã hội. Ngoài ra, chi nhánh tại Việt Nam của tổ chức kinh tế nước ngoài, văn phòng điều hành của nhà thầu nước ngoài hoạt động tại Việt Nam cũng được xếp vào nhóm tổ chức là người cư trú.",

    # Chunk 10: Không liên quan/Độ tương đồng thấp nhất - Chức năng của phòng CBD
    "Phòng kinh doanh khách hàng doanh nghiệp (Corporate Business Department - CBD) có nhiệm vụ chính là huy động tiền gửi và cho vay đối với khách hàng doanh nghiệp. Các nhân viên tại đây thực hiện tư vấn sản phẩm, quản lý kết quả kinh doanh, hướng dẫn và hỗ trợ dịch vụ khách hàng. Họ phối hợp với các trung tâm giao dịch toàn cầu, phòng thanh toán quốc tế và thiết kế triển khai chiến lược kinh doanh cho mảng khách hàng tổ chức nhằm đáp ứng nhu cầu vốn và thanh toán đa dạng của doanh nghiệp."
]


# TEST CASE 2: Tài khoản vốn đầu tư trực tiếp (DICA)
query_2 = "ai được phép mở DICA, mục đích sử dụng và các quy định cần lưu ý khi giao dịch qua tài khoản này là gì"

documents_2 = [
    # Chunk 1: Độ tương đồng cao nhất - Định nghĩa và quy tắc mở DICA
    "Tài khoản vốn đầu tư trực tiếp (DICA) được mở cho Doanh nghiệp FDI (có vốn đầu tư nước ngoài), nhà đầu tư nước ngoài tham gia hợp đồng BCC, hoặc thực hiện dự án PPP. Đặc điểm quan trọng nhất của DICA là mỗi loại ngoại tệ hoặc VNĐ khách hàng chỉ được mở và sử dụng tại 1 (một) Ngân hàng được phép duy nhất. Nếu một doanh nghiệp FDI muốn mở DICA bằng VND, họ chỉ được mở một tài khoản này, việc mở tài khoản nhằm tuân thủ quy định kiểm soát dòng vốn đầu tư nước ngoài vào Việt Nam.",

    # Chunk 2: Độ tương đồng rất cao - Chu trình sử dụng tài khoản DICA
    "Mục đích sử dụng của Tài khoản DICA trải qua quy trình khép kín bắt đầu từ Giai đoạn trước đầu tư (Pre-investment) như mở tài khoản ngoại biên (Offshore Account), chuyển vốn đầu tư vào DICA (Capital Investment). Nguồn vốn sau đó được dùng cho hoạt động kinh doanh, thanh toán (Capital using qua DICA/DDA), nhận tín dụng (Credit granting & payment qua DICA/FBRA). Cuối cùng, khi doanh nghiệp dừng hoạt động hoặc thoái vốn, dòng tiền lợi nhuận sẽ được chuyển trả lại thông qua tài khoản DICA ở bước đóng dự án (Business closing).",

    # Chunk 3: Độ tương đồng cao - Các trường hợp thực tế (Case study) mở DICA
    "Trong thực tế, khi một tổ chức kinh tế thành lập mới và được cấp Giấy chứng nhận đăng ký đầu tư (IRC), dù nhà đầu tư nước ngoài chỉ chiếm 1% vốn, doanh nghiệp đó vẫn được xem là doanh nghiệp FDI và được phép mở tài khoản DICA. Nếu một công ty 100% vốn Việt Nam bán 50% cổ phần cho nhà đầu tư nước ngoài, công ty này tuy trở thành công ty có vốn nước ngoài nhưng không cần mở DICA vì tỷ lệ sở hữu nước ngoài không trên 50%, lúc này được tính là DN Non-FDI.",

    # Chunk 4: Tương đồng khá - Hồ sơ pháp lý yêu cầu khi mở DICA
    "Để mở Tài khoản vốn đầu tư trực tiếp (DICA), khách hàng doanh nghiệp cần cung cấp hồ sơ pháp lý chứng minh tư cách đầu tư. Hồ sơ này bao gồm Giấy chứng nhận đăng ký doanh nghiệp (ERC), Giấy chứng nhận đăng ký đầu tư (IRC) dành cho nhà đầu tư nước ngoài góp vốn. Đối với dự án PPP hoặc hợp đồng BCC, hồ sơ sẽ bao gồm Hợp đồng nguyên tắc, hợp đồng BCC và các giấy phép thành lập dự án được cơ quan nhà nước có thẩm quyền tại Việt Nam phê duyệt.",

    # Chunk 5: Tương đồng trung bình - Phân biệt DICA và IIA (Tài khoản đầu tư gián tiếp)
    "Khác với DICA dành cho đầu tư trực tiếp, Tài khoản đầu tư gián tiếp (IIA) phục vụ cho nhà đầu tư nước ngoài không cư trú thực hiện hoạt động mua bán chứng khoán, góp vốn dưới 51% không trực tiếp tham gia quản lý. Mỗi nhà đầu tư nước ngoài chỉ được mở 01 (một) tài khoản IIA bằng VND tại 01 Ngân hàng được phép. Việc nhận ủy thác đầu tư từ nước ngoài, mua bán trái phiếu, cổ phiếu niêm yết trên thị trường chứng khoán Việt Nam đều phải thông qua tài khoản IIA này, không được dùng DICA.",

    # Chunk 6: Tương đồng trung bình thấp - Các loại tài khoản khác (Offshore và FBRA)
    "Ngoài DICA, các khách hàng FDI hoặc Non-FDI khi vay vốn nước ngoài phải mở Tài khoản vay và trả nợ nước ngoài (FBRA - Foreign Borrowing and Repayment Account). Với FBRA, một khoản vay chỉ được sử dụng 01 tài khoản tại một ngân hàng, nhưng một tài khoản có thể dùng cho nhiều khoản vay khác nhau. Trong khi đó, Offshore Account là tài khoản mở tại nước ngoài, áp dụng cho tổ chức thành lập và hoạt động ở nước ngoài dùng làm chuẩn bị đầu tư trước khi chính thức chuyển tiền vào DICA tại Việt Nam.",

    # Chunk 7: Tương đồng thấp - Chức năng của Tài khoản thanh toán (DDA)
    "Đối với các giao dịch nội địa thông thường không liên quan đến vốn đầu tư, khách hàng sử dụng Tài khoản thanh toán (Demand Deposit Account - DDA). Tài khoản này được dùng chủ yếu để thực hiện các giao dịch thanh toán nhanh chóng, an toàn như chuyển/nhận tiền, nộp thuế điện tử, trả lương nhân viên, thanh toán hóa đơn điện/nước/internet và quản lý dòng tiền qua ngân hàng số. Khách hàng tổ chức cư trú hay không cư trú hợp pháp đều có thể mở tài khoản DDA bằng VND hoặc ngoại tệ (USD, EUR, JPY, KRW...).",

    # Chunk 8: Tương đồng rất thấp - Nghiệp vụ trả lương cho nhân viên qua tài khoản
    "Dịch vụ chi lương là một trong những tiện ích mà doanh nghiệp (DN) sử dụng qua tài khoản thanh toán của mình. Ngân hàng sẽ thực hiện trích nợ từ tài khoản DDA của DN để chuyển tiền lương, tiền thưởng, bảo hiểm xã hội vào tài khoản cá nhân của từng cán bộ nhân viên. DN cần ký hợp đồng mở tài khoản lương với ngân hàng, nộp bảng kê lương, và nhân viên ngân hàng sẽ hạch toán tự động chuyển tiền theo danh sách đã được đối chiếu để đảm bảo quyền lợi cho người lao động.",

    # Chunk 9: Tương đồng cực thấp - Chức năng quản lý rủi ro và xác thực
    "Trong quá trình quản lý tài khoản, bao gồm cả DICA và DDA, ngân hàng cung cấp các dịch vụ xác thực bổ sung để bảo vệ tài sản của khách hàng. Một trong số đó là Dịch vụ giao dịch qua Fax/Email có đăng ký trước. Giao dịch này yêu cầu khách hàng phải thiết lập mật khẩu, đăng ký các địa chỉ email doanh nghiệp hợp lệ. Nhân viên ngân hàng khi nhận lệnh sẽ gọi điện xác thực qua điện thoại ghi âm trước khi hạch toán, nhằm tránh các rủi ro giả mạo chứng từ chuyển tiền.",

    # Chunk 10: Không liên quan/Độ tương đồng thấp nhất - Định nghĩa Doanh nghiệp
    "Về mặt kiến thức chung, Doanh nghiệp là tổ chức kinh tế có tên riêng, có tài sản, có trụ sở giao dịch ổn định, được đăng ký thành lập theo quy định của pháp luật nhằm mục đích thực hiện các hoạt động kinh doanh. Luật Doanh nghiệp 2020 quy định các loại hình phổ biến tại Việt Nam gồm Công ty TNHH một hoặc hai thành viên trở lên, Công ty Cổ phần, Công ty Hợp danh và Doanh nghiệp tư nhân. Mỗi loại hình sẽ có cơ cấu tổ chức và trách nhiệm pháp lý đối với khoản nợ khác nhau."
]


# TEST CASE 3: Khác biệt giữa TDA và Chứng chỉ tiền gửi (CD)
query_3 = "khác biệt giữa Tài khoản tiền gửi có kỳ hạn và Chứng chỉ tiền gửi dành cho khách hàng tổ chức tại ngân hàng là gì về tính chất pháp lý và quy định rút trước hạn"

documents_3 = [
    # Chunk 1: Độ tương đồng cao nhất - Định nghĩa CD và TDA, sự khác biệt cốt lõi
    "Sự khác biệt cốt lõi giữa Tài khoản tiền gửi có kỳ hạn (TDA) và Chứng chỉ tiền gửi (CD) nằm ở tính chất pháp lý và sự linh hoạt. TDA là một tài khoản ghi nhận số dư tiền gửi có kỳ hạn của tổ chức. Trong khi đó, CD là một hình thức giấy tờ có giá, là bằng chứng xác nhận nghĩa vụ trả nợ của ngân hàng đối với người mua trong một thời hạn nhất định. CD được phát hành theo mệnh giá tối thiểu 100 triệu VNĐ (hoặc 10 triệu USD) và tuân theo quy định riêng từng đợt phát hành.",

    # Chunk 2: Độ tương đồng rất cao - Đặc điểm tự động tái tục và trả lãi
    "Đối với Tài khoản tiền gửi có kỳ hạn (TDA), khách hàng có thể lựa chọn phương thức trả lãi (Đầu kỳ, Hàng tháng, Cuối kỳ) và có tùy chọn tự động tái tục gốc và lãi, tự động tái tục gốc, hoặc không tái tục (ghi có gốc và lãi vào tài khoản liên kết). Ngược lại, Chứng chỉ tiền gửi (CD) có ngày đáo hạn cố định và hoàn toàn KHÔNG được phép tự động tái tục. Vào ngày đến hạn, ngân hàng sẽ trả tiền lãi và gốc của CD trực tiếp vào tài khoản thanh toán (DDA) của công ty mua CD.",

    # Chunk 3: Độ tương đồng cao - Quy định rút tiền trước hạn đối với TDA và CD
    "Về quy định rút tiền, TDA cho phép khách hàng được rút trước hạn hoặc rút một phần số tiền gửi. Lãi suất khi rút trước hạn của TDA sẽ áp dụng lãi suất không kỳ hạn thấp nhất do ngân hàng công bố tại thời điểm rút. Đối với Chứng chỉ tiền gửi (CD), quy định rút trước hạn sẽ phụ thuộc vào điều kiện cụ thể của từng đợt phát hành, nhưng nhìn chung CD thường bị hạn chế rút một phần, và nếu được phép rút trước hạn thì cũng phải tuân thủ nghiêm ngặt các điều khoản cam kết lúc mua chứng chỉ.",

    # Chunk 4: Tương đồng khá - Công thức tính lãi suất tiền gửi
    "Phương pháp tính lãi suất cho cả TDA và CD thường dựa trên số dư thực tế và số ngày duy trì. Công thức cơ bản là: Lãi tiền gửi = (Tổng số dư cuối ngày x Lãi suất tương ứng x Số ngày duy trì thực tế) / 365. Riêng đối với tài khoản Shinhan The More MMDA (tài khoản tiền gửi VND rút linh hoạt), lãi suất được tính theo bậc số dư cuối ngày. Ví dụ, dư nợ dưới 300 triệu VND áp dụng mức lãi 0.1%, dư nợ trên 3 tỷ VND áp dụng lãi 0.5%, tiền lãi được cộng dồn hàng ngày.",

    # Chunk 5: Tương đồng trung bình - Tài khoản phong tỏa và Tài khoản đồng chủ sở hữu
    "Bên cạnh TDA và CD, khách hàng còn sử dụng Tài khoản ký quỹ (Escrow Account) và Tài khoản chung (Joint Account). Tài khoản ký quỹ dùng để phong tỏa một khoản tiền theo thỏa thuận nhằm đảm bảo thực hiện nghĩa vụ giữa các bên, tiền chỉ được giải tỏa khi các điều kiện hợp đồng được hoàn tất. Tài khoản chung là tài khoản có từ hai chủ thể trở lên cùng đứng tên, việc rút tiền hoặc hạch toán đòi hỏi sự đồng thuận của tất cả các chủ tài khoản bằng chữ ký hợp lệ trên chứng từ giao dịch.",

    # Chunk 6: Tương đồng trung bình thấp - Quy định đóng và phong tỏa tài khoản thanh toán
    "Nếu công ty xảy ra mâu thuẫn nội bộ dẫn đến việc các cổ đông yêu cầu ngân hàng phong tỏa tài khoản thanh toán, ngân hàng có quyền thực hiện phong tỏa một phần hoặc toàn bộ số dư. Tuy nhiên, việc phong tỏa này chỉ được thực hiện nếu trước đó có thỏa thuận bằng văn bản giữa các cổ đông và ngân hàng về điều kiện phong tỏa liên quan đến mâu thuẫn. Không có thỏa thuận trước, ngân hàng không tự ý khóa tài khoản TDA, CD hay DDA của doanh nghiệp chỉ dựa trên yêu cầu đơn phương từ một nhóm cổ đông.",

    # Chunk 7: Tương đồng thấp - Nhận lệnh giao dịch qua Fax/Email
    "Trong quá trình giao dịch các khoản tiền gửi, khách hàng tổ chức có nhu cầu nhận, gửi lệnh qua Fax/Email để tiết kiệm thời gian. Dịch vụ này yêu cầu khách hàng đăng ký rõ địa chỉ Email thuộc tên miền riêng của doanh nghiệp. Ngân hàng sẽ không chấp nhận các email công cộng (như Gmail, Yahoo). Nhân viên giao dịch kiểm tra thông tin thanh toán, chữ ký, mẫu dấu trên bản scan, gọi điện cho người liên hệ thứ nhất để xác thực trước khi hạch toán rút tiền hoặc chuyển tiền từ tài khoản DDA hay TDA.",

    # Chunk 8: Tương đồng rất thấp - Chứng từ hợp lệ cho giao dịch ngân hàng
    "Mọi giao dịch liên quan đến gửi tiền, rút tiền, hay mua bán CD đều cần chứng từ hợp lệ. Hình thức chứng từ có thể là bản giấy (bản chính, sao y có chứng thực, sao y cấp từ sổ gốc) hoặc văn bản điện tử. Chứng từ bị tẩy xóa, mờ nét, nhòe, rách nát sẽ bị coi là chứng từ không hợp lệ. Đối với chứng từ do cơ quan nước ngoài cấp, tài liệu phải được hợp pháp hóa lãnh sự, dịch sang tiếng Việt và chứng thực theo quy định pháp luật trước khi nộp cho ngân hàng.",

    # Chunk 9: Tương đồng cực thấp - Tư cách pháp nhân của doanh nghiệp
    "Tất cả các chủ thể tham gia mở TDA và CD đều phải có tư cách pháp nhân hợp lệ, trừ trường hợp cá nhân mở tài khoản riêng. Tư cách pháp nhân được cấp cho tổ chức thành lập theo đúng trình tự pháp luật, có tài sản độc lập và tự chịu trách nhiệm bằng tài sản của mình. Tại Việt Nam, Công ty Cổ phần, Công ty TNHH (1 thành viên hoặc 2 thành viên trở lên) đều có tư cách pháp nhân. Ngược lại, Doanh nghiệp tư nhân không có tư cách pháp nhân độc lập với chủ sở hữu.",

    # Chunk 10: Không liên quan/Độ tương đồng thấp nhất - Phân cấp quản lý trong ngân hàng
    "Bộ máy tổ chức quản lý của một pháp nhân thường bao gồm Người đại diện theo pháp luật, Giám đốc, Kế toán trưởng. Đối với hệ thống nội bộ của ngân hàng Shinhan phục vụ doanh nghiệp, Khối Ngân hàng Doanh nghiệp và Đầu tư (CIB) bao gồm nhiều phòng ban như Trung tâm kinh doanh khách hàng FDI, Trung tâm giao dịch toàn cầu, Phòng thanh toán quốc tế và Dịch vụ chứng khoán. Việc quản lý KPI và đánh giá kết quả kinh doanh của các trung tâm này được đo lường thông qua hệ thống báo cáo quản trị MIS định kỳ."
]


# TEST CASE 4: Giao dịch qua Fax/Email
query_4 = "điều kiện đăng ký và quy trình thực hiện dịch vụ giao dịch qua Email đối với KH tổ chức"

documents_4 = [
    # Chunk 1: Độ tương đồng cao nhất - Điều kiện và phê duyệt dịch vụ Fax/Email
    "Dịch vụ giao dịch qua Fax/Email là phương thức tiện lợi cho khách hàng doanh nghiệp yêu cầu hạch toán thông qua bản chụp lệnh giao dịch. Để sử dụng, khách hàng phải nộp đơn đăng ký và Giám đốc hoặc Phó Giám đốc Chi nhánh ngân hàng phải cân nhắc kỹ lưỡng, cố gắng thuyết phục khách hàng sử dụng dịch vụ Internet Banking thay thế do tính rủi ro. Nếu được chấp thuận, địa chỉ Email đăng ký phải là email có đuôi tên miền riêng của doanh nghiệp, không sử dụng email cá nhân công cộng. Tổ chức phải hoạt động ít nhất 2 năm hoặc đạt hạng tín dụng từ BB+ trở lên.",

    # Chunk 2: Độ tương đồng rất cao - Quy trình hạch toán giao dịch Fax/Email
    "Quy trình thực hiện giao dịch qua Fax/Email gồm 5 bước chặt chẽ. Bước 1: Khách hàng gửi chứng từ thanh toán dạng scan/ảnh chụp qua địa chỉ email đã đăng ký. Bước 2: Nhân viên giao dịch kiểm tra tính hợp lệ của thông tin, so khớp chữ ký và mẫu dấu. Bước 3: Nhân viên bắt buộc gọi điện thoại ghi âm cho Người liên hệ thứ nhất đã đăng ký để xác thực thông tin. Bước 4: Thực hiện hạch toán lệnh trên hệ thống. Bước 5: Khách hàng phải bổ sung chứng từ gốc bản cứng trong vòng 05 ngày làm việc, Deposit Manager sẽ lưu hồ sơ.",

    # Chunk 3: Độ tương đồng cao - Trách nhiệm kiểm tra chữ ký và mẫu dấu của Teller
    "Đối với bất kỳ giao dịch nào, kể cả Fax/Email, nhân viên Teller phải thực hiện nghiệp vụ đối chiếu chữ ký và con dấu. Giao dịch viên có trách nhiệm kiểm tra chữ ký trên lệnh chuyển tiền phải khớp đúng với mẫu chữ ký của Kế toán trưởng và Người đại diện theo pháp luật lưu trên hệ thống. Nếu chữ ký không khớp, hồ sơ không hợp lệ, nhân viên có quyền từ chối lệnh và yêu cầu khách hàng làm lại chứng từ. Việc này đảm bảo tính an ninh, chống gian lận và bảo vệ quỹ tiền gửi của doanh nghiệp.",

    # Chunk 4: Tương đồng khá - Điều kiện chứng từ hợp lệ
    "Một chứng từ được coi là hợp lệ để tiến hành giao dịch thanh toán phải đáp ứng các nguyên tắc cơ bản. Chứng từ giấy phải là bản chính hoặc bản sao có chứng thực theo quy định pháp luật. Chứng từ không được rách nát, tẩy xóa, mờ nét. Với chứng từ điện tử, file gửi qua email phải rõ ràng, đảm bảo tính vẹn toàn dữ liệu. Khách hàng khi nộp bổ sung chứng từ gốc của giao dịch Fax phải nộp đúng bản chính đã đóng dấu mộc đỏ và chữ ký sống tươi, không dùng dấu scan in màu.",

    # Chunk 5: Tương đồng trung bình - Tạm dừng giao dịch do hết hạn giấy tờ
    "Trong quá trình kiểm soát rủi ro giao dịch từ xa, nếu hệ thống cảnh báo giấy tờ tùy thân (CCCD/Hộ chiếu) của Người có liên quan trong công ty đã hết hiệu lực, ngân hàng sẽ chặn các giao dịch. Nếu CCCD của Kế toán trưởng hết hạn, toàn bộ giao dịch thanh toán bị tạm dừng cho đến khi khách hàng nộp CCCD mới và cập nhật lên hệ thống. Giao dịch Fax/Email cũng không được thực hiện trong thời gian tài khoản bị khóa do nguyên nhân này, nhằm tuân thủ Thông tư 17/2024/TT-NHNN.",

    # Chunk 6: Tương đồng trung bình thấp - Thay đổi Kế toán trưởng
    "Một tình huống có thể ảnh hưởng đến việc xử lý lệnh qua Fax/Email là khi doanh nghiệp có Quyết định thay đổi Kế toán trưởng. Nếu Giám đốc ra quyết định bổ nhiệm Kế toán trưởng mới, khách hàng phải tiến hành đăng ký cập nhật thông tin chữ ký với ngân hàng ngay. Các lệnh chuyển tiền mang chữ ký của Kế toán trưởng cũ nộp sau thời điểm quyết định thay thế có hiệu lực sẽ bị từ chối thanh toán. Ngân hàng chỉ duyệt lệnh cho các cá nhân đang có thẩm quyền hợp lệ trên hệ thống CIF.",

    # Chunk 7: Tương đồng thấp - Quy trình định danh và mở CIF
    "Việc nhận lệnh và kiểm tra thông tin khách hàng dựa trên dữ liệu thu thập từ quá trình Định danh mở tài khoản (KYC). Trong bước này, ngân hàng đối chiếu thông tin pháp lý của doanh nghiệp từ website Đăng ký kinh doanh quốc gia. Nhân viên ngân hàng phải chụp sinh trắc học người đại diện pháp luật, nhập liệu mã số thuế, kiểm tra FATCA/AML để mở mã CIF. Một khi CIF được duyệt, khách hàng mới có thể đăng ký các dịch vụ giá trị gia tăng như giao dịch qua email, mở tài khoản lương hoặc Internet Banking.",

    # Chunk 8: Tương đồng rất thấp - Dịch vụ chi lương (Salary Transfer)
    "Lệnh yêu cầu qua Fax/Email thường được doanh nghiệp sử dụng để gửi Bảng kê chi lương cho nhân viên. Doanh nghiệp nộp bảng kê có đóng dấu kèm lệnh chuyển tiền tổng. Ngân hàng mở tài khoản lương cho cán bộ nhân viên công ty, thiết lập mã hợp đồng chi lương trên hệ thống. Khi nhận được lệnh tổng hợp, nhân viên ngân hàng sẽ hạch toán ghi nợ tổng số tiền trên tài khoản công ty và tự động phân bổ tiền thưởng, tiền lương vào từng tài khoản cá nhân tương ứng theo danh sách đính kèm.",

    # Chunk 9: Tương đồng cực thấp - Chức năng của tài khoản tích lũy (ISA)
    "Tài khoản tiền gửi tích lũy (ISA) là sản phẩm huy động vốn phục vụ nhu cầu gửi góp định kỳ. Đặc điểm quan trọng là số dư tối thiểu để mở tài khoản chỉ cần 100.000 VNĐ hoặc 10 USD, với các kỳ hạn linh hoạt từ 6 tháng đến 60 tháng. Tài khoản này không được phép chuyển nhượng hay cầm cố tự do nếu chưa được sự đồng ý của ngân hàng. Thường thì khách hàng sẽ chuyển tiền định kỳ từ DDA sang ISA để hưởng lãi suất cao hơn, lệnh này có thể thiết lập trích nợ tự động.",

    # Chunk 10: Không liên quan/Độ tương đồng thấp nhất - Tài khoản vốn đầu tư gián tiếp (IIA)
    "Tài khoản vốn đầu tư gián tiếp (IIA) là tài khoản thanh toán bằng đồng Việt Nam do nhà đầu tư nước ngoài không cư trú mở tại ngân hàng thương mại để thực hiện hoạt động đầu tư chứng khoán, mua cổ phần tại Việt Nam. Nguyên tắc sử dụng tài khoản này là mọi giao dịch mua bán, nhận cổ tức đều phải thông qua IIA. Các văn bản hướng dẫn như Thông tư 05/2014/TT-NHNN và Thông tư 06/2019/TT-NHNN đã quy định chặt chẽ về việc kiểm soát dòng tiền ngoại hối cho mục đích đầu tư gián tiếp này."
]


# TEST CASE 5: Người cư trú và Người không cư trú
query_5 = "xác định một tổ chức là Resident hay Non-resident và các quy định về tài khoản đối với họ"

documents_5 = [
    # Chunk 1: Độ tương đồng cao nhất - Tiêu chí xác định cư trú và không cư trú
    "Tình trạng cư trú của một tổ chức quyết định các loại tài khoản họ được phép mở. Theo Pháp lệnh ngoại hối, Tổ chức là người cư trú (Resident) bao gồm: Tổ chức tín dụng, tổ chức kinh tế thành lập hoạt động tại VN, cơ quan nhà nước, đơn vị lực lượng vũ trang, tổ chức chính trị-xã hội tại VN. Các chi nhánh tại VN của tổ chức kinh tế nước ngoài, văn phòng điều hành nhà thầu nước ngoài cũng là người cư trú. Ngược lại, Tổ chức là người không cư trú (Non-resident) gồm: Cơ quan ngoại giao, lãnh sự, văn phòng đại diện quốc tế tại VN và các tổ chức hoạt động hoàn toàn ở nước ngoài.",

    # Chunk 2: Độ tương đồng rất cao - Mở tài khoản cho người không cư trú
    "Tổ chức không cư trú (Non-resident) có thể mở Tài khoản thanh toán (DDA) tại ngân hàng Việt Nam bằng VND hoặc ngoại tệ để phục vụ các giao dịch vãng lai, chuyển tiền quốc tế hợp pháp. Tuy nhiên, theo quy định, đối với tổ chức không cư trú, việc gửi Tiền gửi có kỳ hạn (TDA) gặp một số hạn chế nhất định. Thời hạn gửi tiền TDA của tổ chức không cư trú không được dài hơn thời hạn hiệu lực còn lại của Giấy phép thành lập/Giấy chứng nhận đăng ký do cơ quan có thẩm quyền cấp.",

    # Chunk 3: Độ tương đồng cao - Tài khoản đầu tư gián tiếp (IIA) cho không cư trú
    "Để thực hiện đầu tư góp vốn, mua cổ phần tại Việt Nam mà không trực tiếp tham gia quản lý, Tổ chức nước ngoài (Người không cư trú) phải mở Tài khoản vốn đầu tư gián tiếp (IIA). Tài khoản IIA bắt buộc phải mở bằng đồng Việt Nam (VND), không được mở bằng ngoại tệ. Mọi dòng tiền chuyển vào VN để đầu tư chứng khoán phải chuyển đổi sang VND và nộp vào IIA. Nhà đầu tư không cư trú chỉ được mở duy nhất 01 tài khoản IIA tại 01 ngân hàng thương mại được phép tại Việt Nam.",

    # Chunk 4: Tương đồng khá - Doanh nghiệp FDI là tổ chức cư trú
    "Khi tổ chức nước ngoài thực hiện đầu tư trực tiếp vào Việt Nam và thành lập Doanh nghiệp FDI (có vốn đầu tư nước ngoài), doanh nghiệp FDI này được cấp Giấy chứng nhận đăng ký doanh nghiệp (ERC) và hoạt động tại Việt Nam. Do đó, Doanh nghiệp FDI được pháp luật công nhận là Tổ chức cư trú (Resident). Doanh nghiệp FDI bắt buộc phải mở Tài khoản vốn đầu tư trực tiếp (DICA) bằng ngoại tệ hoặc VND để nhận vốn góp, vay vốn nước ngoài và chuyển lợi nhuận hợp pháp về nước.",

    # Chunk 5: Tương đồng trung bình - Quy định Kế toán trưởng đối với tổ chức không cư trú
    "Đối với một số hình thức tổ chức đặc thù của nước ngoài tại Việt Nam, quy định về chữ ký trên tài khoản được nới lỏng. Cụ thể, Văn phòng đại diện của Nhà đầu tư nước ngoài (thuộc tổ chức không cư trú hoặc có đặc thù riêng) và Văn phòng điều hành của nhà thầu nước ngoài không bắt buộc phải bố trí, đăng ký chữ ký của Kế toán trưởng trên bộ hồ sơ mở tài khoản và chứng từ giao dịch thanh toán. Lệnh chuyển tiền chỉ cần chữ ký của Trưởng văn phòng đại diện (Người đại diện pháp luật) là hợp lệ.",

    # Chunk 6: Tương đồng trung bình thấp - Tài khoản vay trả nợ nước ngoài (FBRA) và Offshore
    "Trường hợp tổ chức kinh tế cư trú tại Việt Nam vay vốn từ nước ngoài, họ phải sử dụng tài khoản DICA (đối với doanh nghiệp FDI) hoặc tài khoản vay trả nợ nước ngoài FBRA (đối với doanh nghiệp Non-FDI). Ngược lại, Tài khoản ngoại biên (Offshore account) được quy định mở cho tổ chức thành lập và hoạt động ở nước ngoài (tổ chức không cư trú) phục vụ quá trình chuẩn bị đầu tư. Đồng tiền sử dụng cho FBRA có thể là ngoại tệ hoặc VND tùy thuộc vào hợp đồng vay tín dụng quốc tế.",

    # Chunk 7: Tương đồng thấp - Tài khoản phong tỏa/Ký quỹ (Escrow)
    "Trong các giao dịch dân sự, kinh tế hoặc mua bán sáp nhập, tổ chức cư trú và không cư trú đều có thể sử dụng Tài khoản ký quỹ (Escrow account). Đây là tài khoản mà tiền gửi vào bị ngân hàng phong tỏa nhằm đảm bảo thực hiện nghĩa vụ theo đúng thỏa thuận hợp đồng giữa các bên (ví dụ: nghĩa vụ thanh toán tiền mua cổ phần, nghĩa vụ bảo hành). Tài khoản chung (Joint Account) cũng có cơ chế tương tự nhưng là sở hữu đồng thời của nhiều pháp nhân, yêu cầu chữ ký phê duyệt từ tất cả các bên liên quan.",

    # Chunk 8: Tương đồng rất thấp - Pháp nhân và loại hình doanh nghiệp
    "Việc hiểu rõ pháp nhân rất quan trọng trong định danh khách hàng. Công ty Trách nhiệm hữu hạn, Công ty Cổ phần được thành lập theo luật Việt Nam đều có Tư cách pháp nhân. Tuy nhiên, Doanh nghiệp tư nhân (do một cá nhân làm chủ) không có tư cách pháp nhân độc lập với chủ tài sản. Chi nhánh hay Văn phòng đại diện cũng không có tư cách pháp nhân riêng mà chỉ hoạt động dưới sự ủy quyền của pháp nhân công ty mẹ, dù là công ty mẹ trong nước (cư trú) hay nước ngoài (không cư trú).",

    # Chunk 9: Tương đồng cực thấp - Yêu cầu hợp pháp hóa lãnh sự hồ sơ
    "Mọi tổ chức nước ngoài (người không cư trú) khi cung cấp giấy phép thành lập, giấy chứng nhận đăng ký kinh doanh, hoặc văn bản ủy quyền do cơ quan nhà nước sở tại cấp để nộp cho ngân hàng Việt Nam đều phải tuân thủ nghiêm ngặt quy định về xác thực chứng từ. Chứng từ này phải được cơ quan ngoại giao Việt Nam Hợp pháp hóa lãnh sự, sau đó đem dịch thuật công chứng sang tiếng Việt. Nếu thiếu bước hợp pháp hóa này, bộ hồ sơ pháp lý sẽ bị từ chối tiếp nhận vì không đủ tính pháp lý.",

    # Chunk 10: Không liên quan/Độ tương đồng thấp nhất - Hoạt động của Khối CBD
    "Ngân hàng chia cấu trúc hỗ trợ doanh nghiệp nhằm tối ưu hóa phục vụ. Khối Ngân hàng doanh nghiệp và Đầu tư được điều hành bởi Phó Tổng Giám đốc. Các phòng ban chức năng như Trung tâm kinh doanh khách hàng lớn, Phòng Dịch vụ chứng khoán, Quản lý chính sách tiền gửi có trách nhiệm thiết kế, triển khai các chiến lược bán chéo sản phẩm. Đội ngũ Teller hỗ trợ hạch toán chứng từ nhanh chóng, đảm bảo an toàn bảo mật dữ liệu, tuân thủ chặt chẽ cẩm nang hướng dẫn phòng chống rửa tiền (AML) nội bộ hàng năm."
]

dic = {
    query_1: documents_1,
    query_2: documents_2,
    query_3: documents_3,
    query_4: documents_4,
    query_5: documents_5,
}

# extreme evaluation store
corp_dep_eval = []

# Test models
bge_m3 = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=bge_m3,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          #gpu_memory_utilization=0.4,
          disable_log_stats=True,
          )
evaluate_model(bge_m3, llm, dic, corp_dep_eval)

jina_v3 = "jinaai/jina-reranker-v3"
llm = LLM(
    model=jina_v3,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=2024,
    #gpu_memory_utilization=0.6,
    disable_log_stats=True,
)
evaluate_model(jina_v3, llm, dic, corp_dep_eval)

qwen3_06 = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_06,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.4,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_06, llm, dic, corp_dep_eval)

qwen3_4 = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_4,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.8,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_4, llm, dic, corp_dep_eval)

INFO 06-24 04:12:30 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}
INFO 06-24 04:12:30 [model.py:555] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-24 04:12:30 [model.py:2015] Downcasting torch.float32 to torch.float16.
INFO 06-24 04:12:31 [model.py:1680] Using max model len 1024
WARNING 06-24 04:12:31 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-24 04:12:31 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-24 04:12:31 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-24 04:12:31 [vllm.py:1089] Cudagraph is disabled under eager mod

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (14 outputs left out)

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

###4. Card Policy

In [ ]:
# ==========================================
# TEST CASE 1: TRUY VẤN LỊCH SỬ TÍN DỤNG (CIC & KCB)
# ==========================================
query_1 = "quy trình, các bước thao tác trên hệ thống và các quy định về thời hạn hiệu lực khi thực hiện truy vấn thông tin lịch sử tín dụng CIC và KCB cho khách hàng cá nhân và nước ngoài."

documents_1 = [
    # Chunk 1: High Similarity (Trực tiếp trả lời quy trình và điều kiện CIC/KCB)
    """Bước đầu tiên trong quy trình thẩm định là truy vấn lịch sử tín dụng của khách hàng. Nhân viên vào màn hình MH  CIC Information Request trên hệ thống Aither. Tại đây, nhập số CMND, CCCD hoặc Hộ chiếu (đối với người nước ngoài) vào trường ID No để tra cứu. Đối với khách hàng Hàn Quốc, bắt buộc phải tra cứu thêm KCB (Korea Credit Bureau) bằng cách nhập ID Hàn Quốc hoặc Hộ chiếu Hàn Quốc. Thời hạn hiệu lực của báo cáo CIC và KCB là 30 ngày kể từ ngày truy vấn. Nếu trong vòng 30 ngày đã có kết quả tra cứu, hệ thống sẽ tự động tải lại dữ liệu cũ mà không tính phí tra cứu mới. Nhân viên cần kiểm tra kỹ thông tin Tên, Ngày sinh và Số giấy tờ tùy thân khớp với hồ sơ gốc. Đối với KCB, điểm số phải đạt từ 531 điểm trở lên (Hạng KCB <= 7) mới đủ điều kiện cơ bản để xem xét cấp thẻ tín dụng hoặc khoản vay. Nếu khách hàng chưa có lịch sử tín dụng (No hit), quy trình vẫn tiếp tục nhưng sẽ áp dụng các chính sách cấp hạn mức và đánh giá rủi ro nội bộ khác của ngân hàng.""",

    # Chunk 2: High Similarity (Xử lý lỗi và các trường hợp đặc biệt khi tra cứu)
    """Trong quá trình tra cứu trên màn hình MH , nếu hệ thống báo lỗi không tìm thấy dữ liệu do sai lệch thông tin cá nhân, nhân viên cần thực hiện truy vấn lại bằng chức năng nhập tên thiếu (ví dụ nhập "RI YUNG HYUK" thay vì họ tên đầy đủ do giới hạn ký tự) hoặc tra cứu theo số giấy tờ cũ nếu khách hàng vừa đổi từ CMND 9 số sang CCCD 12 số gắn chip. Tại mục Report Type, chọn "1-FULLFORM" đối với thẻ tín dụng và các khoản vay thông thường để lấy toàn bộ lịch sử dư nợ, thẻ tín dụng, thế chấp và lịch sử trả nợ chậm. Sau khi nhấn "Inquiry", trạng thái ban đầu có thể là "Received", nhân viên cần chờ một lúc để hệ thống cập nhật kết quả. Đặc biệt lưu ý, các lỗi phát sinh như sai cấu trúc tên, sai mã chi nhánh yêu cầu (Request Branch) cần được xử lý ngay lập tức. Cần đối chiếu lại toàn bộ số liệu dư nợ các thẻ đang mở tại các TCTD khác so với khai báo của khách hàng trên Đơn yêu cầu mở thẻ để phát hiện rủi ro gian lận.""",

    # Chunk 3: High-Medium Similarity (Các tiêu chí từ chối tự động dựa trên CIC)
    """Sau khi có kết quả CIC, hệ thống sẽ chạy qua màng lọc Filtering Items để tự động đánh giá. Khách hàng sẽ bị từ chối mở thẻ nếu vi phạm một trong các điều kiện sau: Có nợ nhóm 2 tại bất kỳ tổ chức tín dụng nào trong vòng 12 tháng gần nhất (mã lỗi DL020000 >= 2); Có khoản nợ vay thuộc nhóm 3 đến nhóm 5 trong vòng 5 năm gần nhất (mã lỗi DL010003); Có từ 4 lần nợ vay nhóm 2 trong vòng 12 tháng gần nhất; Có lịch sử trễ hạn thẻ tín dụng từ 30 ngày trở lên trong vòng 12 tháng gần nhất (DC031012). Tổng dư nợ thẻ tín dụng đang có không được vượt quá số lượng thẻ quy định (ví dụ khách hàng đang có 4 thẻ tín dụng tại các ngân hàng khác nhau có thể bị từ chối do vượt quá rủi ro - LN019006 >= 4). Đồng thời, nếu khách hàng có các khoản vay mua trả góp tại các tổ chức tài chính (như FE Credit, Home Credit) với dư nợ cao và lịch sử trả chậm, hệ thống cũng sẽ ghi nhận và áp dụng bộ quy tắc Cut-off (R-Clips Rejected).""",

    # Chunk 4: Medium Similarity (Kết quả Screening và R-clips)
    """Kết quả thẩm định cuối cùng sau khi tra cứu CIC và nhập dữ liệu vào màn hình MH  Applying and Screening of consumer application sẽ hiển thị tại phần Processing Division. Nếu tình trạng là "Confirmation", nghĩa là đang chờ phê duyệt. Nếu "Screening Failed", nghĩa là không thỏa điều kiện của ngân hàng. R-Clips Approved Result sẽ cho biết kết quả đánh giá tự động: "Approved" (Đồng ý) hoặc "Rejected" (Từ chối). Lý do từ chối (Screening Denial Reason) có thể là: Thu nhập không đủ (Monthly Salary Underqualified), Không đạt chuẩn chấm điểm (ASS/BSS Standard Underqualified member), hoặc rớt bộ lọc tự động do nợ xấu trên CIC (Filtering items: Rejected). Hạn mức tín dụng được duyệt (Applied Credit Limit) có thể sẽ được điều chỉnh thấp hơn hạn mức yêu cầu ban đầu (Requested Credit Limit) dựa trên đánh giá rủi ro tổng thể và khả năng trả nợ (DSR) của khách hàng. Tất cả các trường hợp bị từ chối đều phải được lưu trữ lý do rõ ràng trên hệ thống Aither để phục vụ kiểm toán nội bộ.""",

    # Chunk 5: Medium Similarity (Tra cứu mã khách hàng CIF)
    """Trước khi tạo thẻ tín dụng hoặc tra cứu CIC, giao dịch viên phải đảm bảo khách hàng đã có mã Khách hàng (CIF) duy nhất trên hệ thống Core Banking. Quy trình bắt đầu tại màn hình MH [1] Manage Customer Information hoặc MH [2] Inquire of Customer Information. Chọn CIF Type là "2-Person" cho khách hàng cá nhân và nhập số CMND/CCCD/Hộ chiếu vào trường ID. No. Nếu hệ thống báo "Not Duplicated", nghĩa là khách hàng chưa từng giao dịch tại Shinhan Bank, nhân viên tiến hành tạo CIF mới tại màn hình MH [3] Open Account. Nếu khách hàng đã có CIF, cần kiểm tra kỹ tình trạng khách hàng có nằm trong danh sách đen (Blacklist Check) hoặc danh sách cấm vận AML, FATCA hay không. Việc cập nhật chính xác số điện thoại, địa chỉ cư trú, email và chữ ký điện tử vào hồ sơ CIF là bắt buộc vì đây là nền tảng để hệ thống gửi SMS OTP, báo cáo thẻ tín dụng điện tử (e-statement) và các liên lạc thu hồi nợ sau này.""",

    # Chunk 6: Low-Medium Similarity (Hướng dẫn chụp ứng dụng VssID)
    """Bên cạnh CIC, một nguồn dữ liệu quan trọng để thẩm định là ứng dụng VssID (Bảo hiểm xã hội số). Ứng dụng này giúp ngân hàng xác minh lịch sử làm việc và mức lương đóng bảo hiểm xã hội của khách hàng mà không cần yêu cầu giấy tờ bản cứng. Nhân viên hướng dẫn khách hàng đăng nhập VssID, vào mục Quá trình tham gia > Bảo hiểm xã hội. Cần chụp toàn bộ màn hình hiển thị rõ Tên công ty, chức vụ, mức lương đóng BHXH (Salary amount for social insurance contribution) và tổng thời gian đóng. Đối với người lao động nước ngoài không có VssID, phải nộp Giấy phép lao động (Work Permit) và Hợp đồng lao động còn hạn. Việc đối chiếu chéo giữa số tiền lương trên VssID, mức lương thực nhận qua sao kê tài khoản ngân hàng và khai báo trên hệ thống là bước kiểm soát gian lận thiết yếu để xác định chính xác thu nhập ròng làm cơ sở cấp hạn mức thẻ.""",

    # Chunk 7: Low Similarity (Điều kiện thẻ doanh nghiệp)
    """Đối với thẻ tín dụng doanh nghiệp, quy trình thẩm định khác biệt hoàn toàn so với cá nhân. Tổ chức xin cấp thẻ phải là pháp nhân được thành lập hợp pháp theo quy định của pháp luật Việt Nam. Doanh nghiệp phải có xếp hạng tín dụng nội bộ còn hiệu lực tại SHBVN từ hạng BBB- trở lên. Các ngành nghề kinh doanh bị hạn chế bao gồm: kinh doanh vũ khí, sản xuất hóa chất độc hại, sòng bạc, và một số loại hình giải trí đặc thù. Việc phát hành thẻ tín dụng doanh nghiệp không có tài sản đảm bảo yêu cầu doanh nghiệp phải nộp Báo cáo tài chính đã được kiểm toán trong 3 năm gần nhất và không có bất kỳ khoản nợ quá hạn nào tại tổ chức tín dụng khác (CIC không có nợ xấu). Hạn mức thẻ doanh nghiệp sẽ được phê duyệt bởi Giám đốc Chi nhánh, Ban tín dụng Hội sở (CO Committee) hoặc Hội đồng tín dụng (MCOC) tùy thuộc vào tổng mức cấp tín dụng (Exposure) và xếp hạng rủi ro của doanh nghiệp.""",

    # Chunk 8: Low Similarity (Hồ sơ chứng minh nơi cư trú)
    """Để chứng minh địa chỉ thường trú và nơi ở hiện tại, ngân hàng yêu cầu khách hàng cung cấp các chứng từ hợp lệ. Đối với người Việt Nam, chứng từ bao gồm CCCD có chip (quét mã QR để lấy thông tin nơi cư trú), ứng dụng VNeID tài khoản định danh mức 2 (phải hiển thị chi tiết Thông tin cư trú bao gồm Nơi thường trú và Nơi ở hiện tại), hoặc Giấy xác nhận thông tin về cư trú (Mẫu CT07/CT08) được cấp bởi cơ quan Công an trong vòng 2 tháng gần nhất. Trong trường hợp khách hàng ở nhà thuê, cần bổ sung Hợp đồng thuê nhà có chữ ký của chủ nhà kèm theo Hóa đơn tiện ích (điện, nước, internet) đứng tên khách hàng trong 3 tháng gần nhất tại địa chỉ đó. Nếu thông tin địa chỉ trên các hệ thống không khớp, nhân viên thẩm định bắt buộc phải tiến hành thực địa (Site visit) và chụp ảnh nhà ở, tọa độ GPS để lưu hồ sơ.""",

    # Chunk 9: Lowest Similarity (Chính sách cho thẻ ưu tiên PWM)
    """Private Wealth Management (PWM) là phân khúc khách hàng cao cấp nhất của ngân hàng. Để đạt hạng PWM Platinum hoặc Signature, khách hàng phải duy trì số dư tiền gửi trung bình (AUM) lớn. Cụ thể, khách hàng có tổng số dư tiền gửi, tiền gửi tích lũy và tiền gửi tiết kiệm từ 1 tỷ đồng trở lên (hạng Platinum) hoặc từ 2 tỷ đồng trở lên (hạng Signature). Hạn mức thẻ tín dụng cấp cho nhóm khách hàng này được tính dựa trên số dư tiền gửi. Ví dụ: Dư nợ khoản vay tiêu dùng cầm cố tiền gửi/ vay cầm cố thẻ tiết kiệm được trừ đi, phần hạn mức còn lại có thể lên tới tối đa 150 triệu đồng cho Platinum và 300 triệu đồng cho Signature. Khách hàng PWM được miễn nhiều loại phí dịch vụ, có chuyên viên chăm sóc riêng (RM) và các đặc quyền sử dụng phòng chờ sân bay, đánh golf. Hồ sơ mở thẻ của PWM cũng được tối giản, không yêu cầu cung cấp các chứng từ chứng minh thu nhập phức tạp như khách hàng đại trà.""",

    # Chunk 10: Lowest Similarity (Tính toán thu nhập chung)
    """Khả năng thanh toán (Loan redemption capability) được tính bằng Thu nhập trung bình hàng tháng trừ đi các Khoản chi trả hàng tháng. Hệ số nhân để xác định hạn mức tín dụng phụ thuộc vào điểm AS (Application Scorecard) của khách hàng, thường là 2, 2.5, 3.3 hoặc 4. Thu nhập cuối cùng dùng để xét duyệt được tính theo công thức: Thu nhập cố định + 75% * (Thu nhập trung bình 3 tháng gần nhất - Thu nhập cố định). Việc chênh lệch thu nhập giữa các tháng phải được giải trình rõ ràng, nếu chênh lệch trên 120% cần có xác nhận thưởng năng suất từ công ty. Mức kiểm soát rủi ro dư nợ tín chấp tối đa (MUE) cho một khách hàng cá nhân không vượt quá 900 triệu đồng (bao gồm cả dư nợ thẻ và vay tiêu dùng). Hạn mức cấp dựa trên thu nhập hàng tháng (MI) thường dao động từ MI x 2 đến MI x 3 tùy hạng khách hàng và lịch sử tín dụng."""
]

# ==========================================
# TEST CASE 2: CÁCH TÍNH THU NHẬP VÀ HẠN MỨC (MI / MUE)
# ==========================================
query_2 = "Công thức tính Final Income cho khách hàng cá nhân, quy định về chênh lệch thu nhập các tháng và phương pháp tính Hạn mức tín dụng tối đa dựa trên thu nhập (MI) và dư nợ không tài sản đảm bảo (MUE)."

documents_2 = [
    # Chunk 1: High Similarity (Công thức Final Income và chênh lệch)
    """Theo chính sách thẩm định, Thu nhập cuối cùng (Final Income) dùng để cấp thẻ tín dụng được tính toán cẩn thận để tránh rủi ro biến động thu nhập của khách hàng. Công thức chuẩn là: Thu nhập cuối cùng = Thu nhập cố định (Lương cơ bản + Phụ cấp cố định như tiền ăn, điện thoại) + 75% * (Thu nhập trung bình 3 tháng gần nhất – Thu nhập cố định). Ngân hàng đặc biệt lưu ý đến mức chênh lệch thu nhập giữa các tháng. Chênh lệch được tính bằng (Tháng lương cao nhất / Tháng lương thấp nhất) * 100. Nếu tỷ lệ chênh lệch thu nhập của 3 tháng gần nhất vượt quá 120% (đối với khách hàng thông thường hoặc làm việc dưới 1 năm) hoặc vượt quá 150% (đối với khách hàng VIP, khách hàng làm trên 1 năm), khoản thu nhập biến động (ví dụ tiền làm thêm giờ, hoa hồng doanh số) sẽ bị cắt giảm theo tỷ lệ để đảm bảo an toàn tín dụng. Nếu một tháng thu nhập thực nhận giảm dưới mức tối thiểu theo quy định hạng thẻ (ví dụ dưới 8 triệu cho thẻ Classic), hồ sơ có thể bị từ chối.""",

    # Chunk 2: High Similarity (Cách tính Hạn mức dựa trên MI và MUE)
    """Hạn mức tín dụng (Credit Limit) cấp cho thẻ cá nhân được áp dụng theo mức thấp hơn giữa Hạn mức đề xuất của hệ thống (dựa trên điểm tín dụng) và Hạn mức tính toán theo Thu nhập hàng tháng (MI). Cụ thể, đối với khách hàng có MI < 20 triệu đồng, hạn mức thẻ được cấp tối đa bằng MI x 2. Đối với khách hàng có MI >= 20 triệu đồng, hạn mức thẻ tối đa bằng MI x 3. Tuy nhiên, tổng hạn mức thẻ cấp ra phải tuân thủ nghiêm ngặt Mức kiểm soát rủi ro (MUE - Maximum Unsecured Exposure). MUE được định nghĩa là Tổng dư nợ khoản vay tín chấp + Hạn mức thẻ tín dụng hiện tại và thẻ đang mở. MUE đối với một khách hàng cá nhân không được vượt quá 21 lần thu nhập (MUE <= 21 x MI) và số tiền tuyệt đối không được vượt quá 900 triệu đồng. Nếu khách hàng đã có dư nợ cao tại Shinhan hoặc ngân hàng khác, hệ thống Aither sẽ tự động giới hạn hoặc từ chối cấp thẻ mới để tránh vỡ nợ.""",

    # Chunk 3: High-Medium Similarity (Hạn mức dựa trên tài sản xe hơi/nhà)
    """Bên cạnh thu nhập từ lương, Shinhan Bank cấp hạn mức thẻ dựa trên tài sản sở hữu như Xe hơi và Bất động sản. Đối với xe hơi, hạn mức được tính dựa trên giá trị xe hiện tại. Xe từ 1 tỷ đồng trở lên có thể được cấp hạn mức tối đa 100 triệu đồng; xe từ 500 triệu đến dưới 1 tỷ cấp 70 triệu; và xe từ 300 triệu đến dưới 500 triệu cấp 50 triệu. Điều kiện là xe phải sản xuất trong vòng 5 năm, thời gian sử dụng từ 2 năm trở xuống (giá trị xe > 80%) và mua bảo hiểm thân vỏ tự nguyện. Đối với bất động sản (sổ hồng, sổ đỏ), diện tích sàn nhà trên đất phải từ 40m2 (không chấp nhận đất trống), hoặc căn hộ chung cư phải từ 60m2. Khách hàng sở hữu nhà dưới 60m2 được cấp hạn mức 50 triệu, từ 60m2 đến 90m2 được cấp 70 triệu và trên 90m2 có thể được cấp tối đa 100 triệu đồng. Chứng từ yêu cầu là Giấy chứng nhận quyền sử dụng đất bản sao y chứng thực và Hóa đơn tiện ích chứng minh có người ở.""",

    # Chunk 4: Medium Similarity (Hạn mức cho nhóm khách hàng ưu tiên VGC và CEP)
    """Nhóm khách hàng có rủi ro thấp như Vietnamese Good Credit (VGC) và Corporate Employee Program (CEP) được hưởng các chính sách hạn mức linh hoạt hơn. Khách hàng làm việc tại các Công ty Tốt (danh sách do ngân hàng phê duyệt) thuộc nhóm VGC có thể được cấp thẻ tín dụng không cần tài sản đảm bảo với hạn mức tối đa lên tới 100 triệu đồng. Tuy nhiên, tổng hạn mức hệ thống quy định là 180 triệu. Hạn mức cấp thẻ dựa trên nguyên tắc: lấy mức trung bình cộng của hạn mức các thẻ tín dụng đang hoạt động tại các ngân hàng khác (như HSBC, UOB, VIB) sau đó đối chiếu với mức tối đa quy định của nhóm khách hàng (ví dụ 100 triệu). Ngân hàng sẽ lấy giá trị nhỏ nhất (MIN) giữa các hạn mức tính toán này để đưa ra quyết định cuối cùng. Do đó, dù thu nhập (MI) của khách hàng rất cao nhưng nếu bị giới hạn bởi trần chính sách của nhóm, hạn mức phê duyệt cuối vẫn không vượt quá trần quy định.""",

    # Chunk 5: Medium Similarity (Đánh giá giới hạn hệ thống - Cut Off và Filtering)
    """Hệ thống chấm điểm thẻ (Card Scorecard) tự động hóa việc tính toán hạn mức thông qua bộ lọc Filtering Items và Cut-off. Sau khi Điểm AS (Application Score) được xác định, hệ thống sẽ nhân điểm khả năng thanh toán (Loan Redemption Capability) với hệ số nhân dựa trên xếp hạng rủi ro (hệ số 2, 2.5, 3.3, 4). Nếu điểm giới hạn hệ thống (Cut-off Result) trả về "Rejected", hồ sơ sẽ bị từ chối ngay lập tức bất kể thu nhập tính toán bằng công thức Final Income có cao đến đâu. Các biến số quản lý rủi ro này được giám sát nghiêm ngặt bởi bộ phận Quản lý Rủi ro. Trong trường hợp hệ thống có đánh giá khác biệt so với tính toán thủ công của nhân viên, quyết định của hệ thống Aither sẽ được ưu tiên. Nếu rớt Cut-off nhưng thuộc nhóm khách hàng ngoại lệ (ví dụ VIP), chi nhánh phải làm Tờ trình (Memorandum) gửi cấp phê duyệt độc lập.""",

    # Chunk 6: Low-Medium Similarity (Kiểm tra và truy vấn màn hình 8600)
    """Để cập nhật và phê duyệt mức thu nhập tính toán, giao dịch viên truy cập MH  Applying and Screening of consumer application. Tại tab "Application Information", nhân viên điền các thông tin về thu nhập cố định, phụ cấp, chức vụ và tên công ty. Cần lưu ý việc tính toán thu nhập này bị ảnh hưởng nếu khách hàng chưa ký hợp đồng lao động chính thức hoặc thời hạn hợp đồng còn lại dưới 6 tháng. Đối với khách hàng nước ngoài (ví dụ người Hàn Quốc), nếu thời hạn cư trú (visa/thẻ tạm trú) còn dưới 1 tháng hoặc HĐLĐ còn dưới 6 tháng thì hồ sơ không thỏa điều kiện đầu vào (Reject). Hạn mức thẻ sẽ không được cấp nếu các trường bắt buộc trên màn hình 8600 bị bỏ trống hoặc thông tin thu nhập không khớp với hồ sơ chứng từ nộp kèm (Sao kê lương/VssID). Mọi sửa đổi thông tin sau khi đã nhấn "Submit" phải thông qua chức năng "Inquiry for Re-screening" để hệ thống tái thẩm định lại.""",

    # Chunk 7: Low Similarity (Hồ sơ yêu cầu chứng minh thu nhập)
    """Để ngân hàng có cơ sở tính toán MI và MUE, khách hàng cá nhân phải nộp đủ bộ Hồ sơ chứng minh thu nhập. Tùy từng đối tượng, hồ sơ có thể là: Hợp đồng lao động (bản sao y có mộc công ty hoặc công chứng), Sao kê tài khoản ngân hàng thể hiện lương 3 tháng gần nhất (bản gốc có dấu mộc của ngân hàng phát hành sao kê), Giấy xác nhận lương (nếu nhận lương tiền mặt, đi kèm chứng từ đóng thuế TNCN). Gần đây, Shinhan Bank cho phép sử dụng ứng dụng Bảo hiểm xã hội số (VssID) để thay thế cho một số loại giấy tờ này. Nhân viên sẽ chụp màn hình quá trình đóng BHXH trên VssID của khách hàng, đảm bảo quá trình đóng liên tục tại công ty hiện tại từ 6 tháng trở lên. Mức lương đóng BHXH ghi nhận trên VssID sẽ được sử dụng làm cơ sở để đối chiếu với sao kê lương ngân hàng, nhằm ngăn chặn tình trạng khách hàng làm giả sao kê để đẩy hạn mức tín dụng lên cao.""",

    # Chunk 8: Low Similarity (CIC của Chủ doanh nghiệp)
    """Việc tính MI và MUE cho nhóm đối tượng là Chủ doanh nghiệp phức tạp hơn so với người lao động nhận lương (Salaried worker). Doanh nghiệp do khách hàng làm chủ phải hoạt động tối thiểu 1 năm. Ngân hàng sẽ tra cứu thông tin thuế từ trang gdt.gov.vn để kiểm tra xem mã số thuế công ty còn hoạt động hay không. Thu nhập của chủ doanh nghiệp thường được đánh giá thông qua báo cáo tài chính, tờ khai thuế giá trị gia tăng, hoặc quy đổi từ số dư bình quân trên tài khoản thanh toán của doanh nghiệp và cá nhân. Chủ doanh nghiệp không có Hợp đồng lao động, nên họ không chịu các giới hạn về hệ số nhân lương như nhân viên thông thường, nhưng họ chịu mức trần rủi ro MUE khắt khe hơn. Hơn nữa, lịch sử tín dụng (CIC) của công ty và cá nhân chủ doanh nghiệp đều phải trong sạch; không có nợ quá hạn hoặc thẻ tín dụng bị hủy do nợ xấu tại các TCTD khác.""",

    # Chunk 9: Lowest Similarity (Quy trình thẩm định thẻ tín dụng doanh nghiệp)
    """Khác hoàn toàn với cách tính thu nhập (MI) của khách hàng cá nhân, hạn mức thẻ tín dụng doanh nghiệp không dựa trên bảng lương của Giám đốc mà dựa trên năng lực tài chính của pháp nhân. Quy trình bắt đầu tại màn hình MH  New Corporate Member/Corporate Card Screening. Giao dịch viên nhập mã số thuế công ty và thông tin Tổng Giám đốc. Các công ty thuộc nhóm SOHO (doanh nghiệp siêu nhỏ) sẽ bị từ chối nếu không có doanh thu ổn định hoặc vi phạm Danh sách đen (Blacklist). Xếp hạng tín dụng nội bộ của doanh nghiệp (ví dụ AAA, BBB+) đóng vai trò quyết định mức Hạn mức tín dụng tối đa (Corporate Limit) mà công ty được cấp. Hạn mức này sau đó có thể được phân bổ ra nhiều thẻ phụ (Thẻ tín dụng doanh nghiệp cấp cho nhân viên) để phục vụ mục đích công tác, tiếp khách, thanh toán vé máy bay. Tổng hạn mức các thẻ phụ không được vượt quá Hạn mức tổng đã cấp cho doanh nghiệp.""",

    # Chunk 10: Lowest Similarity (Kiểm tra e-KYC và VNeID)
    """Bước đầu tiên trước khi tính toán bất kỳ thu nhập nào là xác định danh tính (KYC). Hiện nay, Shinhan Bank áp dụng kiểm tra tài khoản định danh điện tử qua VNeID. Khách hàng phải mở ứng dụng VNeID, đăng nhập bằng sinh trắc học hoặc mật khẩu. Nhân viên xem trực tiếp màn hình ứng dụng để kiểm tra thông tin Thẻ Căn cước công dân gắn chip điện tử. Hình ảnh trên VNeID bao gồm số CCCD, họ tên, ngày tháng năm sinh, đặc điểm nhận dạng và nơi cư trú sẽ được chụp lại để lưu trữ dưới dạng e-document. Phương pháp này giúp ngăn chặn triệt để tình trạng sử dụng CMND giả hoặc mượn giấy tờ của người khác để mở thẻ tín dụng. Việc xác thực danh tính hoàn tất mới là cơ sở pháp lý để ký Đơn đề nghị mở thẻ và áp dụng các công thức tính toán hạn mức, lãi suất cho khoản vay tín chấp của khách hàng đó."""
]

# ==========================================
# TEST CASE 3: PHÁT HÀNH THẺ TÍN DỤNG DOANH NGHIỆP
# ==========================================
query_3 = "Điều kiện, hồ sơ yêu cầu và quy trình thao tác trên hệ thống Aither để thẩm định và phát hành thẻ tín dụng doanh nghiệp không có tài sản đảm bảo."

documents_3 = [
    # Chunk 1: High Similarity (Điều kiện phát hành thẻ DN không TSĐB)
    """Để phát hành thẻ tín dụng doanh nghiệp không có tài sản đảm bảo, công ty phải đáp ứng các tiêu chuẩn tài chính khắt khe. Tổ chức phải là pháp nhân được thành lập và hoạt động hợp pháp theo quy định của pháp luật Việt Nam. Doanh nghiệp được phép mở nhiều thẻ tín dụng nhưng tổng hạn mức cấp cho các cá nhân (nhân viên lãnh đạo) không được vượt quá hạn mức tối đa cấp cho doanh nghiệp. Một điều kiện tiên quyết là doanh nghiệp phải được xếp hạng tín dụng nội bộ của ngân hàng (Shinhan Bank) từ hạng BBB- trở lên, và xếp hạng này còn hiệu lực tại thời điểm nộp hồ sơ. Các doanh nghiệp thuộc danh sách đen, danh sách cấm vận hoặc thuộc các ngành nghề kinh doanh bị hạn chế (như cơ sở karaoke, sòng bạc, kinh doanh vũ khí) sẽ tự động bị từ chối. Đối với doanh nghiệp mới thành lập, yêu cầu bổ sung là phải cung cấp Báo cáo tài chính trong 3 năm gần nhất, và nếu bị đánh giá là mô hình mSOHO tại thời điểm nộp hồ sơ mở thẻ, hồ sơ sẽ bị loại.""",

    # Chunk 2: High Similarity (Quy trình thao tác màn hình Corporate Screening 8520/8521)
    """Quy trình thao tác trên hệ thống Aither bắt đầu bằng việc thẩm định pháp nhân tại màn hình MH  New Corporate Member/Corporate Card Screening. Tại đây, giao dịch viên chọn tab "Corporate and CEO's Basic info", điền Mã số thuế, tên công ty, thông tin người đại diện theo pháp luật và nhấn kiểm tra "Blacklist Check". Sau khi hồ sơ doanh nghiệp được đăng ký, chuyển sang MH  Corporate Credit Card Authorization Request and Processed Result Registration để thực hiện yêu cầu cấp hạn mức. Nhập số tiền Hạn mức tín dụng yêu cầu (Requested Credit Limit) vào mục "Non-collateral" (Không tài sản đảm bảo). Tích chọn các mục bắt buộc như Delinquency History Confirmation (Xác nhận lịch sử nợ quá hạn), Required Documents Confirmation (Xác nhận hồ sơ yêu cầu). Hệ thống sẽ tự động chạy xếp hạng tín dụng (Credit Rating - ví dụ: BBB-). Nếu tất cả các thông tin hợp lệ, giao dịch viên nhấn "Submit" để trình hồ sơ cho cấp có thẩm quyền (Giám đốc chi nhánh hoặc Hội đồng tín dụng) phê duyệt kết quả Screening.""",

    # Chunk 3: High-Medium Similarity (Phát hành thẻ doanh nghiệp MH 8613/8582)
    """Sau khi hạn mức thẻ tín dụng doanh nghiệp đã được phê duyệt tại MH  (trạng thái Branch Screening/ Approval completed), quy trình tiếp tục chuyển sang giai đoạn phát hành thẻ vật lý. Nhân viên truy cập MH  Corporate Member Card Issuance. Tại đây, nhập mã Phòng ban (Department No.) cần phát hành thẻ, điền chi tiết thông tin nhân viên được ủy quyền sử dụng thẻ (như tên nhân viên kế toán, phòng ban tài chính, ngày sinh, giới hạn thẻ cá nhân). Thẻ tín dụng cấp cho nhân viên công ty không được vượt mức đã thỏa thuận. Giao dịch viên nhập số điện thoại, địa chỉ nhận sao kê, tỷ lệ thanh toán và chọn "Submit". Để kiểm tra lại tình trạng phát hành, có thể sử dụng màn hình MH  Corporate Card Member's Department Information Inquiry, nhập số ID của doanh nghiệp, chọn mã bộ phận và kiểm tra danh sách các thẻ đã được in (trạng thái Before Card Issuance hoặc Active). Thẻ sẽ được giao trực tiếp cho Người đại diện theo ủy quyền của doanh nghiệp.""",

    # Chunk 4: Medium Similarity (Hồ sơ chứng từ yêu cầu đối với pháp nhân)
    """Bộ hồ sơ yêu cầu để mở thẻ tín dụng doanh nghiệp không có tài sản đảm bảo phức tạp hơn nhiều so với cá nhân. Các chứng từ bắt buộc bao gồm: Đơn đề nghị mở thẻ tín dụng doanh nghiệp theo mẫu của Shinhan Bank; Chứng từ pháp lý của công ty (Giấy chứng nhận đăng ký doanh nghiệp/Giấy chứng nhận đầu tư/Quyết định thành lập pháp nhân); Bản sao Điều lệ công ty; Chứng từ chứng minh tư cách Người đại diện theo pháp luật (CCCD/Hộ chiếu của Giám đốc). Ngoài ra, công ty phải nộp Báo cáo tài chính đã kiểm toán 3 năm gần nhất, Tờ khai thuế VAT 6 tháng gần nhất, và các hợp đồng kinh tế lớn chứng minh hoạt động sản xuất kinh doanh ổn định. Nếu doanh nghiệp chỉ định nhân viên làm chủ thẻ (Cardholder), phải có Văn bản ủy quyền hợp lệ từ Người đại diện pháp luật, kèm theo giấy tờ tùy thân và Hợp đồng lao động của cá nhân được ủy quyền đó. Toàn bộ hồ sơ bản sao phải có mộc đóng dấu giáp lai của công ty.""",

    # Chunk 5: Medium Similarity (Phân quyền và thẩm quyền phê duyệt hạn mức thẻ DN)
    """Thẩm quyền phê duyệt hạn mức thẻ tín dụng doanh nghiệp (Credit limit approval authority) được phân cấp chặt chẽ dựa trên quy mô rủi ro và xếp hạng tín dụng của công ty (Exempt rating, AAA ~ A-, BBB+ ~ BBB-, B+ ~ B-). Đối với các thẻ tín dụng doanh nghiệp có tài sản đảm bảo (tiền gửi ký quỹ), quy trình phê duyệt đơn giản hơn và thường do Giám đốc Chi nhánh (Branch General Manager) quyết định. Tuy nhiên, đối với thẻ không có tài sản đảm bảo (Unsecured corporate credit card), các công ty có hạng tín dụng cao (AAA ~ A-) có thể được cấp hạn mức lên tới 5.2 tỷ đồng thông qua phê duyệt của Hội đồng tín dụng (CO Committee hoặc MCOC). Doanh nghiệp hạng thấp hơn (B+ ~ B-) chỉ được cấp tối đa 2 tỷ đồng. Bất kỳ khoản cấp tín dụng nào vượt quá thẩm quyền của Chi nhánh đều phải được đẩy lên trình tự duyệt tập trung tại Hội sở thông qua hệ thống luồng công việc tự động trên Aither, có kèm theo báo cáo tóm tắt tình hình tài chính của doanh nghiệp.""",

    # Chunk 6: Low-Medium Similarity (Tra cứu danh sách đen và cấm vận đối với doanh nghiệp)
    """Một bước không thể thiếu trong thẩm định thẻ doanh nghiệp là rà soát Danh sách đen (Blacklist) và Danh sách nghề nghiệp bị hạn chế (Restricted Industry List). Theo danh mục quy định, nhân viên phải sử dụng hệ thống tra cứu nội bộ (mã 6308) để sàng lọc. Công ty kinh doanh trong các lĩnh vực rủi ro cao như: Thủy thủ và những người làm việc trên tàu chuyên tuyến ngoài khơi, Hiệu cầm đồ, kinh doanh Karaoke, quán Bar, vũ trường, hoạt động cá cược, sòng bạc, kinh doanh vũ khí, sản xuất hóa chất độc hại sẽ không được mở thẻ. Thêm vào đó, nếu doanh nghiệp đang bị phân loại Nợ quá hạn, Nợ xấu (Nhóm 2 trở lên) trên CIC, hoặc có lãnh đạo từng dính líu đến gian lận tài chính, rửa tiền (AML), hệ thống sẽ đánh dấu đỏ và chặn đứng quy trình phê duyệt. Chức năng Blacklist Check trên màn hình MH  áp dụng cho cả pháp nhân và cá nhân là thành viên ban giám đốc, đảm bảo không có rủi ro pháp lý nào liên đới tới ngân hàng.""",

    # Chunk 7: Low Similarity (Truy vấn CIC cho thẻ cá nhân)
    """Đối với việc truy vấn tín dụng cá nhân, giao dịch viên vào màn hình MH  CIC Information Request. Thao tác này đòi hỏi sự chính xác về họ tên và số CMND/CCCD. Kết quả trả về cho thấy lịch sử vay mượn của cá nhân tại các ngân hàng khác, các khoản vay tiêu dùng, dư nợ thẻ hiện tại và các cảnh báo nợ xấu (nếu có). Trong khi đó, báo cáo CIC doanh nghiệp phức tạp hơn, hiển thị cả dư nợ trung dài hạn, dư nợ bảo lãnh, dư nợ LC và xếp hạng doanh nghiệp của Trung tâm thông tin tín dụng quốc gia. Nếu khách hàng cá nhân bị từ chối do CIC, lý do thường là vượt giới hạn số thẻ được mở (ví dụ đang có từ 4 thẻ trở lên) hoặc tỷ lệ nợ trên thu nhập (DTI) quá cao. Báo cáo CIC là tài liệu mật, nhân viên ngân hàng bị cấm sao chép, chụp ảnh gửi cho khách hàng hoặc chia sẻ cho bên thứ ba dưới bất kỳ hình thức nào.""",

    # Chunk 8: Low Similarity (Mở thẻ cho chủ doanh nghiệp cá nhân - SOHO)
    """Trường hợp cá nhân là Chủ doanh nghiệp hoặc người tự doanh (Self-employed) muốn mở thẻ tín dụng cá nhân (không phải thẻ công ty), họ được phân loại vào nhóm khách hàng SOHO. Yêu cầu bắt buộc là cơ sở kinh doanh/công ty phải thành lập và hoạt động ít nhất 1 năm. Ngân hàng sẽ yêu cầu nộp Giấy phép đăng ký kinh doanh và chứng từ đóng thuế. Nhân viên phải tự kiểm tra chéo Mã số thuế doanh nghiệp trên trang thông tin của Tổng Cục Thuế (www.gdt.gov.vn) hoặc Cổng thông tin quốc gia về đăng ký doanh nghiệp để xác nhận trạng thái công ty vẫn "Đang hoạt động". Thu nhập của nhóm này được xác định dựa trên một tỷ lệ phần trăm doanh thu hoặc lợi nhuận sau thuế, thay vì sử dụng sao kê lương thông thường. Do tính chất rủi ro cao của nhóm tự doanh, hạn mức tín dụng cá nhân cấp cho họ thường được kiểm soát chặt chẽ và yêu cầu đi thực địa (Site visit) chụp ảnh cơ sở kinh doanh thực tế.""",

    # Chunk 9: Lowest Similarity (Điều kiện sử dụng ứng dụng VssID và VNeID)
    """Trong các năm gần đây, Shinhan Bank đã tích hợp việc kiểm tra chứng từ điện tử qua VssID và VNeID để đơn giản hóa hồ sơ cho khách hàng cá nhân. VssID là ứng dụng của Bảo hiểm Xã hội Việt Nam, cung cấp thông tin về quá trình đóng BHXH, quá trình hưởng chế độ và thẻ BHYT điện tử. Nhân viên ngân hàng có thể hướng dẫn khách hàng mở ứng dụng, vào mục "Quá trình tham gia" để xem chi tiết mức lương đóng bảo hiểm, tên đơn vị sử dụng lao động, từ đó xác thực Hợp đồng lao động và thu nhập mà không cần giấy xác nhận lương. VNeID là ứng dụng định danh điện tử của Bộ Công an, giúp thay thế Thẻ CCCD vật lý và sổ hộ khẩu. Việc kiểm tra tình trạng cư trú thông qua VNeID cấp độ 2 được chấp nhận như một bằng chứng hợp lệ về địa chỉ hiện tại của khách hàng. Tất cả các màn hình tra cứu này phải được chụp ảnh màn hình (screenshot) và lưu trữ tuân thủ đúng Hướng dẫn chụp màn hình của ngân hàng.""",

    # Chunk 10: Lowest Similarity (Chính sách phát hành Thẻ phụ cá nhân)
    """Quy trình phát hành thẻ phụ cá nhân (Secondary/Family Member Card) được thực hiện trên màn hình MH . Thẻ phụ chỉ được phát hành khi Thẻ chính (Primary Member Card) đã được duyệt và đang trong trạng thái hoạt động bình thường. Tại màn hình 8612, giao dịch viên phải nhập Số thẻ chính (Primary Member Card Number), sau đó điền thông tin của chủ thẻ phụ bao gồm Tên, số CMND/CCCD, Ngày sinh và Mối quan hệ với chủ thẻ chính (Vợ/chồng, con cái, cha mẹ). Hạn mức của thẻ phụ là hạn mức dùng chung (Shared limit) với thẻ chính, và khách hàng có thể chỉ định một mức trần sử dụng riêng cho thẻ phụ (ví dụ: giới hạn 20 triệu/tháng trong tổng hạn mức 100 triệu của thẻ chính). Chủ thẻ phụ phải đủ từ 15 tuổi trở lên. Nếu từ 15 đến dưới 18 tuổi, bắt buộc phải là con của chủ thẻ chính và được sự đồng ý của người đại diện hợp pháp. Không yêu cầu chứng minh thu nhập riêng đối với chủ thẻ phụ."""
]

# ==========================================
# TEST CASE 4: THẨM ĐỊNH THU NHẬP VÀ XÁC THỰC QUA VSSID/VNEID
# ==========================================
query_4 = "Cách sử dụng ứng dụng VssID và VNeID để xác minh danh tính, lịch sử đóng bảo hiểm xã hội, nơi cư trú và các quy định về việc chụp màn hình lưu hồ sơ chứng từ điện tử thay cho bản cứng."

documents_4 = [
    # Chunk 1: High Similarity (Cách thao tác và kiểm tra ứng dụng VssID)
    """VssID là ứng dụng số của Bảo hiểm Xã hội Việt Nam, được Shinhan Bank chấp thuận thay thế cho một số chứng từ chứng minh thu nhập và việc làm bản cứng. Để xác minh, nhân viên yêu cầu khách hàng đăng nhập ứng dụng VssID trên điện thoại. Tại màn hình chính, chọn "Quá trình tham gia" (Quá trình tham gia BHXH). Hệ thống sẽ hiển thị lịch sử làm việc, tên cơ quan/công ty đóng BHXH (Employer), thời gian đóng (Accumulated time of contribution) và Mức lương đóng BHXH (Salary amount for social insurance contribution). Nhân viên kiểm tra xem khách hàng có đóng BHXH liên tục tại công ty hiện tại từ 6 tháng trở lên hay không. Số tiền lương đóng BHXH trên VssID sẽ được đối chiếu với sao kê lương qua ngân hàng để xem xét hạn mức. Nhân viên phải thực hiện thao tác chụp màn hình (Screenshot) toàn bộ các thông tin này, đảm bảo hiển thị rõ tên khách hàng, mã số BHXH và quá trình đóng chi tiết nhất. Hình ảnh phải liền mạch và không có dấu hiệu chỉnh sửa.""",

    # Chunk 2: High Similarity (Quy định chụp màn hình VssID và VNeID)
    """Quá trình chụp ảnh màn hình (screenshot) các ứng dụng điện tử như VssID, VNeID, hoặc cổng thông tin baohiemxahoi.gov.vn phải tuân thủ nghiêm ngặt Hướng dẫn chụp màn hình (Screenshot Guideline) của Phòng Quản lý rủi ro (SCMC). Điện thoại sử dụng để chụp phải là điện thoại của khách hàng hoặc ipad của chi nhánh mở ứng dụng trực tiếp, không được dùng ảnh khách hàng tự chụp gửi qua Zalo. Ảnh chụp màn hình phải hiển thị rõ logo ứng dụng, tên tài khoản, thời gian trên thanh trạng thái điện thoại (để chứng minh thời điểm tra cứu là hiện tại). Đối với VNeID, do ứng dụng có chính sách bảo mật không cho phép chức năng chụp màn hình tự động của điện thoại ở một số dòng Android, nhân viên có thể dùng điện thoại khác để chụp lại thiết bị của khách hàng (Take a photo of the screen). Bản sao chứng từ in ra từ ảnh chụp phải có chữ ký nháy của nhân viên thẩm định (Xác nhận đã đối chiếu bản chính) và dấu mộc của cấp Quản lý kênh (SCMC) hoặc chi nhánh.""",

    # Chunk 3: High-Medium Similarity (Sử dụng VNeID xác thực cư trú và nhân thân)
    """VNeID là ứng dụng định danh điện tử của Bộ Công an, có giá trị pháp lý thay thế cho CCCD vật lý và Sổ hộ khẩu/Giấy tạm trú. Khách hàng sử dụng tài khoản VNeID cấp độ 2 (đã đăng ký sinh trắc học tại công an) được chấp nhận làm Hồ sơ nhân dạng và Hồ sơ chứng minh nơi cư trú. Nhân viên hướng dẫn khách hàng mở VNeID, vào mục "Ví giấy tờ", chọn "Căn cước công dân" để xem hình ảnh CCCD 2 mặt và chọn "Thông tin cư trú" để kiểm tra địa chỉ Thường trú và Nơi ở hiện tại. Nếu địa chỉ Nơi ở hiện tại hiển thị đầy đủ 4 cấp (Tỉnh/Thành phố, Quận/Huyện, Phường/Xã, Số nhà/Đường) và khách hàng sinh sống tại TP.HCM hoặc Hà Nội, hồ sơ sẽ được miễn cung cấp Hóa đơn điện/nước hoặc chứng từ tạm trú (CT07). Việc sử dụng VNeID giúp khách hàng không cần mang theo nhiều giấy tờ cứng, rút ngắn thời gian xử lý hồ sơ mở thẻ tín dụng tại quầy.""",

    # Chunk 4: Medium Similarity (Trường hợp không dùng được VssID/VNeID)
    """Trong trường hợp khách hàng không cài đặt VssID, VNeID, hoặc ứng dụng bị lỗi hệ thống (bảo trì), ngân hàng bắt buộc khách hàng phải cung cấp hồ sơ bản cứng theo quy trình truyền thống. Đối với chứng minh nơi cư trú: Nộp CCCD gắn chip (bản sao y có mộc), Giấy xác nhận thông tin về cư trú mẫu CT07/CT08 do công an cấp trong vòng 2 tháng, hoặc Hợp đồng thuê nhà kèm Hóa đơn điện nước 3 tháng gần nhất. Đối với chứng minh công việc và thu nhập: Khách hàng nộp Hợp đồng lao động (bản sao y công ty hoặc có đóng dấu treo, chữ ký giáp lai), kèm theo Sao kê lương ngân hàng 3 tháng gần nhất (có mộc đỏ của ngân hàng phát hành sao kê). Đối với người nước ngoài không có VssID, yêu cầu thẻ tạm trú (TRC) còn hạn, Hộ chiếu (Passport) bản gốc để đối chiếu, và Giấy phép lao động (Work permit) tại Việt Nam.""",

    # Chunk 5: Medium Similarity (Khách hàng miễn trừ và ngoại lệ)
    """Một số nhóm khách hàng không bắt buộc phải cung cấp VssID hoặc Hợp đồng lao động bản cứng do tính chất rủi ro thấp. Ví dụ: Cán bộ nhân viên làm việc trực tiếp tại Shinhan Bank Việt Nam (SHBVN) hoặc các công ty con thuộc tập đoàn tài chính Shinhan (Shinhan Finance, Shinhan Securities, Shinhan Life). Đối với nhóm này, chỉ cần Xác nhận nhân sự nội bộ (HR Confirmation) thay cho hồ sơ việc làm. Khách hàng VIP thuộc nhóm Private Wealth Management (PWM) hạng Platinum hoặc Signature, có số dư tiền gửi tại ngân hàng trên 1 tỷ đồng cũng được miễn trừ chứng minh thu nhập từ lương, do đó không cần kiểm tra VssID. Các khách hàng thuộc chương trình Corporate Employee Program (CEP) tại các công ty có ký thỏa thuận hợp tác với Shinhan Bank cũng được giảm thiểu chứng từ, thay VssID bằng Danh sách nhân viên được nhân sự công ty đối tác đóng dấu xác nhận gửi định kỳ cho ngân hàng.""",

    # Chunk 6: Low-Medium Similarity (Hướng dẫn tra cứu CIC màn hình 2133)
    """Việc tra cứu lịch sử tín dụng (CIC) trên màn hình 2133 (CIC Information Request) là bắt buộc và hoàn toàn độc lập với việc kiểm tra thu nhập qua VssID. Sau khi nhân viên đã xác thực danh tính khách hàng bằng VNeID, họ sẽ lấy số CCCD đó nhập vào màn hình 2133 trên Aither để kéo dữ liệu dư nợ từ Trung tâm thông tin tín dụng. Các thông tin từ CIC như tổng dư nợ hiện tại, số lượng thẻ tín dụng đang có, lịch sử nợ quá hạn (nếu có) sẽ được tổng hợp cùng với mức lương ghi nhận trên VssID để tạo thành bộ hồ sơ tín dụng hoàn chỉnh. Điểm khác biệt là VssID kiểm tra thông tin thu nhập đầu vào, còn CIC kiểm tra tình trạng nợ đầu ra. Nếu khách hàng làm giả VssID nhưng báo cáo CIC ghi nhận nợ xấu nhóm 3, hồ sơ chắc chắn vẫn sẽ bị hệ thống tự động từ chối (Screening Failed).""",

    # Chunk 7: Low Similarity (Quy trình thẩm định khách hàng doanh nghiệp)
    """Trong khi khách hàng cá nhân sử dụng VssID và VNeID để e-KYC, khách hàng doanh nghiệp lại có quy trình thẩm định khác biệt hoàn toàn. Doanh nghiệp không có chứng minh nhân dân hay bảo hiểm xã hội cá nhân. Thẩm định viên sẽ tra cứu pháp nhân qua Cổng thông tin Đăng ký doanh nghiệp quốc gia để xác nhận tình trạng hoạt động của công ty. Yêu cầu nộp Báo cáo tài chính, Tờ khai thuế VAT và Sao kê tài khoản doanh nghiệp (Corporate Bank Account Statement) để phân tích dòng tiền và khả năng sinh lời. Xếp hạng tín dụng nội bộ doanh nghiệp (Corporate Credit Rating) từ AAA đến B- sẽ quyết định hạn mức thẻ tín dụng. Thẻ doanh nghiệp chủ yếu dùng cho các khoản chi phí chung như mua sắm vật tư, thanh toán phí quảng cáo, và tiếp khách, do đó rủi ro được kiểm soát trên bình diện dòng tiền tổng thể của công ty chứ không phải thu nhập của một cá nhân.""",

    # Chunk 8: Low Similarity (Tính hạn mức thẻ tín dụng MI/MUE)
    """Hạn mức thẻ tín dụng cho khách hàng cá nhân được tính toán tự động dựa trên Điểm AS (Application Score) và Thu nhập hàng tháng (MI). Thu nhập này được xác định từ các hồ sơ chứng minh thu nhập (sao kê lương, VssID). Nếu MI < 20 triệu đồng, hạn mức tối đa thường là MI x 2. Nếu MI >= 20 triệu đồng, hạn mức có thể lên tới MI x 3. Tuy nhiên, bất kể thu nhập cao đến đâu, khách hàng bị giới hạn bởi Mức dư nợ tín chấp tối đa (MUE - Maximum Unsecured Exposure). MUE không được vượt quá 900 triệu đồng hoặc 21 lần thu nhập cuối cùng. Nếu khách hàng có các thẻ tín dụng tại ngân hàng khác hoặc các khoản vay tiêu dùng, dư nợ này sẽ được trừ đi khi tính toán hạn mức còn lại có thể cấp tại Shinhan. Điều này đảm bảo tỷ lệ Nợ trên Thu nhập (DTI) của khách hàng ở mức an toàn.""",

    # Chunk 9: Lowest Similarity (Màn hình hệ thống 8600)
    """Màn hình MH  Applying and Screening of consumer application là nơi giao dịch viên nhập liệu toàn bộ đơn xin mở thẻ của khách hàng cá nhân. Tại tab "Application Information", các thông tin về nhân thân, địa chỉ gửi sao kê (Statement sending method), số tài khoản trích nợ tự động (Payment Bank Account) được điền đầy đủ. Tại tab "Screening", hệ thống sẽ chạy đánh giá rủi ro (Cut-off Result, Filtering Result). Nếu kết quả là "Approved", hạn mức tín dụng được duyệt (Applied Credit Limit) sẽ hiển thị. Trong trường hợp khách hàng muốn mở thẻ phụ cho người thân, nhân viên sẽ chuyển sang màn hình MH  Secondary/Family Member Consumer Card Issuance. Tại đây, nhập mã số thẻ chính (Primary Member Card Number), hệ thống sẽ liên kết hai thẻ với nhau và cho phép thẻ phụ chia sẻ hạn mức chung với thẻ chính. Quy trình này không cần chạy lại bộ lọc rủi ro phức tạp cho thẻ phụ.""",

    # Chunk 10: Lowest Similarity (Thực địa và xác minh qua điện thoại)
    """Ngoài việc thu thập chứng từ, ngân hàng thực hiện Thẩm định qua điện thoại (Tele-verification) và Thẩm định thực địa (Site visit) để chống gian lận. Bộ phận thẩm định sẽ gọi điện thoại đến số điện thoại bàn của công ty khách hàng để xác minh tình trạng việc làm, chức vụ, và gọi cho khách hàng để xác nhận các thông tin trên Đơn mở thẻ. Nếu công ty không có số điện thoại bàn hợp lệ, hoặc khách hàng thuộc nhóm rủi ro cao (như làm việc trong ngành nghề nhạy cảm, địa chỉ cư trú trên VNeID không rõ ràng, có dấu hiệu giả mạo hồ sơ), nhân viên ngân hàng sẽ phải đi thực địa trực tiếp đến nơi ở hoặc nơi làm việc của khách hàng. Tại đó, nhân viên sẽ chụp ảnh khách hàng tại địa điểm thực tế, ảnh biển hiệu công ty, tọa độ GPS trên Google Maps để tải lên hệ thống Aither làm bằng chứng lưu trữ hồ sơ, đảm bảo khách hàng là người có thật và đang sinh sống, làm việc tại địa chỉ đã khai báo."""
]

# ==========================================
# TEST CASE 5: NHÓM KHÁCH HÀNG TỐT (VGC) VÀ KHÁCH HÀNG VIP (PWM)
# ==========================================
query_5 = "Chính sách, điều kiện hồ sơ và cách xác định hạn mức tín dụng ưu đãi dành riêng cho nhóm khách hàng làm việc tại Công ty Tốt (VGC) và khách hàng cao cấp Private Wealth Management (PWM)."

documents_5 = [
    # Chunk 1: High Similarity (Điều kiện nhóm VGC và cách tính hạn mức)
    """Nhóm Vietnamese Good Credit (VGC) bao gồm các khách hàng cá nhân người Việt Nam đang làm việc tại các Công ty/Tập đoàn lớn thuộc "Danh sách công ty Tốt" do Shinhan Bank (SHBVN) phê duyệt. Lợi ích lớn nhất của nhóm VGC là chính sách cấp hạn mức tín dụng thẻ ưu đãi không cần tài sản đảm bảo. Hạn mức cấp cho VGC được tính toán theo nguyên tắc: Hạn mức thẻ tối đa có thể cấp bằng Giá trị trung bình cộng hạn mức của các thẻ tín dụng đang hoạt động tại các ngân hàng khác (thỏa điều kiện), nhưng không được vượt quá trần quy định. Cụ thể, trần hạn mức tối đa cấp theo nhóm VGC cho khách làm tại công ty tốt là 100 triệu đồng; nếu khách hàng làm việc tại công ty thuộc nhóm khác có thể là 60 triệu đồng. Tuy nhiên, tổng hạn mức hệ thống cấp tín chấp không được vượt quá 180 triệu đồng. Ngân hàng sẽ áp dụng hạn mức thấp nhất (MIN) giữa (Trung bình hạn mức các thẻ khác), (Trần hạn mức VGC), và (Trần hạn mức hệ thống) để phê duyệt hạn mức thẻ cuối cùng.""",

    # Chunk 2: High Similarity (Điều kiện và hạn mức nhóm PWM)
    """Private Wealth Management (PWM) là phân khúc khách hàng cá nhân cao cấp nhất. Để đạt tiêu chuẩn PWM Platinum hoặc Signature, khách hàng phải duy trì số dư tiền gửi, tiền gửi tích lũy và tiền gửi tiết kiệm rất lớn tại ngân hàng. Cụ thể: Khách hàng phải có số dư tiền gửi tại một thời điểm hoặc tổng số dư tiền gửi 3 tháng liên tiếp từ 1 tỷ đồng trở lên (để vào hạng Platinum) hoặc từ 2 tỷ đồng trở lên (để vào hạng Signature). Hạn mức thẻ tín dụng cấp cho nhóm PWM rất cao: Dư nợ khoản vay cầm cố tiền gửi/vay cầm cố thẻ tiết kiệm được khấu trừ vào hạn mức. Hạn mức thẻ trong hệ thống tính toán (Credit Limit) đối với khách hàng có số dư >= 1 tỷ đồng dao động từ 60 triệu đến 150 triệu đồng; đối với khách hàng có số dư >= 2 tỷ đồng (hạng Signature), hạn mức thẻ được cấp có thể lên tới 300 triệu đồng. Nhóm này được định danh tại chi nhánh và có mã định danh nội bộ (Customer Group Type).""",

    # Chunk 3: High-Medium Similarity (Hồ sơ yêu cầu cho nhóm VGC và PWM)
    """Do thuộc nhóm rủi ro thấp và có tài chính vững mạnh, hồ sơ mở thẻ của VGC và PWM được tối giản đáng kể. Đối với khách hàng VGC: Yêu cầu CCCD gắn chip (bản sao y hoặc xác thực VNeID), Hợp đồng lao động và chứng từ chứng minh thu nhập (sao kê lương hoặc VssID). Khách hàng VGC được miễn thẩm định thực địa nếu địa chỉ rõ ràng. Đối với khách hàng PWM: Chỉ cần Đơn đề nghị mở thẻ, CCCD/Hộ chiếu bản sao y, và Chứng từ tiền gửi (Sổ tiết kiệm/Xác nhận số dư tài khoản tại SHBVN). Khách hàng PWM hoàn toàn được miễn chứng minh thu nhập từ lương (không cần hợp đồng lao động, không cần VssID, không cần sao kê lương hàng tháng). Hệ thống sẽ tự động tra soát số dư tiền gửi nội bộ [4] để xác nhận điều kiện PWM. Tuy nhiên, cả hai nhóm vẫn phải bắt buộc trải qua quá trình tra cứu CIC để đảm bảo không có nợ xấu.""",

    # Chunk 4: Medium Similarity (Khách hàng Top Club)
    """Khách hàng Top Club là nhóm khách hàng thân thiết được đánh giá định kỳ hàng quý dựa trên các tiêu chí giao dịch tổng hợp. Hệ thống phân chia các hạng Top Club: Classic, Best, Ace và Premier. Điểm xếp hạng được cập nhật vào các mốc thời gian cụ thể trong năm (ví dụ: ngày 15/03, 15/06, 15/09, 15/12). Khách hàng thuộc hạng Best, Ace và Premier duy trì trong 3 quý liên tiếp sẽ được áp dụng các chính sách ưu đãi mở thẻ tín dụng tương đương nhóm khách hàng có thu nhập ổn định. Hạn mức thẻ tín dụng không tài sản đảm bảo cấp cho nhóm Top Club dựa trên hạng: Hạng Classic tối đa 20 triệu, Best 30 triệu, Ace 40 triệu và Premier tối đa 50 triệu đồng. Hồ sơ chứng từ đơn giản, chỉ cần CCCD và kiểm tra thông tin hạng thành viên trực tiếp trên hệ thống Aither màn hình MH  tại mục Ref Information - Tops Club Performance.""",

    # Chunk 5: Medium Similarity (Quy định tra cứu CIC chung cho các nhóm)
    """Mặc dù khách hàng VGC, PWM và Top Club được ưu tiên về chứng từ và hạn mức, họ vẫn phải tuân thủ quy định tra cứu Trung tâm thông tin tín dụng quốc gia (CIC). Nhân viên ngân hàng vào màn hình MH  để tra cứu. Nếu khách hàng có nợ nhóm 2 (Nợ cần chú ý) từ 2 lần trở lên trong 12 tháng gần nhất, hoặc có nợ nhóm 3 đến nhóm 5 (Nợ dưới tiêu chuẩn, Nợ nghi ngờ, Nợ có khả năng mất vốn) trong vòng 3 đến 5 năm gần nhất, hệ thống Filtering của ngân hàng sẽ tự động đánh rớt (Screening Failed) bất kể khách hàng có số dư tiết kiệm hàng tỷ đồng hay làm Giám đốc tại công ty danh tiếng. Việc duy trì điểm tín dụng tốt là điều kiện tiên quyết cho mọi chính sách ưu đãi không tài sản đảm bảo.""",

    # Chunk 6: Low-Medium Similarity (Hướng dẫn chụp màn hình VssID)
    """Một số khách hàng VGC nếu không có sao kê lương qua ngân hàng hợp lệ có thể dùng ứng dụng VssID để bổ sung hồ sơ. Nhân viên yêu cầu khách hàng đăng nhập VssID, vào phần "Quá trình tham gia BHXH". Tại đây, mức lương đóng bảo hiểm (thường là lương cơ bản theo hợp đồng) phải phản ánh đúng chức vụ và thu nhập kê khai. Nhân viên chụp ảnh màn hình toàn bộ quá trình đóng, đảm bảo hiển thị rõ tên đơn vị đóng bảo hiểm (phải khớp với tên Công ty Tốt trong danh sách VGC), số tháng đóng liên tục và tên khách hàng. Ảnh chụp phải tuân thủ chuẩn Screenshot Guideline, hiển thị rõ thanh pin và đồng hồ điện thoại để chứng minh là ảnh chụp trực tiếp từ ứng dụng của khách hàng, không qua chỉnh sửa photoshop.""",

    # Chunk 7: Low Similarity (Mở thẻ cho người nước ngoài và Hàn Quốc)
    """Đối với khách hàng cá nhân là người nước ngoài, đặc biệt là người Hàn Quốc (KCB >= 531), ngân hàng có chính sách riêng. Hạn mức tín dụng được cấp dựa trên Điểm KCB (Korean Credit Bureau) thay vì chỉ đánh giá MUE nội địa. Ví dụ: Khách hàng có KCB hạng 1, 2 (điểm từ 904 đến 1000) có thể được cấp thẻ với hạn mức tối đa 100 triệu đồng; hạng 3, 4 cấp 70 triệu; hạng 5, 6 cấp 50 triệu và hạng 7 cấp 30 triệu đồng. Điều kiện đi kèm là khách hàng phải có Giấy phép lao động, Hợp đồng lao động còn hạn trên 6 tháng, Thẻ tạm trú (TRC) còn hiệu lực và có Hộ chiếu bản gốc để đối chiếu. Khách hàng nước ngoài không được áp dụng các chính sách VGC hay Top Club dành riêng cho người Việt Nam, trừ khi họ làm quản lý cấp cao có tham gia góp vốn doanh nghiệp.""",

    # Chunk 8: Low Similarity (Cách tính thu nhập Final Income)
    """Dù thuộc nhóm VGC hay khách hàng đại trà, quy tắc tính Thu nhập cuối cùng (Final Income) trên hệ thống Aither đều giống nhau (ngoại trừ nhóm PWM dùng sổ tiết kiệm). Thu nhập cuối cùng = Thu nhập cố định + 75% * (Thu nhập trung bình 3 tháng gần nhất - Thu nhập cố định). Khoản thu nhập cố định bao gồm lương cơ bản ghi trên hợp đồng và các khoản phụ cấp cố định hàng tháng (phụ cấp cơm, điện thoại, đi lại). Khoản thu nhập không cố định như tiền làm thêm giờ (OT), thưởng KPI, hoa hồng doanh số chỉ được tính 75% giá trị trung bình để phòng ngừa rủi ro thu nhập giảm sút trong tương lai. Nếu thu nhập tính toán dưới mức yêu cầu của hạng thẻ, giao dịch viên không được phép "Submit" hồ sơ trên màn hình MH .""",

    # Chunk 9: Lowest Similarity (Thẻ doanh nghiệp có tài sản đảm bảo)
    """Khác với cá nhân, khách hàng doanh nghiệp nếu muốn mở thẻ tín dụng có Hạn mức cao mà không đủ điều kiện tín chấp (ví dụ báo cáo tài chính yếu, mới thành lập, rớt Cut-off) thì phải dùng phương án Thẻ doanh nghiệp có tài sản đảm bảo (Secured corporate credit card). Tài sản đảm bảo thường là tiền gửi ký quỹ, sổ tiết kiệm đứng tên doanh nghiệp hoặc đứng tên Người đại diện theo pháp luật. Hạn mức thẻ sẽ được cấp bằng 90% đến 100% giá trị tài sản đảm bảo. Quy trình phê duyệt cho loại thẻ này rất nhanh chóng, chủ yếu do Giám đốc Chi nhánh/Phòng giao dịch ký duyệt mà không cần trình Hội đồng tín dụng trung ương. Điều kiện tra cứu CIC đối với doanh nghiệp vẫn phải thực hiện trên màn hình 2133, nhưng chỉ để kiểm tra nợ xấu, không đánh giá khắt khe về các chỉ số tài chính (như tỷ lệ thanh toán hiện hành, vòng quay vốn) như thẻ tín chấp.""",

    # Chunk 10: Lowest Similarity (Gia hạn thẻ doanh nghiệp MH 8521)
    """Đối với thẻ tín dụng doanh nghiệp, khi thẻ sắp hết hạn, giao dịch viên tiến hành quy trình Gia hạn thẻ (Renewal) thông qua MH  Corporate Credit Card Authorization Request. Chọn Type of ID là "Renewal/Condition Change". Quá trình gia hạn yêu cầu doanh nghiệp phải nộp lại báo cáo tài chính mới nhất, báo cáo thuế để chứng minh công ty vẫn đang hoạt động bình thường. Nếu doanh nghiệp có doanh thu sụt giảm nghiêm trọng, ngân hàng có quyền giảm Hạn mức tín dụng hoặc từ chối gia hạn. Nếu gia hạn thành công, nhân viên vào lại MH  Corporate Member Card Issuance để phát hành phôi thẻ mới. Tình trạng thẻ cũ sẽ chuyển sang "Closed" khi thẻ mới được kích hoạt. Không có chính sách tự động gia hạn đối với thẻ tín dụng doanh nghiệp do yếu tố biến động kinh doanh phức tạp, khác với thẻ cá nhân được tự động gửi về nhà nếu lịch sử trả nợ tốt."""
]

dic = {
    query_1: documents_1,
    query_2: documents_2,
    query_3: documents_3,
    query_4: documents_4,
    query_5: documents_5,
}

# extreme evaluation store
card_po_eval = []

# Test models
bge_m3 = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=bge_m3,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          #gpu_memory_utilization=0.4,
          disable_log_stats=True,
          )
evaluate_model(bge_m3, llm, dic, card_po_eval)

jina_v3 = "jinaai/jina-reranker-v3"
llm = LLM(
    model=jina_v3,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=2024,
    #gpu_memory_utilization=0.6,
    disable_log_stats=True,
)
evaluate_model(jina_v3, llm, dic, card_po_eval)

qwen3_06 = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_06,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.4,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_06, llm, dic, card_po_eval)

qwen3_4 = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_4,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.8,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_4, llm, dic, card_po_eval)

INFO 06-24 04:15:24 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}
INFO 06-24 04:15:24 [model.py:555] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-24 04:15:24 [model.py:2015] Downcasting torch.float32 to torch.float16.
INFO 06-24 04:15:24 [model.py:1680] Using max model len 1024
WARNING 06-24 04:15:24 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-24 04:15:24 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-24 04:15:24 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-24 04:15:24 [vllm.py:1089] Cudagraph is disabled under eager mod

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (14 outputs left out)

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

###5. Retail & SOHO Loan Products

In [ ]:
# TEST CASE 1: Nghiệp vụ Vay mua xe ô tô (Car Loan) - Điều kiện & LTV
query_1 = "điều kiện cho vay và LTV đối với retail used car loan là bao nhiêu"

documents_1 = [
    # 1. Trả lời trực tiếp và đầy đủ nhất về LTV xe cũ
    """Sản phẩm vay mua xe ô tô đã qua sử dụng (Retail Used Car Loan): Hạn mức vay được tính bằng tỷ lệ LTV * Min(Giá trị mua bán theo HĐMB/Hóa đơn, Giá trị định giá của xe). Tỷ lệ cho vay (LTV) tối đa đối với xe du lịch từ 0-9 chỗ đã qua sử dụng là 70%. Thời hạn vay phụ thuộc vào tuổi xe, ví dụ xe có tuổi từ lúc đăng ký đến lúc vay là 4 năm thì thời hạn vay tối đa 5 năm, và không áp dụng cho xe thương mại cỡ lớn.""",

    # 2. Bổ sung điều kiện về giá trị xe
    """Đối với xe đã qua sử dụng, số tiền giải ngân tối đa phải thỏa mãn điều kiện: Số tiền vay <= Giá trị xe bao gồm VAT thể hiện trên hóa đơn VAT hoặc Hợp đồng mua bán có công chứng. Khách hàng bắt buộc phải nộp hồ sơ chứng minh số tiền vốn tự có (phần chênh lệch giữa giá trị xe và số tiền vay) đã được thanh toán cho bên bán trước khi ngân hàng tiến hành giải ngân khoản vay.""",

    # 3. So sánh với xe mới để làm rõ bối cảnh
    """Tỷ lệ LTV đối với xe ô tô mới (biển trắng) có sự khác biệt so với xe cũ. Đối với xe du lịch 0-9 chỗ mua mới, LTV tối đa có thể lên tới 80% (trong khi xe cũ là 70%) và thời gian vay tối đa lên tới 8 năm (96 tháng). Chương trình vay bù đắp tài chính chỉ áp dụng cho mua xe ô tô mới, không áp dụng cho các dòng xe tải lớn hoặc xe khách trên 16 chỗ.""",

    # 4. Yêu cầu về bảo hiểm đi kèm
    """Điều kiện bắt buộc trước khi giải ngân khoản vay mua ô tô (mới và cũ): Bên vay phải mua bảo hiểm vật chất xe và chỉ định SHBVN là bên thụ hưởng đầu tiên đối với mọi rủi ro bồi thường. Thời gian hiệu lực của bảo hiểm vật chất phải được duy trì liên tục và kéo dài cho đến khi khách hàng hoàn tất việc trả hết nợ vay cho ngân hàng. Tỷ lệ bồi thường là 100% giá trị xe.""",

    # 5. Quy trình giải ngân cho xe
    """Quy trình hợp tác đối với vay mua xe qua showroom: Sau khi khách hàng ký hợp đồng mua bán với showroom và thanh toán vốn tự có, ngân hàng sẽ phát hành thông báo cho vay. Số tiền giải ngân khoản vay bắt buộc phải được chuyển khoản trực tiếp vào tài khoản của bên bán (showroom hoặc người bán cá nhân đối với xe cũ) chứ không giải ngân bằng tiền mặt cho người vay.""",

    # 6. Liên quan về lãi suất (không trực tiếp trả lời LTV)
    """Lãi suất cho vay mua xe áp dụng theo gói lãi suất (Pricing scheme) do Khối Bán lẻ (RSD) công bố từng thời kỳ. Gói lãi suất thông thường bao gồm thời gian cố định lãi suất (Fixed IR period) từ 1 đến 3 năm đầu tiên, và sau đó chuyển sang áp dụng mức lãi suất thả nổi (Floating IR) theo tỷ giá Shinhan Prime Rate (SPR) cộng với một mức biên độ (spread) nhất định.""",

    # 7. Thông tin quy trình chung của mảng tín dụng bán lẻ
    """Các bước quy trình thực hiện tín dụng bán lẻ tại chi nhánh bao gồm 4 giai đoạn: Consultation (Tư vấn và thu thập hồ sơ), Loan Assessment (Thẩm định khoản vay do LOM hoặc RM thực hiện), Loan Decision (Ra quyết định phê duyệt bởi Giám đốc chi nhánh hoặc Ủy ban tín dụng), và Follow Ups (Kiểm tra và lưu trữ hồ sơ sau giải ngân do bộ phận LOO phụ trách).""",

    # 8. Nghiệp vụ hoàn toàn khác (Vay tiêu dùng)
    """Chương trình SGI Bond Loan là sản phẩm vay tiêu dùng tín chấp hợp tác với công ty bảo hiểm SGI. Hạn mức vay tối đa của sản phẩm này có thể lên tới 500 triệu đồng. Số tiền SGI bảo lãnh cho khoản vay tương đương 104% số tiền giải ngân ban đầu. Sản phẩm này không yêu cầu khách hàng phải thế chấp tài sản như bất động sản hay ô tô cá nhân.""",

    # 9. Nghiệp vụ hoàn toàn khác (Vay SOHO)
    """Đối với khách hàng đăng ký khoản vay SOHO (dành cho Hộ kinh doanh và Doanh nghiệp siêu nhỏ), điều kiện tiên quyết là cơ sở kinh doanh phải được thành lập và hoạt động liên tục ít nhất 12 tháng tính đến ngày đề nghị vay vốn. Ngân hàng sẽ yêu cầu cung cấp báo cáo thuế năm gần nhất và sao kê tài khoản ngân hàng để chứng minh doanh thu dòng tiền.""",

    # 10. Nghiệp vụ hoàn toàn khác (Bất động sản)
    """Diện tích tối thiểu của tài sản đảm bảo nhận thế chấp cho các khoản vay Home Loan là 30m2 đối với nhà đất hoặc chung cư tại khu vực Hà Nội và TP.HCM (không bao gồm phần vi phạm quy hoạch). Các tài sản không đáp ứng tiêu chuẩn này sẽ bị xếp vào loại Tài sản chuyên biệt (Specific collateral) với tỷ lệ LTV bị giảm trừ đáng kể."""
]

# TEST CASE 2: Nghiệp vụ Vay thế chấp BĐS (Home Loan) - Quy trình & Thẩm quyền
query_2 = "quy trình vay thế chấp bất động sản và thẩm quyền phê duyệt khoản vay"

documents_2 = [
    # 1. Khái quát quy trình 4 bước và người thực hiện
    """Quy trình cho vay thế chấp bất động sản tại SHBVN được chia làm 4 giai đoạn chính. Bước 1: Consultation (Tư vấn khách hàng do RM thực hiện). Bước 2: Loan Assessment (Thẩm định hồ sơ tín dụng do nhóm Assessment/DGM đảm nhận). Bước 3: Loan Decision (Ra quyết định phê duyệt bởi GM, Branch Loan Committee hoặc Hội sở). Bước 4: Follow Ups (Các công việc kiểm tra, lưu trữ sau giải ngân do Back Office/LOO xử lý).""",

    # 2. Quy định cụ thể về thẩm quyền phê duyệt (Delegation of Authority)
    """Thẩm quyền phê duyệt khoản vay thế chấp bất động sản phụ thuộc vào phân loại khoản vay, loại tài sản (Secured/Unsecured) và tỷ lệ LTV. Ví dụ: Dưới hạn mức tiêu chuẩn 50,000 USD (Unsecured) hoặc 400,000 USD (Secured), Giám đốc Chi nhánh (GM) có thể trực tiếp phê duyệt. Vượt hạn mức này, hồ sơ phải được trình lên Ban tín dụng Hội sở (CO Comm.) hoặc Hội đồng tín dụng Hội sở cấp cao (MCOC) phê duyệt.""",

    # 3. Nguyên tắc độc lập trong quy trình thẩm định
    """Trong giai đoạn Loan Assessment, nhân viên LOM/DGM hoặc thành viên được GM phân công sẽ thực hiện công việc thẩm định độc lập. Về mặt nguyên tắc tuân thủ, thành viên tham gia thẩm định tín dụng không được phép thực hiện các nhiệm vụ quản lý quan hệ khách hàng (RM), cũng như không được tham gia vào quyết định phê duyệt cuối cùng để tránh xung đột lợi ích.""",

    # 4. Thủ tục giải ngân gắn với tài sản bảo đảm
    """Trước khi khoản vay được giải ngân, khách hàng phải hoàn tất ký Hợp đồng tín dụng, Hợp đồng thế chấp và thực hiện đăng ký giao dịch bảo đảm tại cơ quan chức năng. Chi nhánh SHBVN chỉ được phép giải ngân tiền khi đã nắm giữ bản chính Giấy chứng nhận quyền sở hữu đã cập nhật nội dung thế chấp hoặc có Giấy biên nhận nộp hồ sơ hợp lệ từ cơ quan đăng ký đất đai.""",

    # 5. Thông tin sản phẩm Home Loan
    """Sản phẩm vay thế chấp bất động sản (Home Loan) bao gồm nhiều gói vay linh hoạt như: Home Mortgage Loan (Vay mua nhà/đất), Home Equity Loan (Vay tiêu dùng thế chấp BĐS), Project Housing Loan (Vay mua nhà dự án chưa có sổ), và Home Loan Refinance (Vay bù đắp chi phí hoặc nhận chuyển nhượng khoản vay từ Tổ chức tín dụng khác).""",

    # 6. Các chỉ số rủi ro RCVR và LTV (Thông tin phụ trợ)
    """Tỷ lệ định giá tài sản được công nhận (RCVR) và Tỷ lệ cho vay trên giá trị (LTV) là các yếu tố cốt lõi trong quy trình. LTV tối đa có thể lên đến 80% đối với tài sản tiêu chuẩn. Đối với Tài sản chuyên biệt (như hẻm nhỏ, diện tích hẹp), LTV và RCVR sẽ bị điều chỉnh giảm xuống mức 50% - 60% để đảm bảo biên độ an toàn cho ngân hàng.""",

    # 7. Tính toán năng lực trả nợ (DBR)
    """Khi phê duyệt hồ sơ vay, nhân viên tín dụng phải tính toán Tỷ lệ gánh nợ (DBR - Debt Burden Ratio). DBR = Tổng nghĩa vụ nợ phải trả hàng tháng / Tổng thu nhập được công nhận hàng tháng. Thu nhập có thể đến từ lương, kinh doanh, cho thuê tài sản hoặc cổ tức. Ngân hàng yêu cầu DBR duy trì ở mức <= 60% hoặc <= 70% tùy thuộc vào tổng thu nhập.""",

    # 8. Nghiệp vụ hoàn toàn khác (Vay mua ô tô)
    """Quy trình hợp đồng ba bên đối với khoản vay mua xe ô tô: Showroom ký hợp đồng trực tiếp với khách hàng và ngân hàng. Khách hàng đóng phần vốn đối ứng trước. Sau khi xe xuất hóa đơn và đăng ký biển số thành công, ngân hàng sẽ giải ngân trực tiếp số tiền vay vào tài khoản của showroom chứ không giao tiền mặt cho người mua xe.""",

    # 9. Nghiệp vụ hoàn toàn khác (Cash-Backed Loan)
    """Vay cầm cố sổ tiết kiệm (Cash-backed Loan) có quy trình đơn giản hơn vay bất động sản rất nhiều. GM có thể phê duyệt hầu hết các khoản vay này. LTV tối đa đối với VND lên tới 95%. Hệ thống tự động thiết lập hạn mức tín dụng trên Aither ngay sau khi sổ tiết kiệm hoặc chứng chỉ tiền gửi của khách hàng bị phong tỏa trên hệ thống lõi.""",

    # 10. Nghiệp vụ hoàn toàn khác (Interest Rate)
    """Shinhan Prime Rate (SPR) là tỷ giá cơ sở dùng để tính lãi suất thả nổi cho các sản phẩm vay bán lẻ. Lãi suất vay được cấu trúc gồm giai đoạn Fixed IR (cố định lãi suất) và sau đó là Floating IR. Sự điều chỉnh lãi suất được hệ thống tự động chạy theo chu kỳ 3 tháng hoặc 6 tháng dựa trên biến động của MFC (Market Funding Cost)."""
]

# TEST CASE 3: Nghiệp vụ Vay kinh doanh SOHO - Hồ sơ chứng từ tài chính
query_3 = "đối với khách hàng vay SOHO, danh mục hồ sơ tài chính để chứng minh thu nhập và mục đích vay vốn gồm"

documents_3 = [
    # 1. Chứng từ tài chính lõi cho Hộ kinh doanh
    """Danh mục chứng từ tài chính yêu cầu đối với Hộ kinh doanh vay vốn SOHO bao gồm: Sao kê tài khoản ngân hàng trong vòng 3 tháng hoặc 6 tháng gần nhất, sổ sách ghi chép bán hàng có chữ ký, dữ liệu kết xuất từ hệ thống bán hàng điện tử. Ngoài ra, cần bổ sung các chứng từ đầu ra/đầu vào như Hợp đồng mua bán, hóa đơn VAT hoặc hóa đơn điện tử hợp lệ.""",

    # 2. Chứng từ tài chính lõi cho Doanh nghiệp tư nhân
    """Đối với pháp nhân là Doanh nghiệp siêu nhỏ (Micro Enterprise) vay gói SOHO, hồ sơ chứng minh năng lực tài chính bắt buộc bao gồm: Báo cáo tài chính, báo cáo thuế năm gần nhất thể hiện hoạt động kinh doanh có lợi nhuận. Bên cạnh đó, ngân hàng yêu cầu sao kê tài khoản chính của doanh nghiệp, hợp đồng kinh tế đầu ra/đầu vào lớn và biên lai nộp thuế.""",

    # 3. Chứng từ chứng minh mục đích sử dụng vốn
    """Để giải ngân khoản vay SOHO Facility Loan (đầu tư tài sản cố định), khách hàng phải nộp hồ sơ chứng minh mục đích vay vốn. Hồ sơ này bao gồm: Hợp đồng mua bán thiết bị, máy móc; Giấy phép xây dựng hoặc Hợp đồng thi công đối với mục đích xây dựng/sửa chữa nhà xưởng, mặt bằng kinh doanh; và các hóa đơn, ủy nhiệm chi thể hiện việc thanh toán vốn tự có.""",

    # 4. Lưu ý thẩm định dòng tiền SOHO
    """Hướng dẫn thẩm định dòng tiền SOHO: Nhân viên tín dụng cần kiểm tra chéo các nguồn dữ liệu. Sao kê tài khoản phải phản ánh đúng quy mô kinh doanh. Nếu sử dụng tài khoản cá nhân của vợ/chồng để giao dịch, phải có Giấy đăng ký kết hôn để chứng minh. Cần đối chiếu hóa đơn VAT nộp cho cơ quan thuế thông qua cổng thông tin điện tử eTax.""",

    # 5. Thông tin chung về điều kiện SOHO
    """Khách hàng đăng ký SOHO Loan phải đáp ứng các điều kiện cơ bản: Độ tuổi từ 18 đến 70 tính đến thời điểm đáo hạn khoản vay. Cơ sở kinh doanh (Hộ kinh doanh/Doanh nghiệp) phải có thời gian thành lập và hoạt động thực tế tối thiểu 12 tháng. Không có nợ xấu (nợ nhóm 3 trở lên) trong vòng 5 năm gần nhất tại bất kỳ tổ chức tín dụng nào.""",

    # 6. Các loại sản phẩm SOHO
    """Phân khúc SOHO được chia thành các dòng sản phẩm: SOHO Operation Loan (cho vay bổ sung vốn lưu động ngắn hạn, hình thức vay theo món hoặc cấp hạn mức tín dụng quay vòng) và SOHO Facility Loan (Vay trung dài hạn để đầu tư nhà xưởng, tài sản cố định). Khách hàng có thể thế chấp bằng bất động sản (SOHO Prime) hoặc vay tín chấp một phần.""",

    # 7. Thông tin phụ: Điểm tín dụng mSOHO
    """Hệ thống ngân hàng áp dụng mô hình chấm điểm tín dụng mSOHO để đánh giá khách hàng. Điểm xếp hạng tín dụng (Credit Rating) sẽ quyết định tỷ lệ cấp tín dụng và hạn mức tối đa. Những khách hàng đạt hạng từ B+ trở lên mới đủ điều kiện áp dụng các chính sách cấp tín dụng thông thường hoặc ưu đãi về tỷ lệ LTV.""",

    # 8. Nghiệp vụ hoàn toàn khác (Vay tiêu dùng LEL)
    """Sản phẩm Loyal Employee Loan (LEL) dành cho cá nhân làm việc tại các Công ty đối tác của SHBVN. Khách hàng chỉ cần cung cấp Hợp đồng lao động và sao kê lương 3 tháng gần nhất để chứng minh thu nhập. Hạn mức vay tối đa lên tới 18 lần thu nhập hoặc 900 triệu VND mà không cần phải lập phương án kinh doanh.""",

    # 9. Nghiệp vụ hoàn toàn khác (Vay cầm cố sổ tiết kiệm)
    """Hồ sơ giải ngân Vay cầm cố tiền gửi cực kỳ đơn giản vì không yêu cầu thẩm định nguồn thu nhập kinh doanh. Khách hàng chỉ cần ký Đơn đề nghị vay vốn kiêm Hợp đồng tín dụng, đồng thời nộp bản gốc thẻ tiết kiệm. Ngân hàng sẽ tạo khoản vay trên hệ thống và thiết lập tỷ lệ Margin of safety để giám sát khoản vay tự động.""",

    # 10. Nghiệp vụ hoàn toàn khác (Giải ngân đa kỳ hạn - Home Loan)
    """Cơ chế điều chỉnh lãi suất cho các khoản vay thế chấp giải ngân nhiều đợt (multi-disbursement): Dù khách hàng nhận giải ngân thành 3 hay 4 đợt khác nhau, thời gian áp dụng lãi suất cố định (Fixed IR period) sẽ đồng loạt kết thúc vào cùng một ngày tính từ thời điểm giải ngân đợt đầu tiên. Sau đó toàn bộ dư nợ sẽ chuyển sang lãi suất thả nổi."""
]

# TEST CASE 4: Nghiệp vụ Vay cầm cố tiền gửi (Cash-Backed Loan) - LTV & Margin Call
query_4 = "LTV đối với Cash-Backed Loan là bao nhiêu và chính sách quản lý rủi ro khi có Margin call"

documents_4 = [
    # 1. Trả lời chính xác về LTV các loại tiền tệ
    """Tỷ lệ cho vay (LTV) của sản phẩm Vay cầm cố tiền gửi (Cash-Backed Loan) được quy định chặt chẽ. Đối với tài sản cầm cố là sổ tiết kiệm VND tại SHBVN, LTV tối đa đạt 95%. Nếu cầm cố bằng sổ USD, tỷ lệ LTV là 85%. Tương tự, đối với các ngoại tệ khác như EUR hoặc AUD, ngân hàng cũng chỉ giải ngân tối đa 85% giá trị quy đổi.""",

    # 2. Định nghĩa Margin Call và biên độ an toàn
    """Chính sách quản lý rủi ro tỷ giá (Margin Call) đối với vay cầm cố tiền gửi ngoại tệ được thực hiện tự động: Hệ thống theo dõi Tỷ lệ biên độ an toàn (Margin of Safety) = Tổng dư nợ thực tế hiện tại / Hạn mức vay tối đa quy đổi theo tỷ giá hàng ngày. Khoản vay ở trạng thái An toàn (Active) khi tỷ lệ này <= 100%.""",

    # 3. Các hành động xử lý Margin Call theo từng mức độ
    """Khi Tỷ lệ biên độ biến động: Nếu tỷ lệ nằm trong ngưỡng 100% < x <= 105% (trạng thái Theo dõi - Maintain), khách hàng có 15 ngày làm việc để bổ sung tài sản. Nếu chạm mức 105% < x <= 110% (trạng thái Cận biên - Margin Call), khách hàng chỉ có tối đa 7 ngày để nộp thêm tiền ký quỹ hoặc thanh toán bớt nợ để đưa tỷ lệ về mức an toàn.""",

    # 4. Điều khoản xử lý tài sản bảo đảm
    """Khách hàng vay cầm cố bắt buộc ký thỏa thuận cho phép SHBVN có quyền tự động trích tiền từ tài khoản thanh toán hoặc tiến hành tất toán trước hạn sổ tiết kiệm đang cầm cố để thu hồi nợ gốc và lãi. Điều khoản này được kích hoạt ngay lập tức nếu khách hàng vi phạm cam kết Margin Call hoặc không thanh toán nợ khi đến hạn đáo hạn.""",

    # 5. Điều kiện về tài sản nhận cầm cố
    """Tài sản cầm cố hợp lệ bao gồm chứng chỉ tiền gửi hoặc sổ tiết kiệm đang có hiệu lực tại Ngân hàng Shinhan Việt Nam hoặc tại 4 ngân hàng quốc doanh (Agribank, BIDV, Vietcombank, Vietinbank). Tuy nhiên, tiền gửi bằng ngoại tệ tại các ngân hàng khác không được SHBVN chấp nhận làm tài sản bảo đảm cho khoản vay Cash-backed Loan.""",

    # 6. Phương pháp tính lãi suất khoản vay
    """Lãi suất đối với vay cầm cố tiền gửi bằng VND được xác định bằng công thức: Lãi suất khoản vay = Lãi suất của sổ tiết kiệm cầm cố + Biên độ (Spread) do Khối Bán lẻ quy định. Tiền lãi phát sinh hàng tháng được tính trên số dư nợ gốc thực tế và số ngày vay. Khách hàng có thể chọn trả lãi hàng tháng và gốc cuối kỳ.""",

    # 7. Thẩm quyền phê duyệt khoản vay đơn giản
    """Nhờ bản chất rủi ro tín dụng cực kỳ thấp, thẩm quyền phê duyệt các khoản vay Cash-backed Loan chủ yếu được giao cho Giám đốc chi nhánh (GM). Khách hàng không cần phải cung cấp hồ sơ chứng minh thu nhập, báo cáo tài chính SOHO hay báo cáo thuế. Quá trình giải ngân có thể hoàn tất trong vòng 1-2 giờ làm việc.""",

    # 8. Nghiệp vụ hoàn toàn khác (Vay Mua Ô Tô)
    """Trong quy trình thu hồi nợ quá hạn của các khoản vay mua xe, chi nhánh phải gửi thông báo yêu cầu bổ sung tài sản hoặc thực hiện thu hồi tài sản thế chấp (xe ô tô) nếu khách hàng trễ hạn thanh toán. Ngân hàng sẽ làm việc với hệ thống định vị GPS gắn trên xe để hỗ trợ công tác xử lý nợ.""",

    # 9. Nghiệp vụ hoàn toàn khác (Định giá BĐS)
    """Tỷ lệ RCVR (Tỷ lệ giá trị tài sản được công nhận) đối với các loại đất trống hoặc nhà đang xây dựng (không có giấy phép) được áp dụng ở mức 50%. Đây là tài sản chuyên biệt (Specific Collateral). Việc đánh giá loại tài sản này hoàn toàn khác biệt và rủi ro hơn nhiều so với việc định giá một sổ tiết kiệm VND thanh khoản cao.""",

    # 10. Nghiệp vụ hoàn toàn khác (Quy trình SOHO)
    """Khi xét duyệt khoản vay SOHO Operation Loan, nhân viên LO phải nhập dữ liệu vào mô hình chấm điểm mSOHO. Hạn mức cấp không được vượt quá 1/2 doanh thu phòng tính toán. Không có cơ chế Margin Call tự động hàng ngày đối với bất động sản thế chấp trong gói SOHO như cơ chế áp dụng cho khoản vay ngoại tệ cầm cố sổ tiết kiệm."""
]

# TEST CASE 5: Nghiệp vụ Lãi suất (Interest Rate & Pricing Scheme)
query_5 = "Cơ chế tính lãi suất cho vay bán lẻ của ngân hàng"

documents_5 = [
    # 1. Cơ chế cấu trúc lãi suất 2 giai đoạn
    """Chính sách giá (Pricing Scheme) của Khối Bán lẻ (RRM) cấu trúc lãi suất vay thành 2 giai đoạn: Thời gian cố định lãi suất (Fixed IR period) và Thời gian thả nổi (Floating IR period). Trong giai đoạn đầu (thường 1-3 năm), lãi suất giữ nguyên không đổi. Sau đó, khoản vay tự động chuyển sang lãi suất thả nổi với công thức: Lãi suất áp dụng = Shinhan Prime Rate (SPR) + Biên độ (Spread).""",

    # 2. Khái niệm Shinhan Prime Rate (SPR)
    """Shinhan Prime Rate (SPR) là tỷ giá cơ sở trọng yếu của ngân hàng, được áp dụng làm tham chiếu cho tất cả các khoản vay bán lẻ (cá nhân và SOHO). Mức SPR này được tính toán dựa trên chi phí vốn (MFC) và định kỳ thay đổi (thường chu kỳ 3 tháng, 6 tháng, hoặc 12 tháng một lần) theo biến động của thị trường tài chính.""",

    # 3. Các thành phần tạo nên biểu lãi suất (MFC & FTP)
    """Chi phí vốn thị trường (MFC - Market Funding Cost) được xác định trên cơ sở lãi suất tiết kiệm huy động của Shinhan và các ngân hàng đối thủ. Giá vốn điều chuyển nội bộ (FTP - Funding Transfer Price) là mức giá mua bán vốn giữa Hội sở và các Chi nhánh. Cả MFC và FTP là yếu tố đầu vào để Hội đồng tín dụng quyết định mức Spread biên độ cho từng phân khúc khách hàng.""",

    # 4. Đặc thù với khoản giải ngân nhiều lần
    """Quy định đặc biệt về thời hạn Fixed IR đối với các khoản vay giải ngân làm nhiều đợt (Ví dụ: Vay xây dựng nhà ở). Ngày kết thúc thời gian hưởng lãi suất cố định của tất cả các khế ước giải ngân đợt 2, đợt 3... sẽ được đồng bộ và tính theo ngày kết thúc Fixed IR của đợt giải ngân đầu tiên (1st Disbursement).""",

    # 5. Hệ thống và quy trình công bố lãi suất
    """Biểu lãi suất bán lẻ (Retail Loan Pricing Scheme) được cập nhật liên tục và cấu hình trực tiếp trên hệ thống tín dụng lõi Aither. Các thay đổi về chính sách giá, ưu đãi giảm lãi suất cho khách hàng VIP (PWM) sẽ được Phòng Quản lý Tín dụng Bán lẻ (RSD) gửi thông báo chính thức qua cổng thông tin nội bộ SHBVN Express.""",

    # 6. Quy định về phí phạt trả nợ trước hạn liên quan đến lãi suất
    """Trong giai đoạn áp dụng lãi suất cố định (Fixed IR), nếu khách hàng thực hiện tất toán khoản vay trước hạn, hệ thống sẽ tự động tính Phí trả nợ trước hạn (Early Repayment Fee). Tỷ lệ phí phạt này sẽ giảm dần qua các năm và thường về mức 0% khi khoản vay đã chuyển sang giai đoạn áp dụng lãi suất thả nổi hoàn toàn.""",

    # 7. Lãi suất áp dụng khi nợ quá hạn
    """Nếu khách hàng không thanh toán đúng hạn gốc hoặc lãi, khoản vay sẽ rơi vào nợ quá hạn. Khi đó, mức lãi suất quá hạn (Overdue Interest Rate) sẽ bị áp dụng, thường được hệ thống tự động tính bằng 150% mức lãi suất đang áp dụng trong hạn, tính trên số dư thực tế và số ngày chậm trả của khách hàng.""",

    # 8. Nghiệp vụ hoàn toàn khác (Hồ sơ pháp lý BĐS)
    """Khi nhận thế chấp bất động sản, diện tích tài sản phải đạt tối thiểu 30m2. Nếu đất có công trình xây dựng nhưng không có Giấy phép xây dựng hợp lệ hoặc hoàn công chưa cập nhật trên Giấy chứng nhận quyền sử dụng đất, phần tài sản trên đất sẽ không được định giá (RCVR = 0%) để cấp hạn mức tín dụng.""",

    # 9. Nghiệp vụ hoàn toàn khác (Vay tiêu dùng bằng lương)
    """Sản phẩm Elite Loan áp dụng riêng cho Khách hàng là công dân Hàn Quốc hoặc cán bộ quản lý cấp cao. Hạn mức vay được duyệt dựa trên MUE (Hạn mức thu nhập). Thu nhập chuyển khoản tối thiểu yêu cầu là 20 triệu VND. Khoản vay tín chấp này có thể giải ngân thẳng vào tài khoản lương của khách hàng tại SHBVN.""",

    # 10. Nghiệp vụ hoàn toàn khác (Mua xe ô tô cũ)
    """Thời hạn cho vay tối đa đối với sản phẩm xe ô tô đã qua sử dụng phụ thuộc vào thời điểm đăng ký xe lần đầu. Ví dụ, xe đăng ký được 4 năm thì thời gian vay tối đa chỉ còn 5 năm. Hạn mức cho vay (LTV) tính trên giá trị định giá xe ô tô không vượt quá 70% đối với biển trắng cá nhân."""
]

dic = {
    query_1: documents_1,
    query_2: documents_2,
    query_3: documents_3,
    query_4: documents_4,
    query_5: documents_5,
}

# extreme evaluation store
loan_eval = []

# Test models
bge_m3 = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=bge_m3,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          #gpu_memory_utilization=0.4,
          disable_log_stats=True,
          )
evaluate_model(bge_m3, llm, dic, loan_eval)

jina_v3 = "jinaai/jina-reranker-v3"
llm = LLM(
    model=jina_v3,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=2024,
    #gpu_memory_utilization=0.6,
    disable_log_stats=True,
)
evaluate_model(jina_v3, llm, dic, loan_eval)

qwen3_06 = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_06,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.4,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_06, llm, dic, loan_eval)

qwen3_4 = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_4,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.8,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_4, llm, dic, loan_eval)

INFO 06-24 04:19:15 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}
INFO 06-24 04:19:15 [model.py:555] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-24 04:19:15 [model.py:2015] Downcasting torch.float32 to torch.float16.
INFO 06-24 04:19:15 [model.py:1680] Using max model len 1024
WARNING 06-24 04:19:15 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-24 04:19:15 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-24 04:19:15 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-24 04:19:15 [vllm.py:1089] Cudagraph is disabled under eager mod

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (14 outputs left out)

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

### 6. POS

In [ ]:
clear_gpu_memory()

In [ ]:
# ==========================================
# TEST CASE 1: HỒ SƠ VÀ THỦ TỤC ĐĂNG KÝ ĐVCNT
# ==========================================
query_1 = "quy định về khách hàng và hồ sơ, thủ tục giấy tờ cần thiết để mở mới ĐVCNT"

documents_1 = [
    # Tương đồng cao nhất: Trực tiếp trả lời các yêu cầu hồ sơ cốt lõi
    """Hồ sơ yêu cầu để mở Đơn vị chấp nhận thẻ (ĐVCNT) bao gồm các chứng từ bắt buộc: Giấy phép đăng ký kinh doanh hoặc Quyết định thành lập (Bản sao y công chứng còn thời hạn hiệu lực). Đối với các ngành nghề có điều kiện như y tế, giáo dục, cần thêm Giấy phép kinh doanh có điều kiện. Ngoài ra, bắt buộc phải có CCCD/Hộ chiếu của người đại diện pháp luật (Sao y công chứng). Đơn đăng ký kiêm khảo sát ĐVCNT và Hợp đồng Thanh toán Thẻ (mỗi loại 2 bản gốc, điền đầy đủ thông tin, có chữ ký và con dấu). Hồ sơ cũng yêu cầu cung cấp hình ảnh thực tế của cửa hàng, bao gồm bảng hiệu, địa chỉ và khu vực đặt máy POS.""",

    """Để trở thành ĐVCNT, khách hàng phải là tổ chức hoặc cá nhân có CIF và tài khoản thanh toán tại Ngân hàng Shinhan. Tổ chức này không được thuộc Danh sách ngành nghề kinh doanh không được phép triển khai theo Chính sách thanh toán thẻ của Ngân hàng, không thuộc danh sách sau theo Chính sách thanh toán thẻ: Danh sách hạn chế và cảnh báo của các TCTQT, và không thuộc Danh sách đen (Blacklist). Địa điểm kinh doanh phải cố định, nằm trong phạm vi lãnh thổ Việt Nam và được đăng ký hoạt động tại tỉnh/thành phố có hoặc lân cận chi nhánh phòng giao dịch của Ngân hàng.""",

    """Trong bộ hồ sơ thủ tục, việc kiểm tra hình ảnh khảo sát là rất quan trọng. Hình ảnh khảo sát cửa hàng và địa điểm đặt máy POS phải thể hiện rõ: Hình ảnh mặt tiền cửa hàng (chụp rõ thông tin về Biển hiệu, địa chỉ...); Hình ảnh khu vực bên trong của hàng của ĐVCNT; Hình ảnh cụ thể về mặt hàng kinh doanh hoặc dịch vụ cung cấp của ĐVCNT; và cuối cùng là Hình ảnh quầy thanh toán tiền và vị trí dự kiến đặt máy POS. Nhân viên cần chụp ảnh thực tế tại cửa hàng để đối chiếu.""",

    # Tương đồng mức trung bình: Liên quan đến quy trình nộp và xử lý hồ sơ
    """Khi tiến hành chuẩn bị hồ sơ cho ĐVCNT, đối với các trường hợp người đại diện theo pháp luật ủy quyền cho cá nhân khác ký kết, hồ sơ phải bổ sung thêm Giấy Ủy quyền (Bản gốc, ghi rõ thông tin của người được ủy quyền, phạm vi ủy quyền). Đồng thời, cần cung cấp CCCD/Hộ chiếu của người được ủy quyền (Sao y công chứng). Tài khoản thanh toán tổ chức mở tại Shinhan (HKD/DNTN/CTY) phải được đăng ký để nhận tiền thanh toán từ các giao dịch quẹt thẻ.""",

    """Quy trình triển khai dịch vụ POS bắt đầu bằng việc tiếp cận ĐVCNT, sau đó tư vấn dịch vụ và thu thập hồ sơ. Nhân viên sẽ đánh giá hồ sơ và gửi kiểm tra VMSS. Kiểm tra VMSS (bao gồm việc kiểm tra ĐVCNT có nằm trong Blacklist hay không). Nếu hồ sơ hợp lệ (Y), sẽ tiến hành ký kết hợp đồng. Tiếp theo, khai báo thông tin ĐVCNT mục tiêu, Upload hồ sơ, khai báo thông tin ĐVCNT lên hệ thống. Gửi phê duyệt hệ thống qua các cấp. Cuối cùng, khi hệ thống Approve, sẽ tiến hành lắp đặt và hướng dẫn sử dụng cho ĐVCNT.""",

    # Tương đồng mức thấp: Đề cập đến hệ thống khai báo hồ sơ (MBCS)
    """Trên hệ thống MBCS, thao tác đăng ký mới hồ sơ ĐVCNT yêu cầu Maker phải tải chứng từ tại Tab Documents. Maker phải hoàn thành thao tác đăng ký thông tin Sales ở màn hình Sales Target Merchant & Sales Activity. Sau đó nhấn vào Register Document ở góc trái, loại hình nghiệp vụ chọn Company Type, loại chứng từ chọn Document Type. Sau khi hoàn tất tải file đính kèm, nhấn nút Save, và nhấn vô Submit để tiếp tục thao tác khai báo thông tin Merchant ở các Tab Profile và Tab MDR.""",

    """Để hoàn tất quy trình hồ sơ trên hệ thống, Maker tạo yêu cầu đăng ký thông qua màn hình MC0201100 - Merchant Registration. Hệ thống sẽ lưu trữ các thông tin cơ bản về hồ sơ. Sau khi Maker nhấn Submit, yêu cầu sẽ chuyển đến Checker (Status: 40 - application checker). Checker cần vào màn hình MC0201400 để kiểm tra hồ sơ và phê duyệt. Nếu hồ sơ đầy đủ, chính xác, Checker nhấn Approve. Nếu không đầy đủ, cần chỉnh sửa, Checker gửi hoàn trả với status Return & Reason return.""",

    # Không liên quan: Các chủ đề khác (Biểu phí, Vận hành máy)
    """Biểu phí dịch vụ POS (MDR) áp dụng cho các ĐVCNT được phân loại theo nhóm ngành (MCC). Nhóm 1 bao gồm Xăng dầu, Chính phủ, Dịch vụ vận tải hành khách, Trường học, Siêu thị với mức phí ưu đãi. Nhóm 2 bao gồm Hàng điện tử, Nhà thuốc, Bệnh viện. Nhóm 3 gồm Nhà hàng và các ngành khác. Mức phí áp dụng khác nhau tùy thuộc vào loại thẻ được phát hành tại Việt Nam (Thẻ nội địa, Thẻ ghi nợ, Visa, MasterCard, JCB/UPI, Napas Debit, Napas Credit) hay thẻ phát hành tại nước ngoài.""",

    """Máy POS (Point of Sale) do Shinhan Bank cung cấp bao gồm nhiều dòng máy như POS truyền thống, Mobile POS, và Smart POS/Mini Smart POS. Các máy SmartPOS hiện đại có khả năng kết nối internet qua WiFi hoặc mạng di động 4G để thực hiện các giao dịch thanh toán. Các máy này hỗ trợ nhiều tính năng quan trọng như Thanh toán, Hủy giao dịch (Void), Tổng hợp giao dịch (Settlement), In lại hóa đơn, hỗ trợ đọc thẻ Chip, đọc thẻ từ, thanh toán Contactless (chạm) và thanh toán QR Code.""",

    """Trong quá trình hỗ trợ, chăm sóc ĐVCNT, chính sách bảo hành và sửa chữa thiết bị được quy định rõ. Lắp đặt máy POS được miễn phí. Thời gian lắp đặt tại khu vực Hà Nội và TP.HCM là 02 ngày làm việc, khu vực khác là 04 ngày làm việc. Phí sử dụng POS là 220,000 VND/POS/Tháng (bao gồm VAT). Thiết bị được bảo hành miễn phí trong vòng 12 tháng. Các trường hợp từ chối bảo hành bao gồm thiết bị rách tem, hư hỏng do rơi vỡ, nước vào, do lỗi bảo quản của ĐVCNT."""
]

# ==========================================
# TEST CASE 2: BIỂU PHÍ MDR VÀ TRÌNH GIẢM PHÍ
# ==========================================
query_2 = "chính sách biểu phí dịch vụ POS (MDR) hiện tại, quy trình dự tính P&L và thẩm quyền phê duyệt khi chi nhánh muốn đề xuất Discount MDR cho Merchant diễn ra thế nào"

documents_2 = [
    # Tương đồng cao nhất: Biểu phí, P&L, Thẩm quyền
    """Khi đề xuất và đàm phán mức phí áp dụng cho ĐVCNT, nếu mức phí đề xuất (Proposed MDR) thấp hơn mức phí tiêu chuẩn của Shinhan, chi nhánh cần thực hiện Draft trình giảm phí. Việc này đòi hỏi phải tính toán P&L (Báo cáo Lãi/Lỗ). Dự tính P&L dựa trên doanh số thanh toán thẻ dự kiến theo tháng của từng loại thẻ (Visa, Master, JCB/UPI, NAPAS Debit/Credit, Abroad Card) nhân với mức chênh lệch phí (+/- MDR Gap Percentage). Kết quả Lời/Lỗ dự kiến cuối cùng sẽ quyết định xem đề xuất giảm phí có mang lại lợi nhuận hay không.""",

    """Thẩm quyền phê duyệt trình giảm phí (Discount MDR) được phân cấp dựa trên quy mô ĐVCNT và mức phí đề xuất. Đối với ĐVCNT quy mô vừa, nhỏ và siêu nhỏ (doanh số thanh toán thẻ dưới 1 tỷ/tháng), Giám đốc chi nhánh (GM/DGM) có quyền phê duyệt mức phí Visa/MasterCard từ X >= 0.1% đến X >= 0.5% tùy loại ĐVCNT. PBD Director, Head và DGD sẽ phê duyệt các mức giảm sâu hơn. Đối với ĐVCNT quy mô lớn (doanh số từ 1 tỷ/tháng trở lên), thẩm quyền xét duyệt dựa trên số tiền giảm do chênh lệch mức phí áp dụng so với tiêu chuẩn (Y). Y <= 5 triệu/tháng do GM/DGM duyệt, tăng dần lên PBD Director, Head, DGD và cao nhất là GD cho mức Y > 30 triệu/tháng.""",

    """Biểu phí dịch vụ (MDR - Merchant Discount Rate) quy định tỷ lệ phần trăm phí mà ĐVCNT phải trả cho ngân hàng trên mỗi giao dịch. Mức phí này được phân loại theo mã MCC. Ví dụ: Nhóm 1 (Trường học, Siêu thị, Cửa hàng tiện lợi) thường có mức phí thấp, khoảng 1.21% - 1.54% cho Visa/MasterCard. Nhóm 2 (Hàng điện tử, Nhà thuốc) khoảng 1.87%. Thẻ phát hành ở nước ngoài (Issued in Abroad) thường chịu mức phí cao nhất, mặc định là 2.60% hoặc tùy thuộc vào sự đàm phán giữa ngân hàng và ĐVCNT.""",

    # Tương đồng mức trung bình: Các yếu tố liên quan đến khảo sát và đánh giá để giảm phí
    """Để thực hiện trình giảm phí hợp lý, chi nhánh phải khảo sát và đánh giá ĐVCNT dựa trên 3 tiêu chí: Quy mô (Lĩnh vực kinh doanh MCC, Số lượng cửa hàng, Phạm vi hoạt động), Doanh số (Doanh số thanh toán tổng theo tháng/năm, Doanh số phân bổ qua các dòng thẻ Visa/Master/JCB/Napas), và Mức phí (Mức phí đang áp dụng cho thẻ phát hành trong nước và ngoài nước). Việc lấy thông tin từ máy POS của đối thủ (POS Provider Competitor) cũng rất quan trọng làm cơ sở so sánh.""",

    """Trên hệ thống MBCS, quy trình cài đặt mức phí (Set line approval) đối với các trường hợp trình giảm phí yêu cầu chi nhánh thực hiện: Bảng tính P&L và Nội dung trình, gửi cho phòng Phát triển Kinh doanh Thanh toán (PBD) kiểm tra trước khi submit. Luồng phê duyệt trên hệ thống tùy thuộc thẩm quyền: Nếu thẩm quyền GM/DGM duyệt thì luồng là RM -> MANAGER -> DGD/GM (approval) -> PBD (receiving). Nếu thẩm quyền DGD duyệt thì luồng là RM -> MANAGER -> DGD/GM (approval) -> PBD (parallel) -> Head + DGD (approval).""",

    # Tương đồng mức thấp: Việc cài đặt phí trên hệ thống MBCS
    """Sau khi có phê duyệt giảm phí, Maker thực hiện khai báo thông tin mức phí trên hệ thống MBCS tại màn hình MC0201100, Tab MDR. Maker nhập các thông tin như Rate/Amount (chọn Rate), sau đó nhập tỷ lệ MDR đã được duyệt vào cột Rate Value tương ứng cho từng loại thẻ (Visa, Master, JCB, Napas Debit/Credit). Mục Settlement Days sẽ chọn mặc định D+0. Cuối cùng, hoàn tất khai báo, kiểm tra lại thông tin và nhấn nút Submit để gửi yêu cầu phê duyệt đến Checker.""",

    """Thông tin về biểu phí dịch vụ (Service Fee) cũng phải được thể hiện rõ trên Đơn đăng ký kiêm khảo sát ĐVCNT (bản cứng). Tại mục 2. Phí dịch vụ, ĐVCNT phải chọn loại thẻ chấp nhận và đồng ý với mức phí chiết khấu (MDR) cho Thẻ nội địa, Thẻ quốc tế phát hành tại Việt Nam (Thẻ tín dụng/Thẻ ghi nợ) và Thẻ quốc tế phát hành nước ngoài. Đơn đăng ký cũng cần ghi rõ phí sử dụng dịch vụ thanh toán QR (nếu có) và phí này chưa bao gồm VAT.""",

    # Không liên quan: Thao tác sử dụng máy, hồ sơ đăng ký
    """Khi máy POS bị hỏng, ĐVCNT cần liên hệ Hotline kỹ thuật hoặc Zalo OA của Shinhan để được hỗ trợ. Các lỗi thường gặp như rách tem bảo hành, hư hỏng do rơi vỡ, nước vào, do lỗi bảo quản, hoặc thiết bị bị mất/thất lạc sẽ không được bảo hành miễn phí. ĐVCNT sẽ phải chịu chi phí thay thế linh kiện như: Trục kéo giấy in, màn hình LCD, Module đọc thẻ từ, Pin sạc (Lithium battery), với mức giá dao động từ 200.000 VNĐ đến gần 3 triệu VNĐ tùy linh kiện.""",

    """Kinh doanh POS là hoạt động cho phép Doanh nghiệp chấp nhận thanh toán qua thẻ ngân hàng và các phương thức thanh toán kỹ thuật số cho các giao dịch mua bán hàng hóa, dịch vụ. Các chủ thể tham gia bao gồm: Ngân hàng thanh toán (Acquirer) xử lý giao dịch; Ngân hàng phát hành (Issuer) cấp phép giao dịch; ĐVCNT (Merchant) cung cấp hàng hóa/dịch vụ; Chủ thẻ (Cardholder) thực hiện thanh toán; và Tổ chức thẻ (Card Scheme) cung cấp mạng lưới thanh toán.""",

    """Hướng dẫn sử dụng thanh toán QR trên máy SmartPOS: 1. Nhập số tiền cần thanh toán. 2. Chọn phương thức thanh toán QR trên màn hình. 3. Mã QR sẽ được hiển thị trên màn hình POS. 4. Khách hàng sử dụng ứng dụng ngân hàng hoặc ví điện tử để quét mã QR thành công. 5. Giao dịch thành công, máy in hóa đơn. Đây là phương thức thanh toán không tiếp xúc tiện lợi, phù hợp cho các giao dịch nhỏ lẻ tại cửa hàng tiện lợi, nhà hàng."""
]

# ==========================================
# TEST CASE 3: LOẠI MÁY POS & LOẠI THẺ CHẤP NHẬN
# ==========================================
query_3 = "đang cung cấp máy POS nào, loại thẻ nào được chấp nhận thanh toán và máy POS hỗ trợ những hình thức công nghệ gì?"

documents_3 = [
    # Tương đồng cao nhất: Loại máy, loại thẻ, công nghệ thanh toán
    """Ngân hàng Shinhan hiện đang cung cấp các dòng máy POS đa dạng phục vụ nhu cầu của ĐVCNT, bao gồm: POS truyền thống, Mobile POS (nhỏ gọn, di động), và các dòng máy hiện đại như Smart POS, Mini Smart POS. Nổi bật là máy Quẹt Thẻ PAX SmartPOS A910S (Android SmartPOS Mobile) có thiết kế màn hình cảm ứng lớn, hoạt động trên hệ điều hành Android, có khả năng kết nối internet qua WiFi hoặc mạng di động 4G (cần lắp SIM 4G) để thực hiện các giao dịch thanh toán nhanh chóng.""",

    """Các thiết bị POS của Shinhan Bank chấp nhận thanh toán đa dạng các loại thẻ trên thị trường. Về Tổ chức phát hành thẻ (Card Scheme), máy hỗ trợ thanh toán thẻ VISA (Visa Gold), MasterCard (MasterCard Gold), JCB (JCB Card), UnionPay, và thẻ NAPAS nội địa. Về loại thẻ, ĐVCNT có thể nhận thanh toán từ cả Thẻ ghi nợ (Debit Card) và Thẻ tín dụng (Credit Card). Về khu vực phát hành, máy chấp nhận cả thẻ phát hành trong nước và thẻ phát hành tại nước ngoài. Các thẻ này có thể phân hạng từ Classic, Gold, Platinum đến Infinity.""",

    """Trên các dòng máy Shinhan SmartPOS hiện đại, máy hỗ trợ nhiều hình thức đọc thẻ đa dạng để nâng cao trải nghiệm khách hàng. Các hình thức bao gồm: Đọc thẻ từ (quẹt thẻ truyền thống khe hông), Đọc thẻ Chip (cắm thẻ vào khe dưới máy), và đặc biệt là công nghệ Contactless (Thanh toán không tiếp xúc - khách hàng chỉ cần chạm thẻ vào màn hình/đầu đọc của máy). Ngoài ra, máy còn hỗ trợ các phương thức thanh toán di động như Google Pay, Samsung Pay, Apple Pay và hiển thị mã QR Code (VNPAY/VietQR) để khách hàng quét thanh toán.""",

    # Tương đồng mức trung bình: Các tính năng khác trên máy POS
    """Bên cạnh chức năng thanh toán cơ bản (Sale authorization), máy POS của Shinhan Bank còn cung cấp các tính năng quan trọng khác giúp ĐVCNT quản lý giao dịch. Bao gồm: Hủy giao dịch (Void authorization - dùng để hủy giao dịch vừa thực hiện trong ngày chưa chốt bạt), Hoàn trả (Refund), In lại hóa đơn (Trans reprint), Xem lịch sử giao dịch (Trans history), và Tổng hợp giao dịch cuối ngày (Settlement). Máy SmartPOS còn có khả năng kết nối tích hợp với hầu hết các Phần mềm bán hàng (KiotViet, POS 365, iPOS...).""",

    """Để thực hiện chức năng HỦY GIAO DỊCH trên máy SmartPOS, nhân viên thu ngân cần thực hiện các bước sau: 1. Truy cập nút Menu. 2. Chọn tính năng "Hủy giao dịch". 3. Nhập mật khẩu để hủy giao dịch (mật khẩu mặc định thường là 1400). 4. Tìm kiếm giao dịch cần hủy (nhập số hóa đơn/số trace). 5. Màn hình hiển thị "BẠN CÓ MUỐN TIẾP TỤC HỦY KHÔNG?", nhấn XÁC NHẬN để hoàn tất việc hủy. Hóa đơn hủy sẽ được in ra.""",

    # Tương đồng mức thấp: Kết nối máy, sửa chữa máy
    """Để máy SmartPOS hoạt động được, cần đảm bảo kết nối mạng. Hướng dẫn kết nối WiFi: Bật máy quét thẻ, truy cập Menu Cài đặt, chọn "Kết nối WiFi", chọn mạng WiFi và nhập mật khẩu, xác nhận kết nối. Lưu ý máy chỉ hỗ trợ băng tần WiFi 2.4G. Đối với kết nối Mạng di động 4G: Đảm bảo SIM 4G đã được lắp vào máy đúng chiều, truy cập Menu Cài đặt, chọn "Kết nối mạng di động", cấu hình APN nếu cần và xác nhận kết nối để máy online báo cáo máy chủ.""",

    """Khi sử dụng thiết bị POS, ĐVCNT cần lưu ý bảo quản máy cẩn thận. Phí sử dụng POS là 220,000 VND/tháng. Trong trường hợp máy bị thất lạc, ĐVCNT sẽ phải chịu thời gian khấu hao 36 tháng và bồi thường theo giá trị khấu hao còn lại của thiết bị. Giấy in nhiệt (kích thước 57mm x 37mm) được cung cấp miễn phí theo nhu cầu thực tế của ĐVCNT, thường cấp 5 cuộn giấy in nhiệt khi lắp đặt máy POS ban đầu.""",

    # Không liên quan: MBCS System, Hồ sơ ĐVCNT
    """Để kiểm tra trạng thái của hồ sơ đăng ký mới hoặc hồ sơ yêu cầu thay đổi thông tin ĐVCNT trên hệ thống MBCS, user cần truy cập Màn hình MC0201200 - Merchant Application List. Tại mục Classify, nếu muốn tra cứu hồ sơ đăng ký mới, chọn "New Acquisition". Nếu muốn tra cứu trạng thái hồ sơ thay đổi thông tin, chọn "Information Change". Sau khi chọn Registration Date và Classify, nhấn nút Search để hiển thị danh sách hồ sơ cùng Status tương ứng (ví dụ: 1 - Approved, 7D - Approval Completed).""",

    """Hồ sơ đăng ký ĐVCNT mới bắt buộc phải có Hợp đồng thanh toán thẻ được ký kết giữa Ngân hàng Shinhan và người đại diện theo pháp luật của ĐVCNT. Hợp đồng này quy định rõ quyền và nghĩa vụ của các bên, bao gồm việc ĐVCNT đồng ý chấp nhận thanh toán thẻ, mức phí chiết khấu MDR phải trả, quy định về bảo mật thông tin thẻ, và cam kết không thu thêm phụ phí (surcharge) đối với chủ thẻ khi thanh toán bằng thẻ so với thanh toán tiền mặt.""",

    """Màn hình tra cứu thông tin của Đơn vị chấp nhận thẻ hiện hữu là MC0302000 - Merchant Information. Mục đích của màn hình này là để tra cứu tất cả thông tin đã được ghi nhận của một ĐVCNT. Cách thực hiện: Chọn Identification Code để tra cứu (có thể chọn Merchant ID (MID), Tax Identification No, hoặc Merchant name), nhập giá trị tìm kiếm vào ô tương ứng, sau đó nhấn nút Search. Màn hình sẽ hiển thị các tab Profile, Member Bank, EDC Feature, MDR chứa thông tin chi tiết."""
]

# ==========================================
# TEST CASE 4: THAO TÁC ĐĂNG KÝ ĐVCNT TRÊN MBCS (MAKER/CHECKER/APPROVER)
# ==========================================
query_4 = "quy trình và các thao tác màn hình trên hệ thống MBCS để đăng ký mới một ĐVCNT"

documents_4 = [
    # Tương đồng cao nhất: Flow tổng quan và thao tác Maker ở các màn hình đầu tiên
    """Lưu đồ đăng ký mới ĐVCNT trên hệ thống MBCS tuân theo nguyên tắc phân quyền Maker - Checker - Approver. Quy trình bắt đầu từ Maker (nhân viên Sales/RM): Tạo yêu cầu tại Sales Target Merchant (MC0201100), cập nhật Sales Activity (MC020300). Sau đó chuyển sang Merchant Registration (MC0201100) thực hiện bước 10 - DOC registration (tải chứng từ). Nhấn Submit. Tiếp tục khai báo thông tin Profile, EDC, MDR ở bước 20 - Data Entry. Nhấn Submit gửi Checker. Checker thực hiện 40 - Application check. Nếu Approve, hồ sơ sang Approver (Analyst) thực hiện 50 - Merchant Analyst, 61 - Member Bank Approval 1 và 62 - Member Bank Approval 2.""",

    """Thao tác của Maker bắt đầu tại màn hình MC0201100 - Sales Target Merchant. Tại tab này, mục Acquisition channel chọn trị 7-Member Bank. Nhập Merchant ID/Name và Address (Địa chỉ lắp đặt cửa hàng). Nhập thông tin người giới thiệu (Recommender Name, ID, Phone) nếu có. Chuyển sang màn hình MC0200300 - Sales Activity để cập nhật tình trạng tiếp cận khách hàng. Nhấn nút Add, nhập Visit Date/Time (thời gian RM đến cửa hàng) và Visit Result (chọn kết quả đánh giá). Sau khi hoàn tất, nhấn nút Save trên màn hình để ghi nhận thông tin đã cập nhật.""",

    """Quay lại màn hình MC0201100, Maker thực hiện Upload hồ sơ chứng từ tại Tab Documents. Maker chọn Company Type, Document Type tương ứng, upload file pdf/ảnh và nhấn Save. Tiếp đó, Maker khai báo thông tin ĐVCNT tại Tab Profile. Lưu ý mục Sales Manager - Branch chọn đúng tên CN/PGD quản lý; Postal address Other #1,2,3 nhập đúng địa chỉ lắp đặt; nhập MCC; Phone No nhập số điện thoại liên hệ trên biên lai; Business Environment chọn SHOPS; Posting method chọn Auto Posting. Nhấn Save. Chuyển sang tab Member Bank, nhập Account No (số tài khoản ĐVCNT), Owner, Brand Info. Nhấn Save.""",

    # Tương đồng mức trung bình: Khai báo EDC, MDR và luồng Checker, Approver
    """Trên màn hình MC0201100, Maker tiếp tục khai báo thông tin máy POS tại Tab EDC Feature - Terminal. Nhập Terminal Type (chọn loại máy, ví dụ WIFI), Number of EDC (số lượng máy thiết bị cần lắp), Merchant Name (Tên ĐVCNT). Ở mục EDC Feature choice, tích chọn 3 dòng đầu tiên (Domestic: Credit/Debit Card, International: All Card). Chuyển sang Tab MDR, nhập mức phí chiết khấu bằng cách chọn Rate, nhập Rate Value theo phê duyệt P&L, Settlement Days chọn D+0. Nhấn Save, sau đó nhấn vô nút Submit để chuyển hồ sơ đến Checker (Trạng thái chuyển sang 40 - application checker).""",

    """Checker sẽ kiểm tra hồ sơ và phê duyệt tại màn hình MC0201400 - Application Registration Approval. Checker kiểm tra thông tin nhập liệu của Maker bằng cách chọn Application ở góc trái. Nếu hồ sơ đầy đủ, chính xác, Checker đồng ý phê duyệt bằng cách nhấn Approve, hệ thống pop-up Confirm, chọn Status: Approved và Submit. Nếu thông tin không đầy đủ, cần chỉnh sửa, Checker từ chối và gửi hoàn trả cho Maker: Nhấn Approve, chọn Status: Returned & nhập Reason return. Hồ sơ sẽ quay về Maker để sửa đổi.""",

    """Sau khi Checker phê duyệt, hồ sơ chuyển đến cấp Approver. Maker (hoặc bộ phận xử lý) thực hiện đánh giá và phân tích hồ sơ trên màn hình MC0201600 - Application Registration Analyst. Một số lưu ý: Tích tất cả các mục ở Analyst result. Hoàn tất thông tin yêu cầu của Phone Verification. Nhấn Save để ghi nhận. Nhấn Check của Member Bank Acceptance trả kết quả (Normal/Exception). Nếu Maker phát hiện sai sót, có thể Return về lại bước 20 - Data entry để chỉnh sửa. Nếu đúng, nhấn Submit gửi yêu cầu phê duyệt cuối cùng đến GM/DGM.""",

    # Tương đồng mức thấp: Phê duyệt cuối cùng và các màn hình tra cứu
    """Bước phê duyệt cuối cùng được thực hiện bởi Approver (GM/DGM) tại màn hình MC0201400. Màn hình này xử lý Status: 61 - Member Bank Approval 1 & 62 - Member Bank Approval 2. Approver kiểm tra thông tin nhập liệu của Maker (nhấn vô Application) và thông tin của Analyst (nhấn vô Analyst). Nếu đồng ý, nhấn nút Approve -> Chọn status: Approved. Nếu từ chối phê duyệt, hủy hồ sơ, chọn status: Rejected. Nếu cần sửa, chọn Returned & Reason return.""",

    """Để truy cập vào hệ thống MBCS, người dùng cần có tài khoản được cấp quyền. Hướng dẫn đăng nhập: Mở ứng dụng MBCS, nhập UserID (ví dụ: 0087900025) và Password vào cửa sổ Member Bank. Sau khi đăng nhập thành công, giao diện Tree-menu hiển thị các chức năng. Để tra cứu một màn hình cụ thể, có hai cách: Cách 1 là nhập trực tiếp mã màn hình (ví dụ: MC0201100) vào ô tìm kiếm và nhấn ENTER. Cách 2 là nhấn nhóm màn hình ở Tree-menu (như Merchant -> Merchant Application) rồi chọn tên màn hình cần truy cập.""",

    # Không liên quan: Các chủ đề P&L, Sử dụng POS
    """Trong quá trình triển khai máy POS mới, tính năng Thống kê giao dịch trên máy POS giúp ĐVCNT kiểm tra nhanh doanh số. Thao tác: Truy cập menu Cài đặt, chọn Thống kê giao dịch. Giao diện hiển thị, người dùng có thể chọn khung giờ và chọn ngày. Thời gian tra cứu cho phép trong vòng 30 ngày kể từ ngày hiện tại. Nhấn XÁC NHẬN để máy in ra biên lai thống kê trong khoảng thời gian tra cứu, hiển thị tổng số tiền thanh toán, tổng hoàn tiền theo từng loại thẻ.""",

    """Chương trình Marketing dành cho Micro Merchant Campaign (POSPay) cung cấp giải pháp quản lý bán hàng thiết kế đặc biệt dành cho doanh nghiệp siêu nhỏ và nhỏ trong lĩnh vực F&B. Tính năng nổi bật là thanh toán - xuất hóa đơn điện tử ngay trên thiết bị Shinhan POS, giúp tuân thủ quy định pháp luật. Áp dụng mức phí ưu đãi 1.1% cho thẻ Visa/Mastercard trong nước nếu doanh số thẻ không vượt quá 1,9 tỷ/năm. Miễn phí hệ thống năm đầu tiên, từ năm thứ 2 thu 80,000 VND/tháng."""
]

# ==========================================
# TEST CASE 5: THAO TÁC THAY ĐỔI THÔNG TIN ĐVCNT TRÊN MBCS
# ==========================================
query_5 = "Khi ĐVCNT hiện hữu có nhu cầu thay đổi thông tin, thao tác trên màn hình MBCS được thực hiện ra sao"

documents_5 = [
    # Tương đồng cao nhất: Flow tổng quan và màn hình MC0302100
    """Quy trình thay đổi thông tin ĐVCNT (Information Change) trên hệ thống MBCS bắt đầu bằng việc Maker tạo yêu cầu thay đổi tại màn hình MC0302100 - Merchant Information Change. Maker tải hồ sơ, chứng từ yêu cầu thay đổi (như Đơn đề nghị, Phụ lục hợp đồng) tại Tab Documents. Sau đó Maker thực hiện nhập thông tin mới trên các Tab tương ứng (Profile, Member Bank, EDC/Feature, MDR). Nhấn nút Save. Gửi yêu cầu qua Checker thực hiện Application check. Cuối cùng chuyển lên Approver duyệt Information Change Approval tại MC0302300.""",

    """Tại màn hình MC0302100 - Merchant Information Change, Maker nhập Merchant ID (MID) của ĐVCNT cần thay đổi và nhấn Search. Màn hình sẽ hiển thị thông tin hiện tại. Ở Tab Documents, Maker chọn Document Type tương ứng với nội dung thay đổi (ví dụ: Business Registration Certificate, Other Documents), upload File ID/File Name và nhấn Save. Một số lưu ý quan trọng trước khi Maker gửi yêu cầu phê duyệt: Cần xác định chính xác thông tin cần thay đổi nằm ở Tab nào và bắt buộc phải TÍCH CHỌN CHECKBOX tương ứng trên Tab Documents.""",

    """Sau khi cập nhật xong chứng từ ở Tab Documents, Maker chuyển sang các Tab thông tin (Profile, Member Bank, MDR...). Thực hiện thay đổi thông tin, phải ĐÍNH KÈM CHỨNG TỪ liên quan đến yêu cầu thực hiện trên Tab Documents. Trong quá trình thao tác, Maker có thể truy cập màn hình MC0201200 để tra cứu trạng thái yêu cầu. Sau khi đã nhập liệu và kiểm tra, Maker nhấn Save, sau đó nhấn nút Submit. Hệ thống hiển thị Confirm, Maker chọn Submit và OK. Yêu cầu sẽ được chuyển đến Checker (trạng thái 40 - application checker).""",

    # Tương đồng mức trung bình: Thao tác của Checker và Approver cho luồng Change
    """Đối với luồng thay đổi thông tin, Checker sẽ phê duyệt yêu cầu tại màn hình MC0302300 - Information Change Approval. Checker cần kiểm tra thông tin nhập liệu của Maker bằng cách nhấn vào mục Detail-New. Nếu các thông tin cập nhật hợp lệ và khớp với chứng từ đính kèm, Checker đồng ý phê duyệt: Nhấn nút Approve -> Chọn Status: Submit. Nếu thông tin không đầy đủ hoặc sai sót, Checker gửi hoàn trả: Nhấn Approve -> Chọn Status: Return & Reason return. Yêu cầu thay đổi tài khoản nhận tiền phải cực kỳ cẩn trọng khi duyệt.""",

    """Đặc biệt lưu ý đối với trường hợp yêu cầu thay đổi thông tin là Thu hồi thiết bị (Withdrawal request), Checker ở cấp chi nhánh/phòng giao dịch là người phê duyệt cuối cùng. Đối với các trường hợp thay đổi thông tin khác (thay đổi địa chỉ, tên, tài khoản, biểu phí MDR, MCC), yêu cầu sẽ được chuyển tiếp từ Checker lên cấp Approver (GM/DGM) là người phê duyệt cuối cùng trên hệ thống.""",

    """Approver (GM/DGM) thực hiện phê duyệt cuối cùng yêu cầu thay đổi thông tin ĐVCNT tại màn hình MC0302300. Ở bước này, Approver kiểm tra hồ sơ ở 2 cột (Approved Department & Approved Department 1). Nếu Cột Approved Department 1 trống: Approver chỉ duyệt 61-Member Bank Approval 1. Nếu Cột Approved Department 1 có: Approver phải duyệt thêm 62-Member Bank Approval 2. Approver nhấn vào Detail để xem nội dung thay đổi, chọn Approve và Status: Submit/Approved để hoàn tất quy trình cập nhật thông tin trên MBCS.""",

    # Tương đồng mức thấp: Tra cứu lịch sử thay đổi
    """Để tra cứu lại lịch sử các lần thay đổi thông tin của một ĐVCNT, người dùng sử dụng màn hình MC0302500 - Merchant Information Change History. Tại đây, nhập MID của Merchant và nhấn Search. Màn hình sẽ hiển thị danh sách các lần yêu cầu thay đổi trong khoảng thời gian đã chọn. Tại Detail List ở bên dưới, hệ thống cung cấp thông tin chi tiết: Old Value (Giá trị cũ trước khi thay đổi), New Value (Giá trị mới sau khi thực hiện thay đổi thông tin), và Change Type (thông tin được thay đổi, ví dụ: MCC, Merchant Name).""",

    """Việc thay đổi thông tin từ phía ĐVCNT phải được lập thành văn bản. ĐVCNT phải điền vào "ĐƠN YÊU CẦU DÀNH CHO ĐƠN VỊ CHẤP NHẬN THẺ" (REQUEST FOR MERCHANT). Trong đơn này, tại mục II. YÊU CẦU THAY ĐỔI/BỔ SUNG THÔNG TIN, khách hàng sẽ tích chọn các nội dung như: Thay đổi địa chỉ đăng ký kinh doanh, địa chỉ email, số điện thoại, hoặc Thông tin tài khoản thụ hưởng mới (kèm số tài khoản mới). Đại diện hợp pháp của ĐVCNT phải ký tên, đóng dấu hợp lệ trước khi nộp cho Ngân hàng.""",

    # Không liên quan: Hoàn trả giao dịch, Cài đặt hệ thống
    """Trong trường hợp có tranh chấp hoặc nhầm lẫn, ĐVCNT có thể yêu cầu Hoàn trả giao dịch (Refund transaction). ĐVCNT điền Đơn đề nghị, mục "Yêu cầu hoàn trả/thu giao dịch", cung cấp Số thẻ, Ngày giao dịch, Số tiền giao dịch, Mã chuẩn chi (Approval code), Số tiền hoàn trả, và Lý do hoàn trả. Chứng từ kèm theo bắt buộc bao gồm: Hóa đơn giao dịch (POS bill receipt), Hóa đơn bán hàng, và các chứng từ khác như CCCD/PP, Hợp đồng đặt cọc để ngân hàng đối soát và xử lý.""",

    """Tính năng đổi ngôn ngữ hiển thị trên máy POS: Trên máy quẹt thẻ PAX SmartPOS, người dùng có thể dễ dàng chuyển đổi ngôn ngữ. Thao tác: 1. Truy cập menu CÀI ĐẶT, chọn CẤU HÌNH. 2. Kéo xuống tìm và chọn mục NGÔN NGỮ (Language). 3. Lựa chọn ngôn ngữ mong muốn trong danh sách pop-up. Hiện tại, máy POS của Shinhan hỗ trợ các ngôn ngữ bao gồm: Tiếng Việt, English (Tiếng Anh), và Tiếng Hàn (한국어). Việc đổi ngôn ngữ giúp thu ngân người nước ngoài dễ dàng thao tác thanh toán hơn."""
]

dic = {
    query_1: documents_1,
    query_2: documents_2,
    query_3: documents_3,
    query_4: documents_4,
    query_5: documents_5,
}

# extreme evaluation store
pos_eval = []

# Test models
bge_m3 = "BAAI/bge-reranker-v2-m3"
llm = LLM(model=bge_m3,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          #gpu_memory_utilization=0.4,
          disable_log_stats=True,
          )
evaluate_model(bge_m3, llm, dic, pos_eval)

jina_v3 = "jinaai/jina-reranker-v3"
llm = LLM(
    model=jina_v3,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=2024,
    #gpu_memory_utilization=0.6,
    disable_log_stats=True,
)
evaluate_model(jina_v3, llm, dic, pos_eval)

qwen3_06 = "Qwen/Qwen3-Reranker-0.6B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_06,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.4,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_06, llm, dic, pos_eval)

qwen3_4 = "Qwen/Qwen3-Reranker-4B"
llm = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_4,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2024,
        #gpu_memory_utilization=0.8,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )
evaluate_model(qwen3_4, llm, dic, pos_eval)

INFO 06-24 04:30:26 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 1024, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'BAAI/bge-reranker-v2-m3'}
INFO 06-24 04:30:27 [model.py:555] Resolved architecture: XLMRobertaForSequenceClassification
INFO 06-24 04:30:27 [model.py:2015] Downcasting torch.float32 to torch.float16.
INFO 06-24 04:30:27 [model.py:1680] Using max model len 1024
WARNING 06-24 04:30:27 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-24 04:30:27 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-24 04:30:27 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-24 04:30:27 [vllm.py:1089] Cudagraph is disabled under eager mod

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

... (14 outputs left out)

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

###7. Funding

In [ ]:
# Test Case 1: Quản lý Thanh khoản và Liên ngân hàng của Khối Nguồn vốn (Treasury ALM & Funding)
query_1 = "how Treasiry Department manage Asset and Liability Management, inter-bank lending, and fund liquidity"
documents_1 = [
    "The Treasury department is responsible for ALM and Funding by setting up, announcing, and managing the Fund Transfer Price to Business Units. [1] It manages Asset & Liability, gives direction, and plans to assure the prudential ratio compiled with SBV and Mother Bank. [1] Treasury cooperates with Funding, cross-checks all ratios, collects proposals for ALCO meetings, and supports Business Units in cross-checking FTP. [1] Additionally, they register and manage interest rates like MFC, SPR, and TermSolf in the system. [1] For Fund Management, the bank controls funds within the Nostro and Vostro credit line. [1] This ensures the bank liquidity and fund demand for all system operations are met, alongside checking the updated interest rate at Nostro. [1]",
    "In the Inter-Bank money market, Treasury operations maintain and develop fund relationships with inter-bank partners. [1] The team actively looks for new good partners to take or offer more credit lines for money market transactions, cooperating closely with the FI team. [1] They control Liquidity, Loan-to-Deposit Ratio (LDR), and must strictly comply with State Bank regulations. [1] Furthermore, they control the Compulsory reserve with the State Bank in both VND and foreign currencies. [1] Negotiating Master agreements and contracts with inter-banks on money market transactions is also a key daily task. [1] Controlling fee concerning with deal tracker and daily money market credit line complies with SHB's guidelines. [1]",
    "Regarding Nostro and Fund Management, there are strict cut-off times and processes for outward remittances. [2] Branches are requested to inform the Treasury Dept when doing outward remittance as follows: VND amounts greater than or equal to 20 billion, USD greater than or equal to 200,000, and KRW greater than or equal to 50 million. [2] For other currencies like JPY, CHF, NZD, AUD, SGD, EUR, GBP, CAD, HKD, notification is required at any amount. [2] The funding notification must include Branch Name, Nostro account, Amount, CIF No, and Source of Fund. [2] Transactions arising after cut-off time will be processed case by case, and Treasury must be informed immediately if a payment order is cancelled by the customer. [2]",
    "Fund transfer pricing is essential for establishing the internal pricing mechanism for transferring funds between different Head Office and Business Units within a financial institution. [3] This practice is vital for accurately assessing the profitability of each unit and effectively managing financial resources across the organization. [3] The primary aim of Fund Transfer Pricing is to motivate business units to manage funds efficiently, directly impacting profitability. [4] It also serves as a crucial tool for performance evaluation, helping in assessing departmental performance for better resource allocation. [4] A well-structured FTP mechanism encourages accountability for funding costs and promotes efficient capital allocation. [5]",
    "The Treasury Securities Investment Team handles security investment in Government bonds, Government Guaranteed bonds, SBV's bills, and FI bonds. [6] They are responsible for fixed income evaluation, implementing bond restructure plans, and controlling bond limits as set up by the Mother bank and bond risk ratios. [6] They update fixed income policy guidelines, conduct market research, provide fixed income updates and forecasts for daily, weekly, and monthly reports. [6] Developing fixed income products and managing bond price calculations are key to maximizing investment income. [6] They also manage the credit rating and CE limit annually while updating legal documents and financial information. [6]",
    "Understanding the various elements that affect funding components is crucial for any financial institution. [7] These factors play a vital role in cost management and profitability strategies. [7] Customer Time Deposits impact the cost of funds significantly and take the main portion in the structure of funding. [7] Money Market Borrowing involves mobilizing funds from the interbank market. [7] The Monthly Funding Strategy determines the proportion of each funding component. [7] Market Conditions, Banking Competition, and Central Bank Policies also influence overall interest rates, subsequently affecting the MFC, SPR, and FTP. [8]",
    "The Standard Settlement Instruction (SSI) is critical for managing operations across Domestic and Overseas Banks. [9, 10] For domestic banks like State Bank of Vietnam, Vietcombank, BIDV, and Vietinbank, accounts are held in VND, USD, EUR, JPY, GBP, and other currencies. [9] For instance, Vietcombank HO maintains multiple accounts for varying currencies, linked to specific SWIFT codes like BFTVVNVX. [9] Overseas Banks such as Shinhan Bank Seoul, JP Morgan Chase New York, and Sumitomo Mitsui Banking Corp also have designated NOSTRO accounts and standard settlement instructions for cross-border transactions. [10] Managing these accounts ensures smooth liquidity flows across borders. [10]",
    "The interbank overnight interest rate experienced a sharp surge, peaking at 17.25% per annum on February 3, 2026, before dropping to 2.8% at the end of February. [11] In response to this volatility, the State Bank of Vietnam (SBV) proactively injected liquidity into the system by purchasing valuable papers (OMO) totaling approximately 161 trillion VND across two sessions on February 2nd and 3rd, 2026. [11] This was done alongside FX forward swap operations to ensure the OMO rate is kept within the 4.5% target. [11] This direct market intervention helped stabilize the rapid spikes in short-term borrowing costs. [11]",
    "Inflation, interest rates, and exchange rates are interconnected elements that influence the economic landscape. [12] When one of these variables shifts, it triggers a cascade of effects on the others, leading to significant economic variations. [12] Central banks increase interest rates during inflation spikes to control the money supply. [13] Understanding their relationships is crucial for policymakers and financial analysts to navigate the complexities of economic dynamics. [12] Changes in inflation, interest rates, and exchange rates trigger chain reactions, requiring ongoing monitoring to effectively respond to economic changes. [14] Policy implications are essential for effective monetary strategies that promote stability. [14]",
    "Vietnam's economy faces macroeconomic challenges, such as oil price escalation to $120/barrel, which could trigger significant imported inflation, affecting consumer price stability and manufacturing costs. [15] As a net oil importer, Vietnam faces a widening current account deficit with sustained high energy prices impacting external balances. [15] Energy-intensive industries, including textiles, electronics, and steel production, face margin compression from elevated input costs. [15] Higher fuel and transportation costs will reduce household disposable income, potentially dampening domestic consumption growth. [15] Prolonged regional instability may redirect FDI flows away from Southeast Asia toward perceived safer investment destinations. [15]"
]

# Test Case 2: Cơ chế định giá điều chuyển vốn nội bộ (FTP Framework & Margin)
query_2 = "framework for Fund Transfer Pricing, how to calculate internal pricing margins for deposit and loan products"
documents_2 = [
    "Fund transfer pricing is essential for establishing the internal pricing mechanism for transferring funds between different Head Office and Business Units within a financial institution. [3] The primary aim of Fund Transfer Pricing is to motivate business units to manage funds efficiently, impacting profitability. [4] It also serves as a crucial tool for performance evaluation, helping in assessing departmental performance for better resource allocation. [4] The method is used to set internal prices for funds transferred within banks and institutions, promoting transparency and regulatory compliance. [4, 5] It encourages accountability for funding costs and promotes efficient capital allocation. [5]",
    "The components of Fund Transfer Pricing include the Average Time Deposit Rate, Yield of Certificate of Deposit, and Interbank VNIBOR. [16] The Average Time Deposit Rate reflects the interest rate applicable to time deposits opened within 30 days, serving as a benchmark for determining the cost of funds. [16] The yield from certificates of deposit issued by banks to one another, collected from VBMA, represents an important component in assessing the cost-effectiveness of funding for terms from 3m above. [16] The Interbank Vietnam Interbank Offered Rate, sourced from Refinitiv LSEG, indicates the average rate at which banks lend to one another. [16]",
    "For Deposit FTP, the system calculates the Basic Announced Rate applied for counter transactions and the Final FTP based on the term and deposit type. [17] The Final FTP for favor cases is the same as the Final FTP for normal cases, and thus the margin rate will be reduced accordingly compared to normal cases. [17] For example, a 1-month VND Time Deposit might have a Basic Announced Rate of 1.80%, a Max rate under GM/DGM approval of 2.30%, and a Final FTP of 3.80%, resulting in a Margin Rate of 2.00%. [17] Deposit FTP in the Aither System includes product FTP and stable funding premiums. [18]",
    "Loan FTP calculations differ based on the product type, incorporating the Market Base Rate, ALM spread, and Liquidity Premium. [19, 20] For a Lending Consumer loan with a fixed rate over a 5-year term, the Loan Internal Interest Rate Inquiry combines these factors. [19] For instance, a Market Base Rate of 4.60% added to an ALM spread of 1.6494% yields a final FTP of 6.2494%. [19] Conversely, a Lending Special operation loan over a 3-month term with a fixed rate might have a Market Base Rate of 2.10% and an ALM spread of 2.25%, resulting in a final FTP of 3.35% due to a negative product adjustment. [20]",
    "When processing transactions, checking the Fund Transfer Price is a mandatory step before offering rates to customers. [21] RM, CRM, and Operation Staff must strictly check the daily Fund Transfer Price before proceeding. [21] They must always check the rate displayed on screen 9627 for accurate information. [21] For odd terms, staff must input data into screen 9626 to run specific simulations and calculate the correct FTP. [21] Accurate data entry into the system is necessary to derive the correct base rate and adjust for any specific funding premiums or liquidity costs. [21]",
    "The Market Funding Cost (MFC) is defined as a kind of floating loan rate, mainly applied for Corporate loans and SOHO loans. [22] It is announced by the Treasury Department every at the beginning of the month. [22] The Floating Rate Shinhan Prime Rate (SPR) serves as a variable interest rate particularly designed for retail loans. [23] This rate is revised on a monthly basis by the Treasury Department and is calculated based on the average retail deposit rates across various banks, along with interbank interest rates. [23]",
    "Several external and internal factors influence the Fund Transfer Price. [8] Central Bank Policies and changes in monetary policy by the central bank influence overall interest rates, consequently affecting the MFC, SPR, and FTP. [8] Banking Competition also plays a role, as the rates offered by competing banks impact the Fund Transfer Price while banks strive to attract more customers. [8] Furthermore, Market Conditions and economic indicators such as inflation, exchange rate, and economic growth directly affect the floating rate and its underlying components. [8]",
    "Mobilization from Market 1, which includes deposits from customers, operates under strict regulations relating to interest rates, specifically Circular 46/2024, 47/2024, and 48/2024. [24] The State Bank of Vietnam dictates the ceiling rate applied for the kind of interest paid at maturity. [24] For example, VND demand and terms under 1 month face a 0.5% ceiling, while terms from 1 month to under 6 months have a 4.75% ceiling. [24] USD deposits are capped at 0% for all terms. [24] These regulatory caps directly influence how banks calculate the FTP for short-term deposit products. [24]",
    "The interbank interest rate dynamics play a crucial role in market liquidity and impact funding costs. [25] The interbank interest rate is essential for maintaining liquidity across the banking sector. [25] Rate fluctuations due to central bank policies and economic conditions create volatility. [25] Higher interbank rates increase funding costs, which subsequently affect businesses and consumers by driving up the overall cost of borrowing. [25] This interbank volatility is factored into the daily VNIBOR updates and overall FTP calculations. [25]",
    "In the capital markets, corporate bond issuing values have shown varying trends over the years. [26] Public issuance means the bond is issued through mass media, whereas private bond issuance refers to private placement to professional investors. [26] In early 2026, corporate bonds issued by tenors showed a large concentration in real estate and banking sectors. [26] The corporate bond redeeming value chart illustrates the redemption schedules facing the market, which impacts broader systemic liquidity but does not directly alter the daily retail deposit FTP margins. [26]"
]

# Test Case 3: Can thiệp của Ngân hàng Nhà nước vào Tỷ giá và Thị trường Liên ngân hàng (SBV Interventions)
query_3 = "Ngân hàng Nhà nước điều hành tỷ giá VND/USD và thị trường liên ngân hàng như thế nào để ổn định thanh khoản và kiểm soát lạm phát"
documents_3 = [
    "SBV is using flexible policy to control the foreign exchange market and interest rates. [27] The exchange rate VND/USD is float but under managed within +/- 5%, and the SBV can intervene by buying or selling foreign currencies to stabilize the exchange rate when necessary. [27] To reduce exchange rate pressure and contain imported inflation, the SBV may raise policy rates or withdraw liquidity, which leads to an interbank interest rates increase. [27] The State Bank of Vietnam may have to raise interest rates to adjust managing interest rates and money supply to control inflation and support economic growth. [27]",
    "The interbank overnight interest rate experienced a sharp surge, peaking at 17.25% per annum on February 3, 2026, before dropping to 2.8% at the end of February. [11] In response to this liquidity crunch, the State Bank of Vietnam (SBV) proactively injected liquidity into the system by purchasing valuable papers (OMO) totaling approximately 161 trillion VND across two sessions on February 2nd and 3rd, 2026. [11] This operation was conducted alongside FX forward swap operations. [11] Throughout this period, the OMO rate was successfully kept within the 4.5% target, stabilizing short-term borrowing costs. [11]",
    "In the foreign exchange market, there is a noticeable Market Divergence between official and unofficial rates. [28] The spread between the official bank rate and the free market rate, currently around 600 VND, indicates high local demand and potential speculation. [28] With the free market rate approaching 27,000 VND, this serves as a critical resistance level that may trigger further intervention from the State Bank of Vietnam (SBV). [28] As of March 6-7, 2026, the USD/VND exchange rate at commercial banks remains anchored at elevated levels, quoted between 25,962 and 26,031 VND. [28]",
    "The geopolitical landscape, specifically the US-Iran conflict, exerts upward pressure on the exchange rate, causing VND/USD to increase. [29] A stronger USD means exports decline, as the supply of USD from trade activities decreases. [29] Concurrently, exporting enterprises still need to purchase USD to repay debts or repatriate profits, leading to demand exceeding supply and the depreciation of the VND. [29] Speculation and capital outflows occur as heightened trade risks shift investors to safe-haven assets, resulting in stronger USD outflows from Vietnam. [29] This creates a risk of a balance of payments deficit and downward pressure on the VND. [29]",
    "The interaction with exchange rates is deeply tied to inflation and interest rate influences. [30] Increased interest rates often attract foreign investment, boosting demand for the currency and causing it to appreciate, thus countering inflation pressures. [30] Conversely, higher inflation can weaken a currency's value as domestic goods become more expensive, leading to reduced foreign demand and a potential decline in exchange rates. [30] Market expectations about inflation and interest rate changes can lead to currency speculation, further influencing exchange rate movements based on anticipated economic shifts. [30]",
    "There is a positive relationship between inflation and interest rates. [13] Inflation generally leads to higher interest rates, affecting savings and borrowing behavior. [13] Central banks implement tight monetary policy, increasing interest rates during inflation spikes to control the money supply. [13] Investors look for nominal interest rate increases to maintain positive real returns amid rising inflation. [13] The interplay of inflation, interest rates, and exchange rates are interconnected elements that influence the economic landscape; a shift in one triggers a cascade of effects on the others. [12]",
    "According to the Vietnam Macro Economic Brief, Vietnam's GDP in 2025 is estimated to reach approximately $514 billion, marking an impressive growth rate. [31] However, Vietnam's Consumer Price Index (CPI) in February 2026 rose by 1.14% month-on-month, primarily driven by surging demand for consumer goods and services during the Lunar New Year holiday. [31] Compared to February 2025, the CPI increased by 3.35%, bringing the average growth for the first two months of the year to 2.91%. [31] This inflation pressure requires active management from the central bank. [31]",
    "Vietnam's economic implications are challenged by global macroeconomic shocks. [15] Oil price escalation to $120/barrel could trigger significant imported inflation, affecting Vietnam's consumer price stability and manufacturing costs. [15] As a net oil importer, Vietnam faces a widening current account deficit with sustained high energy prices impacting external balances. [15] Energy-intensive industries face margin compression from elevated input costs. [15] These macroeconomic challenges limit the flexibility of the SBV, as higher fuel and transportation costs reduce household disposable income and potentially dampen domestic consumption growth. [15]",
    "In Market 1, deposit interest rate dynamics show systemic liquidity disparities. [32] In Mar 2026, the average quoted deposit interest rate of term 12m of big 4 banks stood at 5.2%, while big local banks offered 5.4% and smaller banks offered 5.9%. [32] However, a gap between credit growth at 19.01% and deposit growth estimated at 15% has led to a shortage of funds. [32] This pushes the deposit rate favor to a very high level for 6m-12m terms at 7.0% - 8.0%, especially as local small banks without strong CASA may offer above 8.0%. [32]",
    "The primary market for G-Bonds sees issues firstly sold through auctions to investors, while the secondary market is where G-bonds are bought and sold among investors. [33] The Government bond issuing value and issuing value by tenor are closely monitored metrics sourced from VBMA. [33] While G-bond yields affect the overall risk-free rate environment, the secondary market trading of long-term tenors (like 10Y, 15Y, 20Y) has a limited direct and immediate effect on the SBV's daily FX intervention tactics and overnight interbank liquidity management. [33]"
]

# Test Case 4: Lãi suất thả nổi (Floating Rates SPR/MFC & Loan Application)
query_4 = "How are floating interest rates defined, and how are they applied to variable-rate loans?"
documents_4 = [
    "The Floating Rate Shinhan Prime Rate (SPR) serves as a variable interest rate particularly designed for retail loans. [23] This rate is revised on a monthly basis by the Treasury Department and is calculated based on the average retail deposit rates across various banks, along with interbank interest rates. [23] Similarly, the Market Funding Cost (MFC) is a kind of floating loan rate, mainly applied for Corporate loans and SOHO loans. [22] Like the SPR, the MFC is announced by the Treasury Dept. every at the beginning of the month to reflect the latest cost of capital. [22]",
    "The components of the Floating rate MFC are carefully structured. [34] It includes the Average Interest rates of Corporate and Individual time deposits of three big local banks, specifically Shinhan Bank with terms 1m, 3m, 12m. [34] Another critical component is the Interbank VNIBOR Rate for 1m and 3m, with data taken from LSEG. [34] Lastly, it incorporates the Government Bond Yield, with data taken from VBMA. [34] These elements combine to ensure the MFC accurately represents the true cost of funding from both retail markets and wholesale interbank sources. [34]",
    "The history of interest rates, including MFC, SPR, VNIBOR, and MM rates, can be inquired in system screen 9647. [35] MFC and SPR are updated every time at the beginning of the month, whereas VNIBOR and MM rates are updated daily intraday. [35] The Major Market Rate Indexes table shows the Market Average Mobilization Expense (MFC) across terms like 1M, 2M, 3M up to 12M. [36] The Shinhan Prime Rate (SPR) is similarly tiered across 3M, 6M, and 12M terms. [36] For instance, as of April 2025, the 3M SPR was 7.44%, while the 12M SPR was 8.44%. [36]",
    "Floating rates are practically applied in products like the SOHO operation loan. [37] For a Lending SOHO operation with a term of 6 months, the bank applies the Floating MFC 3M rate. [37] In this setup, the Interest Rate Change Cycle is set to 3 Months, meaning the applied MFC rate will adjust quarterly. [37] The Loan Internal Interest Rate Inquiry calculates the final FTP by combining the Market Base Rate of 2.10%, an ALM spread of 2.25%, and a Liquidity Premium of 0.1285%, resulting in a final FTP of 4.4785% applied to the loan. [37]",
    "For long-term retail products, a hybrid rate structure is often used. [38] A Lending Housing loan may feature a fixed rate for the first 3 years and then switch to a floating SPR6M rate for the remainder of its 25-year term. [38] The calculation separates the term into 25 years minus the initial 3 years, leaving a 22-year floating period. [38] The FTP Final applied for this subsequent 11-year block (term/2 for principal payment) uses a Market Base Rate of 3.00% plus an ALM spread of 1.89% and a Liquidity Premium of 0.9329%, totaling 5.8229%. [38]",
    "Variable rates have a direct impact on borrowers. [39] As the rate is floating, borrowers may experience varying repayment amounts based on the rate adjustments made monthly. [39] There are potential cost benefits; if interest rates decline, borrowers can benefit from lower payments, potentially saving money over the loan's term. [39] Conversely, there is a risk of increasing costs. [39] If rates rise, borrowers may face higher costs, making it vital for them to monitor market trends closely to anticipate changes in their monthly financial obligations. [39]",
    "The Loan FTP application accounts for the specific characteristics of floating rate schedules. [40] For a Lending Housing loan fixed for 3 years and then floating SPR6M, the first final FTP calculation specifically targets the initial fixed period. [40] For the first period of 3 years, the term of fund is considered 3 years. [40] The system uses a Market Base Rate of 4.60% and an ALM spread of 1.90%, yielding an applied FTP of 6.50% for this initial segment. [40] Subsequent periods will automatically query the prevailing SPR6M rate from the Treasury updates. [40]",
    "Not all loans use floating rates. [19] A Lending Consumer loan often utilizes a fixed rate over a 5-year term with monthly principal payments. [19] The Loan FTP for this fixed structure locks in the Market Base Rate on the execution date, for instance, 4.60%, and an ALM spread of 1.6494%, yielding a fixed final FTP of 6.2494%. [19] Because the rate is fixed, the Repayment Method is set to equal principal installments, and the customer does not face the varying repayment amounts associated with SPR or MFC. [19]",
    "On the liability side, Deposit FTP relies on a completely different pricing matrix. [17] The Deposit FTP Margin Table for the Business Division utilizes a Basic Announced Rate and a Max rate under GM/DGM approval. [17] It calculates the Final FTP and the corresponding Margin Rate for individual and corporate deposits. [17] A 6-month time deposit might yield a margin of 2.60%, depending on the specific product tier. [17] This fixed-term deposit pricing forms the foundational cost of funds but functions independently of the floating loan rate calculations. [17]",
    "The global economy is currently navigating a period of steady resilience amid divergent forces. [41] While overall growth remains stable, the drivers of that growth vary significantly by region. [41] Global growth is projected to hold at 3.3% for 2026. [41] Meanwhile, the AI Tailback phenomenon shows heavy investment in Artificial Intelligence and digital infrastructure providing a significant productivity floor in North America and parts of Asia. [41] These broad global macroeconomic projections do not detail the specific mechanical calculations of local bank floating rates like SPR or MFC. [41]"
]

# Test Case 5: Huy động vốn và Chênh lệch Tín dụng - Tiền gửi (Deposit Mobilization Strategy)
query_5 = "Chiến lược và quy định huy động vốn từ Market 1 được thực hiện ra sao trong bối cảnh chênh lệch tăng trưởng tín dụng và tiền gửi?"
documents_5 = [
    "Mobilization from Market 1, which involves deposits from customers, is strictly governed by regulations relating to Interest rates, specifically Circular 46/2024, 47/2024, and 48/2024. [24] The State Bank of Vietnam dictates the ceiling rate applied for the kind of interest paid at maturity. [24] Demand deposits and terms under 1 month are capped at 0.5%, while terms from 1 month to under 6 months have a ceiling of 4.75%. [24] USD deposits remain at 0% for all terms. [24] This regulatory framework establishes the boundaries within which the bank executes its retail and corporate deposit mobilization strategies. [24]",
    "The market is currently experiencing significant systemic liquidity disparities. [32] In Mar 2026, the average quoted deposit interest rate for a 12m term at big 4 banks was 5.2%, big local banks at 5.4%, and smaller banks at 5.9%. [32] However, a substantial gap exists between credit growth at 19.01% and deposit growth estimated at 15%. [32] This gap leads to a shortage of funds, pushing deposit rates to very high levels, with 6m-12m terms reaching 7.0% - 8.0%. [32] Local small banks lacking strong CASA strength may even offer rates above 8.0% to attract crucial liquidity. [32]",
    "To manage deposit profitability, the bank utilizes a Deposit FTP Margin Table. [17, 42] The Basic Announced Rate is applied for counter transactions, while the Final FTP for favor cases is matched to normal cases, reducing the margin rate accordingly to share with customers. [17] For instance, a 12-month Corporate deposit might carry a Basic Announced Rate of 4.12%, but a Final FTP of 5.15%, generating a 1.03% margin. [17] By adjusting the GM/DGM approval rates, the bank can dynamically incentivize branches to mobilize specific tenors based on liquidity needs. [17]",
    "Within the Aither System 9627, the Deposit FTP is broken down into specific components to reflect funding stability. [18] The Final FTP for Corporate and Individual deposits incorporates a Basic FTP, a Product FTP adjustment, and crucially, a Stable Funding Premium. [18] For example, a 6-month term may have a Basic FTP of 4.89% and a Product FTP of 0.6%, resulting in an adjusted rate that incentivizes longer-term, sticky deposits. [18] The inclusion of the Stable funding premium ensures that stable, long-term mobilization is properly rewarded in the internal P&L. [18]",
    "When managing individual customer deposits, branches utilize systems to Inquire Account Details for Installment and Time Deposits. [43, 44] Information such as the Open date, Maturity Date, Contract Amount, and Apply Rate are tracked meticulously. [43] For a Time Deposit of 3.16 billion VND, the system reflects the Fixed Interest Rate applied, for example, 5.40%. [43] Operational procedures require staff to keep physical certificates in the vault and, importantly, notice the maturity date of the deposit in advance to proactively contact the customer and encourage rollover, securing the mobilized funds. [43]",
    "Customer Time Deposits play a vital role in cost management and profitability strategies. [7] Understanding the various elements that affect funding components is crucial, and deposits impact the cost of funds significantly. [7] Customer deposits take the main portion in the Structure of Funding for the financial institution. [7] Therefore, determining the proportion of each funding component through a Monthly Funding Strategy is necessary. [7] During periods of high credit growth, maximizing the volume of stable customer time deposits is the primary defense against interbank volatility. [7]",
    "The Treasury Department's ALM & Funding team actively manages the Time deposit rate offered to customers. [1] They set up, announce, and manage the Fund Transfer Price distributed to Business Units to align branch behavior with the bank's macro liquidity strategy. [1] By giving direction and planning to assure prudential ratios comply with SBV and Mother Bank guidelines, the Treasury ensures the bank doesn't over-leverage. [1] They collect proposals from different departments for ALCO meetings to adjust the mobilization strategy in real-time. [1]",
    "When Market 1 mobilization falls short of rapid credit expansion, banks turn to Market 2, securing funds from the Interbank market. [45] Market 2 involves lending and borrowing terms with Credit Institutions, typically maximized under 1 year. [45] Deposit terms in this wholesale market are generally maximized from 3 months and below, with interest rates being negotiated per transaction rather than set by public boards. [45] Finance and Finance Leasing companies are treated similarly to Credit Institutions under Head Office approved transaction limits and policies. [45]",
    "Public Investment Disbursement Progress shows that as of the end of February, public investment disbursement reached 5.6% of the 2026 annual plan. [46] Concurrently, Disbursed FDI reached $3.21 billion for Jan-Feb, an 8.8% increase YoY, marking the highest two-month disbursement in five years. [46] International Trade turnover reached $155.7 billion, with a trade deficit of $1.04 billion in February. [46] While these macro indicators reflect strong economic momentum, they do not dictate the specific tactical deposit pricing or ceiling rate compliance within retail banking branches. [46]",
    "The global economy faces significant geopolitical shocks, such as the US-Iran conflict's impact on global markets. [47] Sharp volatility driven by supply concerns could push Brent crude to $100-130/barrel due to risks like a Hormuz Strait blockade. [47] This causes a flight to quality and safe-haven demand surges in Gold and Precious Metals. [47] Government Bonds experience safe-haven inflows, leading to a rally amid risk aversion. [47] While this affects global risk sentiment, it is completely disconnected from the bank's internal policies for mobilizing local VND deposits from domestic retail customers. [47]"
]

### Loan Assessment

In [ ]:
# TEST CASE 1: THẨM ĐỊNH THU NHẬP TỪ LƯƠNG VÀ THƯỞNG
query_1 = "quy định, chứng từ yêu cầu và phương pháp tính thu nhập ròng hàng tháng đối với khách hàng hưởng lương và thưởng"
documents_1 = [
    # High Similarity
    "Thu nhập ròng hàng tháng = Thu nhập cố định + Thu nhập không cố định (trong khoảng thời gian thu thập chứng từ) [1]. Thu nhập cố định là tiền lương và phụ cấp cố định hàng tháng [1]. Phần thu nhập vượt quá 120% thu nhập cố định thấp nhất sẽ được tính theo quy định của thu nhập không cố định [1]. Thu nhập không cố định bao gồm tiền thưởng, hoa hồng, phụ cấp không cố định nhận được trong kỳ thu thập chứng từ [1]. Các khoản này sẽ được chia cho 12 tháng và giảm trừ ít nhất 25% [1]. Yêu cầu khoảng thời gian thu thập chứng từ tối thiểu 3 tháng gần nhất đối với thu nhập cố định và tối thiểu 6 tháng gần nhất nếu có áp dụng cho một số công việc thù lao không cố định [1].",
    "Đối với chứng từ yêu cầu, khách hàng cần cung cấp hợp đồng lao động hoặc các chứng từ tương đương như quyết định bổ nhiệm, thư xác nhận có thể hiện thông tin nơi làm việc và mức lương hiện tại [2]. Loại hợp đồng chính thức phải có thời hạn ít nhất 12 tháng và còn hiệu lực [2]. Nếu hợp đồng đã hết hạn hoặc thời gian còn lại của hợp đồng dưới 15 ngày tại thời điểm nộp hồ sơ, bắt buộc phải có điều khoản gia hạn tự động rõ ràng [2]. Hoặc cần bổ sung hợp đồng lao động mới tại thời điểm giải ngân [2].",
    "Trường hợp khách hàng cung cấp sao kê tài khoản ngân hàng, yêu cầu nộp bản chính hoặc bản sao kê ngân hàng có mộc dấu của ngân hàng trước thời điểm giải ngân [3]. Chấp nhận bản gốc sao kê ViettelPay/Viettel Money có dấu hoặc bản chụp màn hình hệ thống chi lương trên ứng dụng ViettelPay/Viettel Money nếu công ty thuộc Tập đoàn Viettel [3]. Nếu lương nhận qua tài khoản ngân hàng trực tiếp, sao kê không cần thể hiện thông tin 'lương' [4]. Trường hợp lương nhận từ công ty liên quan, bắt buộc bổ sung giấy phép kinh doanh [4].",
    "Đối với thu nhập được trả bằng tiền mặt, khách hàng cần cung cấp phiếu chi lương bản chính có đóng mộc công ty và chữ ký người có thẩm quyền [3]. Hoặc có thể dùng sổ bảo hiểm xã hội đối với mức lương từ 20 triệu đến 40 triệu VND, hoặc chứng từ quyết toán thuế thu nhập cá nhân năm trước đó đối với mức lương trên 40 triệu VND [3]. Tiền thưởng sẽ không được dùng để tính tỷ lệ DBR nếu thu nhập được trả bằng tiền mặt [5]. Khách hàng phải qua thời gian thử việc tại công ty hiện tại, ít nhất 3 tháng nếu không gồm thời gian thử việc [6].",
    "Chứng từ từ nơi làm việc tại công ty trước để chứng minh thời gian làm việc ít nhất 12 tháng và khoảng thời gian gián đoạn giữa 2 công ty không vượt quá 6 tháng [2]. Thời gian gián đoạn được tính bằng (1) - (2), trong đó (1) là Ngày bắt đầu hợp đồng lao động chính thức/ngày nhận quyết định bổ nhiệm của nhân sự và (2) là Ngày thôi việc [2]. Đây là yêu cầu bắt buộc khi thẩm định lịch sử làm việc của khách hàng vay [2].",
    # Medium Similarity
    "Trường hợp thời gian làm việc tại công ty hiện tại dưới 3 tháng, khách hàng phải cung cấp thêm chứng từ thu nhập tối thiểu 3 tháng gần nhất tại công ty trước đó [5]. Thu nhập để tính DBR lúc này sẽ là giá trị Min (bình quân thu nhập tại công ty hiện tại; bình quân thu nhập tại công ty trước đó) [5]. Chỉ áp dụng đối với sản phẩm cho vay có tài sản thế chấp [5].",
    "Khi thu thập chứng từ điện tử, hướng dẫn xác thực đối với những 'hợp đồng lao động hoặc chứng từ tương đương được ký bởi mộc dấu hoặc chữ ký điện tử' như sau: Phương án 1 là không xác thực được bằng phiên bản điện tử thì cần có dấu mộc của công ty và chữ ký điện tử, dấu treo giáp lai [7]. Phương án 2 là xác thực được bằng phiên bản điện tử thì dùng Foxit Reader hoặc tra trang web https://neac.gov.vn/vi [7].",
    "Các loại phụ cấp thường xuyên có thể được cộng vào lương để tính thu nhập ròng. Trong ví dụ tính toán, khách hàng có lương 30,000,000 VND và phụ cấp 5,000,000 VND mỗi tháng [8]. Phần thưởng (Hoa hồng) nhận hàng tháng cũng được liệt kê và phân loại thành thu nhập cố định và thu nhập không cố định [8]. Tỷ lệ giảm trừ 25% chỉ áp dụng đối với thu nhập không cố định [8].",
    # Low Similarity (Noise/Distractors)
    "Đối với doanh nghiệp, cần thu thập thông tin quy mô, mã số thuế, số lượng nhân viên, vốn điều lệ và địa chỉ trụ sở chính [9]. Báo cáo tài chính hoặc kiểm toán nội bộ công ty phải nộp đủ trong vòng 2 năm gần nhất để đánh giá quy mô [9]. Công ty tư nhân và cổ phần có các tiêu chí tính điểm khác nhau [10].",
    "Quy định về tài sản thế chấp (Collateral Information): Mô tả chi tiết tài sản đảm bảo bao gồm địa chỉ nhà/đất, số lượng diện tích (m2), người đứng tên sở hữu, cấu trúc nhà hoặc giấy phép xây dựng [11]. Nếu là xe ô tô, phải kiểm tra năm sản xuất và hợp đồng mua bán với showroom có hợp tác cùng SHBVN [11]."
]

# TEST CASE 2: TÍNH TOÁN DBR VÀ KHẢ NĂNG TRẢ NỢ
query_2 = "Công thức tính toán Tỷ lệ gánh nặng nợ (DBR), cách xác định tổng nghĩa vụ nợ, tổng thu nhập ròng và chi phí sinh hoạt."
documents_2 = [
    # High Similarity
    "Công thức tính Tỷ lệ trả nợ (DBR: Debt burden Ratio) là: DBR = Tổng nghĩa vụ nợ / Tổng thu nhập ròng [12]. Theo quy định, nếu tổng thu nhập để trả nợ hàng tháng >= 50 triệu VND thì DBR <= 70% [12]. Nếu tổng thu nhập để trả nợ hàng tháng < 50 triệu VND thì DBR <= 60% [12]. Riêng đối với thu nhập từ Hộ kinh doanh, điều kiện chặt chẽ hơn: DBR <= 60% [12]. Khả năng thanh toán = Tổng thu nhập ròng - Tổng nợ phải trả - Chi phí sinh hoạt [12]. Số tiền còn lại sau khi thanh toán phải bằng hoặc lớn hơn 0 [12].",
    "Tổng nghĩa vụ nợ được tính bằng (1) Nợ gốc và lãi vay ngân hàng + (2) các khoản nợ khác + (3) Nợ thẻ tín dụng ước tính [13]. Nợ ngân hàng bao gồm khoản vay hiện tại và khoản vay đang đề xuất [13]. Thanh toán thấu chi phải ít nhất 5% số dư cao nhất thể hiện trong chứng từ thu thập được [13]. Nợ thẻ tín dụng phải ít nhất 5% số dư bình quân trong 3 tháng gần nhất đối với khoản vay không có tài sản đảm bảo, hoặc trung bình dư nợ trong 12 tháng gần đây đối với khoản vay có tài sản đảm bảo [13].",
    "Việc tính toán số tiền gốc và lãi phải trả dựa vào thông tin trên báo cáo CIC và áp dụng theo công thức: Nợ phải trả tại ngân hàng = PMT (lãi suất cơ bản/12, thời hạn cho vay, - số tiền vay) [14]. Đối với khoản vay không có tài sản đảm bảo, Lãi suất thả nổi sau thời kỳ ưu đãi của chính khoản vay đó tính trung bình/dài hạn: 15%/năm, ngắn hạn: 12%/năm [14]. Đối với khoản vay có tài sản đảm bảo, áp dụng SPR 12 tháng + 2% [14].",
    "Chi phí sinh hoạt cho 1 gia đình được quy định cụ thể tại các khu vực: 5 triệu đồng/tháng tại Hà Nội, TP.HCM và 4 triệu đồng/tháng đối với các tỉnh khác [12]. Phải đảm bảo sau khi trừ các khoản nợ phải trả (Kỳ trả nợ vay đề xuất, dư nợ hiện hữu và ước tính nợ thẻ tín dụng) cùng chi phí sinh hoạt, khách hàng vẫn còn số dư khả dụng (Số tiền còn lại >= 0) [15].",
    "Trong các trường hợp đặc biệt, nghĩa vụ thanh toán nợ gốc/lãi có thể được loại trừ: Thời gian vay của khoản vay đề xuất nằm trong thời gian ân hạn vốn/lãi của khoản vay hiện hữu (hợp đồng tín dụng phải được bổ sung) [16]. Dư nợ thẻ tháng gần nhất của từng thẻ có thể được loại trừ nếu khách hàng cung cấp chứng từ đóng thẻ trước giải ngân [16].",
    # Medium Similarity
    "Ví dụ bài tập tình huống: Khách hàng B vay Home Loan, thu nhập từ lương người vợ 39 triệu/tháng, thu nhập hộ kinh doanh người chồng 150 triệu/tháng [17]. Nợ phải trả khoản vay đề xuất 15 triệu, ước tính nợ thẻ tín dụng 1 triệu. Tổng thu nhập là 189 triệu, tổng nợ phải trả là 116 triệu [15]. Tỷ lệ DBR sẽ được tính toán trên tổng các nguồn này để quyết định phê duyệt [15].",
    "Thu nhập ròng hàng tháng để đưa vào công thức DBR sẽ tính tổng từ Lương, Cho thuê tài sản, Doanh nghiệp công ty, và cổ tức [18]. Mục Available Fund = Total net monthly income - Loan Repayment - Living expenses [19]. Mục Loan Repayment bao gồm khoản vay mới SHB, khoản vay hiện hành khác và trả nợ thẻ tín dụng [19].",
    "Trong bảng đánh giá khách hàng SOHO, khả năng trả nợ (Repayment ability) cũng được đo lường bằng Tỷ lệ Nợ ròng/EBITDA với ngưỡng rủi ro cao là > 5.0 hoặc EBITDA <= 0 [20]. Hệ số thanh toán nhanh (Quick ratio) và Tỷ lệ phụ thuộc nợ vay (Borrowing dependency ratio) là các tiêu chí khác bổ sung ngoài DBR thông thường của khối Khách hàng cá nhân [21].",
    # Low Similarity (Noise/Distractors)
    "Theo quy định định giá tài sản, Giá trị tài sản đảm bảo (Collateral Value) cho Nhà và Đất (LUR & H) thường được nhân với hệ số RCVR là 60%, trong khi đối với Ô tô là 30% [22]. Tài sản đảm bảo nhằm mục đích bù đắp rủi ro nếu chỉ số DBR vượt ngưỡng [22].",
    "Trường hợp xác minh qua điện thoại không thực hiện được, thẩm định có thể thực hiện qua địa chỉ email xác thực hoặc người phụ trách thẩm định phải đi thực địa để kiểm tra [23]. Thẩm định viên sẽ chọn YES/NO cho từng điều kiện quy định trong Verification Report [24]."
]

# TEST CASE 3: THẨM ĐỊNH THU NHẬP TỪ CHO THUÊ TÀI SẢN
query_3 = "Quy định tỷ lệ chấp nhận, chứng từ yêu cầu và phương pháp tính thu nhập từ cho thuê bất động sản và phương tiện đi lại."
documents_3 = [
    # High Similarity
    "Thu nhập cho thuê bất động sản được chấp nhận đến 100% thu nhập tính vào DBR [25]. Đối với thu nhập cho thuê phương tiện đi lại, chấp nhận đến 90% thu nhập tính vào DBR [25]. Thu nhập cho thuê xe hơi không bao gồm chi phí tài xế, nhiên liệu và chi phí bảo trì [25].",
    "Thu nhập này phải phát sinh định kỳ hàng tháng và áp dụng cho khoản vay có tài sản đảm bảo [26]. Chỉ chấp nhận nguồn thu được thanh toán qua tài khoản ngân hàng [26]. Thu nhập từ lương hưu được coi là nguồn thu nhập bổ sung và khách hàng cần cung cấp thêm (các) nguồn thu nhập phù hợp với Hướng dẫn cho vay bán lẻ [26].",
    "Về chứng từ yêu cầu, khách hàng phải nộp Bản sao Giấy chứng nhận quyền sở hữu tài sản cho thuê đứng tên khách hàng, được đối chiếu với bản gốc [27]. Hợp đồng cho thuê phải được ký ít nhất 3 tháng kể từ ngày làm đơn vay vốn, nếu không phải nộp bổ sung hợp đồng thuê trước đó [27]. Thời hạn cho thuê ít nhất 12 tháng và thời hạn còn lại ít nhất 1 tháng [27].",
    "Đối với dịch vụ cho thuê phòng trọ, hợp đồng không cần phải tuân thủ nghiêm ngặt các tiêu chí trên và có thể được thay thế bằng sổ tạm trú nhưng bắt buộc phải đi thẩm định thực tế [27]. Chứng từ thanh toán: thu nhập cho thuê từ các bên liên quan được chấp nhận trong trường hợp có chứng từ rõ ràng của giao dịch (sao kê ngân hàng/chuyển khoản ngân hàng) hoặc tờ khai thuế [28]. Chi nhánh phải thực hiện KYC cẩn thận để xác nhận thực tế kinh doanh [28].",
    "Mức giá cho thuê nào thận trọng hơn giữa giá thuê trung bình trên thị trường và hợp đồng cho thuê sẽ được sử dụng để tính thu nhập [28]. Chi nhánh phải tiến hành quá trình khảo sát thị trường trên internet, báo giấy hoặc hỏi giá xung quanh khu vực để đánh giá giá thuê trung bình [28]. Không có lỗ lũy kế và không có lỗ trong 02 năm gần nhất [29].",
    # Medium Similarity
    "Bài tập tình huống: Khách hàng cung cấp hợp đồng thuê xe. Xe đứng tên sở hữu của con gái khách hàng. Hợp đồng thể hiện giá thuê 15 triệu/tháng thanh toán qua tài khoản [30]. Ngân hàng sẽ không chấp nhận nguồn thu nhập này vì tài sản không đứng tên của chính khách hàng vay, mà đứng tên con gái [30].",
    "Nếu tài sản cho thuê là bất động sản đứng tên khách hàng, giá thuê cố định 50 triệu/tháng, và khách hàng tự trả các chi phí khác như điện, nước, thuế [31]. Theo quy định cho vay cá nhân, thu nhập cho thuê tối đa được tính vào DBR sẽ là 45 triệu/tháng (giảm trừ thuế và các chi phí ước tính nếu không có chứng từ cụ thể, hoặc theo tỷ lệ 90% nếu là xe hơi) [31].",
    "Trong mẫu nhập liệu tín dụng (Appendix 4.2), thông tin về Rental Income yêu cầu chi tiết: Số lượng, Địa chỉ/Nhãn hiệu xe, Tên người thuê, Thời hạn thuê, Giá thuê, Phương pháp chiết khấu, Kỳ thanh toán, và Loại hợp đồng [32]. NVTD phải nhập đầy đủ vào màn hình thẩm định 9023 [32].",
    # Low Similarity (Noise/Distractors)
    "Đối với thu nhập từ cổ tức: Các doanh nghiệp nằm trong top 500 công ty ở Việt Nam hiện tại (nguồn www.vnr500.com.vn) hoặc các công ty được niêm yết trên thị trường chứng khoán Việt Nam mới được tính [33]. Thu nhập loại này không được vượt quá 40% tổng thu nhập [33].",
    "Kiểm tra dư nợ tín dụng trên CIC: Phân loại nợ (Nhóm 1 đến 5). Nếu khách hàng có lịch sử nợ nhóm 2 trong 12 tháng gần nhất, hoặc nợ xấu nhóm 3, 4, 5 trong 5 năm gần nhất, khoản vay sẽ bị từ chối hoặc cần trình cấp ngoại lệ đặc biệt [34]."
]

# TEST CASE 4: THẨM ĐỊNH THU NHẬP DOANH NGHIỆP TƯ NHÂN VÀ SOHO
query_4 = "Phương pháp tính thu nhập ròng, xác định quy mô doanh nghiệp và chứng từ yêu cầu đối với khách hàng là Công ty TNHH/Cổ phần hoặc Hộ kinh doanh."
documents_4 = [
    # High Similarity
    "Đánh giá thu nhập từ doanh nghiệp (Công ty TNHH/Cổ phần), thu nhập ròng được tính theo công thức: Thu nhập ròng = (((Trung bình doanh thu trong 2 năm gần nhất * Tỷ suất lợi nhuận ước tính) - thuế phải trả trong năm gần nhất) * tỷ lệ cổ phần) / 12 [35]. Tỷ suất lợi nhuận ước tính phụ thuộc vào Quy mô công ty: Nhỏ (10%), Vừa (7.5%), Lớn (5%) [36].",
    "Quy mô công ty được phân loại dựa trên Doanh thu hàng năm, Số lượng nhân viên và Vốn điều lệ [10]. Doanh thu < 5 tỷ VND là Nhỏ, từ 5 tỷ - 20 tỷ VND là Vừa, trên 20 tỷ VND là Lớn [10]. Số lượng nhân viên tương ứng là < 30, 30-69, và 70-100 người [10].",
    "Điều kiện cho công ty đủ tiêu chuẩn: Thành lập hợp pháp và hoạt động ít nhất 12 tháng tính đến ngày đề nghị vay vốn [37]. Doanh thu báo cáo thuế năm gần nhất phải lớn hơn 0 [37]. Có lợi nhuận trong năm gần nhất và không có lỗ lũy kế [37]. Chứng từ thanh toán thuế VAT hoặc tờ khai VAT trong 6 tháng gần nhất phải được cung cấp để tính toán doanh thu thực tế [29].",
    "Đối với thu nhập từ Hộ kinh doanh (SOHO), phương pháp tính Lợi nhuận ròng = Doanh thu - Chi phí giá vốn hàng bán - Chi phí hoạt động [38]. Thu nhập của hộ gia đình = Lợi nhuận ròng * % cổ phần * (1 - X%) [38]. Tỷ lệ giảm trừ (X) tối thiểu 25% có thể điều chỉnh dựa trên tỷ lệ lợi nhuận của khách hàng nhận được và tỷ lệ tái đầu tư [38].",
    "Thu nhập doanh nghiệp Hộ kinh doanh cần tập trung phân tích: Doanh thu chứng từ đầu ra, Chi phí giá vốn hàng bán (COGS) tối đa 70% với sản xuất và 85% với thương mại, lĩnh vực dịch vụ tối đa 50% [39]. Đối với dịch vụ cho thuê bất động sản mà khách hàng là chủ tài sản cho thuê thì chi phí giá vốn được cân nhắc là 25% [39]. Chi phí hoạt động bao gồm tiền lương, điện nước, thuế mặt bằng [39].",
    # Medium Similarity
    "Chứng từ đầu vào được chứng minh bởi bất kỳ loại hóa đơn VAT, hợp đồng mua theo chứng từ đầu ra, sổ ghi chép quản lý hàng tồn kho trong 3 tháng gần nhất hoặc 6 tháng [40]. Hóa đơn VAT/Hợp đồng mua bán là chứng từ có giá trị pháp lý cao nhất [41]. Việc thẩm định nguồn thu từ doanh nghiệp tư nhân phải được thực hiện qua các bước bắt buộc như thẩm định thực tế [42].",
    "Không chấp nhận thu nhập hộ gia đình dưới tên của khách hàng vay/vợ hoặc chồng đối với loại hình kinh doanh nhạy cảm: tiệm cầm đồ, dịch vụ massage [43]. Lưu ý: Nếu khách hàng có nhiều loại thu nhập, tỷ lệ của thu nhập này nếu chiếm từ 60% trở lên thì phải áp dụng quy định riêng của SOHO [43].",
    "Về kiểm tra hoạt động kinh doanh SOHO: Địa điểm kinh doanh cần ổn định. Cơ sở vật chất phục vụ hoạt động phải có diện tích, mặt bằng, nhà xưởng, tình trạng tài sản sở hữu hay đi thuê hợp lệ [44]. NVTD đánh giá kết quả kinh doanh trong 3 năm gần nhất, kiểm tra tồn kho và sổ sách ghi chép nội bộ [45].",
    # Low Similarity (Noise/Distractors)
    "Đối với thẻ tín dụng, nhập thông tin nợ thẻ tín dụng, dư nợ bình quân trong 12 tháng gần nhất từ tất cả các ngân hàng (ví dụ: XY Bank cấp hạn mức 20,000,000 VND, Outstanding 2,000,000 VND) [46]. Thông tin dư nợ sẽ tự động tính 5% vào mục Nghĩa vụ nợ (Estimated Repayment) [46].",
    "Chứng từ quyết toán thuế thu nhập cá nhân của năm trước đó đối với mức lương từ VND 40 triệu trở lên [3]. Phiếu chi lương phải là bản chính, có đóng mộc công ty và chữ ký bởi người có thẩm quyền với tên và chức vụ rõ ràng [3]. Phải đính kèm ảnh chụp màn hình hệ thống Viettel Money [3]."
]

# TEST CASE 5: ĐÁNH GIÁ RỦI RO KHÁCH HÀNG SOHO (HỘ KINH DOANH)
query_5 = "Các chỉ số đánh giá rủi ro kinh doanh, tài chính và điểm tín dụng dành riêng cho khách hàng SOHO."
documents_5 = [
    # High Similarity
    "Đánh giá khách hàng SOHO tập trung vào rủi ro hoạt động kinh doanh do khả năng tài chính và vốn tương đối yếu, uy tín và khả năng điều hành phụ thuộc vào cá nhân chủ doanh nghiệp [47]. Rủi ro cao trong nguồn vốn cá nhân bị hòa lẫn với nguồn vốn kinh doanh [47]. Số tiền vay tương đối nhỏ nhưng tệp khách hàng lớn và mức độ cạnh tranh giữa các tổ chức tài chính rất cao [47].",
    "Bảng đánh giá giá trị SOHO (SOHO Assessment Table) đo lường các tiêu chí phi tài chính: Chuyên môn kinh nghiệm (thời gian làm việc trong cùng ngành >= 10 năm được điểm tối đa), Tài sản sở hữu (Tổng tài sản > 20 tỷ VND), Khả năng gây quỹ khẩn cấp (tiền gửi của chủ doanh nghiệp) [48]. Độ tin cậy được đánh giá bằng Lịch sử nợ (Nhóm 1 trong 16 tháng gần nhất) và Tình trạng trả nợ [48].",
    "Về mặt tài chính, rủi ro SOHO được phân tích qua Tỷ lệ nợ ròng/EBITDA. Ngưỡng rủi ro cao là > 5.0 hoặc EBITDA <= 0 [20]. Tỷ lệ phụ thuộc nợ vay (Tổng nợ vay / Tổng tài sản) > 50% cũng bị đánh giá là rủi ro cao [20]. Các chỉ số này giúp xác định dòng tiền thực tế có đủ bù đắp nghĩa vụ nợ (EBITDA / Lãi vay) [20].",
    "Hệ số thanh toán nhanh (Tài sản ngắn hạn - Hàng tồn kho) / Nợ ngắn hạn nếu < 100% được xem là dấu hiệu cảnh báo [49]. Tăng trưởng doanh thu (Doanh thu năm nay - Doanh thu năm trước) / Doanh thu năm trước nếu < 0% (tăng trưởng âm) sẽ đánh dấu ngưỡng rủi ro cao [50]. Dòng tiền sau khi thanh toán nợ vay < 0 yêu cầu phải xem xét lại mức cấp tín dụng [51].",
    "Một lưu ý quan trọng khác khi cấp vốn SOHO là Hạn mức vốn có thể vay = Nhu cầu vốn cho mỗi vòng quay vốn lưu động - Hạn mức vay vốn lưu động hiện tại [52]. Trong đó, Doanh thu kế hoạch = Doanh thu năm trước + (Doanh thu năm trước * Tăng trưởng bình quân 3 năm trước) [53]. Thời gian quay vòng vốn lưu động = Các khoản phải thu + Tồn kho - Các khoản phải trả [53].",
    # Medium Similarity
    "Để kiểm tra tính xác thực, bắt buộc phải thẩm định thực tế tình trạng hoạt động của SOHO [54]. Các phương pháp thẩm định thu nhập chéo bao gồm đối chiếu hệ thống sổ sách ghi chép, sổ phụ ngân hàng, hoặc hệ thống bán hàng đăng ký cơ quan thuế [54]. Đặc biệt trong 3 tháng gần nhất, nếu doanh thu giảm mạnh trên 20%, phải sử dụng doanh thu năm gần nhất để tính toán [35].",
    "Trong phần phân tích đối tượng khách hàng, SOHO có 3 mức độ giám sát: Warning, Caution, và Non-criteria. Những ngành nghề rủi ro cao như Môi giới bất động sản, Xây dựng hoặc phân phối vật liệu xây dựng, Dịch vụ Grab taxi nếu chưa có giấy phép vệ sinh thực phẩm/phòng cháy chữa cháy sẽ bị cảnh báo (Warning) hoặc từ chối (Reject) [55].",
    "Hệ thống Bảng đánh giá sẽ cho ra Final Score (ví dụ: 85.6) [56]. Dựa trên điểm số này, hệ thống CCRM sẽ ra quyết định Credit rating (ví dụ: B) và Approval Authority. Khách hàng có DBR > 60% có thể bị Reject tự động trừ khi có cấp ngoại lệ từ Risk Department [56].",
    # Low Similarity (Noise/Distractors)
    "Đối với chứng từ vay mua xe hơi, khách hàng cần nộp Hợp đồng mua bán xe, Giấy chứng nhận đăng ký xe (cà vẹt) hoặc Giấy biên nhận thế chấp từ tổ chức tín dụng [57]. Xe dùng cho mục đích cá nhân sẽ có RCVR 80% so với xe dùng để kinh doanh [22].",
    "Lãi suất cho khoản vay bán lẻ mua nhà thường được cố định năm đầu (First fixed rate) hoặc năm thứ 2 (Second fixed rate). Sau đó sẽ áp dụng lãi suất thả nổi = SPR + Biên độ (Margin) [58]. Phương thức thanh toán gốc có thể là Hàng tháng hoặc Hàng quý [58]."
]

# TEST CASE 6: THẨM ĐỊNH TÀI SẢN BẢO ĐẢM (COLLATERAL ASSESSMENT)
query_6 = "Hướng dẫn ghi nhận, phân loại, tỷ lệ cho vay trên giá trị (LTV/RCVR) và các yêu cầu đối với tài sản bảo đảm."
documents_6 = [
    # High Similarity
    "Tài sản bảo đảm (Collateral) được phân loại chính thành Bất động sản (LUR & H - Quyền sử dụng đất và nhà ở) và Xe ô tô (Car) [22]. Tỷ lệ RCVR (Tỷ lệ cho vay trên giá trị tài sản) áp dụng khác nhau: Ví dụ, LUR & H tại các quận trung tâm HCMC thường áp dụng RCVR là 60% hoặc 70%, trong khi Car có tỷ lệ RCVR thấp hơn, ví dụ 30% [22].",
    "Total Loan to Value (%) (LTV) được tính bằng cách lấy Tổng số tiền vay (Total loan amount) / Tổng giá trị định giá của tài sản đảm bảo (Total Collateral Value) [59]. Ví dụ: Tổng giá trị tài sản 10 tỷ VND, RCVR 60% -> Giá trị tài sản thông thường (Regular Value) là 6 tỷ VND. Khoản vay hiện hữu 2 tỷ VND, Khoản vay đề xuất 3 tỷ VND -> Tổng dư nợ thế chấp là 5 tỷ VND, LTV = 50% [59].",
    "Mô tả chi tiết tài sản bảo đảm (Appendix 2) yêu cầu các thông tin: Chủ sở hữu (Owners name), Mối quan hệ với khách hàng vay (Relationship), Tình trạng vật lý (Physical status - phải khớp với Giấy chứng nhận sở hữu), Diện tích đất (Land Area), Mục đích sử dụng đất (Land using purpose - ví dụ: Urban land), Diện tích xây dựng (House Area) và Cấu trúc nhà [11].",
    "Điều kiện giải ngân (Branch conditions): Trước khi giải ngân (Before disbursement), thủ tục thế chấp và đăng ký giao dịch bảo đảm phải được hoàn tất (Mortgage procedure and secured registration must be completed) [60]. Phải nộp đủ CMND/CCCD có công chứng, Giấy chứng nhận kết hôn và Hợp đồng cho thuê nhà (nếu dùng nguồn thu từ cho thuê) [60]. Chuyển khoản trực tiếp vào tài khoản người bán đối với mục đích mua nhà [60].",
    "Sau khi giải ngân (After disbursement): Bắt buộc nộp Giấy chứng nhận Quyền sử dụng đất và tài sản gắn liền với đất (LUR&House Certificate) thể hiện tên khách hàng mua nhà trong vòng 02 tháng kể từ ngày giải ngân [60]. Ngân hàng SHBVN phải được ghi nhận là Người thụ hưởng thứ nhất (1st Beneficiary) đối với Bảo hiểm tài sản [60].",
    # Medium Similarity
    "Đối với mục đích vay (Loan purpose description): Nhập thông tin chi tiết về bất động sản dự định mua như địa chỉ, chủ sở hữu hiện tại (Ownership No), Cấu trúc nhà, Trị giá hợp đồng mua bán (Purchasing value) [61]. Tỷ lệ tự tài trợ (Self-financed fund) và Tỷ lệ cấp vốn (Financing ratio) phải được chứng minh bằng lịch sử thanh toán hoặc hóa đơn chuyển tiền [61].",
    "Trong quá trình xác minh (Verification report), RM và LO phải chọn 'YES/NO' cho các thông tin tài sản đảm bảo: Tình trạng vật lý, Người cư trú hiện tại, Mối quan hệ với chủ sở hữu. Kết luận phải Match với tài liệu pháp lý của tài sản [24]. NVTD phải đi thực địa xác minh độc lập [24].",
    "Trường hợp SOHO sử dụng tài sản bảo đảm, việc phân bổ dòng vốn được thẩm định thông qua Bảng cân đối kế toán. Cần đánh giá tài sản có bị cầm cố quá mức không, giá trị sổ sách so với giá trị thị trường có chênh lệch lớn không [62].",
    # Low Similarity (Noise/Distractors)
    "Đối với thu nhập từ Hộ kinh doanh cá thể: Không chấp nhận lĩnh vực kinh doanh nhạy cảm (Cầm đồ, Massage). Chủ kinh doanh phải có cổ phần trong Hộ kinh doanh. Thời gian hoạt động phải tuân theo Hướng dẫn Khách hàng cá nhân của Shinhan [63].",
    "Chỉ số nợ thẻ tín dụng được tính 5% dựa trên Dư nợ trung bình (Average using balance) nếu hạn mức thẻ được cấp từ các ngân hàng khác [46]. Lịch sử nợ xấu từ Nhóm 2 trở lên sẽ tự động bị rớt (Reject) bởi hệ thống CCRM trừ khi có lý do chính đáng được xác nhận bởi Trưởng bộ phận [34]."
]

# TEST CASE 7: KIỂM TRA LỊCH SỬ TÍN DỤNG (CIC)
query_7 = "Quy trình kiểm tra dư nợ, phân loại nhóm nợ, lịch sử chậm trả và thẻ tín dụng trên hệ thống CIC."
documents_7 = [
    # High Similarity
    "Màn hình kiểm tra lịch sử tín dụng (3. Credit history) hiển thị phân loại nhóm nợ (Debt classification) và Dư nợ VND (VND Balance) của khách hàng [34]. Lịch sử nợ xấu (History of bad debt - Nhóm 3, 4, 5) trong 5 năm gần nhất và Lịch sử nợ nhóm 2 (Special mentioned debt) trong 12 tháng gần nhất sẽ được kiểm tra chi tiết bao gồm Nhóm nợ, Số dư và Thời gian báo cáo gần nhất [34].",
    "Lịch sử thẻ tín dụng (Credit card information) theo dõi chi tiết số ngày chậm thanh toán tối đa (maximum credit card delinquency days) trong 12 tháng, 24 tháng, 36 tháng và 60 tháng qua [34]. Thông tin này phải được nhập chính xác từ kết quả truy vấn hệ thống CIC. Đánh giá về lịch sử tín dụng cần phân tích nguyên nhân quá hạn nếu có [34].",
    "Khi tính toán tổng nghĩa vụ nợ trả hàng tháng (Total Debt), bên cạnh các khoản vay dài hạn (Existing Loan Repayment), NVTD bắt buộc phải cộng thêm Ước tính thanh toán thẻ tín dụng (Credit card repayment) [19]. Nghĩa vụ này bằng 5% nhân với dư nợ bình quân thẻ tín dụng trong 12 tháng [46]. Khai báo thiếu dư nợ thẻ là lỗi nghiêm trọng ảnh hưởng trực tiếp đến DBR [46].",
    "Hệ thống sẽ tự động tổng hợp Thông tin dư nợ vay (Credit Exposure Information) từ Appendix 1, bao gồm khoản vay mới đang đề xuất (Applying) và các khoản vay hiện hữu (Existing) [46]. 'Actual payment' là số tiền thực trả dựa trên chứng từ khách hàng cung cấp. Lưu ý: 'Actual payment' có thể nhỏ hơn 'Estimated payment amount' (được tính bằng công thức PMT), lúc này hệ thống sẽ tự chọn mức cao hơn để tính DBR cho an toàn [46].",
    "Khách hàng có lịch sử nợ từ Nhóm 2 trở lên hoặc nợ xấu tại VAMC sẽ bị trừ điểm nặng trong Bảng đánh giá rủi ro (SOHO Assessment) [48]. Cụ thể, 'Debt group 2 or above in the last 12 months' hoặc 'Having loan balance sold to VAMC' sẽ bị điểm thấp nhất, ảnh hưởng đến Quyết định phê duyệt tự động [48].",
    # Medium Similarity
    "Đánh giá năng lực của khách hàng (SOHO) thông qua Tiêu chí phi tài chính: Mức độ tin cậy được đo lường thông qua Lịch sử tín dụng của cả vợ và chồng (Credit record bao gồm trường hợp Hộ kinh doanh do vợ/chồng đứng tên) [48]. Lịch sử CIC sách sẽ giúp nâng điểm Credit rating của KH lên mức B hoặc A [48].",
    "Phần Nhóm nợ, Dư nợ ròng/EBITDA và Hệ số thanh toán nhanh là 3 trụ cột của rủi ro tài chính [20]. Nhưng nếu Lịch sử tín dụng CIC (rủi ro phi tài chính) không đạt yêu cầu, hồ sơ vay vốn sẽ bị từ chối trước khi bước vào thẩm định thu nhập kinh doanh [20].",
    "Chỉ tiêu Phải trả thẻ tín dụng (Dòng 11) trong Bảng cân đối dòng tiền phải được đối chiếu trùng khớp với số liệu thu thập từ màn hình kiểm tra CIC [64]. Điều kiện bắt buộc là Thu nhập ròng (Dòng 13) trừ đi Tổng nợ phải trả lớn hơn 0 [64].",
    # Low Similarity (Noise/Distractors)
    "Tỷ lệ LTV (Loan to Value) của khoản vay không có thế chấp (Unsecured amount) tính toán rủi ro phần vốn ngân hàng không được đảm bảo bằng tài sản [59]. Việc tăng tỷ lệ Tự tài trợ (Self-financed) làm giảm DBR, tăng cơ hội duyệt vay [61].",
    "Trong phương pháp tính thu nhập từ Cổ tức: Thu nhập = (Cổ tức của một năm / Số lượng cổ phiếu của năm trước) * Số lượng cổ phiếu hiện tại / 12 [65]. Yêu cầu phải cung cấp Biên bản họp Hội đồng quản trị về chia cổ tức hoặc Sao kê ngân hàng thể hiện khoản thu nhập này [65]."
]

# TEST CASE 8: BÁO CÁO XÁC MINH THỰC ĐỊA VÀ QUA ĐIỆN THOẠI (VERIFICATION REPORT)
query_8 = "Quy trình, các hạng mục cần xác nhận trong Báo cáo xác minh thực địa và xác minh qua điện thoại (Verification Report)."
documents_8 = [
    # High Similarity
    "Báo cáo xác minh (Verification report) yêu cầu Chuyên viên Quan hệ Khách hàng (RM) và Chuyên viên Thẩm định (LO) xác nhận 'YES/NO' cho các hạng mục thực địa [24]. Bước đầu tiên: Xác minh trực tiếp với khách hàng (Customer Identification), đảm bảo thông tin trên Đơn xin vay khớp với lời khai, giấy tờ tùy thân là bản gốc và hành vi của khách hàng đáng tin cậy (Trustworthy, honest, cooperative) [24].",
    "Xác nhận Nơi cư trú (Customer's Resident Site) có thể thực hiện bằng cách Gọi điện thoại (Checked by phone call) hoặc Đi thực địa (Checked by on-site visit) [24]. Phải kiểm tra tính tương đồng giữa hình thức ngôi nhà và thu nhập khai báo. Đánh giá về hàng xóm, điều kiện sống (Living standard) [24].",
    "Xác nhận Nơi làm việc/Công ty (Customer's Working Place/Company) bao gồm kiểm tra hợp đồng lao động, gọi điện thoại cho công ty để xác minh chức vụ, thời gian làm việc. Nếu gọi điện (Checked by phone call), phải ghi rõ Tên người trả lời, Chức vụ (như nhân sự HR), và xác nhận xem công ty/nhân viên đó có thực sự đang hoạt động/làm việc tại đó không [24].",
    "Xác nhận Tài sản đảm bảo (Collateral Information): Kiểm tra địa chỉ, tình trạng vật lý hiện tại của tài sản, tổng số người đang cư trú trên tài sản đó và Mối quan hệ với chủ sở hữu (Relationship with owner). Đưa ra Kết luận: Khớp với tài liệu pháp lý của tài sản (Match with collateral's legal documents) [24].",
    "Nếu xác minh qua điện thoại không thực hiện được (do công ty không có máy bàn hoặc HR không cung cấp thông tin), việc thẩm định có thể thay thế bằng Email xác thực bằng email domain của công ty, hoặc người phụ trách thẩm định bắt buộc phải đi thực địa (On-site visit) tại nơi làm việc của KH để kiểm tra chéo [23].",
    # Medium Similarity
    "Xác minh thực tế đối với Hộ kinh doanh (SOHO) cực kỳ quan trọng vì rủi ro chứng từ giả mạo cao. RM cần đánh giá quy mô cơ sở vật chất (nhà xưởng, thiết bị, bảng hiệu) có tương xứng với mức Doanh thu khai báo không [44]. Xác định địa điểm kinh doanh là tự sở hữu hay đi thuê (ảnh hưởng trực tiếp đến chi phí hoạt động) [44].",
    "RM phải kiểm tra chéo (Cross-check) Doanh thu ước tính dựa trên thực tế bằng cách đếm số lượng khách hàng tại một thời điểm, kiểm tra sổ sách viết tay/sổ phụ ngân hàng hoặc hệ thống POS/Phần mềm bán hàng [54]. Trường hợp biên lợi nhuận cao bất thường so với mức trung bình ngành, phải thu thập chứng từ giải trình từ khách hàng [54].",
    "Về kiểm tra CIC, LO phải tiến hành tra cứu thêm thông tin từ các Trung tâm Thông tin Tín dụng ngoài CIC (nếu có) để đảm bảo không bị sót dư nợ của KH và vợ/chồng [66]. Đánh giá toàn bộ lịch sử hình thành, thay đổi chủ hộ kinh doanh qua các mốc thời gian [66].",
    # Low Similarity (Noise/Distractors)
    "Đối với khoản vay theo tỷ lệ LTV linh hoạt, hạn mức phê duyệt ngoại lệ (Exceptional Items) phải tuân theo hướng dẫn rủi ro (Retail Guideline). Tổng số tiền vay khách hàng (Total customer loan amount) > 400,000 USD cần có cấp thẩm quyền phê duyệt COC [60].",
    "Thu nhập từ cho thuê bất động sản được tính bằng 100% dòng tiền hàng tháng [25]. Nếu giá thuê cố định là 50 triệu/tháng, hợp đồng rõ ràng, nhưng tài sản thuộc về con gái khách hàng thì nguồn thu nhập này bị Reject (Không chấp nhận) vì không đứng tên khách hàng vay [30]."
]

# TEST CASE 9: THẨM ĐỊNH THU NHẬP TỪ CỔ TỨC (DIVIDEND INCOME)
query_9 = "Điều kiện áp dụng, tỷ lệ tối đa và phương pháp tính thu nhập từ cổ tức của cổ đông."
documents_9 = [
    # High Similarity
    "Thu nhập từ cổ tức (Dividend Income) chỉ được tính vào DBR với các điều kiện: Doanh nghiệp nằm trong Top 500 công ty ở Việt Nam hiện tại (nguồn www.vnr500.com.vn) hoặc các công ty được niêm yết trên thị trường chứng khoán Việt Nam [33]. Thu nhập loại này sẽ không được tính vượt quá 40% tổng thu nhập của khách hàng (Max 40% of debtor's total income) [33].",
    "Chỉ áp dụng thu nhập cổ tức đối với khoản vay có tài sản đảm bảo (Only apply for loan with collateral) [67]. Khoản thu nhập này phải phát sinh ít nhất 1 năm trước ngày nộp đơn xin vay vốn [67]. Phương pháp tính toán: Thu nhập cổ tức hàng tháng ước tính = (Cổ tức của một năm trước đó / Số lượng cổ phiếu của năm trước đó) * (Số lượng cổ phiếu hiện tại) / 12 [65].",
    "Chứng từ yêu cầu bắt buộc: Biên bản cuộc họp Đại hội đồng cổ đông (chính sách chi trả cổ tức) [65]. Phiếu chi, Tài khoản chứng khoán hoặc Sao kê ngân hàng (bắt buộc chứng từ nào có thể chứng minh công ty đã trả cổ tức cho khách hàng) [65]. Giấy chứng nhận cổ đông và xác nhận bản gốc của công ty trong vòng 1 tháng [65].",
    "Màn hình nhập liệu Appendix 4.5 (Dividends) yêu cầu tick chọn Yes/No cho các tiêu chí: Thuộc VN500/Niêm yết, Không quá 40% tổng thu nhập, Có TSĐB. Yêu cầu nhập Mã số thuế (Tax code), Số lượng cổ phiếu từng năm (Number of shares), Cổ tức đã trả (Dividend paid), và Số cổ phần hiện tại đang sở hữu [68].",
    "Sau khi tính toán, Thu nhập từ cổ tức ước tính hàng tháng (Estimated monthly dividend income) sẽ được cộng trực tiếp vào mục (1d) Dividend incomes trong bảng Tổng thu nhập ròng hàng tháng (Total net monthly Income) [19]. Lưu ý kiểm tra chéo dòng tiền cổ tức này có thực sự chạy vào tài khoản ngân hàng cá nhân của khách hàng hay không [3].",
    # Medium Similarity
    "Thu nhập từ cổ tức khác với Thu nhập từ phần vốn góp trong công ty TNHH/Cổ phần chưa niêm yết. Đối với công ty chưa niêm yết, thu nhập ròng được tính bằng lợi nhuận của doanh nghiệp nhân với Tỷ lệ sở hữu (Shareholder %) [69]. Phần thu nhập này dựa trên báo cáo tài chính nội bộ hoặc thuế, không phải dựa trên quyết định chia cổ tức thực tế [69].",
    "Cần lưu ý về Giới hạn rủi ro tập trung cấp tín dụng (Group credit exposure limit). Phải tìm kiếm cả chủ doanh nghiệp và các công ty liên quan (Related parties) xem có khoản vay dư nợ gốc tại hệ thống SHBVN không [70]. Mối quan hệ nhóm khách hàng cần được khai báo rõ trên Appendix 5 [70].",
    "Một số khoản thu nhập khác (Other income) cũng được thẩm định tương tự cổ tức, chỉ áp dụng cho khoản vay có tài sản đảm bảo và phải phát sinh ổn định. Cần nhập kỳ nhận (Collecting period), Tỷ lệ chiết khấu (Discount ratio) để ra Thu nhập cuối cùng (Final income) [67].",
    # Low Similarity (Noise/Distractors)
    "Đối với thẻ tín dụng, DBR (Debt Burden Ratio) ước tính một khoản tương đương 5% số dư trung bình thẻ. Nếu khách hàng chậm trả thẻ tín dụng (Delinquency days), thẻ đó sẽ làm điểm uy tín giảm mạnh trên CIC [34]. Thẩm định viên ghi chú lý do chậm trả [34].",
    "Phân tích vòng quay vốn lưu động cho SOHO: Các khoản phải thu (Accounts Receivable) + Tồn kho (Inventory) - Các khoản phải trả (Accounts Payable) [53]. Công thức này giúp LO xác định nhu cầu vay vốn thực sự của tiểu thương, phòng ngừa rủi ro dùng sai mục đích vốn [53]."
]

# TEST CASE 10: XỬ LÝ NGOẠI LỆ VÀ PHÊ DUYỆT TÍN DỤNG (EXCEPTIONAL ITEMS)
query_10 = "Các quy định về điều khoản ngoại lệ, cấp thẩm quyền phê duyệt tín dụng (Approval Authority) và giới hạn cấp tín dụng."
documents_10 = [
    # High Similarity
    "Quy định nhập các điểm ngoại lệ (Exceptional Items) và cấp thẩm quyền phê duyệt tương ứng trên tờ trình tín dụng [60]. Ví dụ: Nếu Tổng số tiền vay của khách hàng (Total customer loan amount) > USD 400,000, bắt buộc cấp thẩm quyền phê duyệt phải là COC (Credit Officer Committee) [60]. Các giải trình ngoại lệ (Justification) phải được so sánh đối chiếu với Chính sách hiện hành hoặc Retail Guideline [60].",
    "Thông tin phê duyệt hệ thống (Approval Authority) sẽ được nhập vào màn hình 2347 sau khi khởi tạo CCRM (Hệ thống Xếp hạng Tín dụng Chấm điểm) [71]. Điểm xếp hạng (System rating result) sẽ quyết định ai là người có quyền ký duyệt cuối cùng, phụ thuộc vào Tỷ lệ DBR, LTV và Rủi ro chấm điểm [71].",
    "Đối với hạn mức tín dụng nhóm khách hàng liên quan (Group credit exposure limit), cần báo cáo vốn chủ sở hữu SHB (SHB owner's equity) và cập nhật thông báo của Risk Department từng thời kỳ [70]. Liệt kê danh sách Nhóm khách hàng (Group Customer) và Tỷ lệ trên vốn tự có (% of equity capital) để đảm bảo không vi phạm tỷ lệ tập trung tín dụng [70].",
    "Trên Bảng Đánh Giá SOHO (SOHO Assessment Table), hệ thống sẽ tính Điểm Tổng (Final Score) từ tiêu chí tài chính và phi tài chính [56]. Nếu Credit rating đạt hạng B hoặc thấp hơn (C, D), Quyết định phê duyệt tín dụng tự động sẽ Không được phép (Credit rating is B or below, GM - loan approval authority is not allowed) [56]. Lúc này cần trình cấp cao hơn hoặc ban rủi ro [56].",
    "Điều kiện áp dụng lãi suất: Sau khi giải ngân, khách hàng được hưởng Lãi suất ưu đãi năm đầu (First fixed rate), sau đó áp dụng Lãi suất nổi (Second fixed rate) theo công thức SPR + Margin. Phí phạt trả nợ trước hạn (Prepayment fee) được tính theo tỷ lệ quy định đối với dư nợ gốc giảm dần [58]. Các ngoại lệ về Lãi suất cũng phải trình duyệt [58].",
    # Medium Similarity
    "Ngoại lệ về chứng từ thu nhập: Nếu khách hàng nhận tiền mặt nhưng không cung cấp được Chứng từ quyết toán thuế (đối với lương > 40 triệu), hồ sơ sẽ bị Đánh hỏng tự động. Tuy nhiên, nếu khách nộp Sổ BHXH và sao kê tài khoản tiết kiệm định kỳ có thể trình ngoại lệ giám đốc chi nhánh tùy khẩu vị rủi ro [3].",
    "Ngoại lệ về độ tuổi doanh nghiệp SOHO: Điều kiện thông thường công ty thành lập 12-24 tháng. Nếu thời gian hoạt động dưới 12 tháng nhưng chủ hộ kinh doanh có kinh nghiệm quản lý ngành nghề tương đương > 5 năm, LO có thể nộp Justification xin cấp ngoại lệ phi tiêu chuẩn [72].",
    "Ngoại lệ về hạn mức thẻ tín dụng: Nếu tỷ lệ phụ thuộc nợ vay quá 100% nhưng số dư nợ chủ yếu là hạn mức thẻ tín dụng chưa sử dụng (Available Limit), KH có thể làm thủ tục hủy thẻ để giảm DBR và tái đệ trình hồ sơ [16].",
    # Low Similarity (Noise/Distractors)
    "Về kiểm tra tài sản thực địa: Cấu trúc nhà theo giấy tờ là 4 tầng, bê tông cốt thép, diện tích 100m2 (Urban land). LO phải ghi nhận số tầng thực tế, tình trạng nhà có bị đập phá, cơi nới sai phép hay không vào Verification Report [11]. Nếu sai phép, định giá sẽ trừ đi phần xây dựng sai [11].",
    "Thu nhập từ Doanh thu kế hoạch = Doanh thu năm trước cộng (Doanh thu năm trước nhân Tỷ lệ tăng trưởng bình quân 3 năm). Phương pháp này áp dụng riêng để tính toán vòng quay vốn lưu động cho SOHO, không dùng để duyệt vay mua nhà cá nhân [53]."
]

### Digital Banking

In [ ]:
# TEST CASE 1: Đăng ký eKYC và sinh trắc học trên SOL
query_1 = "Quy trình đăng ký tài khoản trực tuyến eKYC và xác thực sinh trắc học trên ứng dụng Shinhan SOL 3.0 diễn ra như thế nào?"
documents_1 = [
    # Top 1: Cực kỳ liên quan (Mô tả chi tiết các bước eKYC và sinh trắc học)
    """Để mở tài khoản trực tuyến (eKYC) trên ứng dụng Shinhan SOL 3.0, khách hàng cần chuẩn bị thẻ Căn cước công dân (CCCD) gắn chip và thiết bị di động hỗ trợ NFC. Quy trình bắt đầu bằng việc chụp ảnh hai mặt CCCD để trích xuất OCR. Tiếp theo, khách hàng đặt thẻ CCCD sát vào mặt lưng điện thoại để quét NFC, giúp ứng dụng đọc dữ liệu từ chip, đảm bảo tính chính xác và chống giả mạo. Sau đó, khách hàng thực hiện xác thực khuôn mặt (Face Verification) để hệ thống đối chiếu ảnh thực tế với dữ liệu của Bộ Công an. Cuối cùng, khách hàng bổ sung thông tin liên lạc và thiết lập tên đăng nhập, mật khẩu, mã PIN để hoàn tất mở tài khoản và đăng ký sinh trắc học.""",
    # Top 2: Rất liên quan (Đề cập đến đối tượng và tính năng eKYC)
    """Tính năng mở tài khoản và dịch vụ trực tuyến (eKYC) áp dụng cho khách hàng Việt Nam cư trú từ 15 tuổi trở lên. Thông qua VNEID và công nghệ NFC, người dùng không cần đến trực tiếp quầy giao dịch mà vẫn có thể mở thẻ ghi nợ hoặc tài khoản thanh toán. Trong quá trình này, sinh trắc học đóng vai trò quan trọng trong việc xác nhận danh tính, tuân thủ quy định an toàn bảo mật. Những khách hàng đã có thông tin sinh trắc học có thể sử dụng Face ID hoặc vân tay để đăng nhập nhanh chóng trong các lần tiếp theo thay vì phải nhập mật khẩu thủ công.""",
    # Top 3: Khá liên quan (Nói về cập nhật sinh trắc học cho khách hàng đã có tài khoản)
    """Đối với khách hàng nước ngoài hoặc những người chưa có CCCD gắn chip, quy trình đăng ký sinh trắc học lần đầu yêu cầu phải thực hiện tại quầy giao dịch (Aither) với sự hỗ trợ của giao dịch viên. Khi khách hàng mang Hộ chiếu hoặc giấy tờ tùy thân hợp lệ đến, nhân viên sẽ tiến hành thu thập hình ảnh khuôn mặt và liên kết với mã khách hàng (CIF). Sau khi hoàn tất tại quầy, khách hàng có thể mở ứng dụng SOL 3.0, chọn mục cập nhật sinh trắc học và làm theo hướng dẫn để đồng bộ hóa dữ liệu.""",
    # Top 4: Liên quan một phần (Trải nghiệm Pre-login trên SOL)
    """Ngay từ màn hình Pre-login (trước khi đăng nhập) của ứng dụng SOL, người dùng đã có thể trải nghiệm nhiều tiện ích mà chưa cần phải là khách hàng hiện hữu. Các tính năng này bao gồm tra cứu tỷ giá ngoại tệ, tính toán lãi suất khoản vay SOHO, tìm kiếm chi nhánh/ATM, và đặc biệt là nút chức năng 'Mở tài khoản trực tuyến'. Đây là điểm chạm đầu tiên, dẫn dắt khách hàng vào luồng eKYC để đăng ký sinh trắc học và trở thành người dùng chính thức của ngân hàng.""",
    # Top 5: Chứa từ khóa nhưng khác ngữ cảnh (Xác thực giao dịch thay vì mở tài khoản)
    """Sau khi đăng nhập (After-login), người dùng có thể thực hiện nhiều loại giao dịch như chuyển tiền, thanh toán hóa đơn, và mở sổ tiết kiệm. Tuy nhiên, đối với các giao dịch có giá trị lớn (ví dụ: chuyển khoản trên 20 triệu VNĐ/lần hoặc tổng hạn mức trên 20 triệu VNĐ/ngày), Ngân hàng Nhà nước yêu cầu bắt buộc phải xác thực bằng sinh trắc học (khuôn mặt). Điều này nhằm tăng cường tính an toàn, bảo vệ tài sản người dùng khỏi các nguy cơ gian lận trực tuyến.""",
    # Top 6: Thông tin cấu hình hệ thống
    """Chức năng quản lý thiết bị bảo mật trên ứng dụng cho phép người dùng thay đổi phương thức đăng nhập, cài đặt mã PIN mới, hoặc hủy dịch vụ khi bị mất thiết bị. Nếu người dùng nhập sai mật khẩu quá 5 lần, tài khoản sẽ tự động bị khóa và họ cần sử dụng tính năng 'Quên mật khẩu' để khôi phục lại, trong đó cũng sẽ cần đến việc quét khuôn mặt nếu đã đăng ký sinh trắc học trước đó.""",
    # Top 7: Ngữ cảnh doanh nghiệp (Không liên quan đến KH cá nhân)
    """Trong hệ thống ngân hàng điện tử dành cho Khách hàng Doanh nghiệp, người dùng Admin cũng cần phải đăng ký thông tin sinh trắc học. Việc này giúp xác minh danh tính người phê duyệt các lệnh chuyển tiền hoặc thay đổi cấu hình tài khoản. Dữ liệu sinh trắc học của người đại diện hợp pháp được thu thập và quản lý nghiêm ngặt nhằm đảm bảo an toàn cho các giao dịch có khối lượng lớn của tổ chức.""",
    # Top 8: Hoàn toàn không liên quan (Dịch vụ thu hộ)
    """Dịch vụ Nhớ thu chủ động (Bulk Collection) là giải pháp giúp doanh nghiệp thu hộ tiền từ nhiều đối tác, đại lý theo lịch trình định sẵn. Ngân hàng sẽ tự động trích nợ từ tài khoản của người mua (đã ký ủy quyền) và chuyển vào tài khoản doanh nghiệp, giúp tiết kiệm thời gian đối soát và giảm thiểu nợ xấu.""",
    # Top 9: Hoàn toàn không liên quan (Firm Banking)
    """Giải pháp Firm Banking kết nối trực tiếp hệ thống ERP của doanh nghiệp với Core Banking của ngân hàng thông qua API hoặc SFTP. Giải pháp này giúp tự động hóa quá trình lập lệnh chuyển tiền, nhận thông báo số dư, và tra cứu giao dịch mà không cần nhân sự kế toán thao tác thủ công trên cổng ngân hàng điện tử trực tuyến.""",
    # Top 10: Hoàn toàn không liên quan (Thẻ tín dụng)
    """Trên màn hình quản lý thẻ của ứng dụng SOL, khách hàng có thể kích hoạt thẻ mới, tra cứu sao kê, đổi mã PIN thẻ ATM/Credit, và đăng ký các dịch vụ trả góp. Khách hàng cũng có thể tạm khóa thẻ khẩn cấp khi nghi ngờ có rủi ro lộ thông tin, nhằm ngăn chặn các giao dịch quẹt thẻ hoặc thanh toán trực tuyến trái phép."""
]

# TEST CASE 2: Phân quyền CMS Doanh nghiệp
query_2 = "Trong hệ thống CMS doanh nghiệp, vai trò của người dùng Admin, Entry và Confirm được phân quyền như thế nào trong quy trình duyệt lệnh?"
documents_2 = [
    # Top 1
    """Trong hệ thống ngân hàng điện tử (CMS) dành cho doanh nghiệp, cơ chế phân quyền đa cấp được áp dụng để đảm bảo tính minh bạch và an toàn. Cụ thể, người dùng 'Admin' (Quản trị viên) có quyền cao nhất, chịu trách nhiệm cài đặt hệ thống, tạo và quản lý người dùng, cũng như thiết lập hạn mức giao dịch cho từng tài khoản. Tuy nhiên, Admin không được phép trực tiếp lập hay duyệt lệnh chuyển tiền. Quyền lập lệnh được giao cho người dùng 'Entry' (Người nhập liệu, thường là kế toán viên). Sau khi Entry tạo lệnh, lệnh sẽ chuyển sang trạng thái chờ duyệt. Lúc này, người dùng 'Confirm' (Người phê duyệt, thường là kế toán trưởng hoặc giám đốc) sẽ kiểm tra thông tin, xác thực bằng Token hoặc OTP và duyệt để lệnh được thực thi.""",
    # Top 2
    """Việc tách biệt chức năng giữa các vai trò Entry và Confirm giúp doanh nghiệp kiểm soát rủi ro giao dịch. Người dùng Entry chỉ có thể xem danh sách tài khoản được chỉ định và khởi tạo các giao dịch như chuyển tiền, thanh toán lương, hoặc nộp thuế. Trong khi đó, người dùng Confirm có thẩm quyền phê duyệt cuối cùng. Hệ thống cho phép thiết lập nhiều cấp duyệt (Ví dụ: Confirm 1, Confirm 2) tùy thuộc vào quy mô và quy trình nội bộ của doanh nghiệp. Một doanh nghiệp có thể đăng ký nhiều người dùng Entry và Confirm, nhưng số lượng Admin thường được giới hạn (tối đa 1 hoặc 2 người).""",
    # Top 3
    """Khi tiến hành đăng ký sử dụng dịch vụ CMS, doanh nghiệp phải điền form đăng ký chỉ định rõ ai sẽ nắm giữ vai trò Admin. Người phụ trách kế toán không được phép đồng thời là người phê duyệt cuối cùng trong các tuyến giao dịch để tránh xung đột lợi ích. Mỗi user khi được tạo trên hệ thống sẽ được cấp một ID riêng và phương thức xác thực bảo mật tương ứng như M-OTP, A-OTP hoặc thiết bị Token vật lý để ký duyệt các giao dịch tài chính.""",
    # Top 4
    """Đối với mô hình doanh nghiệp siêu nhỏ (Micro-Enterprise), cơ chế phân quyền này được đơn giản hóa để giảm bớt rào cản hành chính. Những doanh nghiệp này có thể đăng ký sử dụng dịch vụ ngân hàng điện tử theo gói 'Single User' (Một người dùng duy nhất). Trong mô hình này, người dùng (thường là chủ doanh nghiệp) vừa có quyền lập lệnh vừa có quyền tự phê duyệt lệnh chuyển tiền của chính mình mà không cần qua các bước Confirm phức tạp như doanh nghiệp lớn.""",
    # Top 5
    """Ngoài chức năng chuyển tiền, người dùng được phân quyền trên CMS còn có thể tra cứu sao kê tài khoản, theo dõi tình trạng các khoản vay, và tải xuống các báo cáo tài chính. Tùy thuộc vào thiết lập của Admin, người dùng Entry có thể bị giới hạn chỉ xem được lịch sử giao dịch của một số tài khoản cụ thể (ví dụ: tài khoản chi lương được bảo mật, chỉ có Admin hoặc Giám đốc mới được xem).""",
    # Top 6
    """Giải pháp Firm Banking tự động hóa việc đẩy lệnh từ hệ thống ERP sang hệ thống ngân hàng. Khi sử dụng Firm Banking, quy trình duyệt lệnh Entry/Confirm truyền thống có thể được thực hiện ngay trên phần mềm ERP nội bộ của doanh nghiệp. Lệnh sau khi được duyệt cuối cùng trên ERP sẽ được ký bằng chứng thư số (Digital Signature) và truyền an toàn qua kết nối SFTP/API tới ngân hàng để xử lý tự động.""",
    # Top 7
    """Để hỗ trợ việc nhận diện dòng tiền tự động, ngân hàng cung cấp dịch vụ Mã số định danh (Virtual Account). Khi khách hàng thanh toán vào VA, hệ thống sẽ tự động cập nhật trạng thái 'Đã thu tiền' trên CMS. Người dùng kế toán (Entry) có thể dễ dàng xuất báo cáo đối soát từ hệ thống thay vì phải lọc sao kê thủ công theo nội dung chuyển khoản.""",
    # Top 8
    """Đối với khách hàng cá nhân sử dụng SOL 3.0, việc phân quyền này hoàn toàn không tồn tại. Một tài khoản cá nhân có toàn quyền xem số dư, chuyển khoản, và thanh toán hóa đơn. Ngân hàng chỉ quản lý dựa trên hạn mức giao dịch tối đa trong ngày và yêu cầu xác thực bảo mật mạnh (Sinh trắc học hoặc OTP) đối với các khoản tiền lớn.""",
    # Top 9
    """Ngân hàng triển khai công nghệ eKYC và NFC để hỗ trợ khách hàng cá nhân mở tài khoản từ xa một cách nhanh chóng. Việc đọc dữ liệu trên chip CCCD đảm bảo thông tin cá nhân được trích xuất chính xác 100%, ngăn ngừa các hành vi giả mạo giấy tờ và tuân thủ các quy định về phòng chống rửa tiền.""",
    # Top 10
    """Trên ứng dụng dành cho thiết bị di động, Menu điều hướng (Navigation) được thiết kế trực quan, chia thành các nhóm Dịch vụ Ngân hàng chính, Quản lý Thẻ, Chăm sóc khách hàng và Cài đặt. Khách hàng có thể dễ dàng vuốt và chạm để thao tác các chức năng cần thiết mọi lúc mọi nơi."""
]

# TEST CASE 3: Firm Banking và ERP
query_3 = "Giải pháp Firm Banking kết nối với hệ thống ERP của doanh nghiệp hoạt động như thế nào và mang lại những lợi ích gì trong quản lý dòng tiền?"
documents_3 = [
    # Top 1
    """Firm Banking là hình thức kết nối trực tiếp giữa hệ thống phần mềm quản trị doanh nghiệp (ERP) hoặc phần mềm kế toán của công ty với Core Banking của ngân hàng. Thông qua các phương thức truyền tin bảo mật như SFTP (truyền file dữ liệu mã hóa số lượng lớn) hoặc API (giao tiếp dữ liệu thời gian thực), các lệnh thanh toán, chuyển tiền nội bộ, chuyển tiền liên ngân hàng, và nộp thuế có thể được tự động đẩy từ ERP sang ngân hàng mà không cần kế toán phải đăng nhập thủ công vào CMS. Khi giao dịch hoàn tất, ngân hàng sẽ trả trạng thái (thành công/thất bại) và dữ liệu sao kê ngược lại ERP, giúp doanh nghiệp đồng bộ hóa sổ sách kế toán tức thời.""",
    # Top 2
    """Lợi ích cốt lõi của Firm Banking là giúp doanh nghiệp tăng hiệu quả vận hành, quản lý dòng tiền chính xác và giảm thiểu rủi ro con người. Thay vì phải tải file Excel từ phần mềm kế toán và upload lên hệ thống ngân hàng điện tử, toàn bộ quy trình lập lệnh và duyệt lệnh đều được thực hiện tập trung trên hệ thống nội bộ của công ty. Điều này đặc biệt hữu ích cho các tập đoàn, tổng công ty có số lượng giao dịch khổng lồ (xử lý theo lô) mỗi ngày. Ngoài ra, việc kết nối này giúp bảo mật thông tin tài chính cao hơn do loại bỏ các bước thao tác dữ liệu trung gian.""",
    # Top 3
    """Để triển khai Firm Banking, ngân hàng Shinhan cung cấp tài liệu kỹ thuật chuẩn để đội ngũ IT của doanh nghiệp thực hiện lập trình kết nối. Nếu doanh nghiệp chưa có đội ngũ IT mạnh, họ có thể sử dụng giải pháp từ các đối tác Vendor đã tích hợp sẵn (như FAST, MISA, AMNote, VNPAY). Quy trình đăng ký bao gồm: nộp đơn yêu cầu, kết nối kỹ thuật thử nghiệm (môi trường UAT), và sau khi nghiệm thu thành công sẽ chính thức Golive để chạy dữ liệu thật.""",
    # Top 4
    """Dịch vụ Firm Banking không chỉ hỗ trợ chiều chi (chuyển tiền) mà còn cực kỳ hiệu quả ở chiều thu khi kết hợp với Dịch vụ Mã số định danh (Virtual Account - VA). Khi đối tác thanh toán tiền vào VA, ngân hàng sẽ đẩy webhook (thông báo API) ngay lập tức về ERP. Hệ thống ERP nhận được thông tin sẽ tự động gạch nợ cho đúng đối tác, đúng hóa đơn mà không cần sự can thiệp của kế toán công nợ.""",
    # Top 5
    """Chi phí để thiết lập và duy trì Firm Banking thường bao gồm phí tích hợp hệ thống ban đầu, phí duy trì kết nối hàng tháng, và biểu phí giao dịch chuyển khoản tùy theo thỏa thuận. Các doanh nghiệp lớn thường sử dụng đồng thời Firm Banking cho khối lượng giao dịch tự động và CMS truyền thống để thực hiện các giao dịch mang tính chất bất thường hoặc quản trị hệ thống.""",
    # Top 6
    """Trong hệ thống CMS thông thường (Internet Banking dành cho doanh nghiệp), để xử lý các lô giao dịch lớn như trả lương, người dùng Entry phải tạo mẫu file Excel theo chuẩn của ngân hàng, điền hàng nghìn số tài khoản nhân viên, và tải file lên hệ thống (Bulk Transfer). Sau đó người dùng Confirm đăng nhập để ký duyệt. Cách làm này vẫn mất nhiều bước thao tác thủ công so với Firm Banking.""",
    # Top 7
    """Ngân hàng cung cấp dịch vụ Nhớ thu chủ động (Bulk Collection), cho phép doanh nghiệp định kỳ thu các khoản phí cố định (như phí quản lý, học phí, tiền điện nước) từ tài khoản của khách hàng. Tuy nhiên, để thực hiện được, người thanh toán (payer) phải điền đơn ủy quyền trích nợ tự động nộp cho ngân hàng. Dịch vụ này khác với Firm Banking, dù cũng giúp doanh nghiệp quản lý dòng tiền thu hiệu quả.""",
    # Top 8
    """Đối với việc phân quyền trên hệ thống, Admin của doanh nghiệp chịu trách nhiệm khai báo thông tin người dùng, giới hạn quyền truy cập từng tài khoản. Tính năng phân quyền (Entry, Confirm) vẫn được duy trì nghiêm ngặt để đảm bảo an ninh, tránh việc một cá nhân có thể lạm quyền thực hiện các giao dịch chiếm đoạt tài sản của tổ chức.""",
    # Top 9
    """Người dùng cá nhân không được cấp Firm Banking mà sử dụng ứng dụng SOL 3.0 trên điện thoại. Với SOL, họ có thể sử dụng mã QR để thanh toán tại các cửa hàng, chuyển khoản nhanh Napas 24/7, và quản lý các loại thẻ tín dụng. Trải nghiệm được tối ưu hóa cho màn hình cảm ứng di động với giao diện thân thiện và trực quan.""",
    # Top 10
    """Để tăng cường bảo mật cho khách hàng cá nhân khi sử dụng ngân hàng số, hệ thống tích hợp sinh trắc học thông qua công nghệ eKYC và Face ID. Bằng việc đọc chip trên thẻ CCCD qua NFC, ngân hàng có thể xác thực danh tính khách hàng một cách chính xác tuyệt đối, giảm thiểu rủi ro mạo danh mở tài khoản để thực hiện các hành vi lừa đảo trực tuyến."""
]

# TEST CASE 4: Mã số định danh (Virtual Account)
query_4 = "Mã số định danh (Virtual Account - VA) là gì và làm thế nào để doanh nghiệp sử dụng VA để tự động hóa việc đối soát công nợ?"
documents_4 = [
    # Top 1
    """Mã số định danh (Virtual Account - VA) là một dải mã số được ngân hàng cấp cho doanh nghiệp, từ đó doanh nghiệp có thể tạo ra hàng ngàn 'Tài khoản ảo' gắn với một 'Tài khoản thực' (Tài khoản thanh toán mẹ) của mình. Mỗi khách hàng, đối tác hoặc mỗi đơn hàng của doanh nghiệp sẽ được cấp một mã VA duy nhất (ví dụ: kèm tiền tố mã công ty + số điện thoại khách hàng). Khi khách hàng chuyển tiền thanh toán vào mã VA này, tiền thực chất sẽ chảy trực tiếp vào tài khoản mẹ của doanh nghiệp. Hệ thống ngân hàng ngay lập tức ghi nhận giao dịch, nhận diện chính xác nguồn tiền từ mã VA nào, và gửi thông báo hoặc đẩy dữ liệu báo cáo đối soát tự động cho doanh nghiệp.""",
    # Top 2
    """Lợi ích cốt lõi của Virtual Account là giúp doanh nghiệp tự động phân loại giao dịch, gạch nợ chính xác nguồn tiền thu về đối với từng công nợ cụ thể. Thay vì kế toán phải nhìn vào nội dung chuyển khoản (thường hay bị khách hàng viết sai, thiếu mã đơn hàng) để tìm kiếm và gạch nợ thủ công, VA giúp đối soát chính xác 100%. Tiết kiệm thời gian, giảm thiểu rủi ro sai sót, và không cần người dùng nhập nội dung chuyển khoản phức tạp, giúp chu kỳ thu tiền nhanh hơn.""",
    # Top 3
    """Trên hệ thống quản lý giao dịch nội bộ (Aither) hoặc CMS, người dùng có quyền Admin hoặc quản lý có thể thao tác phát hành mã VA mới (Virtual Code Issuance), tra cứu lịch sử phát hành, và phân bổ VA cho các đại lý. Khi tích hợp với hệ thống ERP thông qua API, doanh nghiệp có thể gọi lệnh tạo mã VA real-time ngay khi khách hàng bấm nút 'Thanh toán' trên website thương mại điện tử, giúp trải nghiệm mua sắm mượt mà.""",
    # Top 4
    """Dịch vụ Thu qua mã số định danh thường được ứng dụng rộng rãi tại các trường học (thu học phí), công ty quản lý tòa nhà (thu phí dịch vụ chung cư), công ty bảo hiểm, hoặc các chuỗi bán lẻ. Với các mô hình này, việc thu tiền lắt nhắt từ hàng ngàn cá nhân mỗi tháng sẽ trở thành gánh nặng khổng lồ nếu không có hệ thống nhận diện tự động như VA.""",
    # Top 5
    """Doanh nghiệp sử dụng dịch vụ VA có thể nhận được báo cáo đối soát chi tiết dưới nhiều hình thức: thông báo trên hệ thống CMS, nhận file báo cáo định kỳ theo giờ/ngày qua email, hoặc nhận dữ liệu trực tiếp vào hệ thống ERP qua đường truyền Firm Banking. Ngân hàng áp dụng biểu phí duy trì dịch vụ VA hàng tháng và phí thu hộ tính trên mỗi giao dịch chuyển tiền vào VA thành công.""",
    # Top 6
    """Khác với VA là hình thức người nộp chủ động chuyển tiền, dịch vụ Nhớ thu chủ động (Bulk Collection) lại cho phép doanh nghiệp (người thu) chủ động tạo lệnh trích tiền từ tài khoản của người nộp. Để sử dụng Bulk Collection, người nộp bắt buộc phải ký một Thỏa thuận ủy quyền trích nợ tự động tại quầy giao dịch, trong khi VA không yêu cầu thủ tục phức tạp từ phía người nộp tiền.""",
    # Top 7
    """Để thực hiện một giao dịch chuyển tiền đi (chi tiền) trên ngân hàng điện tử, người dùng kế toán (Entry) cần tạo lệnh, điền số tài khoản người nhận, số tiền, và chọn tài khoản trích nợ. Nếu giao dịch lớn, lệnh sẽ được chuyển lên cho người dùng Confirm để phê duyệt bằng Token. Việc quản lý luồng chi này cần tuân thủ nghiêm ngặt các quy định phân quyền nội bộ.""",
    # Top 8
    """Bên cạnh các giải pháp thu chi tự động, tính năng thanh toán hóa đơn (Bill Payment) và nộp thuế điện tử trên CMS hỗ trợ doanh nghiệp hoàn thành nghĩa vụ tài chính với nhà nước một cách dễ dàng, thay thế cho việc phải ra nộp tiền mặt tại Kho bạc hoặc quầy giao dịch ngân hàng truyền thống.""",
    # Top 9
    """Với ứng dụng SOL 3.0 dành cho Khách hàng cá nhân, tính năng quét mã QR thanh toán đang trở nên vô cùng phổ biến. Người dùng chỉ cần đưa camera lên quét mã QR chuẩn VietQR để chuyển khoản chính xác tới người bán mà không cần gõ tay số tài khoản hay số tiền, giảm thiểu thao tác trên điện thoại di động.""",
    # Top 10
    """Khi thiết lập bảo mật trên ứng dụng, người dùng có thể kích hoạt Sinh trắc học (vân tay hoặc nhận diện khuôn mặt). Điều này tiện lợi hơn rất nhiều so với việc phải ghi nhớ và nhập mật khẩu tĩnh mỗi lần đăng nhập. Tính năng eKYC kết nối dữ liệu VNEID cũng góp phần tạo nên một hệ sinh thái số an toàn, bảo mật tuyệt đối cho mọi khách hàng."""
]

# TEST CASE 5: Dịch vụ Nhớ thu chủ động (Bulk Collection)
query_5 = "Dịch vụ Nhớ thu chủ động (Bulk Collection) hoạt động ra sao và cần những thủ tục gì để doanh nghiệp có thể tự động trích tiền từ tài khoản người nộp?"
documents_5 = [
    # Top 1
    """Dịch vụ Nhớ thu chủ động (Bulk Collection) là giải pháp giúp doanh nghiệp (Bên thu tiền) chủ động khởi tạo lệnh trích nợ từ các tài khoản của đối tác, khách hàng (Bên thanh toán) đã mở tại ngân hàng Shinhan để thu các khoản phí định kỳ (như học phí, phí dịch vụ, hóa đơn...). Để triển khai, điều kiện bắt buộc là Bên thanh toán phải điền form và ký xác nhận vào văn bản 'Ủy quyền trích nợ tự động' (Debit Mandate) nộp cho ngân hàng. Sau khi ngân hàng ghi nhận ủy quyền lên hệ thống, doanh nghiệp có thể tạo file danh sách số tiền cần thu và tải lên CMS (hoặc đẩy qua API/Firm Banking). Hệ thống sẽ tự động quét và trừ tiền hàng loạt từ tài khoản khách hàng để ghi có vào tài khoản doanh nghiệp.""",
    # Top 2
    """Lợi ích lớn nhất của Bulk Collection là giúp doanh nghiệp chủ động lập kế hoạch dòng tiền, giảm thiểu nợ xấu và tình trạng khách hàng chây ì thanh toán. Kế toán không cần gọi điện nhắc nợ hay chờ đợi khách hàng chuyển khoản. Việc thu tiền được thực hiện hoàn toàn tự động, nhanh chóng thu gọn kỳ thu tiền, và giúp tiết kiệm thời gian, nhân lực xử lý. Phía người nộp cũng không lo bị quên đóng phí dẫn đến bị phạt trễ hạn.""",
    # Top 3
    """Thủ tục đăng ký Bulk Collection bao gồm việc doanh nghiệp ký hợp đồng sử dụng dịch vụ với ngân hàng. Trên hệ thống (màn hình Aither), Giao dịch viên sẽ tạo danh mục đăng ký thu hộ. Đối với người nộp tiền, họ cần cung cấp thông tin tài khoản, CMND/CCCD, và điền Mẫu Đăng ký Dịch vụ Ủy quyền Ghi nợ (Application for Debit Authorization). Nếu số dư tài khoản người nộp không đủ tại thời điểm trích nợ, giao dịch thu hộ sẽ báo lỗi 'Insufficient fund'.""",
    # Top 4
    """Quy trình vận hành Bulk Collection trên màn hình hệ thống bao gồm các bước: Nhân viên ngân hàng (hoặc qua CMS) đăng nhập vào tính năng Quản lý Nhớ thu (Manage Account for Bulk Collection), thực hiện truy vấn các lệnh thu hộ. File danh sách thu sẽ được kiểm tra định dạng Excel, map các cột dữ liệu như Số tài khoản khách hàng, Tên khách hàng, Số tiền, và Nội dung. Sau đó, hệ thống sẽ chạy batch xử lý và xuất ra báo cáo đối soát kết quả thu hộ (Thành công/Thất bại) cho doanh nghiệp.""",
    # Top 5
    """Một giải pháp thu tiền khác là Mã số định danh (Virtual Account - VA). Điểm khác biệt lớn nhất là với VA, khách hàng chủ động thực hiện chuyển tiền (thường qua Internet Banking của bất kỳ ngân hàng nào) vào số VA. Còn với Bulk Collection, doanh nghiệp là bên kéo tiền về. VA phù hợp với thương mại điện tử, bán lẻ; trong khi Bulk Collection phù hợp với các dịch vụ có tính chất chu kỳ cố định như tiện ích, chung cư, viễn thông.""",
    # Top 6
    """Giải pháp Firm Banking hỗ trợ truyền tải cả dữ liệu thu (VA, Bulk Collection) và chi (Bulk Transfer) một cách tự động, bảo mật cao bằng cách kết nối thẳng vào ERP. Điều này biến ERP của doanh nghiệp trở thành một trạm điều khiển tài chính tập trung, giám sát toàn bộ dòng tiền ra vào theo thời gian thực mà không phụ thuộc vào cổng CMS truyền thống.""",
    # Top 7
    """Khi lập lệnh chuyển tiền lô (Bulk Transfer) trên CMS để chi lương cho nhân viên, doanh nghiệp cũng tải lên một file Excel chứa hàng trăm số tài khoản. Người dùng Entry thực hiện import file, hệ thống sẽ kiểm tra tính hợp lệ của số tài khoản thụ hưởng. Sau khi Confirm duyệt lệnh bằng Token, ngân hàng sẽ thực hiện ghi nợ tài khoản công ty một lần và ghi có vào từng tài khoản nhân viên.""",
    # Top 8
    """Các cấp người dùng trong hệ thống CMS được chia thành Admin (Quản trị), Entry (Nhập liệu) và Confirm (Phê duyệt). Đối với các giao dịch tài chính quan trọng, nguyên tắc maker - checker (người tạo - người duyệt) giúp đảm bảo sự minh bạch nội bộ và ngăn ngừa hành vi gian lận tài chính trong doanh nghiệp.""",
    # Top 9
    """Khách hàng cá nhân khi sử dụng ứng dụng SOL có thể sử dụng tính năng Chuyển tiền tự động (Auto Transfer). Tính năng này cho phép cá nhân cài đặt lịch chuyển một số tiền cố định định kỳ hàng tháng cho người thân hoặc thanh toán hóa đơn. Nó hoạt động tương tự cơ chế trích nợ, nhưng hoàn toàn dựa trên sự tự nguyện thiết lập trên điện thoại di động của người dùng.""",
    # Top 10
    """Quy trình eKYC với công nghệ nhận diện khuôn mặt và đọc thẻ CCCD qua NFC giúp nâng cao tính bảo mật. Việc ngân hàng áp dụng bắt buộc sinh trắc học cho các giao dịch lớn giúp chặn đứng rủi ro giả mạo danh tính, bảo vệ tài khoản ngân hàng của khách hàng cá nhân khỏi các thủ đoạn lừa đảo tinh vi."""
]

# TEST CASE 6: Thiết bị bảo mật (Token, OTP)
query_6 = "Sự khác biệt giữa các thiết bị bảo mật M-OTP, A-OTP và Token là gì và chúng được áp dụng cho các hạn mức giao dịch chuyển tiền nào?"
documents_6 = [
    # Top 1
    """Hệ thống ngân hàng sử dụng nhiều phương thức xác thực bảo mật khác nhau tùy thuộc vào đối tượng khách hàng và hạn mức giao dịch. M-OTP (Mobile OTP) là phương thức mặc định cho khách hàng cá nhân, mã OTP được sinh ra ngay trên ứng dụng điện thoại, phục vụ các giao dịch hạn mức tiêu chuẩn. A-OTP (Advanced OTP) là phiên bản nâng cao, thường được kích hoạt cho các giao dịch có hạn mức lớn hơn, kết hợp với nhận diện sinh trắc học để tăng cường an ninh. Trong khi đó, Token (Thiết bị tạo mã cứng) là phương thức bảo mật cấp cao nhất, thường cấp dưới dạng thiết bị vật lý chuyên biệt, bắt buộc dành cho khách hàng Doanh nghiệp (người dùng Confirm) để phê duyệt các lệnh chuyển tiền giá trị cực lớn.""",
    # Top 2
    """Sự phân chia thiết bị bảo mật đi liền với biểu hạn mức giao dịch. Đối với cá nhân, việc sử dụng M-OTP có thể bị giới hạn giao dịch ở mức dưới 500 triệu đồng/ngày. Nếu khách hàng muốn nâng hạn mức lên hàng tỷ đồng (Ví dụ: chuyển tiền thanh toán bất động sản), họ bắt buộc phải đăng ký sử dụng thiết bị A-OTP hoặc áp dụng công nghệ nhận diện khuôn mặt theo quy định mới của NHNN để đảm bảo giao dịch do chính chủ thực hiện. Đối với doanh nghiệp, Token vật lý hoặc Soft Token (ứng dụng sinh mã độc lập) là bắt buộc để mã hóa chữ ký điện tử cho mỗi giao dịch duyệt lệnh.""",
    # Top 3
    """Để cài đặt thiết bị bảo mật trên ứng dụng SOL, người dùng vào phần Cài đặt > Quản lý thiết bị bảo mật. Nếu nhập sai mã PIN hoặc OTP quá 5 lần, thiết bị bảo mật hoặc tài khoản sẽ tự động bị khóa (Accident Code) nhằm ngăn chặn hacker xâm nhập. Khách hàng cá nhân có thể phải sử dụng tính năng 'Quên mật khẩu' hoặc thực hiện eKYC khuôn mặt lại để mở khóa tài khoản, trong khi khách hàng doanh nghiệp có thể yêu cầu Admin reset lại quyền truy cập hoặc cấp lại Token.""",
    # Top 4
    """Trong quy trình vận hành CMS doanh nghiệp, bước thiết lập phân quyền (Entry, Confirm) do Admin thực hiện vô cùng quan trọng. Mỗi người dùng Confirm khi được tạo tài khoản sẽ được cấp một mã số Token tương ứng. Khi Entry tạo xong lệnh chuyển tiền, lệnh đó chỉ nằm ở trạng thái 'Chờ duyệt'. Người dùng Confirm đăng nhập, kiểm tra thông tin, bấm Duyệt và phải nhập chuỗi số động gồm 6 chữ số đang hiển thị trên màn hình máy Token vật lý để hoàn tất.""",
    # Top 5
    """Đối với doanh nghiệp siêu nhỏ sử dụng gói dịch vụ Single User, do chỉ có một người dùng thực hiện cả thao tác tạo lệnh và duyệt lệnh, hệ thống có thể cung cấp giải pháp xác thực bằng sinh trắc học kết hợp SMS OTP hoặc Soft Token trực tiếp trên ứng dụng di động, giúp họ phê duyệt giao dịch một cách linh hoạt, nhanh chóng mà không cần mang theo Token vật lý rườm rà.""",
    # Top 6
    """Nếu khách hàng cá nhân muốn lấy lại mật khẩu đăng nhập, quy trình bắt buộc phải xác thực đa yếu tố. Hệ thống sẽ yêu cầu nhập Tên đăng nhập (ID), quét khuôn mặt Face ID và nhập mã OTP gửi về số điện thoại đã đăng ký (SMS OTP). SMS OTP ở đây đóng vai trò như một kênh xác thực dự phòng ngoài các phương thức M-OTP tích hợp sẵn trong ứng dụng.""",
    # Top 7
    """Giải pháp Firm Banking không sử dụng Token cá nhân để duyệt từng lệnh. Thay vào đó, dữ liệu truyền từ ERP sang Ngân hàng được mã hóa và xác thực bằng Chứng thư số (Digital Certificate) của tổ chức. Hệ thống sẽ tự động đối chiếu chữ ký điện tử này, nếu khớp chuẩn bảo mật, các lệnh thanh toán theo lô sẽ tự động được giải quyết mà không cần ai bấm Token duyệt lệnh nữa.""",
    # Top 8
    """Virtual Account (Mã số định danh) chỉ đóng vai trò là một tài khoản ảo để nhận tiền và phân loại dòng thu. Các giao dịch chuyển tiền từ người thanh toán vào VA được xác thực bảo mật bởi chính ngân hàng của người thanh toán đó, ngân hàng trích nợ không yêu cầu bên nhận (doanh nghiệp) phải sử dụng Token để xác thực nhận tiền.""",
    # Top 9
    """Dịch vụ Nhớ thu chủ động (Bulk Collection) dựa trên cơ chế ủy quyền trước. Khi người nộp đã ký đơn đồng ý cho phép tự động trích nợ, doanh nghiệp chỉ cần đẩy file danh sách thu tiền lên hệ thống. Việc trích tiền từ tài khoản người nộp sẽ diễn ra tự động mà người nộp không cần phải nhập mã OTP cho mỗi lần đóng phí định kỳ.""",
    # Top 10
    """Quy trình eKYC là một bước đột phá giúp xác thực danh tính khách hàng bằng cách so khớp khuôn mặt với dữ liệu hình ảnh lưu trong chip CCCD qua kết nối NFC. Tuy công nghệ này rất an toàn cho việc định danh, nhưng trong vận hành hàng ngày, ngân hàng vẫn duy trì hệ thống mật khẩu và OTP làm hàng rào bảo vệ nhiều lớp cho từng giao dịch biến động số dư."""
]

# TEST CASE 7: Quy trình Quên mật khẩu
query_7 = "Khi khách hàng cá nhân quên mật khẩu đăng nhập ứng dụng SOL 3.0, họ cần thực hiện những bước nào để lấy lại mật khẩu và thiết lập lại bảo mật?"
documents_7 = [
    # Top 1
    """Khi khách hàng cá nhân quên mật khẩu đăng nhập ứng dụng SOL 3.0, họ có thể tự lấy lại mật khẩu bằng cách chọn tính năng 'Quên mật khẩu' ngay trên màn hình Pre-login. Hệ thống sẽ yêu cầu khách hàng nhập User ID (Tên đăng nhập) và Số giấy tờ tùy thân (CCCD/Hộ chiếu). Tiếp theo, để chứng minh danh tính thật, khách hàng phải thực hiện xác thực sinh trắc học (quét khuôn mặt) để so khớp với dữ liệu đã lưu. Sau khi khuôn mặt được xác thực thành công, hệ thống gửi mã OTP xác nhận về số điện thoại. Cuối cùng, khách hàng sẽ được điều hướng đến màn hình thiết lập mật khẩu mới, nhập mật khẩu mới 2 lần để hoàn tất và có thể đăng nhập lại ngay.""",
    # Top 2
    """Việc bắt buộc sử dụng xác thực khuôn mặt trong quy trình cấp lại mật khẩu giúp ngăn chặn triệt để rủi ro kẻ gian đánh cắp điện thoại và cố tình đổi mật khẩu hòng chiếm đoạt tài sản. Trong trường hợp khách hàng nhập sai mật khẩu quá 5 lần hoặc sai mã PIN bảo mật nhiều lần, tài khoản SOL sẽ tự động chuyển sang trạng thái bị khóa. Lúc này, quy trình 'Quên mật khẩu/Mở khóa tài khoản' trực tuyến cũng áp dụng các bước xác thực sinh trắc học tương tự để tự động mở khóa.""",
    # Top 3
    """Nếu khách hàng chưa từng đăng ký sinh trắc học trên ứng dụng (hoặc là khách hàng nước ngoài không có CCCD gắn chip), họ không thể hoàn tất quy trình lấy lại mật khẩu trực tuyến 100%. Trong trường hợp này, ứng dụng sẽ hiện thông báo hướng dẫn khách hàng mang theo giấy tờ tùy thân (Hộ chiếu/CMND) đến chi nhánh hoặc quầy giao dịch gần nhất để giao dịch viên hỗ trợ cấp lại mật khẩu mới hoặc định danh lại dữ liệu sinh trắc học.""",
    # Top 4
    """Trong phần Cài đặt của ứng dụng SOL, người dùng có thể tùy chỉnh các phương thức đăng nhập và bảo mật. Sau khi lấy lại mật khẩu thành công, khách hàng nên truy cập vào menu này để cài đặt đăng nhập bằng Face ID hoặc Vân tay (Biometric Login), giúp các lần truy cập ứng dụng tiếp theo diễn ra thuận tiện, nhanh chóng mà không cần phải ghi nhớ chuỗi mật khẩu phức tạp.""",
    # Top 5
    """Việc ứng dụng công nghệ eKYC đọc thẻ CCCD qua NFC và đối chiếu khuôn mặt với dữ liệu Bộ Công an không chỉ dùng để mở tài khoản mới mà còn được tái sử dụng trong các tác vụ bảo mật quan trọng như cấp lại mật khẩu, nâng hạn mức giao dịch, hoặc thay đổi thiết bị đăng nhập (cài app trên điện thoại mới).""",
    # Top 6
    """Đối với người dùng trong hệ thống CMS Doanh nghiệp (Entry, Confirm), nếu họ quên mật khẩu, quy trình lấy lại mật khẩu sẽ khác biệt. Họ không thể dùng app SOL để quét mặt, mà phải báo cáo cho người quản trị (Admin) của công ty. Admin có quyền truy cập vào hệ thống, chọn tài khoản người dùng tương ứng và nhấn nút 'Reset Password' để hệ thống cấp lại một mật khẩu tạm thời gửi qua email của nhân viên đó.""",
    # Top 7
    """Mọi giao dịch chuyển tiền (nội bộ, ngoài hệ thống, chuyển tiền quốc tế) đều yêu cầu nhập mã PIN bảo mật hoặc xác thực M-OTP/A-OTP ở bước cuối cùng. Do đó, việc bảo vệ mật khẩu đăng nhập và thông tin sinh trắc học là yếu tố then chốt để khóa chặt các cánh cửa truy cập vào tài sản của người dùng.""",
    # Top 8
    """Doanh nghiệp sử dụng Firm Banking được cung cấp một kênh truyền dữ liệu bảo mật kết nối trực tiếp với Core Banking. Quá trình xác thực kênh truyền dựa trên địa chỉ IP, chứng thư số, và phương thức mã hóa, do đó không tồn tại khái niệm 'Quên mật khẩu' cho các lệnh giao dịch API tự động, mọi thiết lập bảo mật đều ở tầng hạ tầng mạng.""",
    # Top 9
    """Sử dụng dịch vụ Nhớ thu chủ động (Bulk Collection), doanh nghiệp cần tải lên các file danh sách thu tiền theo mẫu Excel quy định. Khi trạng thái giao dịch hiển thị lỗi tài khoản không đủ tiền hoặc sai thông tin, doanh nghiệp cần làm việc lại với đối tác thanh toán để cập nhật dữ liệu, chứ không liên quan đến việc reset mật khẩu tài khoản.""",
    # Top 10
    """Giải pháp Virtual Account (Mã số định danh) cấp cho doanh nghiệp hàng ngàn mã số gắn với tài khoản mẹ. Các mã số này có thể được gán tĩnh cho từng khách hàng hoặc sinh động theo từng đơn hàng. Tiện ích này tối ưu hóa việc hạch toán công nợ, tiết kiệm chi phí vận hành cho bộ phận kế toán so với việc dò thủ công bằng mắt nội dung chuyển khoản."""
]

#Summary

NOTE:
- BGE-Reranker-v2-M3 has a large corpus (250,000 tokens) while Jina v3 and Qwen (0.6B, 4B) has smaller corpus (~151.000 tokens) -> with the same Document 800 words: BGE-v2-m3 devides into 900 tokens, Jina/Qwen devides into 1.300 tokens.
- BGE, Jina is Cross-Encoder Architecture and use only 3 tokens like `[CLS] <Query> [SEP] <Document> [SEP]` while Qwen 0.6B / 4B is LLM-based Reranker they use chat_template which is around 150 tokens.



## Extreme 1

In [ ]:
import pandas as pd

print("RERANKER MODELS COMPARISON ACROSS BANKING DOMAINS")
print("-"*60)

df1 = pd.DataFrame(retail_deposit_eval)
print("=== Retail Deposit ===")
print(df1.to_markdown(index=False, numalign="center", stralign="left"))

df2 = pd.DataFrame(consumer_card_eval)
print("=== Consumer Cards ===")
print(df2.to_markdown(index=False, numalign="center", stralign="left"))

df3 = pd.DataFrame(corp_dep_eval)
print("=== Corp Deposit ===")
print(df3.to_markdown(index=False, numalign="center", stralign="left"))

df4 = pd.DataFrame(card_po_eval)
print("=== Card Policy ===")
print(df3.to_markdown(index=False, numalign="center", stralign="left"))

df5 = pd.DataFrame(loan_eval)
print("=== Loan Products ===")
print(df3.to_markdown(index=False, numalign="center", stralign="left"))

df6 = pd.DataFrame(pos_eval)
print("=== POS ===")
print(df3.to_markdown(index=False, numalign="center", stralign="left"))

RERANKER MODELS COMPARISON ACROSS BANKING DOMAINS
------------------------------------------------------------
=== Retail Deposit ===
| model                    |  recall  |  mrr  |  latency  |   qps    |  docs_per_sec  |  used_vram_gb  |
|:-------------------------|:--------:|:-----:|:---------:|:--------:|:--------------:|:--------------:|
| BAAI/bge-reranker-v2-m3  | 66.6667  |   1   |  637.956  | 1.56751  |    15.6751     |    1.50281     |
| jinaai/jina-reranker-v3  | 86.6667  |   1   |  406.003  | 2.46304  |    24.6304     |    14.0184     |
| Qwen/Qwen3-Reranker-0.6B | 86.6667  |   1   |  409.503  | 2.44199  |    24.4199     |    13.9911     |
| Qwen/Qwen3-Reranker-4B   |    80    |   1   |  1228.82  | 0.813789 |    8.13789     |    13.7137     |
=== Consumer Cards ===
| model                    |  recall  |  mrr  |  latency  |   qps   |  docs_per_sec  |  used_vram_gb  |
|:-------------------------|:--------:|:-----:|:---------:|:-------:|:--------------:|:--------------:|
| BAA

## Extreme 2

In [4]:
clear_gpu_memory()
qwen3_4 = "Qwen/Qwen3-Reranker-4B"
llm4 = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_4,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2042,
        gpu_memory_utilization=0.8,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )

bge_m3 = "BAAI/bge-reranker-v2-m3"
llmbge = LLM(model=bge_m3,
          runner="pooling",
          enforce_eager=True,
          trust_remote_code=True,
          max_model_len=1024,
          gpu_memory_utilization=0.1,
          disable_log_stats=True,
          )

INFO 06-26 07:27:42 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 2042, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Reranker-4B'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

INFO 06-26 07:27:44 [config.py:960] Found sentence-transformers tokenize configuration.
INFO 06-26 07:28:06 [model.py:924] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-26 07:28:23 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 06-26 07:28:23 [nixl_utils.py:34] NIXL is not available
WARNING 06-26 07:28:23 [nixl_utils.py:44] NIXL agent config is not available
INFO 06-26 07:28:23 [model.py:555] Resolved architecture: Qwen3ForSequenceClassification


modules.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

... (13 outputs left out)

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

INFO 06-26 07:33:08 [llm.py:394] Supported pooling task: classify


In [11]:
clear_gpu_memory()
qwen3_06 = "Qwen/Qwen3-Reranker-0.6B"
llm06 = LLM(
        # Specify the original model from HuggingFace
        model=qwen3_06,
        # Use pooling runner for score task
        runner="pooling",
        enforce_eager=True,
        trust_remote_code=True,
        max_model_len=2042,
        gpu_memory_utilization=0.4,
        disable_log_stats=True,
        # HuggingFace model configuration overrides required for compatibility
        hf_overrides={
            "architectures": ["Qwen3ForSequenceClassification"],
            "classifier_from_token": ["no", "yes"],
            "is_original_qwen3_reranker": True,
        },
    )

jina_v3 = "jinaai/jina-reranker-v3"
llmjina = LLM(
    model=jina_v3,
    runner="pooling",
    trust_remote_code=True,
    enforce_eager=True,          # Disables memory-hungry CUDA graphs
    max_model_len=2042,
    gpu_memory_utilization=0.5,
    disable_log_stats=True,
)


INFO 06-26 07:38:37 [utils.py:233] non-default args: {'runner': 'pooling', 'trust_remote_code': True, 'max_model_len': 2042, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'hf_overrides': {'architectures': ['Qwen3ForSequenceClassification'], 'classifier_from_token': ['no', 'yes'], 'is_original_qwen3_reranker': True}, 'enforce_eager': True, 'model': 'Qwen/Qwen3-Reranker-0.6B'}


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

INFO 06-26 07:38:37 [config.py:960] Found sentence-transformers tokenize configuration.
INFO 06-26 07:38:37 [model.py:924] Resolved `--convert auto` to `--convert classify`. Pass the value explicitly to silence this message.
INFO 06-26 07:38:37 [model.py:555] Resolved architecture: Qwen3ForSequenceClassification


modules.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

INFO 06-26 07:38:37 [config.py:848] Found sentence-transformers modules configuration.
WARNING 06-26 07:38:37 [model.py:1965] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-26 07:38:37 [model.py:2018] Casting torch.bfloat16 to torch.float16.
INFO 06-26 07:38:37 [model.py:1680] Using max model len 2042
WARNING 06-26 07:38:37 [vllm.py:896] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 06-26 07:38:37 [vllm.py:914] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 06-26 07:38:37 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'])
INFO 06-26 07:38:37 [vllm.py:1089] Cudagraph is disabled under eager mode


... (13 outputs left out)

added_tokens.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

INFO 06-26 07:41:00 [llm.py:394] Supported pooling task: token_embed


### Case 1

In [ ]:
query = "quy trình, các bước thao tác trên hệ thống và các quy định về thời hạn hiệu lực khi thực hiện truy vấn thông tin lịch sử tín dụng CIC và KCB cho khách hàng cá nhân và nước ngoài."

documents = [
    # Chunk 1: High Similarity (Trực tiếp trả lời quy trình và điều kiện CIC/KCB)
    """Bước đầu tiên trong quy trình thẩm định là truy vấn lịch sử tín dụng của khách hàng. Nhân viên vào màn hình MH  CIC Information Request trên hệ thống Aither. Tại đây, nhập số CMND, CCCD hoặc Hộ chiếu (đối với người nước ngoài) vào trường ID No để tra cứu. Đối với khách hàng Hàn Quốc, bắt buộc phải tra cứu thêm KCB (Korea Credit Bureau) bằng cách nhập ID Hàn Quốc hoặc Hộ chiếu Hàn Quốc. Thời hạn hiệu lực của báo cáo CIC và KCB là 30 ngày kể từ ngày truy vấn. Nếu trong vòng 30 ngày đã có kết quả tra cứu, hệ thống sẽ tự động tải lại dữ liệu cũ mà không tính phí tra cứu mới. Đối với KCB, điểm số phải đạt từ 531 điểm trở lên (Hạng KCB <= 7) mới đủ điều kiện cơ bản để xem xét cấp thẻ tín dụng hoặc khoản vay.""",

    # Chunk 2: High Similarity (Xử lý lỗi và các trường hợp đặc biệt khi tra cứu)
    """Trong quá trình tra cứu trên màn hình MH , nếu hệ thống báo lỗi không tìm thấy dữ liệu do sai lệch thông tin cá nhân, nhân viên cần thực hiện truy vấn lại bằng chức năng nhập tên thiếu (ví dụ nhập "RI YUNG HYUK" thay vì họ tên đầy đủ do giới hạn ký tự) hoặc tra cứu theo số giấy tờ cũ nếu khách hàng vừa đổi từ CMND 9 số sang CCCD 12 số gắn chip. Tại mục Report Type, chọn "1-FULLFORM" đối với thẻ tín dụng và các khoản vay thông thường để lấy toàn bộ lịch sử dư nợ, thẻ tín dụng, thế chấp và lịch sử trả nợ chậm. Sau khi nhấn "Inquiry", trạng thái ban đầu có thể là "Received", nhân viên cần chờ một lúc để hệ thống cập nhật kết quả.""",

    # Chunk 3: High-Medium Similarity (Các tiêu chí từ chối tự động dựa trên CIC)
    """Sau khi có kết quả CIC, hệ thống sẽ chạy qua màng lọc Filtering Items để tự động đánh giá. Khách hàng sẽ bị từ chối mở thẻ nếu vi phạm một trong các điều kiện sau: Có nợ nhóm 2 tại bất kỳ tổ chức tín dụng nào trong vòng 12 tháng gần nhất (mã lỗi DL020000 >= 2); Có khoản nợ vay thuộc nhóm 3 đến nhóm 5 trong vòng 5 năm gần nhất (mã lỗi DL010003); Có từ 4 lần nợ vay nhóm 2 trong vòng 12 tháng gần nhất; Có lịch sử trễ hạn thẻ tín dụng từ 30 ngày trở lên trong vòng 12 tháng gần nhất (DC031012). Tổng dư nợ thẻ tín dụng đang có không được vượt quá số lượng thẻ quy định (ví dụ khách hàng đang có 4 thẻ tín dụng tại các ngân hàng khác nhau có thể bị từ chối do vượt quá rủi ro - LN019006 >= 4).""",

    # Chunk 4: Medium Similarity (Kết quả Screening và R-clips)
    """Kết quả thẩm định cuối cùng sau khi tra cứu CIC và nhập dữ liệu vào màn hình MH  Applying and Screening of consumer application sẽ hiển thị tại phần Processing Division. Nếu tình trạng là "Confirmation", nghĩa là đang chờ phê duyệt. Nếu "Screening Failed", nghĩa là không thỏa điều kiện của ngân hàng. R-Clips Approved Result sẽ cho biết kết quả đánh giá tự động: "Approved" (Đồng ý) hoặc "Rejected" (Từ chối). Lý do từ chối (Screening Denial Reason) có thể là: Thu nhập không đủ (Monthly Salary Underqualified), Không đạt chuẩn chấm điểm (ASS/BSS Standard Underqualified member), hoặc rớt bộ lọc tự động do nợ xấu trên CIC (Filtering items: Rejected). """,
# Chunk 1 (Rank 1 - Rất cao): Tổng quan quy trình 9 bước mở CIF/Tài khoản.
    """Quy trình đăng ký CIF và mở tài khoản/tiền gửi bao gồm 9 bước chính: (1) Nhận hồ sơ từ Khách hàng; (2) Tra cứu số CIF bằng màn hình 0030; (3) Kiểm tra hồ sơ Khách hàng (nhận dạng, KYC, FATCA, AML); (4) Tư vấn & cung cấp mẫu biểu cho khách hàng; (5) Kiểm tra & đối chiếu thông tin điền trên form; (6) Đăng ký CIF, đối chiếu thông tin sinh trắc học và mở tài khoản; (7) Cung cấp số tài khoản cho khách; (8) Phê duyệt hồ sơ trên hệ thống; (9) Đăng tải và lưu trữ hồ sơ bằng màn hình 0950 & thư chấp thuận.""",

    # Chunk 4 (Rank 4 - Trung bình): Hồ sơ nhận diện đối với người Việt Nam và khách hàng dưới 14 tuổi.
    """Khi thực hiện quy trình mở CIF, yêu cầu về hồ sơ khách hàng là người Việt Nam phải có Giấy tờ tùy thân hợp lệ (Thẻ căn cước, Thẻ CCCD gắn chip). Hệ thống sẽ yêu cầu đọc thông tin lưu trữ trên chip. Đối với Khách hàng vị thành niên dưới 14 tuổi chưa có thẻ căn cước, có thể sử dụng Giấy khai sinh và số định danh cá nhân để thay thế, kèm theo hồ sơ của người đại diện hợp pháp (bố mẹ/người giám hộ). Người đại diện hợp pháp phải đăng ký CIF và mở tài khoản trước khi thực hiện quy trình cho khách hàng vị thành niên.""",
# Chunk 2: Độ tương đồng rất cao - Quy định về các đối tượng miễn trừ Kế toán trưởng
    "Trong quá trình mở tài khoản và đăng ký mẫu chữ ký, một số tổ chức không bắt buộc phải đăng ký chữ ký Kế toán trưởng hoặc Người phụ trách kế toán. Các đối tượng này bao gồm: Văn phòng đại diện của Nhà đầu tư nước ngoài trong hợp đồng BCC, Văn phòng điều hành của Nhà thầu nước ngoài, Văn phòng đại diện của các công ty nước ngoài và trong nước, Tổ chức phi lợi nhuận, và Ban quản trị nhà chung cư. Các tổ chức này có thể đánh dấu vào tùy chọn không bắt buộc bố trí Kế toán trưởng trên biểu mẫu.",

    # Chunk 7: Tương đồng thấp - Quy trình thực hiện KYC và kiểm tra FATCA
    "Quy trình định danh khách hàng (KYC) yêu cầu nhân viên phải tìm hiểu mục đích mở tài khoản, kiểm tra thông tin khách hàng trên các nguồn thông tin chính phủ như website đăng ký kinh doanh và hệ thống mã số thuế. Khách hàng cũng phải khai báo thông tin FATCA và chi nhánh thực hiện tra soát thông tin theo Luật Phòng, chống rửa tiền. Sau khi thu thập đủ giấy tờ thành lập, giấy tờ tùy thân, chi nhánh sẽ tiến hành tạo mã khách hàng (CIF) trên hệ thống ngân hàng lõi.",
# Chunk 6: Low-Medium Similarity (Tra cứu danh sách đen và cấm vận đối với doanh nghiệp)
    """Một bước không thể thiếu trong thẩm định thẻ doanh nghiệp là rà soát Danh sách đen (Blacklist) và Danh sách nghề nghiệp bị hạn chế (Restricted Industry List). Theo danh mục quy định, nhân viên phải sử dụng hệ thống tra cứu nội bộ (mã 6308) để sàng lọc. Công ty kinh doanh trong các lĩnh vực rủi ro cao như: Thủy thủ và những người làm việc trên tàu chuyên tuyến ngoài khơi, Hiệu cầm đồ, kinh doanh Karaoke, quán Bar, vũ trường, hoạt động cá cược, sòng bạc, kinh doanh vũ khí, sản xuất hóa chất độc hại sẽ không được mở thẻ. Thêm vào đó, nếu doanh nghiệp đang bị phân loại Nợ quá hạn, Nợ xấu (Nhóm 2 trở lên) trên CIC, hoặc có lãnh đạo từng dính líu đến gian lận tài chính, rửa tiền (AML),""",

]

dic = {
    query: documents,
}

In [ ]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy trình, các bước thao tác trên hệ thống và các quy định về thời hạn hiệu lực khi thực hiện truy vấn thông tin lịch sử tín dụng CIC và KCB cho khách hàng cá nhân và nước ngoài.
[1] Score: 0.4199
Bước đầu tiên trong quy trình thẩm định là truy vấn lịch sử tín dụng của khách hàng. Nhân viên vào màn hình MH  CIC Information Request trên hệ thống Aither. Tại đây, nhập số CMND, CCCD hoặc Hộ chiếu (đối với người nước ngoài) vào trường ID No để tra cứu. Đối với khách hàng Hàn Quốc, bắt buộc phải tra cứu thêm KCB (Korea Credit Bureau) bằng cách nhập ID Hàn Quốc hoặc Hộ chiếu Hàn Quốc. Thời hạn hiệu lực của báo cáo CIC và KCB là 30 ngày kể từ ngày truy vấn. Nếu trong vòng 30 ngày đã có kết quả tra cứu, hệ thống sẽ tự động tải lại dữ liệu cũ mà không tính phí tra cứu mới. Đối với KCB, điểm số phải đạt từ 531 điểm trở lên (Hạng KCB <= 7) mới đủ điều kiện cơ bản để xem xét cấp thẻ tín dụng hoặc khoản vay.
----------------------------------------------------------------------
[5] Score:

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy trình, các bước thao tác trên hệ thống và các quy định về thời hạn hiệu lực khi thực hiện truy vấn thông tin lịch sử tín dụng CIC và KCB cho khách hàng cá nhân và nước ngoài.
[1] Score: 0.9964
Bước đầu tiên trong quy trình thẩm định là truy vấn lịch sử tín dụng của khách hàng. Nhân viên vào màn hình MH  CIC Information Request trên hệ thống Aither. Tại đây, nhập số CMND, CCCD hoặc Hộ chiếu (đối với người nước ngoài) vào trường ID No để tra cứu. Đối với khách hàng Hàn Quốc, bắt buộc phải tra cứu thêm KCB (Korea Credit Bureau) bằng cách nhập ID Hàn Quốc hoặc Hộ chiếu Hàn Quốc. Thời hạn hiệu lực của báo cáo CIC và KCB là 30 ngày kể từ ngày truy vấn. Nếu trong vòng 30 ngày đã có kết quả tra cứu, hệ thống sẽ tự động tải lại dữ liệu cũ mà không tính phí tra cứu mới. Đối với KCB, điểm số phải đạt từ 531 điểm trở lên (Hạng KCB <= 7) mới đủ điều kiện cơ bản để xem xét cấp thẻ tín dụng hoặc khoản vay.
----------------------------------------------------------------------
[3] Score:

In [ ]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy trình, các bước thao tác trên hệ thống và các quy định về thời hạn hiệu lực khi thực hiện truy vấn thông tin lịch sử tín dụng CIC và KCB cho khách hàng cá nhân và nước ngoài.
[1] Score: 0.9964
Bước đầu tiên trong quy trình thẩm định là truy vấn lịch sử tín dụng của khách hàng. Nhân viên vào màn hình MH  CIC Information Request trên hệ thống Aither. Tại đây, nhập số CMND, CCCD hoặc Hộ chiếu (đối với người nước ngoài) vào trường ID No để tra cứu. Đối với khách hàng Hàn Quốc, bắt buộc phải tra cứu thêm KCB (Korea Credit Bureau) bằng cách nhập ID Hàn Quốc hoặc Hộ chiếu Hàn Quốc. Thời hạn hiệu lực của báo cáo CIC và KCB là 30 ngày kể từ ngày truy vấn. Nếu trong vòng 30 ngày đã có kết quả tra cứu, hệ thống sẽ tự động tải lại dữ liệu cũ mà không tính phí tra cứu mới. Đối với KCB, điểm số phải đạt từ 531 điểm trở lên (Hạng KCB <= 7) mới đủ điều kiện cơ bản để xem xét cấp thẻ tín dụng hoặc khoản vay.
----------------------------------------------------------------------
[2] Score:

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy trình, các bước thao tác trên hệ thống và các quy định về thời hạn hiệu lực khi thực hiện truy vấn thông tin lịch sử tín dụng CIC và KCB cho khách hàng cá nhân và nước ngoài.
[1] Score: 0.5864
Bước đầu tiên trong quy trình thẩm định là truy vấn lịch sử tín dụng của khách hàng. Nhân viên vào màn hình MH  CIC Information Request trên hệ thống Aither. Tại đây, nhập số CMND, CCCD hoặc Hộ chiếu (đối với người nước ngoài) vào trường ID No để tra cứu. Đối với khách hàng Hàn Quốc, bắt buộc phải tra cứu thêm KCB (Korea Credit Bureau) bằng cách nhập ID Hàn Quốc hoặc Hộ chiếu Hàn Quốc. Thời hạn hiệu lực của báo cáo CIC và KCB là 30 ngày kể từ ngày truy vấn. Nếu trong vòng 30 ngày đã có kết quả tra cứu, hệ thống sẽ tự động tải lại dữ liệu cũ mà không tính phí tra cứu mới. Đối với KCB, điểm số phải đạt từ 531 điểm trở lên (Hạng KCB <= 7) mới đủ điều kiện cơ bản để xem xét cấp thẻ tín dụng hoặc khoản vay.
----------------------------------------------------------------------
[2] Score:

Qwen3 0.6B và Jina nhận định query đang đề cập vấn đề THẨM ĐỊNH và TRUY VẤN CIF, còn 2 model còn lại bị [5] lừa vì nó đề cập ĐĂNG KÝ và MỞ TÀI KHOẢN

### Case 2

In [12]:
query = "đang cung cấp máy POS nào, loại thẻ nào được chấp nhận thanh toán và sử dụng công nghệ gì"

documents = [
    # HIGH
    """Ngân hàng Shinhan hiện đang cung cấp các dòng máy POS đa dạng phục vụ nhu cầu của ĐVCNT, bao gồm: POS truyền thống, Mobile POS (nhỏ gọn, di động), và các dòng máy hiện đại như Smart POS, Mini Smart POS. Nổi bật là máy Quẹt Thẻ PAX SmartPOS A910S (Android SmartPOS Mobile) có thiết kế màn hình cảm ứng lớn, hoạt động trên hệ điều hành Android, có khả năng kết nối internet qua WiFi hoặc mạng di động 4G (cần lắp SIM 4G) để thực hiện các giao dịch thanh toán nhanh chóng.""",

    """Các thiết bị POS của Shinhan Bank chấp nhận thanh toán đa dạng các loại thẻ trên thị trường. Về Tổ chức phát hành thẻ (Card Scheme), máy hỗ trợ thanh toán thẻ VISA (Visa Gold), MasterCard (MasterCard Gold), JCB (JCB Card), UnionPay, và thẻ NAPAS nội địa. Về loại thẻ, ĐVCNT có thể nhận thanh toán từ cả Thẻ ghi nợ (Debit Card) và Thẻ tín dụng (Credit Card). Về khu vực phát hành, máy chấp nhận cả thẻ phát hành trong nước và thẻ phát hành tại nước ngoài. Các thẻ này có thể phân hạng từ Classic, Gold, Platinum đến Infinity.""",

    """Trên các dòng máy Shinhan SmartPOS hiện đại, máy hỗ trợ nhiều hình thức đọc thẻ đa dạng để nâng cao trải nghiệm khách hàng. Các hình thức bao gồm: Đọc thẻ từ (quẹt thẻ truyền thống khe hông), Đọc thẻ Chip (cắm thẻ vào khe dưới máy), và đặc biệt là công nghệ Contactless (Thanh toán không tiếp xúc - khách hàng chỉ cần chạm thẻ vào màn hình/đầu đọc của máy). Ngoài ra, máy còn hỗ trợ các phương thức thanh toán di động như Google Pay, Samsung Pay, Apple Pay và hiển thị mã QR Code (VNPAY/VietQR) để khách hàng quét thanh toán.""",

    """Bên cạnh chức năng thanh toán cơ bản (Sale authorization), máy POS của Shinhan Bank còn cung cấp các tính năng quan trọng khác giúp ĐVCNT quản lý giao dịch. Bao gồm: Hủy giao dịch (Void authorization - dùng để hủy giao dịch vừa thực hiện trong ngày chưa chốt bạt), Hoàn trả (Refund), In lại hóa đơn (Trans reprint), Xem lịch sử giao dịch (Trans history), và Tổng hợp giao dịch cuối ngày (Settlement). Máy SmartPOS còn có khả năng kết nối tích hợp với hầu hết các Phần mềm bán hàng (KiotViet, POS 365, iPOS...).""",
# Others
    "Lãi suất thẻ tín dụng thông thường dao động từ 24.99% đến 31.8% một năm tùy thuộc vào loại thẻ. Phí giao dịch ngoại tệ áp dụng cho các giao dịch quốc tế là 0.99% đối với thẻ PWM và Travel Platinum, 1.99% cho thẻ Signature, và 2.9% đối với các dòng thẻ Platinum thông thường, Gold, Classic và thẻ doanh nghiệp. Phí chậm thanh toán được tính bằng 4% trên số tiền tối thiểu chưa thanh toán, với mức phạt tối thiểu là 250.000 VNĐ và tối đa là 999.000 VNĐ.",
"""Khác với cá nhân, khách hàng doanh nghiệp nếu muốn mở thẻ tín dụng có Hạn mức cao mà không đủ điều kiện tín chấp (ví dụ báo cáo tài chính yếu, mới thành lập, rớt Cut-off) thì phải dùng phương án Thẻ doanh nghiệp có tài sản đảm bảo (Secured corporate credit card). Tài sản đảm bảo thường là tiền gửi ký quỹ, sổ tiết kiệm đứng tên doanh nghiệp hoặc đứng tên Người đại diện theo pháp luật. Hạn mức thẻ sẽ được cấp bằng 90% đến 100% giá trị tài sản đảm bảo. Quy trình phê duyệt cho loại thẻ này rất nhanh chóng, chủ yếu do Giám đốc Chi nhánh/Phòng giao dịch ký duyệt mà không cần trình Hội đồng tín dụng trung ương. Điều kiện tra cứu CIC đối với doanh nghiệp vẫn phải thực hiện trên màn hình 2133, nhưng chỉ để kiểm tra nợ xấu, không đánh giá khắt khe về các chỉ số tài chính (như tỷ lệ thanh toán hiện hành, vòng quay vốn) như thẻ tín chấp.""",
"Để hỗ trợ việc kinh doanh và thu hộ, Shinhan cung cấp Dịch vụ chấp nhận thanh toán thẻ qua máy POS với tỷ lệ chiết khấu cạnh tranh. Khi đăng ký lắp đặt POS, doanh nghiệp có thể tham gia gói SOHO Biz Package hoặc gói ưu đãi Shinhan SOHO Merchant để nhận nhiều quyền lợi miễn phí giao dịch và ưu đãi lãi suất tiền gửi, giúp tiết kiệm chi phí vận hành và quản lý bán hàng dễ dàng hơn.",
"""Trong quá trình hỗ trợ, chăm sóc ĐVCNT, chính sách bảo hành và sửa chữa thiết bị được quy định rõ. Lắp đặt máy POS được miễn phí. Thời gian lắp đặt tại khu vực Hà Nội và TP.HCM là 02 ngày làm việc, khu vực khác là 04 ngày làm việc. Phí sử dụng POS là 220,000 VND/POS/Tháng (bao gồm VAT). Thiết bị được bảo hành miễn phí trong vòng 12 tháng. Các trường hợp từ chối bảo hành bao gồm thiết bị rách tem, hư hỏng do rơi vỡ, nước vào, do lỗi bảo quản của ĐVCNT."""
"Đối với mảng doanh nghiệp, Shinhan Bank hỗ trợ mở tài khoản thanh toán nội địa và ngoại tệ, tiền gửi có kỳ hạn, số tài khoản may mắn. Doanh nghiệp có thể sử dụng các dịch vụ tín dụng doanh nghiệp như Hỗ trợ vay ngắn hạn, trung và dài hạn, dịch vụ bảo lãnh, bao thanh toán và cho vay thế chấp khoản phải thu. Các giao dịch có thể thực hiện thông qua hệ thống CMS dành riêng cho doanh nghiệp.",
"""Khả năng thanh toán (Loan redemption capability) được tính bằng Thu nhập trung bình hàng tháng trừ đi các Khoản chi trả hàng tháng. Hệ số nhân để xác định hạn mức tín dụng phụ thuộc vào điểm AS (Application Scorecard) của khách hàng, thường là 2, 2.5, 3.3 hoặc 4. Thu nhập cuối cùng dùng để xét duyệt được tính theo công thức: Thu nhập cố định + 75% * (Thu nhập trung bình 3 tháng gần nhất - Thu nhập cố định). Việc chênh lệch thu nhập giữa các tháng phải được giải trình rõ ràng, nếu chênh lệch trên 120% cần có xác nhận thưởng năng suất từ công ty. Mức kiểm soát rủi ro dư nợ tín chấp tối đa (MUE) cho một khách hàng cá nhân không vượt quá 900 triệu đồng (bao gồm cả dư nợ thẻ và vay tiêu dùng). Hạn mức cấp dựa trên thu nhập hàng tháng (MI) thường dao động từ MI x 2 đến MI x 3 tùy hạng khách hàng và lịch sử tín dụng."""
    ]

dic = {
    query: documents,
}

In [6]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: đang cung cấp máy POS nào, loại thẻ nào được chấp nhận thanh toán và sử dụng công nghệ gì
[3] Score: 0.4102
Trên các dòng máy Shinhan SmartPOS hiện đại, máy hỗ trợ nhiều hình thức đọc thẻ đa dạng để nâng cao trải nghiệm khách hàng. Các hình thức bao gồm: Đọc thẻ từ (quẹt thẻ truyền thống khe hông), Đọc thẻ Chip (cắm thẻ vào khe dưới máy), và đặc biệt là công nghệ Contactless (Thanh toán không tiếp xúc - khách hàng chỉ cần chạm thẻ vào màn hình/đầu đọc của máy). Ngoài ra, máy còn hỗ trợ các phương thức thanh toán di động như Google Pay, Samsung Pay, Apple Pay và hiển thị mã QR Code (VNPAY/VietQR) để khách hàng quét thanh toán.
----------------------------------------------------------------------
[2] Score: 0.3810
Các thiết bị POS của Shinhan Bank chấp nhận thanh toán đa dạng các loại thẻ trên thị trường. Về Tổ chức phát hành thẻ (Card Scheme), máy hỗ trợ thanh toán thẻ VISA (Visa Gold), MasterCard (MasterCard Gold), JCB (JCB Card), UnionPay, và thẻ NAPAS nội địa. Về loại thẻ,

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: đang cung cấp máy POS nào, loại thẻ nào được chấp nhận thanh toán và sử dụng công nghệ gì
[2] Score: 0.9246
Các thiết bị POS của Shinhan Bank chấp nhận thanh toán đa dạng các loại thẻ trên thị trường. Về Tổ chức phát hành thẻ (Card Scheme), máy hỗ trợ thanh toán thẻ VISA (Visa Gold), MasterCard (MasterCard Gold), JCB (JCB Card), UnionPay, và thẻ NAPAS nội địa. Về loại thẻ, ĐVCNT có thể nhận thanh toán từ cả Thẻ ghi nợ (Debit Card) và Thẻ tín dụng (Credit Card). Về khu vực phát hành, máy chấp nhận cả thẻ phát hành trong nước và thẻ phát hành tại nước ngoài. Các thẻ này có thể phân hạng từ Classic, Gold, Platinum đến Infinity.
----------------------------------------------------------------------
[1] Score: 0.7317
Ngân hàng Shinhan hiện đang cung cấp các dòng máy POS đa dạng phục vụ nhu cầu của ĐVCNT, bao gồm: POS truyền thống, Mobile POS (nhỏ gọn, di động), và các dòng máy hiện đại như Smart POS, Mini Smart POS. Nổi bật là máy Quẹt Thẻ PAX SmartPOS A910S (Android SmartPOS Mobi

In [13]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: đang cung cấp máy POS nào, loại thẻ nào được chấp nhận thanh toán và sử dụng công nghệ gì
[3] Score: 0.9948
Trên các dòng máy Shinhan SmartPOS hiện đại, máy hỗ trợ nhiều hình thức đọc thẻ đa dạng để nâng cao trải nghiệm khách hàng. Các hình thức bao gồm: Đọc thẻ từ (quẹt thẻ truyền thống khe hông), Đọc thẻ Chip (cắm thẻ vào khe dưới máy), và đặc biệt là công nghệ Contactless (Thanh toán không tiếp xúc - khách hàng chỉ cần chạm thẻ vào màn hình/đầu đọc của máy). Ngoài ra, máy còn hỗ trợ các phương thức thanh toán di động như Google Pay, Samsung Pay, Apple Pay và hiển thị mã QR Code (VNPAY/VietQR) để khách hàng quét thanh toán.
----------------------------------------------------------------------
[1] Score: 0.9907
Ngân hàng Shinhan hiện đang cung cấp các dòng máy POS đa dạng phục vụ nhu cầu của ĐVCNT, bao gồm: POS truyền thống, Mobile POS (nhỏ gọn, di động), và các dòng máy hiện đại như Smart POS, Mini Smart POS. Nổi bật là máy Quẹt Thẻ PAX SmartPOS A910S (Android SmartPOS Mob

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: đang cung cấp máy POS nào, loại thẻ nào được chấp nhận thanh toán và sử dụng công nghệ gì
[2] Score: 0.3259
Các thiết bị POS của Shinhan Bank chấp nhận thanh toán đa dạng các loại thẻ trên thị trường. Về Tổ chức phát hành thẻ (Card Scheme), máy hỗ trợ thanh toán thẻ VISA (Visa Gold), MasterCard (MasterCard Gold), JCB (JCB Card), UnionPay, và thẻ NAPAS nội địa. Về loại thẻ, ĐVCNT có thể nhận thanh toán từ cả Thẻ ghi nợ (Debit Card) và Thẻ tín dụng (Credit Card). Về khu vực phát hành, máy chấp nhận cả thẻ phát hành trong nước và thẻ phát hành tại nước ngoài. Các thẻ này có thể phân hạng từ Classic, Gold, Platinum đến Infinity.
----------------------------------------------------------------------
[1] Score: 0.2391
Ngân hàng Shinhan hiện đang cung cấp các dòng máy POS đa dạng phục vụ nhu cầu của ĐVCNT, bao gồm: POS truyền thống, Mobile POS (nhỏ gọn, di động), và các dòng máy hiện đại như Smart POS, Mini Smart POS. Nổi bật là máy Quẹt Thẻ PAX SmartPOS A910S (Android SmartPOS Mobi

### Case 3

In [ ]:
query = "quy trình dự tính P&L và thẩm quyền phê duyệt khi chi nhánh muốn đề xuất Discount MDR cho Merchant"
# ĐVCNT: Đơn vị chấp nhận thẻ
# P&L: báo cáo lãi & lỗ
documents = [
    # HIGH relevant
    """Khi đề xuất và đàm phán mức phí áp dụng cho ĐVCNT, nếu mức phí đề xuất (Proposed MDR) thấp hơn mức phí tiêu chuẩn của Shinhan, chi nhánh cần thực hiện Draft trình giảm phí. Việc này đòi hỏi phải tính toán P&L (Báo cáo Lãi/Lỗ). Dự tính P&L dựa trên doanh số thanh toán thẻ dự kiến theo tháng của từng loại thẻ (Visa, Master, JCB/UPI, NAPAS Debit/Credit, Abroad Card) nhân với mức chênh lệch phí (+/- MDR Gap Percentage). Kết quả Lời/Lỗ dự kiến cuối cùng sẽ quyết định xem đề xuất giảm phí có mang lại lợi nhuận hay không.""",

    """Thẩm quyền phê duyệt trình giảm phí (Discount MDR) được phân cấp dựa trên quy mô ĐVCNT và mức phí đề xuất. Đối với ĐVCNT quy mô vừa, nhỏ và siêu nhỏ (doanh số thanh toán thẻ dưới 1 tỷ/tháng), Giám đốc chi nhánh (GM/DGM) có quyền phê duyệt mức phí Visa/MasterCard từ X >= 0.1% đến X >= 0.5% tùy loại ĐVCNT. PBD Director, Head và DGD sẽ phê duyệt các mức giảm sâu hơn. Đối với ĐVCNT quy mô lớn (doanh số từ 1 tỷ/tháng trở lên), thẩm quyền xét duyệt dựa trên số tiền giảm do chênh lệch mức phí áp dụng so với tiêu chuẩn (Y). Y <= 5 triệu/tháng do GM/DGM duyệt, tăng dần lên PBD Director, Head, DGD và cao nhất là GD cho mức Y > 30 triệu/tháng.""",
"""Để thực hiện trình giảm phí hợp lý, chi nhánh phải khảo sát và đánh giá ĐVCNT dựa trên 3 tiêu chí: Quy mô (Lĩnh vực kinh doanh MCC, Số lượng cửa hàng, Phạm vi hoạt động), Doanh số (Doanh số thanh toán tổng theo tháng/năm, Doanh số phân bổ qua các dòng thẻ Visa/Master/JCB/Napas), và Mức phí (Mức phí đang áp dụng cho thẻ phát hành trong nước và ngoài nước). Việc lấy thông tin từ máy POS của đối thủ (POS Provider Competitor) cũng rất quan trọng làm cơ sở so sánh.""",

    """Trên hệ thống MBCS, quy trình cài đặt mức phí (Set line approval) đối với các trường hợp trình giảm phí yêu cầu chi nhánh thực hiện: Bảng tính P&L và Nội dung trình, gửi cho phòng Phát triển Kinh doanh Thanh toán (PBD) kiểm tra trước khi submit. Luồng phê duyệt trên hệ thống tùy thuộc thẩm quyền: Nếu thẩm quyền GM/DGM duyệt thì luồng là RM -> MANAGER -> DGD/GM (approval) -> PBD (receiving). Nếu thẩm quyền DGD duyệt thì luồng là RM -> MANAGER -> DGD/GM (approval) -> PBD (parallel) -> Head + DGD (approval).""",

    # MEDIUM, LOW, NO relenvant
    """Thẩm quyền phê duyệt hạn mức thẻ tín dụng doanh nghiệp (Credit limit approval authority) được phân cấp chặt chẽ dựa trên quy mô rủi ro và xếp hạng tín dụng của công ty (Exempt rating, AAA ~ A-, BBB+ ~ BBB-, B+ ~ B-). Đối với các thẻ tín dụng doanh nghiệp có tài sản đảm bảo (tiền gửi ký quỹ), quy trình phê duyệt đơn giản hơn và thường do Giám đốc Chi nhánh (Branch General Manager) quyết định. Tuy nhiên, đối với thẻ không có tài sản đảm bảo (Unsecured corporate credit card), các công ty có hạng tín dụng cao (AAA ~ A-) có thể được cấp hạn mức lên tới 5.2 tỷ đồng thông qua phê duyệt của Hội đồng tín dụng (CO Committee hoặc MCOC). Doanh nghiệp hạng thấp hơn (B+ ~ B-) chỉ được cấp tối đa 2 tỷ đồng. Bất kỳ khoản cấp tín dụng nào vượt quá thẩm quyền của Chi nhánh đều phải được đẩy lên trình tự duyệt tập trung tại Hội sở thông qua hệ thống luồng công việc tự động trên Aither, có kèm theo báo cáo tóm tắt tình hình tài chính của doanh nghiệp.""",

    """Nhờ bản chất rủi ro tín dụng cực kỳ thấp, thẩm quyền phê duyệt các khoản vay Cash-backed Loan chủ yếu được giao cho Giám đốc chi nhánh (GM). Khách hàng không cần phải cung cấp hồ sơ chứng minh thu nhập, báo cáo tài chính SOHO hay báo cáo thuế. Quá trình giải ngân có thể hoàn tất trong vòng 1-2 giờ làm việc.""",
    """Thẩm quyền phê duyệt khoản vay thế chấp bất động sản phụ thuộc vào phân loại khoản vay, loại tài sản (Secured/Unsecured) và tỷ lệ LTV. Ví dụ: Dưới hạn mức tiêu chuẩn 50,000 USD (Unsecured) hoặc 400,000 USD (Secured), Giám đốc Chi nhánh (GM) có thể trực tiếp phê duyệt. Vượt hạn mức này, hồ sơ phải được trình lên Ban tín dụng Hội sở (CO Comm.) hoặc Hội đồng tín dụng Hội sở cấp cao (MCOC) phê duyệt.""",
"Dynamic Currency Conversion (DCC) is a service offered by overseas merchants that allows you to pay in your home currency, but it almost always features a significantly worse exchange rate than letting your bank handle the conversion.",
    "If your credit or debit card statement shows duplicate transactions for a single purchase, do not panic immediately. Contact the merchant first to confirm they only received one order, and the duplicate bank hold will expire and fall off your account naturally.",
    "Refunds originating from canceled orders on e-commerce platforms can take up to 45 days to reflect on your credit card statement, depending entirely on the merchant's payment processor and the issuing bank's policies.",

    ]

dic = {
    query: documents,
}

In [ ]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy trình dự tính P&L và thẩm quyền phê duyệt khi chi nhánh muốn đề xuất Discount MDR cho Merchant
[4] Score: 0.4578
Trên hệ thống MBCS, quy trình cài đặt mức phí (Set line approval) đối với các trường hợp trình giảm phí yêu cầu chi nhánh thực hiện: Bảng tính P&L và Nội dung trình, gửi cho phòng Phát triển Kinh doanh Thanh toán (PBD) kiểm tra trước khi submit. Luồng phê duyệt trên hệ thống tùy thuộc thẩm quyền: Nếu thẩm quyền GM/DGM duyệt thì luồng là RM -> MANAGER -> DGD/GM (approval) -> PBD (receiving). Nếu thẩm quyền DGD duyệt thì luồng là RM -> MANAGER -> DGD/GM (approval) -> PBD (parallel) -> Head + DGD (approval).
----------------------------------------------------------------------
[2] Score: 0.4022
Thẩm quyền phê duyệt trình giảm phí (Discount MDR) được phân cấp dựa trên quy mô ĐVCNT và mức phí đề xuất. Đối với ĐVCNT quy mô vừa, nhỏ và siêu nhỏ (doanh số thanh toán thẻ dưới 1 tỷ/tháng), Giám đốc chi nhánh (GM/DGM) có quyền phê duyệt mức phí Visa/MasterCard từ X >= 0.

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy trình dự tính P&L và thẩm quyền phê duyệt khi chi nhánh muốn đề xuất Discount MDR cho Merchant
[2] Score: 0.9626
Thẩm quyền phê duyệt trình giảm phí (Discount MDR) được phân cấp dựa trên quy mô ĐVCNT và mức phí đề xuất. Đối với ĐVCNT quy mô vừa, nhỏ và siêu nhỏ (doanh số thanh toán thẻ dưới 1 tỷ/tháng), Giám đốc chi nhánh (GM/DGM) có quyền phê duyệt mức phí Visa/MasterCard từ X >= 0.1% đến X >= 0.5% tùy loại ĐVCNT. PBD Director, Head và DGD sẽ phê duyệt các mức giảm sâu hơn. Đối với ĐVCNT quy mô lớn (doanh số từ 1 tỷ/tháng trở lên), thẩm quyền xét duyệt dựa trên số tiền giảm do chênh lệch mức phí áp dụng so với tiêu chuẩn (Y). Y <= 5 triệu/tháng do GM/DGM duyệt, tăng dần lên PBD Director, Head, DGD và cao nhất là GD cho mức Y > 30 triệu/tháng.
----------------------------------------------------------------------
[1] Score: 0.9420
Khi đề xuất và đàm phán mức phí áp dụng cho ĐVCNT, nếu mức phí đề xuất (Proposed MDR) thấp hơn mức phí tiêu chuẩn của Shinhan, chi nhánh cần th

In [ ]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy trình dự tính P&L và thẩm quyền phê duyệt khi chi nhánh muốn đề xuất Discount MDR cho Merchant
[2] Score: 0.9972
Thẩm quyền phê duyệt trình giảm phí (Discount MDR) được phân cấp dựa trên quy mô ĐVCNT và mức phí đề xuất. Đối với ĐVCNT quy mô vừa, nhỏ và siêu nhỏ (doanh số thanh toán thẻ dưới 1 tỷ/tháng), Giám đốc chi nhánh (GM/DGM) có quyền phê duyệt mức phí Visa/MasterCard từ X >= 0.1% đến X >= 0.5% tùy loại ĐVCNT. PBD Director, Head và DGD sẽ phê duyệt các mức giảm sâu hơn. Đối với ĐVCNT quy mô lớn (doanh số từ 1 tỷ/tháng trở lên), thẩm quyền xét duyệt dựa trên số tiền giảm do chênh lệch mức phí áp dụng so với tiêu chuẩn (Y). Y <= 5 triệu/tháng do GM/DGM duyệt, tăng dần lên PBD Director, Head, DGD và cao nhất là GD cho mức Y > 30 triệu/tháng.
----------------------------------------------------------------------
[4] Score: 0.9951
Trên hệ thống MBCS, quy trình cài đặt mức phí (Set line approval) đối với các trường hợp trình giảm phí yêu cầu chi nhánh thực hiện: Bảng tính 

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy trình dự tính P&L và thẩm quyền phê duyệt khi chi nhánh muốn đề xuất Discount MDR cho Merchant
[2] Score: 0.4782
Thẩm quyền phê duyệt trình giảm phí (Discount MDR) được phân cấp dựa trên quy mô ĐVCNT và mức phí đề xuất. Đối với ĐVCNT quy mô vừa, nhỏ và siêu nhỏ (doanh số thanh toán thẻ dưới 1 tỷ/tháng), Giám đốc chi nhánh (GM/DGM) có quyền phê duyệt mức phí Visa/MasterCard từ X >= 0.1% đến X >= 0.5% tùy loại ĐVCNT. PBD Director, Head và DGD sẽ phê duyệt các mức giảm sâu hơn. Đối với ĐVCNT quy mô lớn (doanh số từ 1 tỷ/tháng trở lên), thẩm quyền xét duyệt dựa trên số tiền giảm do chênh lệch mức phí áp dụng so với tiêu chuẩn (Y). Y <= 5 triệu/tháng do GM/DGM duyệt, tăng dần lên PBD Director, Head, DGD và cao nhất là GD cho mức Y > 30 triệu/tháng.
----------------------------------------------------------------------
[1] Score: 0.3451
Khi đề xuất và đàm phán mức phí áp dụng cho ĐVCNT, nếu mức phí đề xuất (Proposed MDR) thấp hơn mức phí tiêu chuẩn của Shinhan, chi nhánh cần th

Đánh giá Qwen3 0.6B: bối cảnh của [6] là sản phẩm Vay cầm cố giấy tờ có giá (Cash-backed Loan) của khách hàng cá nhân/SOHO, hoàn toàn không liên quan gì đến biểu phí quẹt thẻ MDR của doanh nghiệp (Merchant). 0.6B đẩy [6] lên cao hơn [3] chỉ vì cụm từ "thẩm quyền phê duyệt", đó là dấu hiệu mô hình đang bị thiên lệch về Keyword Matching thay vì Semantic (Ngữ nghĩa).

### Case 4

In [14]:
query = "Shinhan cung cấp những sản phẩm thẻ tín dụng, gói vay và dịch vụ thanh toán nào cho KHDN và HKD SME"

documents = [
    # High similarity (SME Credit Cards)
    "Dành cho khách hàng doanh nghiệp và hộ kinh doanh, Shinhan cung cấp các dòng thẻ tín dụng chuyên biệt như Thẻ tín dụng Shinhan Business Owner và thẻ Shinhan SME. Thẻ tín dụng doanh nghiệp giúp tách bạch chi tiêu cá nhân và doanh nghiệp, quản lý dòng tiền hiệu quả. Thẻ Business Owner (hạng Bạch Kim) cung cấp ưu đãi hoàn tiền 5% cho ẩm thực và 2% cho quảng cáo trực tuyến, đặc biệt phù hợp cho các chủ cơ sở kinh doanh vừa và nhỏ.",

    # High similarity (Corporate Loans)
    "Về giải pháp tài chính, Shinhan Bank cung cấp đa dạng các gói Vay tài trợ kinh doanh cho SME và Hộ kinh doanh. Các sản phẩm tín dụng nổi bật bao gồm: Hỗ trợ vay bổ sung vốn lưu động ngắn hạn, Vay vốn đầu tư kinh doanh trung và dài hạn. Ngân hàng cũng cung cấp dịch vụ bảo lãnh, dịch vụ bao thanh toán, cho vay thế chấp khoản phải thu, và dịch vụ giải ngân trực tuyến vô cùng thuận tiện trên hệ thống điện tử.",

    # Medium-High similarity (POS and Packages)
    "Để hỗ trợ việc kinh doanh và thu hộ, Shinhan cung cấp Dịch vụ chấp nhận thanh toán thẻ qua máy POS với tỷ lệ chiết khấu cạnh tranh. Khi đăng ký lắp đặt POS, doanh nghiệp có thể tham gia gói SOHO Biz Package hoặc gói ưu đãi Shinhan SOHO Merchant để nhận nhiều quyền lợi miễn phí giao dịch và ưu đãi lãi suất tiền gửi, giúp tiết kiệm chi phí vận hành và quản lý bán hàng dễ dàng hơn.",

    # Medium similarity (Import/Export and FX)
    "Đối với doanh nghiệp hoạt động thương mại quốc tế, Shinhan cung cấp các dịch vụ Xuất Nhập Khẩu toàn diện. Doanh nghiệp có thể thực hiện phát hành và điều chỉnh thư Tín dụng (L/C) trực tuyến thông qua kênh Ngân hàng trực tuyến một cách dễ dàng. Ngân hàng cũng cung cấp dịch vụ giao dịch ngoại hối (mua bán ngoại tệ kỳ hạn, hoán đổi lãi suất, dịch vụ ngoại tệ trực tuyến) để giúp doanh nghiệp quản trị rủi ro tỷ giá.",
# Others
    "Tài khoản S-Payroll cung cấp các ưu đãi miễn phí hoàn toàn cho khách hàng, bao gồm: Miễn phí phát hành thẻ ATM, thẻ tín dụng, thẻ phát hành rút tiền tại ATM Shinhan và các ngân hàng khác; Miễn phí chuyển tiền (chuyển tiền trong nước, chuyển khoản 24/7 qua Internet Banking/Mobile Banking/ATM); Miễn phí thanh toán hóa đơn điện, nước, internet, truyền hình cáp; Miễn phí dịch vụ tin nhắn thông báo qua Zalo; Miễn phí dịch vụ ngân hàng điện tử (Internet Banking, Mobile Banking).""",
"Thông tin về biểu phí dịch vụ (Service Fee) cũng phải được thể hiện rõ trên Đơn đăng ký kiêm khảo sát ĐVCNT (bản cứng). Tại mục 2. Phí dịch vụ, ĐVCNT phải chọn loại thẻ chấp nhận và đồng ý với mức phí chiết khấu (MDR) cho Thẻ nội địa, Thẻ quốc tế phát hành tại Việt Nam (Thẻ tín dụng/Thẻ ghi nợ) và Thẻ quốc tế phát hành nước ngoài. Đơn đăng ký cũng cần ghi rõ phí sử dụng dịch vụ thanh toán QR (nếu có) và phí này chưa bao gồm VAT.",
"Khách hàng doanh nghiệp cần cung cấp giấy phép đăng ký kinh doanh và mẫu dấu khi mở tài khoản thanh toán tại ngân hàng.",
"Việc đăng ký chữ ký mẫu và tài khoản thanh toán doanh nghiệp đòi hỏi quyết định bổ nhiệm Kế toán trưởng. Kế toán trưởng có thể không cần ủy quyền nếu điều lệ công ty đã quy định rõ quyền hạn của họ trong giao dịch ngân hàng.",
    "Hồ sơ mở tài khoản tổ chức yêu cầu Bản sao chứng thực Giấy chứng nhận đăng ký doanh nghiệp, Thông báo mẫu dấu, và Căn cước công dân của tất cả những người có liên quan đến tài khoản.",
    """Khi tiến hành chuẩn bị hồ sơ cho ĐVCNT, đối với các trường hợp người đại diện theo pháp luật ủy quyền cho cá nhân khác ký kết, hồ sơ phải bổ sung thêm Giấy Ủy quyền (Bản gốc, ghi rõ thông tin của người được ủy quyền, phạm vi ủy quyền). Đồng thời, cần cung cấp CCCD/Hộ chiếu của người được ủy quyền (Sao y công chứng). Tài khoản thanh toán tổ chức mở tại Shinhan (HKD/DNTN/CTY) phải được đăng ký để nhận tiền thanh toán từ các giao dịch quẹt thẻ.""",
    ]

dic = {
    query: documents,
}

In [8]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Shinhan cung cấp những sản phẩm thẻ tín dụng, gói vay và dịch vụ thanh toán nào cho KHDN và HKD SME
[2] Score: 0.4685
Về giải pháp tài chính, Shinhan Bank cung cấp đa dạng các gói Vay tài trợ kinh doanh cho SME và Hộ kinh doanh. Các sản phẩm tín dụng nổi bật bao gồm: Hỗ trợ vay bổ sung vốn lưu động ngắn hạn, Vay vốn đầu tư kinh doanh trung và dài hạn. Ngân hàng cũng cung cấp dịch vụ bảo lãnh, dịch vụ bao thanh toán, cho vay thế chấp khoản phải thu, và dịch vụ giải ngân trực tuyến vô cùng thuận tiện trên hệ thống điện tử.
----------------------------------------------------------------------
[1] Score: 0.4487
Dành cho khách hàng doanh nghiệp và hộ kinh doanh, Shinhan cung cấp các dòng thẻ tín dụng chuyên biệt như Thẻ tín dụng Shinhan Business Owner và thẻ Shinhan SME. Thẻ tín dụng doanh nghiệp giúp tách bạch chi tiêu cá nhân và doanh nghiệp, quản lý dòng tiền hiệu quả. Thẻ Business Owner (hạng Bạch Kim) cung cấp ưu đãi hoàn tiền 5% cho ẩm thực và 2% cho quảng cáo trực tuyến, đ

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Shinhan cung cấp những sản phẩm thẻ tín dụng, gói vay và dịch vụ thanh toán nào cho KHDN và HKD SME
[2] Score: 0.9958
Về giải pháp tài chính, Shinhan Bank cung cấp đa dạng các gói Vay tài trợ kinh doanh cho SME và Hộ kinh doanh. Các sản phẩm tín dụng nổi bật bao gồm: Hỗ trợ vay bổ sung vốn lưu động ngắn hạn, Vay vốn đầu tư kinh doanh trung và dài hạn. Ngân hàng cũng cung cấp dịch vụ bảo lãnh, dịch vụ bao thanh toán, cho vay thế chấp khoản phải thu, và dịch vụ giải ngân trực tuyến vô cùng thuận tiện trên hệ thống điện tử.
----------------------------------------------------------------------
[1] Score: 0.9876
Dành cho khách hàng doanh nghiệp và hộ kinh doanh, Shinhan cung cấp các dòng thẻ tín dụng chuyên biệt như Thẻ tín dụng Shinhan Business Owner và thẻ Shinhan SME. Thẻ tín dụng doanh nghiệp giúp tách bạch chi tiêu cá nhân và doanh nghiệp, quản lý dòng tiền hiệu quả. Thẻ Business Owner (hạng Bạch Kim) cung cấp ưu đãi hoàn tiền 5% cho ẩm thực và 2% cho quảng cáo trực tuyến, đ

In [15]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Shinhan cung cấp những sản phẩm thẻ tín dụng, gói vay và dịch vụ thanh toán nào cho KHDN và HKD SME
[2] Score: 0.9880
Về giải pháp tài chính, Shinhan Bank cung cấp đa dạng các gói Vay tài trợ kinh doanh cho SME và Hộ kinh doanh. Các sản phẩm tín dụng nổi bật bao gồm: Hỗ trợ vay bổ sung vốn lưu động ngắn hạn, Vay vốn đầu tư kinh doanh trung và dài hạn. Ngân hàng cũng cung cấp dịch vụ bảo lãnh, dịch vụ bao thanh toán, cho vay thế chấp khoản phải thu, và dịch vụ giải ngân trực tuyến vô cùng thuận tiện trên hệ thống điện tử.
----------------------------------------------------------------------
[5] Score: 0.9597
Tài khoản S-Payroll cung cấp các ưu đãi miễn phí hoàn toàn cho khách hàng, bao gồm: Miễn phí phát hành thẻ ATM, thẻ tín dụng, thẻ phát hành rút tiền tại ATM Shinhan và các ngân hàng khác; Miễn phí chuyển tiền (chuyển tiền trong nước, chuyển khoản 24/7 qua Internet Banking/Mobile Banking/ATM); Miễn phí thanh toán hóa đơn điện, nước, internet, truyền hình cáp; Miễn phí dịch

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Shinhan cung cấp những sản phẩm thẻ tín dụng, gói vay và dịch vụ thanh toán nào cho KHDN và HKD SME
[1] Score: 0.3521
Dành cho khách hàng doanh nghiệp và hộ kinh doanh, Shinhan cung cấp các dòng thẻ tín dụng chuyên biệt như Thẻ tín dụng Shinhan Business Owner và thẻ Shinhan SME. Thẻ tín dụng doanh nghiệp giúp tách bạch chi tiêu cá nhân và doanh nghiệp, quản lý dòng tiền hiệu quả. Thẻ Business Owner (hạng Bạch Kim) cung cấp ưu đãi hoàn tiền 5% cho ẩm thực và 2% cho quảng cáo trực tuyến, đặc biệt phù hợp cho các chủ cơ sở kinh doanh vừa và nhỏ.
----------------------------------------------------------------------
[2] Score: 0.3153
Về giải pháp tài chính, Shinhan Bank cung cấp đa dạng các gói Vay tài trợ kinh doanh cho SME và Hộ kinh doanh. Các sản phẩm tín dụng nổi bật bao gồm: Hỗ trợ vay bổ sung vốn lưu động ngắn hạn, Vay vốn đầu tư kinh doanh trung và dài hạn. Ngân hàng cũng cung cấp dịch vụ bảo lãnh, dịch vụ bao thanh toán, cho vay thế chấp khoản phải thu, và dịch vụ giải n

### Case 5

In [ ]:
query = "đăng ký chữ ký của KTT và người đại diện hợp pháp trên chứng từ giao dịch thanh toán của doanh nghiệp được thực hiệc thế nào"

documents = [
    # HIGH relevant
    # Chunk 1: Độ tương đồng cao nhất - Trả lời trực tiếp quy định ký chứng từ và ngoại lệ
    "Theo quy định tại Nghị định 174/2016/NĐ-CP và hướng dẫn nội bộ, chứng từ giao dịch của khách hàng tổ chức phải có đủ chữ ký của Người đại diện theo pháp luật và Kế toán trưởng (hoặc người phụ trách kế toán) cùng dấu của đơn vị. Tại ô dành cho Kế toán trưởng sẽ do người được bổ nhiệm ký. Tuy nhiên, doanh nghiệp siêu nhỏ theo quy định pháp luật về hỗ trợ doanh nghiệp nhỏ và vừa không bắt buộc phải bố trí Kế toán trưởng. Đối với chữ ký, Kế toán trưởng không được ủy quyền lại cho người khác.",

    # Chunk 2: Độ tương đồng rất cao - Quy định về các đối tượng miễn trừ Kế toán trưởng
    "Trong quá trình mở tài khoản và đăng ký mẫu chữ ký, một số tổ chức không bắt buộc phải đăng ký chữ ký Kế toán trưởng hoặc Người phụ trách kế toán. Các đối tượng này bao gồm: Văn phòng đại diện của Nhà đầu tư nước ngoài trong hợp đồng BCC, Văn phòng điều hành của Nhà thầu nước ngoài, Văn phòng đại diện của các công ty nước ngoài và trong nước, Tổ chức phi lợi nhuận, và Ban quản trị nhà chung cư. Các tổ chức này có thể đánh dấu vào tùy chọn không bắt buộc bố trí Kế toán trưởng trên biểu mẫu.",

    # Chunk 3: Độ tương đồng cao - Xử lý khi hết hạn giấy tờ ủy quyền hoặc giấy tờ tùy thân
    "Căn cứ Thông tư 17/2024/TT-NHNN, việc tạm dừng giao dịch thanh toán, rút tiền trên tài khoản tổ chức sẽ được áp dụng khi giấy tờ tùy thân của người có liên quan hết hiệu lực. Cụ thể, nếu giấy tờ tùy thân của Người đại diện hợp pháp hoặc Kế toán trưởng hết hạn, ngân hàng sẽ tạm dừng trên toàn bộ giao dịch thanh toán của tài khoản. Nếu giấy tờ của Người được ủy quyền hết hạn, ngân hàng chỉ tạm dừng các giao dịch do chính người được ủy quyền đó yêu cầu, các người có thẩm quyền khác vẫn giao dịch bình thường.",

    # Chunk 4: Tương đồng khá - Tình huống thay đổi Kế toán trưởng/Chủ tịch HĐTV
    "Trong các trường hợp doanh nghiệp thay đổi nhân sự quản lý như Kế toán trưởng hoặc Chủ tịch Hội đồng thành viên, các quyết định bổ nhiệm cũ có thể vẫn còn hiệu lực cho đến khi có văn bản bãi nhiệm thay thế. Ví dụ, việc công ty thay đổi Chủ tịch HĐTV không tự động làm chấm dứt hiệu lực của Quyết định bổ nhiệm kế toán trưởng do Chủ tịch cũ ký. Tuy nhiên, khách hàng có trách nhiệm đăng ký thay đổi thông tin kế toán trưởng và cung cấp hồ sơ cập nhật cho chi nhánh ngân hàng (SHBVN) theo đúng quy định pháp luật.",

    # MEDIUM, LOW, NO relenvant
    "Bạn không thể tự ý xóa tên người đồng sở hữu khỏi tài khoản chung mà không có sự đồng ý và chữ ký xác nhận của họ tại quầy giao dịch.",
    # Chunk 5: Tương đồng trung bình - Tài khoản phong tỏa và Tài khoản đồng chủ sở hữu
    "Bên cạnh TDA và CD, khách hàng còn sử dụng Tài khoản ký quỹ (Escrow Account) và Tài khoản chung (Joint Account). Tài khoản ký quỹ dùng để phong tỏa một khoản tiền theo thỏa thuận nhằm đảm bảo thực hiện nghĩa vụ giữa các bên, tiền chỉ được giải tỏa khi các điều kiện hợp đồng được hoàn tất. Tài khoản chung là tài khoản có từ hai chủ thể trở lên cùng đứng tên, việc rút tiền hoặc hạch toán đòi hỏi sự đồng thuận của tất cả các chủ tài khoản bằng chữ ký hợp lệ trên chứng từ giao dịch.",
    # Chunk 7: Tương đồng thấp - Nhận lệnh giao dịch qua Fax/Email
    "Trong quá trình giao dịch các khoản tiền gửi, khách hàng tổ chức có nhu cầu nhận, gửi lệnh qua Fax/Email để tiết kiệm thời gian. Dịch vụ này yêu cầu khách hàng đăng ký rõ địa chỉ Email thuộc tên miền riêng của doanh nghiệp. Ngân hàng sẽ không chấp nhận các email công cộng (như Gmail, Yahoo). Nhân viên giao dịch kiểm tra thông tin thanh toán, chữ ký, mẫu dấu trên bản scan, gọi điện cho người liên hệ thứ nhất để xác thực trước khi hạch toán rút tiền hoặc chuyển tiền từ tài khoản DDA hay TDA.",
    # Mở tk, mở CIF (Đăng ký sinh trắc học, mở tài khoản, phê duyệt)
    """Ở Bước 6, nhân viên thực hiện Đăng ký "Chấp thuận về xử lý dữ liệu cá nhân" trên màn hình 0044. Đăng ký thông tin CDD ở màn hình 6803 và FATCA ở màn hình 6316. Kiểm tra đối chiếu khớp đúng thông tin sinh trắc học (màn hình 0042 đối với CMND/CCCD hoặc qua ứng dụng SOL đối với Căn cước công dân gắn chip). Mở tài khoản bằng màn hình 0110. Sau đó, thu thập chữ ký và dấu vân tay sinh trắc học. Bước 8, Giao dịch viên đưa hồ sơ cho Kiểm soát viên phê duyệt trên màn hình và phê duyệt tay trên hồ sơ giấy.""",
    "Tỷ giá mua bán ngoại tệ giao ngay (Spot Rate) được áp dụng cho các giao dịch thanh toán quốc tế có ngày giá trị trong vòng 2 ngày làm việc kể từ ngày giao kết hợp đồng.",
"Một chứng từ được coi là hợp lệ để tiến hành giao dịch thanh toán phải đáp ứng các nguyên tắc cơ bản. Chứng từ giấy phải là bản chính hoặc bản sao có chứng thực theo quy định pháp luật. Chứng từ không được rách nát, tẩy xóa, mờ nét. Với chứng từ điện tử, file gửi qua email phải rõ ràng, đảm bảo tính vẹn toàn dữ liệu. Khách hàng khi nộp bổ sung chứng từ gốc của giao dịch Fax phải nộp đúng bản chính đã đóng dấu mộc đỏ và chữ ký sống tươi, không dùng dấu scan in màu.",

    ]

dic = {
    query: documents,
}

In [ ]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: đăng ký chữ ký của KTT và người đại diện hợp pháp trên chứng từ giao dịch thanh toán của doanh nghiệp được thực hiệc thế nào
[1] Score: 0.4618
Theo quy định tại Nghị định 174/2016/NĐ-CP và hướng dẫn nội bộ, chứng từ giao dịch của khách hàng tổ chức phải có đủ chữ ký của Người đại diện theo pháp luật và Kế toán trưởng (hoặc người phụ trách kế toán) cùng dấu của đơn vị. Tại ô dành cho Kế toán trưởng sẽ do người được bổ nhiệm ký. Tuy nhiên, doanh nghiệp siêu nhỏ theo quy định pháp luật về hỗ trợ doanh nghiệp nhỏ và vừa không bắt buộc phải bố trí Kế toán trưởng. Đối với chữ ký, Kế toán trưởng không được ủy quyền lại cho người khác.
----------------------------------------------------------------------
[2] Score: 0.3445
Trong quá trình mở tài khoản và đăng ký mẫu chữ ký, một số tổ chức không bắt buộc phải đăng ký chữ ký Kế toán trưởng hoặc Người phụ trách kế toán. Các đối tượng này bao gồm: Văn phòng đại diện của Nhà đầu tư nước ngoài trong hợp đồng BCC, Văn phòng điều hành của Nh

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: đăng ký chữ ký của KTT và người đại diện hợp pháp trên chứng từ giao dịch thanh toán của doanh nghiệp được thực hiệc thế nào
[1] Score: 0.8260
Theo quy định tại Nghị định 174/2016/NĐ-CP và hướng dẫn nội bộ, chứng từ giao dịch của khách hàng tổ chức phải có đủ chữ ký của Người đại diện theo pháp luật và Kế toán trưởng (hoặc người phụ trách kế toán) cùng dấu của đơn vị. Tại ô dành cho Kế toán trưởng sẽ do người được bổ nhiệm ký. Tuy nhiên, doanh nghiệp siêu nhỏ theo quy định pháp luật về hỗ trợ doanh nghiệp nhỏ và vừa không bắt buộc phải bố trí Kế toán trưởng. Đối với chữ ký, Kế toán trưởng không được ủy quyền lại cho người khác.
----------------------------------------------------------------------
[2] Score: 0.3273
Trong quá trình mở tài khoản và đăng ký mẫu chữ ký, một số tổ chức không bắt buộc phải đăng ký chữ ký Kế toán trưởng hoặc Người phụ trách kế toán. Các đối tượng này bao gồm: Văn phòng đại diện của Nhà đầu tư nước ngoài trong hợp đồng BCC, Văn phòng điều hành của Nh

In [ ]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: đăng ký chữ ký của KTT và người đại diện hợp pháp trên chứng từ giao dịch thanh toán của doanh nghiệp được thực hiệc thế nào
[1] Score: 0.9881
Theo quy định tại Nghị định 174/2016/NĐ-CP và hướng dẫn nội bộ, chứng từ giao dịch của khách hàng tổ chức phải có đủ chữ ký của Người đại diện theo pháp luật và Kế toán trưởng (hoặc người phụ trách kế toán) cùng dấu của đơn vị. Tại ô dành cho Kế toán trưởng sẽ do người được bổ nhiệm ký. Tuy nhiên, doanh nghiệp siêu nhỏ theo quy định pháp luật về hỗ trợ doanh nghiệp nhỏ và vừa không bắt buộc phải bố trí Kế toán trưởng. Đối với chữ ký, Kế toán trưởng không được ủy quyền lại cho người khác.
----------------------------------------------------------------------
[10] Score: 0.6862
Một chứng từ được coi là hợp lệ để tiến hành giao dịch thanh toán phải đáp ứng các nguyên tắc cơ bản. Chứng từ giấy phải là bản chính hoặc bản sao có chứng thực theo quy định pháp luật. Chứng từ không được rách nát, tẩy xóa, mờ nét. Với chứng từ điện tử, file gửi 

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: đăng ký chữ ký của KTT và người đại diện hợp pháp trên chứng từ giao dịch thanh toán của doanh nghiệp được thực hiệc thế nào
[1] Score: 0.3953
Theo quy định tại Nghị định 174/2016/NĐ-CP và hướng dẫn nội bộ, chứng từ giao dịch của khách hàng tổ chức phải có đủ chữ ký của Người đại diện theo pháp luật và Kế toán trưởng (hoặc người phụ trách kế toán) cùng dấu của đơn vị. Tại ô dành cho Kế toán trưởng sẽ do người được bổ nhiệm ký. Tuy nhiên, doanh nghiệp siêu nhỏ theo quy định pháp luật về hỗ trợ doanh nghiệp nhỏ và vừa không bắt buộc phải bố trí Kế toán trưởng. Đối với chữ ký, Kế toán trưởng không được ủy quyền lại cho người khác.
----------------------------------------------------------------------
[2] Score: -0.0121
Trong quá trình mở tài khoản và đăng ký mẫu chữ ký, một số tổ chức không bắt buộc phải đăng ký chữ ký Kế toán trưởng hoặc Người phụ trách kế toán. Các đối tượng này bao gồm: Văn phòng đại diện của Nhà đầu tư nước ngoài trong hợp đồng BCC, Văn phòng điều hành của N

Doc [10] chỉ định nghĩa thế nào là một chứng từ thanh toán hợp lệ. Nó mô tả tình trạng vật lý của chứng từ lúc được sử dụng, chứ hoàn toàn không trả lời câu hỏi "làm thủ tục đăng ký **CHỮ KÝ MẪU BAN ĐẦU** như thế nào" -> doc này ở bước SỬ DỤNG chứ không phải bước ĐĂNG KÝ -> cả 4 mô hình đều bị lừa

Doc [4] nên đạt mức NORMAL SIMILARITY vì thứ nhất có chứa các keyword: Kế toán trưởng, đăng ký thay đổi thông tin, và người đại diện/chủ tịch; thứ hai giải quyết một case pháp lý (legal case): "Quyết định bổ nhiệm KTT cũ có bị mất hiệu lực khi thay đổi Giám đốc/Chủ tịch không?" -> Doc [4] dù không hướng dẫn quy trình (How-to), nhưng nó lại trúng tệp ngữ cảnh (Domain)

[5] [6] nói về Tài khoản chung (Joint Account) của cá nhân, hoàn toàn không liên quan đến Doanh nghiệp hay Kế toán trưởng. Mô hình bị "lừa" vì trong text có chứa cụm từ "chữ ký xác nhận" (Doc 5) và "chữ ký hợp lệ trên chứng từ giao dịch" (Doc 6) -> mô hình nên chấm [5] [6] ở mức LOW

### Case 6

In [ ]:
query = "chỉ số đánh giá rủi ro kinh doanh, tài chính và điểm tín dụng dành riêng cho khách hàng SOHO"

documents = [
    # HIGH
    "Đánh giá khách hàng SOHO tập trung vào rủi ro hoạt động kinh doanh do khả năng tài chính và vốn tương đối yếu, uy tín và khả năng điều hành phụ thuộc vào cá nhân chủ doanh nghiệp [47]. Rủi ro cao trong nguồn vốn cá nhân bị hòa lẫn với nguồn vốn kinh doanh [47]. Số tiền vay tương đối nhỏ nhưng tệp khách hàng lớn và mức độ cạnh tranh giữa các tổ chức tài chính rất cao [47].",
    "Bảng đánh giá giá trị SOHO (SOHO Assessment Table) đo lường các tiêu chí phi tài chính: Chuyên môn kinh nghiệm (thời gian làm việc trong cùng ngành >= 10 năm được điểm tối đa), Tài sản sở hữu (Tổng tài sản > 20 tỷ VND), Khả năng gây quỹ khẩn cấp (tiền gửi của chủ doanh nghiệp) [48]. Độ tin cậy được đánh giá bằng Lịch sử nợ (Nhóm 1 trong 16 tháng gần nhất) và Tình trạng trả nợ [48].",
    "Về mặt tài chính, rủi ro SOHO được phân tích qua Tỷ lệ nợ ròng/EBITDA. Ngưỡng rủi ro cao là > 5.0 hoặc EBITDA <= 0 [20]. Tỷ lệ phụ thuộc nợ vay (Tổng nợ vay / Tổng tài sản) > 50% cũng bị đánh giá là rủi ro cao [20]. Các chỉ số này giúp xác định dòng tiền thực tế có đủ bù đắp nghĩa vụ nợ (EBITDA / Lãi vay) [20].",
    "Hệ số thanh toán nhanh (Tài sản ngắn hạn - Hàng tồn kho) / Nợ ngắn hạn nếu < 100% được xem là dấu hiệu cảnh báo [49]. Tăng trưởng doanh thu (Doanh thu năm nay - Doanh thu năm trước) / Doanh thu năm trước nếu < 0% (tăng trưởng âm) sẽ đánh dấu ngưỡng rủi ro cao [50]. Dòng tiền sau khi thanh toán nợ vay < 0 yêu cầu phải xem xét lại mức cấp tín dụng [51].",
    # Others
    # Chunk 5: Tương đồng trung bình - Tạm dừng giao dịch do hết hạn giấy tờ
    "Trong quá trình kiểm soát rủi ro giao dịch từ xa, nếu hệ thống cảnh báo giấy tờ tùy thân (CCCD/Hộ chiếu) của Người có liên quan trong công ty đã hết hiệu lực, ngân hàng sẽ chặn các giao dịch. Nếu CCCD của Kế toán trưởng hết hạn, toàn bộ giao dịch thanh toán bị tạm dừng cho đến khi khách hàng nộp CCCD mới và cập nhật lên hệ thống. Giao dịch Fax/Email cũng không được thực hiện trong thời gian tài khoản bị khóa do nguyên nhân này, nhằm tuân thủ Thông tư 17/2024/TT-NHNN.",
# Chunk 10: Lowest Similarity (Tính toán thu nhập chung)
    """Khả năng thanh toán (Loan redemption capability) được tính bằng Thu nhập trung bình hàng tháng trừ đi các Khoản chi trả hàng tháng. Hệ số nhân để xác định hạn mức tín dụng phụ thuộc vào điểm AS (Application Scorecard) của khách hàng, thường là 2, 2.5, 3.3 hoặc 4. Thu nhập cuối cùng dùng để xét duyệt được tính theo công thức: Thu nhập cố định + 75% * (Thu nhập trung bình 3 tháng gần nhất - Thu nhập cố định). Việc chênh lệch thu nhập giữa các tháng phải được giải trình rõ ràng, nếu chênh lệch trên 120% cần có xác nhận thưởng năng suất từ công ty. Mức kiểm soát rủi ro dư nợ tín chấp tối đa (MUE) cho một khách hàng cá nhân không vượt quá 900 triệu đồng (bao gồm cả dư nợ thẻ và vay tiêu dùng). Hạn mức cấp dựa trên thu nhập hàng tháng (MI) thường dao động từ MI x 2 đến MI x 3 tùy hạng khách hàng và lịch sử tín dụng."""
# Low similarity (Loan products)
    "Bên cạnh thẻ tín dụng, Shinhan Bank cung cấp các sản phẩm vay cá nhân trực tuyến đa dạng nhằm đáp ứng nhu cầu tài chính của khách hàng. Các sản phẩm bao gồm Vay mua nhà, vay mua xe mới và xe đã qua sử dụng, vay tiêu dùng cá nhân. Khách hàng có thể tham gia chương trình trả nợ khoản vay thế chấp bất động sản tại ngân hàng khác để hưởng lãi suất ưu đãi từ Shinhan.",
# 1. Chứng từ tài chính lõi cho Hộ kinh doanh
    """Danh mục chứng từ tài chính yêu cầu đối với Hộ kinh doanh vay vốn SOHO bao gồm: Sao kê tài khoản ngân hàng trong vòng 3 tháng hoặc 6 tháng gần nhất, sổ sách ghi chép bán hàng có chữ ký, dữ liệu kết xuất từ hệ thống bán hàng điện tử. Ngoài ra, cần bổ sung các chứng từ đầu ra/đầu vào như Hợp đồng mua bán, hóa đơn VAT hoặc hóa đơn điện tử hợp lệ.""",
# 2. Chứng từ tài chính lõi cho Doanh nghiệp tư nhân
    """Đối với pháp nhân là Doanh nghiệp siêu nhỏ (Micro Enterprise) vay gói SOHO, hồ sơ chứng minh năng lực tài chính bắt buộc bao gồm: Báo cáo tài chính, báo cáo thuế năm gần nhất thể hiện hoạt động kinh doanh có lợi nhuận. Bên cạnh đó, ngân hàng yêu cầu sao kê tài khoản chính của doanh nghiệp, hợp đồng kinh tế đầu ra/đầu vào lớn và biên lai nộp thuế.""",
# Chunk 3 (Rank 3 - Khá): Cơ chế tính điểm Thẻ tín dụng và Chuyển tiền ngoại tệ.
    """Bên cạnh tiền gửi và khoản vay, điểm Tops Club còn được tính từ giao dịch thẻ tín dụng và chuyển tiền/mua bán ngoại tệ. Đối với Thẻ tín dụng, tổng số tiền thanh toán thẻ tín dụng trong 3 tháng (chỉ tính giao dịch POS) được cộng 1 điểm cho mỗi 500,000 VND thanh toán. Đối với Giao dịch Chuyển khoản, tổng số giao dịch chuyển tiền ngoại tệ trong 3 tháng sẽ mang lại 10 điểm cho mỗi 1 giao dịch. Khách hàng thực hiện mua/bán ngoại tệ mang lại 2 điểm cho mỗi 500 USD giao dịch thành công.""",

    ]

dic = {
    query: documents,
}

In [ ]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: chỉ số đánh giá rủi ro kinh doanh, tài chính và điểm tín dụng dành riêng cho khách hàng SOHO
[2] Score: 0.4023
Bảng đánh giá giá trị SOHO (SOHO Assessment Table) đo lường các tiêu chí phi tài chính: Chuyên môn kinh nghiệm (thời gian làm việc trong cùng ngành >= 10 năm được điểm tối đa), Tài sản sở hữu (Tổng tài sản > 20 tỷ VND), Khả năng gây quỹ khẩn cấp (tiền gửi của chủ doanh nghiệp) [48]. Độ tin cậy được đánh giá bằng Lịch sử nợ (Nhóm 1 trong 16 tháng gần nhất) và Tình trạng trả nợ [48].
----------------------------------------------------------------------
[3] Score: 0.3945
Về mặt tài chính, rủi ro SOHO được phân tích qua Tỷ lệ nợ ròng/EBITDA. Ngưỡng rủi ro cao là > 5.0 hoặc EBITDA <= 0 [20]. Tỷ lệ phụ thuộc nợ vay (Tổng nợ vay / Tổng tài sản) > 50% cũng bị đánh giá là rủi ro cao [20]. Các chỉ số này giúp xác định dòng tiền thực tế có đủ bù đắp nghĩa vụ nợ (EBITDA / Lãi vay) [20].
----------------------------------------------------------------------
[1] Score: 0.3497
Đán

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: chỉ số đánh giá rủi ro kinh doanh, tài chính và điểm tín dụng dành riêng cho khách hàng SOHO
[3] Score: 0.9834
Về mặt tài chính, rủi ro SOHO được phân tích qua Tỷ lệ nợ ròng/EBITDA. Ngưỡng rủi ro cao là > 5.0 hoặc EBITDA <= 0 [20]. Tỷ lệ phụ thuộc nợ vay (Tổng nợ vay / Tổng tài sản) > 50% cũng bị đánh giá là rủi ro cao [20]. Các chỉ số này giúp xác định dòng tiền thực tế có đủ bù đắp nghĩa vụ nợ (EBITDA / Lãi vay) [20].
----------------------------------------------------------------------
[1] Score: 0.9657
Đánh giá khách hàng SOHO tập trung vào rủi ro hoạt động kinh doanh do khả năng tài chính và vốn tương đối yếu, uy tín và khả năng điều hành phụ thuộc vào cá nhân chủ doanh nghiệp [47]. Rủi ro cao trong nguồn vốn cá nhân bị hòa lẫn với nguồn vốn kinh doanh [47]. Số tiền vay tương đối nhỏ nhưng tệp khách hàng lớn và mức độ cạnh tranh giữa các tổ chức tài chính rất cao [47].
----------------------------------------------------------------------
[2] Score: 0.8814
Bảng đánh giá

In [ ]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: chỉ số đánh giá rủi ro kinh doanh, tài chính và điểm tín dụng dành riêng cho khách hàng SOHO
[1] Score: 0.9912
Đánh giá khách hàng SOHO tập trung vào rủi ro hoạt động kinh doanh do khả năng tài chính và vốn tương đối yếu, uy tín và khả năng điều hành phụ thuộc vào cá nhân chủ doanh nghiệp [47]. Rủi ro cao trong nguồn vốn cá nhân bị hòa lẫn với nguồn vốn kinh doanh [47]. Số tiền vay tương đối nhỏ nhưng tệp khách hàng lớn và mức độ cạnh tranh giữa các tổ chức tài chính rất cao [47].
----------------------------------------------------------------------
[3] Score: 0.9911
Về mặt tài chính, rủi ro SOHO được phân tích qua Tỷ lệ nợ ròng/EBITDA. Ngưỡng rủi ro cao là > 5.0 hoặc EBITDA <= 0 [20]. Tỷ lệ phụ thuộc nợ vay (Tổng nợ vay / Tổng tài sản) > 50% cũng bị đánh giá là rủi ro cao [20]. Các chỉ số này giúp xác định dòng tiền thực tế có đủ bù đắp nghĩa vụ nợ (EBITDA / Lãi vay) [20].
----------------------------------------------------------------------
[2] Score: 0.9901
Bảng đánh giá

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: chỉ số đánh giá rủi ro kinh doanh, tài chính và điểm tín dụng dành riêng cho khách hàng SOHO
[1] Score: 0.4002
Đánh giá khách hàng SOHO tập trung vào rủi ro hoạt động kinh doanh do khả năng tài chính và vốn tương đối yếu, uy tín và khả năng điều hành phụ thuộc vào cá nhân chủ doanh nghiệp [47]. Rủi ro cao trong nguồn vốn cá nhân bị hòa lẫn với nguồn vốn kinh doanh [47]. Số tiền vay tương đối nhỏ nhưng tệp khách hàng lớn và mức độ cạnh tranh giữa các tổ chức tài chính rất cao [47].
----------------------------------------------------------------------
[3] Score: 0.2807
Về mặt tài chính, rủi ro SOHO được phân tích qua Tỷ lệ nợ ròng/EBITDA. Ngưỡng rủi ro cao là > 5.0 hoặc EBITDA <= 0 [20]. Tỷ lệ phụ thuộc nợ vay (Tổng nợ vay / Tổng tài sản) > 50% cũng bị đánh giá là rủi ro cao [20]. Các chỉ số này giúp xác định dòng tiền thực tế có đủ bù đắp nghĩa vụ nợ (EBITDA / Lãi vay) [20].
----------------------------------------------------------------------
[2] Score: 0.2053
Bảng đánh giá

Qwen3 0.6B nhận định [8] có HIGH relevant nhưng Jina thì không

Mặc dù Báo cáo tài chính (trong [8]) chính là nơi để trích xuất ra các Chỉ số tài chính (trong Query), nhưng [8] không hề trả lời trực tiếp câu hỏi: "Các chỉ số đó là gì và điểm tín dụng được tính ra sao?".

### Case 7

In [ ]:
query = "sử dụng ứng dụng VssID và VNeID để xác minh danh tính, lịch sử đóng BHXH, nơi cư trú và các quy định về việc chụp màn hình thay cho hồ sơ bản cứng"

documents = [
    # HIGH relevant
    # Chunk 1: High Similarity (Cách thao tác và kiểm tra ứng dụng VssID)
    """VssID là ứng dụng số của Bảo hiểm Xã hội Việt Nam, được Shinhan Bank chấp thuận thay thế cho một số chứng từ chứng minh thu nhập và việc làm bản cứng. Yêu cầu khách hàng đăng nhập ứng dụng VssID trên điện thoại. Tại màn hình chính, chọn "Quá trình tham gia". Hệ thống sẽ hiển thị lịch sử làm việc, tên cơ quan/công ty đóng BHXH, thời gian đóng và Mức lương đóng BHXH. Nhân viên kiểm tra xem khách hàng có đóng BHXH liên tục tại công ty hiện tại từ 6 tháng trở lên hay không. Số tiền lương đóng BHXH trên VssID sẽ được đối chiếu với sao kê lương qua ngân hàng để xem xét hạn mức. Nhân viên phải thực hiện thao tác chụp màn hình (Screenshot) toàn bộ các thông tin này, đảm bảo hiển thị rõ tên khách hàng, mã số BHXH và quá trình đóng chi tiết nhất.""",

    # Chunk 2: High Similarity (Quy định chụp màn hình VssID và VNeID)
    """Quá trình chụp ảnh màn hình (screenshot) các ứng dụng điện tử như VssID, VNeID, hoặc cổng thông tin baohiemxahoi.gov.vn phải tuân thủ nghiêm ngặt Hướng dẫn chụp màn hình của Phòng Quản lý rủi ro. Điện thoại sử dụng để chụp phải là điện thoại của khách hàng hoặc ipad của chi nhánh mở ứng dụng trực tiếp, không được dùng ảnh khách hàng tự chụp gửi qua Zalo. Ảnh chụp màn hình phải hiển thị rõ logo ứng dụng, tên tài khoản, thời gian trên thanh trạng thái điện thoại. Với VNeID, do ứng dụng có chính sách bảo mật không cho phép chức năng chụp màn hình tự động, dùng điện thoại khác để chụp lại thiết bị của KH. Bản sao chứng từ in ra từ ảnh chụp phải có chữ ký nháy của nhân viên thẩm định và dấu mộc của cấp Quản lý kênh hoặc chi nhánh.""",

    # Chunk 3: High-Medium Similarity (Sử dụng VNeID xác thực cư trú và nhân thân)
    """VNeID là ứng dụng định danh điện tử của Bộ Công an, có giá trị pháp lý thay thế cho CCCD vật lý và Sổ hộ khẩu/Giấy tạm trú. Khách hàng sử dụng tài khoản VNeID cấp độ 2 (đã đăng ký sinh trắc học tại công an) được chấp nhận làm Hồ sơ nhân dạng và Hồ sơ chứng minh nơi cư trú. Nhân viên hướng dẫn khách hàng mở VNeID, vào mục "Ví giấy tờ", chọn "Căn cước công dân" để xem hình ảnh CCCD 2 mặt và chọn "Thông tin cư trú" để kiểm tra địa chỉ Thường trú và Nơi ở hiện tại. Nếu địa chỉ Nơi ở hiện tại hiển thị đầy đủ 4 cấp (Tỉnh/Thành phố, Quận/Huyện, Phường/Xã, Số nhà/Đường) và khách hàng sinh sống tại TP.HCM hoặc Hà Nội, hồ sơ sẽ được miễn cung cấp Hóa đơn điện/nước hoặc chứng từ tạm trú (CT07).""",

    # Chunk 4: Medium Similarity (Trường hợp không dùng được VssID/VNeID)
    """Trong trường hợp khách hàng không cài đặt VssID, VNeID, hoặc ứng dụng bị lỗi hệ thống (bảo trì), ngân hàng bắt buộc khách hàng phải cung cấp hồ sơ bản cứng theo quy trình truyền thống. Đối với chứng minh nơi cư trú: Nộp CCCD gắn chip (bản sao y có mộc), Giấy xác nhận thông tin về cư trú mẫu CT07/CT08 do công an cấp trong vòng 2 tháng, hoặc Hợp đồng thuê nhà kèm Hóa đơn điện nước 3 tháng gần nhất. Đối với chứng minh công việc và thu nhập: Khách hàng nộp Hợp đồng lao động (bản sao y công ty hoặc có đóng dấu treo, chữ ký giáp lai), kèm theo Sao kê lương ngân hàng 3 tháng gần nhất (có mộc đỏ của ngân hàng phát hành sao kê).""",

    # MEDIUM, LOW, NO relenvant
    """Bên cạnh CIC, một nguồn dữ liệu quan trọng để thẩm định là ứng dụng VssID (Bảo hiểm xã hội số). Ứng dụng này giúp ngân hàng xác minh lịch sử làm việc và mức lương đóng bảo hiểm xã hội của khách hàng mà không cần yêu cầu giấy tờ bản cứng. Đối với người lao động nước ngoài không có VssID, phải nộp Giấy phép lao động (Work Permit) và Hợp đồng lao động còn hạn. Việc đối chiếu chéo giữa số tiền lương trên VssID, mức lương thực nhận qua sao kê tài khoản ngân hàng và khai báo trên hệ thống là bước kiểm soát gian lận thiết yếu để xác định chính xác thu nhập ròng làm cơ sở cấp hạn mức thẻ.""",
"""Ngoài việc thu thập chứng từ, ngân hàng thực hiện Thẩm định qua điện thoại (Tele-verification) và Thẩm định thực địa (Site visit) để chống gian lận. Bộ phận thẩm định sẽ gọi điện thoại đến số điện thoại bàn của công ty khách hàng để xác minh tình trạng việc làm, chức vụ, và gọi cho khách hàng để xác nhận các thông tin trên Đơn mở thẻ. Nếu công ty không có số điện thoại bàn hợp lệ, hoặc khách hàng thuộc nhóm rủi ro cao (như làm việc trong ngành nghề nhạy cảm, địa chỉ cư trú trên VNeID không rõ ràng, có dấu hiệu giả mạo hồ sơ), nhân viên ngân hàng sẽ phải đi thực địa trực tiếp đến nơi ở hoặc nơi làm việc của khách hàng. Tại đó, nhân viên sẽ chụp ảnh khách hàng tại địa điểm thực tế."""
    # Chunk 2 (Rank 2 - Cao): Giải thích chi tiết về thời hạn cư trú và con dấu nhập cảnh.
    """Người nước ngoài được phép cư trú tại Việt Nam với thời hạn từ 12 tháng (360 ngày) trở lên được xem là người cư trú. Tình trạng cư trú được xác định dựa trên thời hạn cư trú phép tại Việt Nam thể hiện trên giấy tờ tùy thân. Đối với Thẻ tạm trú, tình trạng cư trú có giá trị trong thời hạn của thẻ. Đối với khách hàng người nước ngoài (bao gồm người Hàn Quốc) sử dụng Thị thực điện tử (E-visa), thị thực này được phê duyệt trước khi nhập cảnh với thời hạn tối đa 90 ngày. Con dấu nhập cảnh bao gồm thời hạn cư trú được phép tại Việt Nam. Thời hạn cư trú sẽ là ngày nhập cảnh và ngày cuối cùng được phép cư trú tại Việt Nam trên Con dấu nhập cảnh.""",
# Chunk 9: Tương đồng cực thấp - Định nghĩa tổ chức cư trú
    "Để mở tài khoản đúng mục đích, cần phân biệt rõ trình trạng cư trú. Tổ chức cư trú (Resident) bao gồm các tổ chức tín dụng, chi nhánh ngân hàng nước ngoài được thành lập tại Việt Nam, các tổ chức kinh tế, cơ quan nhà nước, đơn vị lực lượng vũ trang, các tổ chức chính trị - xã hội, quỹ xã hội. Ngoài ra, chi nhánh tại Việt Nam của tổ chức kinh tế nước ngoài, văn phòng điều hành của nhà thầu nước ngoài hoạt động tại Việt Nam cũng được xếp vào nhóm tổ chức là người cư trú.",
# Chunk 1: Độ tương đồng cao nhất - Tiêu chí xác định cư trú và không cư trú
    "Tình trạng cư trú của một tổ chức quyết định các loại tài khoản họ được phép mở. Theo Pháp lệnh ngoại hối, Tổ chức là người cư trú (Resident) bao gồm: Tổ chức tín dụng, tổ chức kinh tế thành lập hoạt động tại VN, cơ quan nhà nước, đơn vị lực lượng vũ trang, tổ chức chính trị-xã hội tại VN. Các chi nhánh tại VN của tổ chức kinh tế nước ngoài, văn phòng điều hành nhà thầu nước ngoài cũng là người cư trú. Ngược lại, Tổ chức là người không cư trú (Non-resident) gồm: Cơ quan ngoại giao, lãnh sự, văn phòng đại diện quốc tế tại VN và các tổ chức hoạt động hoàn toàn ở nước ngoài.",
"Mật khẩu ngân hàng điện tử (Internet Banking) của bạn sẽ bị khóa tự động nếu bạn nhập sai quá 5 lần liên tiếp để bảo vệ tài khoản khỏi các cuộc tấn công brute-force.",

    ]

dic = {
    query: documents,
}

In [ ]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Cách sử dụng ứng dụng VssID và VNeID để xác minh danh tính, lịch sử đóng bảo hiểm xã hội, nơi cư trú và các quy định về việc chụp màn hình lưu hồ sơ chứng từ điện tử thay cho bản cứng
[1] Score: 0.3352
VssID là ứng dụng số của Bảo hiểm Xã hội Việt Nam, được Shinhan Bank chấp thuận thay thế cho một số chứng từ chứng minh thu nhập và việc làm bản cứng. Để xác minh, nhân viên yêu cầu khách hàng đăng nhập ứng dụng VssID trên điện thoại. Tại màn hình chính, chọn "Quá trình tham gia" (Quá trình tham gia BHXH). Hệ thống sẽ hiển thị lịch sử làm việc, tên cơ quan/công ty đóng BHXH (Employer), thời gian đóng (Accumulated time of contribution) và Mức lương đóng BHXH (Salary amount for social insurance contribution). Nhân viên kiểm tra xem khách hàng có đóng BHXH liên tục tại công ty hiện tại từ 6 tháng trở lên hay không. Số tiền lương đóng BHXH trên VssID sẽ được đối chiếu với sao kê lương qua ngân hàng để xem xét hạn mức. Nhân viên phải thực hiện thao tác chụp màn hình (Screenshot) toà

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Cách sử dụng ứng dụng VssID và VNeID để xác minh danh tính, lịch sử đóng bảo hiểm xã hội, nơi cư trú và các quy định về việc chụp màn hình lưu hồ sơ chứng từ điện tử thay cho bản cứng
[1] Score: 0.9818
VssID là ứng dụng số của Bảo hiểm Xã hội Việt Nam, được Shinhan Bank chấp thuận thay thế cho một số chứng từ chứng minh thu nhập và việc làm bản cứng. Để xác minh, nhân viên yêu cầu khách hàng đăng nhập ứng dụng VssID trên điện thoại. Tại màn hình chính, chọn "Quá trình tham gia" (Quá trình tham gia BHXH). Hệ thống sẽ hiển thị lịch sử làm việc, tên cơ quan/công ty đóng BHXH (Employer), thời gian đóng (Accumulated time of contribution) và Mức lương đóng BHXH (Salary amount for social insurance contribution). Nhân viên kiểm tra xem khách hàng có đóng BHXH liên tục tại công ty hiện tại từ 6 tháng trở lên hay không. Số tiền lương đóng BHXH trên VssID sẽ được đối chiếu với sao kê lương qua ngân hàng để xem xét hạn mức. Nhân viên phải thực hiện thao tác chụp màn hình (Screenshot) toà

Test này thể hiện Qwen3 4B bị ngáo, dễ bị lừa bởi keywords

In [ ]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: sử dụng ứng dụng VssID và VNeID để xác minh danh tính, lịch sử đóng BHXH, nơi cư trú và các quy định về việc chụp màn hình thay cho hồ sơ bản cứng
[1] Score: 0.9910
VssID là ứng dụng số của Bảo hiểm Xã hội Việt Nam, được Shinhan Bank chấp thuận thay thế cho một số chứng từ chứng minh thu nhập và việc làm bản cứng. Yêu cầu khách hàng đăng nhập ứng dụng VssID trên điện thoại. Tại màn hình chính, chọn "Quá trình tham gia". Hệ thống sẽ hiển thị lịch sử làm việc, tên cơ quan/công ty đóng BHXH, thời gian đóng và Mức lương đóng BHXH. Nhân viên kiểm tra xem khách hàng có đóng BHXH liên tục tại công ty hiện tại từ 6 tháng trở lên hay không. Số tiền lương đóng BHXH trên VssID sẽ được đối chiếu với sao kê lương qua ngân hàng để xem xét hạn mức. Nhân viên phải thực hiện thao tác chụp màn hình (Screenshot) toàn bộ các thông tin này, đảm bảo hiển thị rõ tên khách hàng, mã số BHXH và quá trình đóng chi tiết nhất.
----------------------------------------------------------------------
[4] Sco

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: sử dụng ứng dụng VssID và VNeID để xác minh danh tính, lịch sử đóng BHXH, nơi cư trú và các quy định về việc chụp màn hình thay cho hồ sơ bản cứng
[1] Score: 0.4861
VssID là ứng dụng số của Bảo hiểm Xã hội Việt Nam, được Shinhan Bank chấp thuận thay thế cho một số chứng từ chứng minh thu nhập và việc làm bản cứng. Yêu cầu khách hàng đăng nhập ứng dụng VssID trên điện thoại. Tại màn hình chính, chọn "Quá trình tham gia". Hệ thống sẽ hiển thị lịch sử làm việc, tên cơ quan/công ty đóng BHXH, thời gian đóng và Mức lương đóng BHXH. Nhân viên kiểm tra xem khách hàng có đóng BHXH liên tục tại công ty hiện tại từ 6 tháng trở lên hay không. Số tiền lương đóng BHXH trên VssID sẽ được đối chiếu với sao kê lương qua ngân hàng để xem xét hạn mức. Nhân viên phải thực hiện thao tác chụp màn hình (Screenshot) toàn bộ các thông tin này, đảm bảo hiển thị rõ tên khách hàng, mã số BHXH và quá trình đóng chi tiết nhất.
----------------------------------------------------------------------
[2] Sco

### Case 8

In [ ]:
query = "quy định, chứng từ yêu cầu và phương pháp tính thu nhập ròng hàng tháng đối với khách hàng hưởng lương và thưởng"

documents = [
    # HIGH
    "Thu nhập ròng hàng tháng = Thu nhập cố định + Thu nhập không cố định (trong khoảng thời gian thu thập chứng từ) [1]. Thu nhập cố định là tiền lương và phụ cấp cố định hàng tháng [1]. Phần thu nhập vượt quá 120% thu nhập cố định thấp nhất sẽ được tính theo quy định của thu nhập không cố định [1]. Thu nhập không cố định bao gồm tiền thưởng, hoa hồng, phụ cấp không cố định nhận được trong kỳ thu thập chứng từ [1]. Các khoản này sẽ được chia cho 12 tháng và giảm trừ ít nhất 25% [1]. Yêu cầu khoảng thời gian thu thập chứng từ tối thiểu 3 tháng gần nhất đối với thu nhập cố định và tối thiểu 6 tháng gần nhất nếu có áp dụng cho một số công việc thù lao không cố định [1].",
    "Đối với chứng từ yêu cầu, khách hàng cần cung cấp hợp đồng lao động hoặc các chứng từ tương đương như quyết định bổ nhiệm, thư xác nhận có thể hiện thông tin nơi làm việc và mức lương hiện tại [2]. Loại hợp đồng chính thức phải có thời hạn ít nhất 12 tháng và còn hiệu lực [2]. Nếu hợp đồng đã hết hạn hoặc thời gian còn lại của hợp đồng dưới 15 ngày tại thời điểm nộp hồ sơ, bắt buộc phải có điều khoản gia hạn tự động rõ ràng [2]. Hoặc cần bổ sung hợp đồng lao động mới tại thời điểm giải ngân [2].",
    "Trường hợp khách hàng cung cấp sao kê tài khoản ngân hàng, yêu cầu nộp bản chính hoặc bản sao kê ngân hàng có mộc dấu của ngân hàng trước thời điểm giải ngân [3]. Chấp nhận bản gốc sao kê ViettelPay/Viettel Money có dấu hoặc bản chụp màn hình hệ thống chi lương trên ứng dụng ViettelPay/Viettel Money nếu công ty thuộc Tập đoàn Viettel [3]. Nếu lương nhận qua tài khoản ngân hàng trực tiếp, sao kê không cần thể hiện thông tin 'lương' [4]. Trường hợp lương nhận từ công ty liên quan, bắt buộc bổ sung giấy phép kinh doanh [4].",
    #MED, LOW, NO
    # Chunk 1: Độ tương đồng cao nhất - Trả lời trực tiếp quy định ký chứng từ và ngoại lệ
    "Theo quy định tại Nghị định 174/2016/NĐ-CP và hướng dẫn nội bộ, chứng từ giao dịch của khách hàng tổ chức phải có đủ chữ ký của Người đại diện theo pháp luật và Kế toán trưởng (hoặc người phụ trách kế toán) cùng dấu của đơn vị. Tại ô dành cho Kế toán trưởng sẽ do người được bổ nhiệm ký. Tuy nhiên, doanh nghiệp siêu nhỏ theo quy định pháp luật về hỗ trợ doanh nghiệp nhỏ và vừa không bắt buộc phải bố trí Kế toán trưởng. Đối với chữ ký, Kế toán trưởng không được ủy quyền lại cho người khác.",
    "Trong quá trình quản lý tài khoản, bao gồm cả DICA và DDA, ngân hàng cung cấp các dịch vụ xác thực bổ sung để bảo vệ tài sản của khách hàng. Một trong số đó là Dịch vụ giao dịch qua Fax/Email có đăng ký trước. Giao dịch này yêu cầu khách hàng phải thiết lập mật khẩu, đăng ký các địa chỉ email doanh nghiệp hợp lệ. Nhân viên ngân hàng khi nhận lệnh sẽ gọi điện xác thực qua điện thoại ghi âm trước khi hạch toán, nhằm tránh các rủi ro giả mạo chứng từ chuyển tiền.",
# Chunk 4: Medium Similarity (Hồ sơ chứng từ yêu cầu đối với pháp nhân)
    """Bộ hồ sơ yêu cầu để mở thẻ tín dụng doanh nghiệp không có tài sản đảm bảo phức tạp hơn nhiều so với cá nhân. Các chứng từ bắt buộc bao gồm: Đơn đề nghị mở thẻ tín dụng doanh nghiệp theo mẫu của Shinhan Bank; Chứng từ pháp lý của công ty (Giấy chứng nhận đăng ký doanh nghiệp/Giấy chứng nhận đầu tư/Quyết định thành lập pháp nhân); Bản sao Điều lệ công ty; Chứng từ chứng minh tư cách Người đại diện theo pháp luật (CCCD/Hộ chiếu của Giám đốc). Ngoài ra, công ty phải nộp Báo cáo tài chính đã kiểm toán 3 năm gần nhất, Tờ khai thuế VAT 6 tháng gần nhất, và các hợp đồng kinh tế lớn chứng minh hoạt động sản xuất kinh doanh ổn định. Nếu doanh nghiệp chỉ định nhân viên làm chủ thẻ (Cardholder), phải có Văn bản ủy quyền hợp lệ từ Người đại diện pháp luật, kèm theo giấy tờ tùy thân và Hợp đồng lao động của cá nhân được ủy quyền đó. Toàn bộ hồ sơ bản sao phải có mộc đóng dấu giáp lai của công ty.""",
# 1. Trả lời chính xác về LTV các loại tiền tệ
    """Tỷ lệ cho vay (LTV) của sản phẩm Vay cầm cố tiền gửi (Cash-Backed Loan) được quy định chặt chẽ. Đối với tài sản cầm cố là sổ tiết kiệm VND tại SHBVN, LTV tối đa đạt 95%. Nếu cầm cố bằng sổ USD, tỷ lệ LTV là 85%. Tương tự, đối với các ngoại tệ khác như EUR hoặc AUD, ngân hàng cũng chỉ giải ngân tối đa 85% giá trị quy đổi.""",
# Chunk 5: Tương đồng trung bình - Quy định Kế toán trưởng đối với tổ chức không cư trú
    "Đối với một số hình thức tổ chức đặc thù của nước ngoài tại Việt Nam, quy định về chữ ký trên tài khoản được nới lỏng. Cụ thể, Văn phòng đại diện của Nhà đầu tư nước ngoài (thuộc tổ chức không cư trú hoặc có đặc thù riêng) và Văn phòng điều hành của nhà thầu nước ngoài không bắt buộc phải bố trí, đăng ký chữ ký của Kế toán trưởng trên bộ hồ sơ mở tài khoản và chứng từ giao dịch thanh toán. Lệnh chuyển tiền chỉ cần chữ ký của Trưởng văn phòng đại diện (Người đại diện pháp luật) là hợp lệ.",
# Chunk 4: Tương đồng khá - Công thức tính lãi suất tiền gửi
    "Phương pháp tính lãi suất cho cả TDA và CD thường dựa trên số dư thực tế và số ngày duy trì. Công thức cơ bản là: Lãi tiền gửi = (Tổng số dư cuối ngày x Lãi suất tương ứng x Số ngày duy trì thực tế) / 365. Riêng đối với tài khoản Shinhan The More MMDA (tài khoản tiền gửi VND rút linh hoạt), lãi suất được tính theo bậc số dư cuối ngày. Ví dụ, dư nợ dưới 300 triệu VND áp dụng mức lãi 0.1%, dư nợ trên 3 tỷ VND áp dụng lãi 0.5%, tiền lãi được cộng dồn hàng ngày.",
"Công thức tính Tỷ lệ trả nợ (DBR: Debt burden Ratio) là: DBR = Tổng nghĩa vụ nợ / Tổng thu nhập ròng [12]. Theo quy định, nếu tổng thu nhập để trả nợ hàng tháng >= 50 triệu VND thì DBR <= 70% [12]. Nếu tổng thu nhập để trả nợ hàng tháng < 50 triệu VND thì DBR <= 60% [12]. Riêng đối với thu nhập từ Hộ kinh doanh, điều kiện chặt chẽ hơn: DBR <= 60% [12]. Khả năng thanh toán = Tổng thu nhập ròng - Tổng nợ phải trả - Chi phí sinh hoạt [12]. Số tiền còn lại sau khi thanh toán phải bằng hoặc lớn hơn 0 [12].",

    ]

dic = {
    query: documents,
}

In [ ]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy định, chứng từ yêu cầu và phương pháp tính thu nhập ròng hàng tháng đối với khách hàng hưởng lương và thưởng
[1] Score: 0.4903
Thu nhập ròng hàng tháng = Thu nhập cố định + Thu nhập không cố định (trong khoảng thời gian thu thập chứng từ) [1]. Thu nhập cố định là tiền lương và phụ cấp cố định hàng tháng [1]. Phần thu nhập vượt quá 120% thu nhập cố định thấp nhất sẽ được tính theo quy định của thu nhập không cố định [1]. Thu nhập không cố định bao gồm tiền thưởng, hoa hồng, phụ cấp không cố định nhận được trong kỳ thu thập chứng từ [1]. Các khoản này sẽ được chia cho 12 tháng và giảm trừ ít nhất 25% [1]. Yêu cầu khoảng thời gian thu thập chứng từ tối thiểu 3 tháng gần nhất đối với thu nhập cố định và tối thiểu 6 tháng gần nhất nếu có áp dụng cho một số công việc thù lao không cố định [1].
----------------------------------------------------------------------
[2] Score: 0.2765
Đối với chứng từ yêu cầu, khách hàng cần cung cấp hợp đồng lao động hoặc các chứng từ tương đương 

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy định, chứng từ yêu cầu và phương pháp tính thu nhập ròng hàng tháng đối với khách hàng hưởng lương và thưởng
[1] Score: 0.9554
Thu nhập ròng hàng tháng = Thu nhập cố định + Thu nhập không cố định (trong khoảng thời gian thu thập chứng từ) [1]. Thu nhập cố định là tiền lương và phụ cấp cố định hàng tháng [1]. Phần thu nhập vượt quá 120% thu nhập cố định thấp nhất sẽ được tính theo quy định của thu nhập không cố định [1]. Thu nhập không cố định bao gồm tiền thưởng, hoa hồng, phụ cấp không cố định nhận được trong kỳ thu thập chứng từ [1]. Các khoản này sẽ được chia cho 12 tháng và giảm trừ ít nhất 25% [1]. Yêu cầu khoảng thời gian thu thập chứng từ tối thiểu 3 tháng gần nhất đối với thu nhập cố định và tối thiểu 6 tháng gần nhất nếu có áp dụng cho một số công việc thù lao không cố định [1].
----------------------------------------------------------------------
[2] Score: 0.3057
Đối với chứng từ yêu cầu, khách hàng cần cung cấp hợp đồng lao động hoặc các chứng từ tương đương 

In [ ]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy định, chứng từ yêu cầu và phương pháp tính thu nhập ròng hàng tháng đối với khách hàng hưởng lương và thưởng
[1] Score: 0.9880
Thu nhập ròng hàng tháng = Thu nhập cố định + Thu nhập không cố định (trong khoảng thời gian thu thập chứng từ) [1]. Thu nhập cố định là tiền lương và phụ cấp cố định hàng tháng [1]. Phần thu nhập vượt quá 120% thu nhập cố định thấp nhất sẽ được tính theo quy định của thu nhập không cố định [1]. Thu nhập không cố định bao gồm tiền thưởng, hoa hồng, phụ cấp không cố định nhận được trong kỳ thu thập chứng từ [1]. Các khoản này sẽ được chia cho 12 tháng và giảm trừ ít nhất 25% [1]. Yêu cầu khoảng thời gian thu thập chứng từ tối thiểu 3 tháng gần nhất đối với thu nhập cố định và tối thiểu 6 tháng gần nhất nếu có áp dụng cho một số công việc thù lao không cố định [1].
----------------------------------------------------------------------
[2] Score: 0.7949
Đối với chứng từ yêu cầu, khách hàng cần cung cấp hợp đồng lao động hoặc các chứng từ tương đương 

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: quy định, chứng từ yêu cầu và phương pháp tính thu nhập ròng hàng tháng đối với khách hàng hưởng lương và thưởng
[1] Score: 0.4962
Thu nhập ròng hàng tháng = Thu nhập cố định + Thu nhập không cố định (trong khoảng thời gian thu thập chứng từ) [1]. Thu nhập cố định là tiền lương và phụ cấp cố định hàng tháng [1]. Phần thu nhập vượt quá 120% thu nhập cố định thấp nhất sẽ được tính theo quy định của thu nhập không cố định [1]. Thu nhập không cố định bao gồm tiền thưởng, hoa hồng, phụ cấp không cố định nhận được trong kỳ thu thập chứng từ [1]. Các khoản này sẽ được chia cho 12 tháng và giảm trừ ít nhất 25% [1]. Yêu cầu khoảng thời gian thu thập chứng từ tối thiểu 3 tháng gần nhất đối với thu nhập cố định và tối thiểu 6 tháng gần nhất nếu có áp dụng cho một số công việc thù lao không cố định [1].
----------------------------------------------------------------------
[2] Score: 0.0515
Đối với chứng từ yêu cầu, khách hàng cần cung cấp hợp đồng lao động hoặc các chứng từ tương đương 

doc [10] mặc định rằng ngân hàng đã tính xong Tổng thu nhập ròng rồi, và bây giờ lấy con số đó ráp vào công thức tính tỷ lệ DBR để xem khách hàng có đủ điều kiện vay hay không. -> mức độ liên quan thấp -> cả Qwen3 0.6B và Jina đều bị lừa

### Case 9

In [ ]:
query = "phương thức thanh toán dư nợ thẻ tín dụng là gì và quy định về việc trích nợ tự động, thời gian cập nhật hạn mức thẻ"

documents = [
    # HIGH relevant
    # High similarity (Auto-debit payment)
    "Phương thức thanh toán tiện lợi nhất là Thanh toán tự động (Auto-debit). Dịch vụ này áp dụng cho khách hàng có tài khoản tiền gửi thanh toán tại Shinhan Bank. Ngân hàng sẽ tự động trích nợ số tiền tương ứng với tỷ lệ thanh toán (tối thiểu hoặc toàn bộ) mà khách hàng đã đăng ký từ tài khoản vào Ngày đến hạn. Khách hàng cần duy trì số dư đủ để thanh toán trước 16h30 ngày đến hạn để tránh ghi nhận thanh toán trễ.",

    # High similarity (Payoo & Over counter)
    "Khách hàng có thể thanh toán dư nợ thẻ tín dụng miễn phí thông qua hệ thống Payoo bằng 3 cách: Nộp tiền mặt trực tiếp tại các cửa hàng liên kết như Circle K, FamilyMart, GS25; Thanh toán qua ứng dụng Payoo trên điện thoại di động; hoặc thanh toán trên website Payoo. Khách hàng chỉ cần nhập mã Khách hàng (Member No.). Ngoài ra, khách hàng có thể nộp tiền mặt thanh toán tại quầy giao dịch chi nhánh Shinhan trước 16h30 ngày đến hạn.",

    # Medium-High similarity (Bank transfer payment)
    "Thanh toán qua hình thức chuyển khoản từ ngân hàng khác cũng được hỗ trợ. Để đảm bảo việc thanh toán đúng hạn, khách hàng nên thực hiện chuyển khoản trước Ngày đến hạn ít nhất 02 ngày làm việc. Khi sử dụng dịch vụ Chuyển tiền nhanh liên ngân hàng qua số thẻ (Express domestic transfer service), số tiền chuyển sẽ được ghi có trực tiếp vào Thẻ tín dụng của khách hàng mà không cần liên kết với tài khoản tiền gửi Shinhan (DDA).",

    # Medium similarity (Limit update timing)
    "Thời gian cập nhật lại hạn mức tín dụng phụ thuộc vào kênh thanh toán. Nếu thanh toán qua dịch vụ Ngân hàng trực tuyến/SOL, hạn mức sẽ được cập nhật ngay lập tức sau 3 phút. Nếu nộp tiền mặt tại quầy hoặc qua Payoo, hạn mức cũng được cập nhật trong khoảng thời gian tương tự. Tuy nhiên, nếu thanh toán vào cuối tuần, ngày lễ hoặc sau 18h00, giao dịch có thể sẽ được hạch toán và cập nhật vào ngày làm việc tiếp theo.",

    # MEDIUM, LOW, NO relenvant
    "Phương thức tính lãi suất cho cả TDA và CD thường dựa trên số dư thực tế và số ngày duy trì. Công thức cơ bản là: Lãi tiền gửi = (Tổng số dư cuối ngày x Lãi suất tương ứng x Số ngày duy trì thực tế) / 365. Riêng đối với tài khoản Shinhan The More MMDA (tài khoản tiền gửi VND rút linh hoạt), lãi suất được tính theo bậc số dư cuối ngày. Ví dụ, dư nợ dưới 300 triệu VND áp dụng mức lãi 0.1%, dư nợ trên 3 tỷ VND áp dụng lãi 0.5%, tiền lãi được cộng dồn hàng ngày.",
    """Cơ chế tính điểm của Tops Club dựa trên nhiều loại hình giao dịch. Đối với tiền gửi Không kỳ hạn, mỗi 500,000 VND số dư trung bình 3 tháng được tính 5 điểm. Đối với tiền gửi Có kỳ hạn/Tiền gửi tích lũy, mỗi 5,000,000 VND số dư trung bình 3 tháng tính 3 điểm. Khách hàng có tài khoản tiền gửi đầu tư hưởng 10 điểm cho mỗi 5,000,000 VND số dư trung bình. Đối với các khoản vay cá nhân, mỗi 500,000 VND dư nợ trung bình 3 tháng quy đổi thành 10 điểm.""",
    # Chunk 2: High Similarity (Xử lý lỗi và các trường hợp đặc biệt khi tra cứu)
    """Trong quá trình tra cứu trên màn hình MH , nếu hệ thống báo lỗi không tìm thấy dữ liệu do sai lệch thông tin cá nhân, nhân viên cần thực hiện truy vấn lại bằng chức năng nhập tên thiếu (ví dụ nhập "RI YUNG HYUK" thay vì họ tên đầy đủ do giới hạn ký tự) hoặc tra cứu theo số giấy tờ cũ nếu khách hàng vừa đổi từ CMND 9 số sang CCCD 12 số gắn chip. Tại mục Report Type, chọn "1-FULLFORM" đối với thẻ tín dụng và các khoản vay thông thường để lấy toàn bộ lịch sử dư nợ, thẻ tín dụng, thế chấp và lịch sử trả nợ chậm. Sau khi nhấn "Inquiry", trạng thái ban đầu có thể là "Received", nhân viên cần chờ một lúc để hệ thống cập nhật kết quả. Đặc biệt lưu ý, các lỗi phát sinh như sai cấu trúc tên, sai mã chi nhánh yêu cầu (Request Branch) cần được xử lý ngay lập tức. Cần đối chiếu lại toàn bộ số liệu dư nợ các thẻ đang mở tại các TCTD khác so với khai báo của khách hàng trên Đơn yêu cầu mở thẻ để phát hiện rủi ro gian lận.""",

    # Chunk 3: High-Medium Similarity (Các tiêu chí từ chối tự động dựa trên CIC)
    """Sau khi có kết quả CIC, hệ thống sẽ chạy qua màng lọc Filtering Items để tự động đánh giá. Khách hàng sẽ bị từ chối mở thẻ nếu vi phạm một trong các điều kiện sau: Có nợ nhóm 2 tại bất kỳ tổ chức tín dụng nào trong vòng 12 tháng gần nhất (mã lỗi DL020000 >= 2); Có khoản nợ vay thuộc nhóm 3 đến nhóm 5 trong vòng 5 năm gần nhất (mã lỗi DL010003); Có từ 4 lần nợ vay nhóm 2 trong vòng 12 tháng gần nhất; Có lịch sử trễ hạn thẻ tín dụng từ 30 ngày trở lên trong vòng 12 tháng gần nhất (DC031012). Tổng dư nợ thẻ tín dụng đang có không được vượt quá số lượng thẻ quy định (ví dụ khách hàng đang có 4 thẻ tín dụng tại các ngân hàng khác nhau có thể bị từ chối do vượt quá rủi ro - LN019006 >= 4). Đồng thời, nếu khách hàng có các khoản vay mua trả góp tại các tổ chức tài chính (như FE Credit, Home Credit) với dư nợ cao và lịch sử trả chậm, hệ thống cũng sẽ ghi nhận và áp dụng bộ quy tắc Cut-off (R-Clips Rejected).""",
# 10. Nghiệp vụ hoàn toàn khác (Giải ngân đa kỳ hạn - Home Loan)
    """Cơ chế điều chỉnh lãi suất cho các khoản vay thế chấp giải ngân nhiều đợt (multi-disbursement): Dù khách hàng nhận giải ngân thành 3 hay 4 đợt khác nhau, thời gian áp dụng lãi suất cố định (Fixed IR period) sẽ đồng loạt kết thúc vào cùng một ngày tính từ thời điểm giải ngân đợt đầu tiên. Sau đó toàn bộ dư nợ sẽ chuyển sang lãi suất thả nổi."""
    # Chunk 3: High-Medium Similarity (Hạn mức dựa trên tài sản xe hơi/nhà)
    """Bên cạnh thu nhập từ lương, Shinhan Bank cấp hạn mức thẻ dựa trên tài sản sở hữu như Xe hơi và Bất động sản. Đối với xe hơi, hạn mức được tính dựa trên giá trị xe hiện tại. Xe từ 1 tỷ đồng trở lên có thể được cấp hạn mức tối đa 100 triệu đồng; xe từ 500 triệu đến dưới 1 tỷ cấp 70 triệu; và xe từ 300 triệu đến dưới 500 triệu cấp 50 triệu. Điều kiện là xe phải sản xuất trong vòng 5 năm, thời gian sử dụng từ 2 năm trở xuống (giá trị xe > 80%) và mua bảo hiểm thân vỏ tự nguyện. Đối với bất động sản (sổ hồng, sổ đỏ), diện tích sàn nhà trên đất phải từ 40m2 (không chấp nhận đất trống), hoặc căn hộ chung cư phải từ 60m2. Khách hàng sở hữu nhà dưới 60m2 được cấp hạn mức 50 triệu, từ 60m2 đến 90m2 được cấp 70 triệu và trên 90m2 có thể được cấp tối đa 100 triệu đồng. Chứng từ yêu cầu là Giấy chứng nhận quyền sử dụng đất bản sao y chứng thực và Hóa đơn tiện ích chứng minh có người ở.""",

    ]

dic = {
    query: documents,
}

In [ ]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: phương thức thanh toán dư nợ thẻ tín dụng là gì và quy định về việc trích nợ tự động, thời gian cập nhật hạn mức thẻ
[4] Score: 0.3694
Thời gian cập nhật lại hạn mức tín dụng phụ thuộc vào kênh thanh toán. Nếu thanh toán qua dịch vụ Ngân hàng trực tuyến/SOL, hạn mức sẽ được cập nhật ngay lập tức sau 3 phút. Nếu nộp tiền mặt tại quầy hoặc qua Payoo, hạn mức cũng được cập nhật trong khoảng thời gian tương tự. Tuy nhiên, nếu thanh toán vào cuối tuần, ngày lễ hoặc sau 18h00, giao dịch có thể sẽ được hạch toán và cập nhật vào ngày làm việc tiếp theo.
----------------------------------------------------------------------
[1] Score: 0.3297
Phương thức thanh toán tiện lợi nhất là Thanh toán tự động (Auto-debit). Dịch vụ này áp dụng cho khách hàng có tài khoản tiền gửi thanh toán tại Shinhan Bank. Ngân hàng sẽ tự động trích nợ số tiền tương ứng với tỷ lệ thanh toán (tối thiểu hoặc toàn bộ) mà khách hàng đã đăng ký từ tài khoản vào Ngày đến hạn. Khách hàng cần duy trì số dư đủ để thanh

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: phương thức thanh toán dư nợ thẻ tín dụng là gì và quy định về việc trích nợ tự động, thời gian cập nhật hạn mức thẻ
[4] Score: 0.8662
Thời gian cập nhật lại hạn mức tín dụng phụ thuộc vào kênh thanh toán. Nếu thanh toán qua dịch vụ Ngân hàng trực tuyến/SOL, hạn mức sẽ được cập nhật ngay lập tức sau 3 phút. Nếu nộp tiền mặt tại quầy hoặc qua Payoo, hạn mức cũng được cập nhật trong khoảng thời gian tương tự. Tuy nhiên, nếu thanh toán vào cuối tuần, ngày lễ hoặc sau 18h00, giao dịch có thể sẽ được hạch toán và cập nhật vào ngày làm việc tiếp theo.
----------------------------------------------------------------------
[2] Score: 0.7916
Khách hàng có thể thanh toán dư nợ thẻ tín dụng miễn phí thông qua hệ thống Payoo bằng 3 cách: Nộp tiền mặt trực tiếp tại các cửa hàng liên kết như Circle K, FamilyMart, GS25; Thanh toán qua ứng dụng Payoo trên điện thoại di động; hoặc thanh toán trên website Payoo. Khách hàng chỉ cần nhập mã Khách hàng (Member No.). Ngoài ra, khách hàng có thể nộ

In [ ]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: phương thức thanh toán dư nợ thẻ tín dụng là gì và quy định về việc trích nợ tự động, thời gian cập nhật hạn mức thẻ
[1] Score: 0.9786
Phương thức thanh toán tiện lợi nhất là Thanh toán tự động (Auto-debit). Dịch vụ này áp dụng cho khách hàng có tài khoản tiền gửi thanh toán tại Shinhan Bank. Ngân hàng sẽ tự động trích nợ số tiền tương ứng với tỷ lệ thanh toán (tối thiểu hoặc toàn bộ) mà khách hàng đã đăng ký từ tài khoản vào Ngày đến hạn. Khách hàng cần duy trì số dư đủ để thanh toán trước 16h30 ngày đến hạn để tránh ghi nhận thanh toán trễ.
----------------------------------------------------------------------
[2] Score: 0.9674
Khách hàng có thể thanh toán dư nợ thẻ tín dụng miễn phí thông qua hệ thống Payoo bằng 3 cách: Nộp tiền mặt trực tiếp tại các cửa hàng liên kết như Circle K, FamilyMart, GS25; Thanh toán qua ứng dụng Payoo trên điện thoại di động; hoặc thanh toán trên website Payoo. Khách hàng chỉ cần nhập mã Khách hàng (Member No.). Ngoài ra, khách hàng có thể nộp t

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: phương thức thanh toán dư nợ thẻ tín dụng là gì và quy định về việc trích nợ tự động, thời gian cập nhật hạn mức thẻ
[4] Score: 0.2167
Thời gian cập nhật lại hạn mức tín dụng phụ thuộc vào kênh thanh toán. Nếu thanh toán qua dịch vụ Ngân hàng trực tuyến/SOL, hạn mức sẽ được cập nhật ngay lập tức sau 3 phút. Nếu nộp tiền mặt tại quầy hoặc qua Payoo, hạn mức cũng được cập nhật trong khoảng thời gian tương tự. Tuy nhiên, nếu thanh toán vào cuối tuần, ngày lễ hoặc sau 18h00, giao dịch có thể sẽ được hạch toán và cập nhật vào ngày làm việc tiếp theo.
----------------------------------------------------------------------
[1] Score: 0.2037
Phương thức thanh toán tiện lợi nhất là Thanh toán tự động (Auto-debit). Dịch vụ này áp dụng cho khách hàng có tài khoản tiền gửi thanh toán tại Shinhan Bank. Ngân hàng sẽ tự động trích nợ số tiền tương ứng với tỷ lệ thanh toán (tối thiểu hoặc toàn bộ) mà khách hàng đã đăng ký từ tài khoản vào Ngày đến hạn. Khách hàng cần duy trì số dư đủ để thanh

Đánh giá Qwen3 0.6B: [3] phải được rank cao hơn so với [8] vì [3] nói về một trong các phương thức thanh toán dư nợ thẻ tín dụng (chuyển khoản liên ngân hàng) -> trả lời được một vế của câu hỏi là "phương thức thanh toán là gì?" và một phần nhỏ về thời gian ("thực hiện trước ít nhất 02 ngày", "ghi có trực tiếp"). [8] đề cập quy trình thẩm định phát hành thẻ (Credit Appraisal / Underwriting) dựa trên lịch sử CIC -> không liên quan gì đến việc khách hàng thanh toán dư nợ hàng tháng như thế nào.

### Case 10

In [16]:
query = "Quy trình đăng ký tài khoản trực tuyến eKYC và xác thực sinh trắc học trên ứng dụng Shinhan SOL diễn ra như thế nào?"

documents = [
    # HIGH relevant
    # Top 1: Cực kỳ liên quan (Mô tả chi tiết các bước eKYC và sinh trắc học)
    """Để mở tài khoản trực tuyến (eKYC) trên ứng dụng Shinhan SOL 3.0, khách hàng cần chuẩn bị thẻ Căn cước công dân (CCCD) gắn chip và thiết bị di động hỗ trợ NFC. Quy trình bắt đầu bằng việc chụp ảnh hai mặt CCCD để trích xuất OCR. Tiếp theo, khách hàng đặt thẻ CCCD sát vào mặt lưng điện thoại để quét NFC, giúp ứng dụng đọc dữ liệu từ chip, đảm bảo tính chính xác và chống giả mạo. Sau đó, khách hàng thực hiện xác thực khuôn mặt (Face Verification) để hệ thống đối chiếu ảnh thực tế với dữ liệu của Bộ Công an. Cuối cùng, khách hàng bổ sung thông tin liên lạc và thiết lập tên đăng nhập, mật khẩu, mã PIN để hoàn tất mở tài khoản và đăng ký sinh trắc học.""",
    # Top 2: Rất liên quan (Đề cập đến đối tượng và tính năng eKYC)
    """Tính năng mở tài khoản và dịch vụ trực tuyến (eKYC) áp dụng cho khách hàng Việt Nam cư trú từ 15 tuổi trở lên. Thông qua VNEID và công nghệ NFC, người dùng không cần đến trực tiếp quầy giao dịch mà vẫn có thể mở thẻ ghi nợ hoặc tài khoản thanh toán. Trong quá trình này, sinh trắc học đóng vai trò quan trọng trong việc xác nhận danh tính, tuân thủ quy định an toàn bảo mật. Những khách hàng đã có thông tin sinh trắc học có thể sử dụng Face ID hoặc vân tay để đăng nhập nhanh chóng trong các lần tiếp theo thay vì phải nhập mật khẩu thủ công.""",
    # Top 3: Khá liên quan (Nói về cập nhật sinh trắc học cho khách hàng đã có tài khoản)
    """Đối với khách hàng nước ngoài hoặc những người chưa có CCCD gắn chip, quy trình đăng ký sinh trắc học lần đầu yêu cầu phải thực hiện tại quầy giao dịch (Aither) với sự hỗ trợ của giao dịch viên. Khi khách hàng mang Hộ chiếu hoặc giấy tờ tùy thân hợp lệ đến, nhân viên sẽ tiến hành thu thập hình ảnh khuôn mặt và liên kết với mã khách hàng (CIF). Sau khi hoàn tất tại quầy, khách hàng có thể mở ứng dụng SOL 3.0, chọn mục cập nhật sinh trắc học và làm theo hướng dẫn để đồng bộ hóa dữ liệu.""",
    # Top 4: Liên quan một phần (Trải nghiệm Pre-login trên SOL)
    """Ngay từ màn hình Pre-login (trước khi đăng nhập) của ứng dụng SOL, người dùng đã có thể trải nghiệm nhiều tiện ích mà chưa cần phải là khách hàng hiện hữu. Các tính năng này bao gồm tra cứu tỷ giá ngoại tệ, tính toán lãi suất khoản vay SOHO, tìm kiếm chi nhánh/ATM, và đặc biệt là nút chức năng 'Mở tài khoản trực tuyến'. Đây là điểm chạm đầu tiên, dẫn dắt khách hàng vào luồng eKYC để đăng ký sinh trắc học và trở thành người dùng chính thức của ngân hàng.""",

    # Others
    "Khi thực hiện eKYC trong điều kiện thiếu sáng, thuật toán Liveness Detection có thể không ghi nhận được độ sâu của khuôn mặt, dẫn đến lỗi từ chối sinh trắc học. Vui lòng di chuyển đến khu vực có ánh sáng tự nhiên và thử lại.",
    "Facial recognition failures during the digital onboarding (eKYC) process are frequently caused by poor illumination, extreme backlighting, or glare on eyeglasses, which prevents the OCR model from extracting key facial landmarks.",
    """Tại Bước 6 của quy trình Đăng ký CIF và Mở tài khoản: Đăng ký CIF, đối chiếu thông tin sinh trắc học và mở tài khoản. Nhân viên cần đăng ký chấp thuận xử lý dữ liệu cá nhân trên màn hình 0044, đăng ký thông tin CIF mới, kiểm tra đối chiếu thông tin sinh trắc học của Chủ tài khoản ở màn hình 6316. Đối với khách hàng người Việt Nam, luồng đối chiếu sinh trắc học thực hiện qua màn hình 0042 hoặc qua ứng dụng Shinhan SOL Mobile Banking."""
"Việc nhận lệnh và kiểm tra thông tin khách hàng dựa trên dữ liệu thu thập từ quá trình Định danh mở tài khoản (KYC). Trong bước này, ngân hàng đối chiếu thông tin pháp lý của doanh nghiệp từ website Đăng ký kinh doanh quốc gia. Nhân viên ngân hàng phải chụp sinh trắc học người đại diện pháp luật, nhập liệu mã số thuế, kiểm tra FATCA/AML để mở mã CIF. Một khi CIF được duyệt, khách hàng mới có thể đăng ký các dịch vụ giá trị gia tăng như giao dịch qua email, mở tài khoản lương hoặc Internet Banking.",
"Ngân hàng điện tử SOL App cung cấp tính năng Quản lý cửa hàng dành cho khách hàng Hộ kinh doanh/SME. Tính năng này giúp minh bạch nguồn thu, quản lý dòng tiền và thao tác tiện lợi, hỗ trợ đắc lực trong kinh doanh thời đại số. Ngoài ra, khách hàng có thể trải nghiệm các dịch vụ thanh toán không chạm như Google Pay, Samsung Pay, Garmin Pay liên kết trực tiếp với thẻ tín dụng Shinhan.",
"Ứng dụng Mobile Banking SOL của Shinhan không chỉ dành cho cá nhân mà còn tích hợp tính năng 'Quản lý cửa hàng' (Store Management) là giải pháp vàng cho kinh doanh thời đại số. Tính năng này cho phép các hộ kinh doanh cá thể quản lý doanh thu minh bạch, theo dõi các giao dịch thanh toán QR và POS dễ dàng ngay trên điện thoại di động, từ đó đưa ra các quyết định kinh doanh kịp thời.",

    ]

dic = {
    query: documents,
}

In [10]:
test_model(qwen3_4, llm4, dic)
test_model(bge_m3, llmbge, dic)


*** Testing Qwen/Qwen3-Reranker-4B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Quy trình đăng ký tài khoản trực tuyến eKYC và xác thực sinh trắc học trên ứng dụng Shinhan SOL diễn ra như thế nào?
[4] Score: 0.4632
Ngay từ màn hình Pre-login (trước khi đăng nhập) của ứng dụng SOL, người dùng đã có thể trải nghiệm nhiều tiện ích mà chưa cần phải là khách hàng hiện hữu. Các tính năng này bao gồm tra cứu tỷ giá ngoại tệ, tính toán lãi suất khoản vay SOHO, tìm kiếm chi nhánh/ATM, và đặc biệt là nút chức năng 'Mở tài khoản trực tuyến'. Đây là điểm chạm đầu tiên, dẫn dắt khách hàng vào luồng eKYC để đăng ký sinh trắc học và trở thành người dùng chính thức của ngân hàng.
----------------------------------------------------------------------
[1] Score: 0.4330
Để mở tài khoản trực tuyến (eKYC) trên ứng dụng Shinhan SOL 3.0, khách hàng cần chuẩn bị thẻ Căn cước công dân (CCCD) gắn chip và thiết bị di động hỗ trợ NFC. Quy trình bắt đầu bằng việc chụp ảnh hai mặt CCCD để trích xuất OCR. Tiếp theo, khách hàng đặt thẻ CCCD sát vào mặt lưng điện thoại để quét NFC, giúp

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Quy trình đăng ký tài khoản trực tuyến eKYC và xác thực sinh trắc học trên ứng dụng Shinhan SOL diễn ra như thế nào?
[1] Score: 0.9996
Để mở tài khoản trực tuyến (eKYC) trên ứng dụng Shinhan SOL 3.0, khách hàng cần chuẩn bị thẻ Căn cước công dân (CCCD) gắn chip và thiết bị di động hỗ trợ NFC. Quy trình bắt đầu bằng việc chụp ảnh hai mặt CCCD để trích xuất OCR. Tiếp theo, khách hàng đặt thẻ CCCD sát vào mặt lưng điện thoại để quét NFC, giúp ứng dụng đọc dữ liệu từ chip, đảm bảo tính chính xác và chống giả mạo. Sau đó, khách hàng thực hiện xác thực khuôn mặt (Face Verification) để hệ thống đối chiếu ảnh thực tế với dữ liệu của Bộ Công an. Cuối cùng, khách hàng bổ sung thông tin liên lạc và thiết lập tên đăng nhập, mật khẩu, mã PIN để hoàn tất mở tài khoản và đăng ký sinh trắc học.
----------------------------------------------------------------------
[2] Score: 0.9020
Tính năng mở tài khoản và dịch vụ trực tuyến (eKYC) áp dụng cho khách hàng Việt Nam cư trú từ 15 tuổi trở lên. 

In [17]:
test_model(qwen3_06, llm06, dic)
test_model(jina_v3, llmjina, dic)


*** Testing Qwen/Qwen3-Reranker-0.6B ***



Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Quy trình đăng ký tài khoản trực tuyến eKYC và xác thực sinh trắc học trên ứng dụng Shinhan SOL diễn ra như thế nào?
[1] Score: 0.9962
Để mở tài khoản trực tuyến (eKYC) trên ứng dụng Shinhan SOL 3.0, khách hàng cần chuẩn bị thẻ Căn cước công dân (CCCD) gắn chip và thiết bị di động hỗ trợ NFC. Quy trình bắt đầu bằng việc chụp ảnh hai mặt CCCD để trích xuất OCR. Tiếp theo, khách hàng đặt thẻ CCCD sát vào mặt lưng điện thoại để quét NFC, giúp ứng dụng đọc dữ liệu từ chip, đảm bảo tính chính xác và chống giả mạo. Sau đó, khách hàng thực hiện xác thực khuôn mặt (Face Verification) để hệ thống đối chiếu ảnh thực tế với dữ liệu của Bộ Công an. Cuối cùng, khách hàng bổ sung thông tin liên lạc và thiết lập tên đăng nhập, mật khẩu, mã PIN để hoàn tất mở tài khoản và đăng ký sinh trắc học.
----------------------------------------------------------------------
[4] Score: 0.9932
Ngay từ màn hình Pre-login (trước khi đăng nhập) của ứng dụng SOL, người dùng đã có thể trải nghiệm nhiều tiện 

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Query 1: Quy trình đăng ký tài khoản trực tuyến eKYC và xác thực sinh trắc học trên ứng dụng Shinhan SOL diễn ra như thế nào?
[1] Score: 0.5398
Để mở tài khoản trực tuyến (eKYC) trên ứng dụng Shinhan SOL 3.0, khách hàng cần chuẩn bị thẻ Căn cước công dân (CCCD) gắn chip và thiết bị di động hỗ trợ NFC. Quy trình bắt đầu bằng việc chụp ảnh hai mặt CCCD để trích xuất OCR. Tiếp theo, khách hàng đặt thẻ CCCD sát vào mặt lưng điện thoại để quét NFC, giúp ứng dụng đọc dữ liệu từ chip, đảm bảo tính chính xác và chống giả mạo. Sau đó, khách hàng thực hiện xác thực khuôn mặt (Face Verification) để hệ thống đối chiếu ảnh thực tế với dữ liệu của Bộ Công an. Cuối cùng, khách hàng bổ sung thông tin liên lạc và thiết lập tên đăng nhập, mật khẩu, mã PIN để hoàn tất mở tài khoản và đăng ký sinh trắc học.
----------------------------------------------------------------------
[7] Score: 0.2436
Tại Bước 6 của quy trình Đăng ký CIF và Mở tài khoản: Đăng ký CIF, đối chiếu thông tin sinh trắc học và mở tài 